<a href="https://colab.research.google.com/github/colodro/perdigueiro-wordpress/blob/main/Perdigueiro_VGG16_Triplet_KFold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Perdigueiro — VGG16, Triplet Loss e validação cruzada

Notebook de pesquisa do TCC. Esta edição preserva **todas as células de código na ordem original**, mas remove saídas e contadores de execução. Algumas células são diagnósticos e recuperações de sessões anteriores e não devem ser executadas indiscriminadamente em sequência.

**Antes de executar:** configurar Google Colab, acesso autorizado aos datasets no Google Drive, caminhos de arquivos e dependências. Os datasets e checkpoints não estão incluídos neste notebook. A reprodução integral em uma sessão nova **não foi validada**.

### Como utilizar este notebook público

Este arquivo documenta o **histórico experimental** do TCC (VGG16, Triplet Loss e validação cruzada com cinco folds). Os blocos foram desenvolvidos em diferentes sessões do Colab, com recuperação de checkpoints. **Não se deve pressupor que “Executar tudo” funcione em uma sessão nova**: há dependências de datasets, checkpoints e arquivos no Google Drive do autor.

1. No Colab, habilite GPU e monte o Drive; configure os caminhos de `TCC_Caes` e dos datasets antes de executar as etapas dependentes de arquivos.
2. Para **reproduzir o treinamento do zero**, utilize as células de preparação, arquitetura e treinamento, adaptando os caminhos e verificando as dependências entre blocos.
3. Para **reavaliar modelos treinados**, disponibilize os checkpoints e os índices/splits correspondentes. Os resultados não são reconstruídos sem esses artefatos.
4. Células identificadas como *diagnóstico* ou *recuperação* registram o processo de desenvolvimento e podem não ser necessárias em uma execução nova. Elas foram mantidas por rastreabilidade.

**Publicação:** não inclua datasets, imagens de pessoas, credenciais ou checkpoints de grande porte no repositório sem revisar permissões e licenças. As saídas de execução foram removidas.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Preparação dos dados e definição experimental

Inventário dos conjuntos de dados, divisão por identidade e configuração dos folds.

In [ ]:
# ============================================================
# TCC 1 — Experimento VGG16 + Triplet Loss + K-Fold
# Bloco de código [1] — Configuração inicial e persistência no Google Drive
# ============================================================

from google.colab import drive
from pathlib import Path
import random
import numpy as np
import tensorflow as tf
import json

drive.mount('/content/drive')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE = Path('/content/drive/MyDrive/TCC_Caes')
EXPERIMENTO = BASE / 'experimentos' / 'tcc1_kfold'

PASTAS = {
    'datasets': BASE / 'datasets',
    'splits': EXPERIMENTO / 'splits',
    'checkpoints': EXPERIMENTO / 'checkpoints',
    'modelos': EXPERIMENTO / 'modelos',
    'logs': EXPERIMENTO / 'logs',
    'resultados': EXPERIMENTO / 'resultados',
}

for nome, caminho in PASTAS.items():
    caminho.mkdir(parents=True, exist_ok=True)
    print(f'{nome}: {caminho}')

print('\nTensorFlow:', tf.__version__)
print('GPUs disponíveis:', tf.config.list_physical_devices('GPU'))

config = {
    'experimento': 'TCC1_VGG16_Triplet_KFold',
    'seed': SEED,
    'datasets': ['DogFaceNet', 'YT-BB-Dog'],
    'test_size': 0.30,
    'n_folds': 5,
    'split_por_identidade': True
}

with open(EXPERIMENTO / 'config.json', 'w') as f:
    json.dump(config, f, indent=4)

print('\nConfiguração salva com sucesso.')


In [ ]:

# ============================================================
# Bloco de código [2] — Localização dos datasets no Drive
# Referência: tcc_caes_colab-oficial.ipynb
# ============================================================

from pathlib import Path

MYDRIVE = Path('/content/drive/MyDrive')

DATASETS_DRIVE = MYDRIVE / 'datasets_extraido'

DOGFACENET_RAIZ = DATASETS_DRIVE / 'DogFaceNet-master'
YTBB_RAIZ = DATASETS_DRIVE / 'YT-BB-dog'

print('=' * 60)
print('VERIFICAÇÃO DOS DATASETS')
print('=' * 60)

print(f'\nDiretório principal: {DATASETS_DRIVE}')
print(f'Existe: {DATASETS_DRIVE.is_dir()}')

IMG_EXT = {'.jpg', '.jpeg', '.png'}

def mostrar_estrutura(raiz, profundidade=3, limite=8):
    """
    Exibe uma amostra da estrutura de diretórios.
    Não percorre todas as imagens do dataset.
    """
    if not raiz.exists():
        print(f'ERRO: caminho não encontrado: {raiz}')
        return

    def percorrer(pasta, nivel):
        if nivel > profundidade:
            return

        try:
            itens = sorted(
                pasta.iterdir(),
                key=lambda p: (not p.is_dir(), p.name.lower())
            )
        except (OSError, PermissionError) as erro:
            print('  ' * nivel, erro)
            return

        print('  ' * nivel + f'[{pasta.name}] '
              + f'({len(itens)} itens)')

        for item in itens[:limite]:
            if item.is_dir():
                percorrer(item, nivel + 1)
            else:
                print('  ' * (nivel + 1) + item.name)

        if len(itens) > limite:
            print('  ' * (nivel + 1)
                  + f'... mais {len(itens) - limite} itens')

    percorrer(raiz, 0)


for nome, raiz in [
    ('DogFaceNet', DOGFACENET_RAIZ),
    ('YT-BB-Dog', YTBB_RAIZ)
]:
    print('\n' + '=' * 60)
    print(f'DATASET: {nome}')
    print('=' * 60)

    mostrar_estrutura(
        raiz,
        profundidade=3,
        limite=5
    )

print('\nBloco [2] concluído.')


In [ ]:

# ============================================================
# Bloco de código [3] — Inventário dos datasets
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd

EXTENSOES = {'.jpg', '.jpeg', '.png', '.webp'}

DOGFACENET = (
    Path('/content/drive/MyDrive/datasets_extraido')
    / 'DogFaceNet-master'
    / 'after_4_bis'
)

YTBB_BASE = (
    Path('/content/drive/MyDrive/datasets_extraido')
    / 'YT-BB-dog'
)

def localizar_diretorios_imagens(raiz):
    """
    Localiza diretórios que contêm imagens diretamente,
    sem abrir ou carregar as imagens na memória.
    """
    diretorios = []

    for pasta in raiz.rglob('*'):
        if not pasta.is_dir():
            continue

        try:
            arquivos = [
                item for item in pasta.iterdir()
                if item.is_file()
                and item.suffix.lower() in EXTENSOES
            ]
        except (OSError, PermissionError):
            continue

        if arquivos:
            diretorios.append({
                'diretorio': str(pasta),
                'quantidade': len(arquivos)
            })

    return diretorios


print('=' * 60)
print('INVENTÁRIO — DOGFACENET')
print('=' * 60)

inventario_dogface = localizar_diretorios_imagens(
    DOGFACENET
)

df_dogface = pd.DataFrame(inventario_dogface)

print('Diretórios com imagens:', len(df_dogface))

if not df_dogface.empty:
    print('Total de imagens:',
          df_dogface['quantidade'].sum())

    print('\nImagens por diretório:')
    print(df_dogface['quantidade'].describe())

    print('\nPrimeiros diretórios:')
    print(df_dogface.head(5).to_string(index=False))


print('\n' + '=' * 60)
print('INVENTÁRIO — YT-BB-DOG')
print('=' * 60)

for variante in ['YT-BB-Dog', 'YT-BB-Dog_random_bckg']:

    raiz_variante = YTBB_BASE / variante

    print(f'\nVariante: {variante}')

    if not raiz_variante.exists():
        print('Diretório não encontrado.')
        continue

    inventario = localizar_diretorios_imagens(
        raiz_variante
    )

    df_variante = pd.DataFrame(inventario)

    print('Diretórios com imagens:', len(df_variante))

    if not df_variante.empty:
        print('Total de imagens:',
              df_variante['quantidade'].sum())

        print('\nPrimeiros diretórios:')
        print(df_variante.head(8).to_string(index=False))

        print('\nEstatísticas:')
        print(df_variante['quantidade'].describe())


# Persistência do inventário DogFaceNet
if not df_dogface.empty:
    destino = PASTAS['resultados'] / 'inventario_dogfacenet.csv'

    df_dogface.to_csv(
        destino,
        index=False
    )

    print('\nInventário salvo em:', destino)

print('\nBloco [3] concluído.')


In [ ]:

# ============================================================
# Bloco de código [4] — Inventário corrigido do YT-BB-Dog
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
import pandas as pd

YTBB_BASE = (
    Path('/content/drive/MyDrive/datasets_extraido')
    / 'YT-BB-dog'
    / 'YT-BB-dog'
)

VARIANTES = {
    'original': YTBB_BASE / 'YT-BB-Dog',
    'random_bckg': YTBB_BASE / 'YT-BB-Dog_random_bckg'
}

inventarios_ytbb = {}

print('=' * 60)
print('INVENTÁRIO CORRIGIDO — YT-BB-DOG')
print('=' * 60)

for nome, raiz in VARIANTES.items():

    print(f'\nVariante: {nome}')
    print('Caminho:', raiz)
    print('Existe:', raiz.is_dir())

    if not raiz.is_dir():
        continue

    inventario = localizar_diretorios_imagens(raiz)

    df = pd.DataFrame(inventario)
    inventarios_ytbb[nome] = df

    if df.empty:
        print('Nenhuma imagem encontrada.')
        continue

    print('Diretórios com imagens:', len(df))
    print('Total de imagens:', df['quantidade'].sum())

    print('\nEstatísticas:')
    print(df['quantidade'].describe())

    print('\nExemplos de diretórios:')
    print(df.head(8).to_string(index=False))

    destino = (
        PASTAS['resultados']
        / f'inventario_ytbb_{nome}.csv'
    )

    df.to_csv(destino, index=False)
    print('\nInventário salvo:', destino)

print('\nBloco [4] concluído.')


In [ ]:

# ============================================================
# Bloco de código [5] — Índice unificado das identidades
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd

EXTENSOES = {'.jpg', '.jpeg', '.png', '.webp'}

DOGFACENET = (
    Path('/content/drive/MyDrive/datasets_extraido')
    / 'DogFaceNet-master'
    / 'after_4_bis'
)

YTBB_ORIGINAL = (
    Path('/content/drive/MyDrive/datasets_extraido')
    / 'YT-BB-dog'
    / 'YT-BB-dog'
    / 'YT-BB-Dog'
)

registros = []

def registrar_identidades(raiz, dataset):
    """
    Cada diretório terminal contendo imagens é tratado
    como uma identidade candidata.

    Não altera os arquivos originais.
    """
    for pasta in sorted(raiz.rglob('*')):
        if not pasta.is_dir():
            continue

        imagens = sorted([
            arq for arq in pasta.iterdir()
            if arq.is_file()
            and arq.suffix.lower() in EXTENSOES
        ])

        if not imagens:
            continue

        identidade_local = pasta.name

        identidade_global = (
            f'{dataset}::{identidade_local}'
        )

        partes = [
            parte.lower()
            for parte in pasta.relative_to(raiz).parts
        ]

        divisao_original = 'nao_identificada'

        if 'train' in partes:
            divisao_original = 'train'
        elif 'test' in partes:
            divisao_original = 'test'

        for imagem in imagens:
            registros.append({
                'dataset': dataset,
                'identidade': identidade_global,
                'identidade_local': identidade_local,
                'divisao_original': divisao_original,
                'caminho': str(imagem)
            })


registrar_identidades(DOGFACENET, 'DogFaceNet')
registrar_identidades(YTBB_ORIGINAL, 'YTBB')

df_imagens = pd.DataFrame(registros)

print('=' * 60)
print('ÍNDICE UNIFICADO')
print('=' * 60)

print('Total de imagens:', len(df_imagens))

print(
    'Identidades globais:',
    df_imagens['identidade'].nunique()
)

print('\nImagens por dataset:')
print(df_imagens.groupby('dataset').size())

print('\nIdentidades por dataset:')
print(
    df_imagens.groupby('dataset')['identidade'].nunique()
)

print('\nDivisões originais:')
print(
    df_imagens.groupby(
        ['dataset', 'divisao_original']
    )['identidade'].nunique()
)

# Verificar possíveis identificadores repetidos
# dentro de diferentes pastas do mesmo dataset

pastas_por_identidade = (
    df_imagens.assign(
        pasta=df_imagens['caminho'].map(
            lambda p: str(Path(p).parent)
        )
    )
    .groupby('identidade')['pasta']
    .nunique()
)

conflitos = pastas_por_identidade[
    pastas_por_identidade > 1
]

print('\nIdentidades em múltiplas pastas:',
      len(conflitos))

if len(conflitos):
    print(conflitos.head(15))

# Quantidade de imagens por identidade

df_identidades = (
    df_imagens.groupby(
        ['dataset', 'identidade'],
        as_index=False
    )
    .agg(
        quantidade_imagens=('caminho', 'size')
    )
)

print('\nDistribuição de imagens por identidade:')
print(
    df_identidades.groupby('dataset')[
        'quantidade_imagens'
    ].describe()
)

print('\nIdentidades com menos de 3 imagens:')
print(
    df_identidades[
        df_identidades['quantidade_imagens'] < 3
    ].groupby('dataset').size()
)

# Salvar índices

df_imagens.to_csv(
    PASTAS['splits'] / 'indice_imagens.csv',
    index=False
)

df_identidades.to_csv(
    PASTAS['splits'] / 'indice_identidades.csv',
    index=False
)

print('\nÍndices salvos em:', PASTAS['splits'])
print('Bloco [5] concluído.')


In [ ]:

# ============================================================
# Bloco de código [6] — Divisão 70/30 + K-Fold
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

SEED = 42
N_FOLDS = 5
TEST_SIZE = 0.30

PASTA_SPLITS = PASTAS['splits']

# Carregar os índices criados no bloco [5]
df_identidades = pd.read_csv(
    PASTA_SPLITS / 'indice_identidades.csv'
)

df_imagens = pd.read_csv(
    PASTA_SPLITS / 'indice_imagens.csv'
)

# Verificações iniciais
assert df_identidades['identidade'].is_unique
assert df_imagens['identidade'].nunique() == len(df_identidades)

# ------------------------------------------------------------
# 1. Separação por identidade: 70% / 30%
# Estratificação pela origem do dataset
# ------------------------------------------------------------

ids_desenvolvimento, ids_teste = train_test_split(
    df_identidades,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
    stratify=df_identidades['dataset']
)

ids_desenvolvimento = ids_desenvolvimento.copy()
ids_teste = ids_teste.copy()

print('=' * 60)
print('DIVISÃO 70% / 30%')
print('=' * 60)

print('Desenvolvimento:', len(ids_desenvolvimento))
print('Teste:', len(ids_teste))

print('\nIdentidades por dataset:')
print('Desenvolvimento:')
print(ids_desenvolvimento['dataset'].value_counts())

print('\nTeste:')
print(ids_teste['dataset'].value_counts())

# Garantir ausência de vazamento de identidades
assert set(ids_desenvolvimento['identidade']).isdisjoint(
    set(ids_teste['identidade'])
)

# Salvar divisão principal
ids_desenvolvimento.to_csv(
    PASTA_SPLITS / 'identidades_desenvolvimento_70.csv',
    index=False
)

ids_teste.to_csv(
    PASTA_SPLITS / 'identidades_teste_30.csv',
    index=False
)

# ------------------------------------------------------------
# 2. Validação cruzada com 5 folds
# Cada fold contém identidades distintas
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

registros_folds = []

for fold, (idx_treino, idx_validacao) in enumerate(
    skf.split(
        ids_desenvolvimento,
        ids_desenvolvimento['dataset']
    ),
    start=1
):
    treino_fold = ids_desenvolvimento.iloc[
        idx_treino
    ].copy()

    validacao_fold = ids_desenvolvimento.iloc[
        idx_validacao
    ].copy()

    assert set(treino_fold['identidade']).isdisjoint(
        set(validacao_fold['identidade'])
    )

    treino_fold.to_csv(
        PASTA_SPLITS / f'fold_{fold}_treino.csv',
        index=False
    )

    validacao_fold.to_csv(
        PASTA_SPLITS / f'fold_{fold}_validacao.csv',
        index=False
    )

    registros_folds.append({
        'fold': fold,
        'treino': len(treino_fold),
        'validacao': len(validacao_fold)
    })

# ------------------------------------------------------------
# 3. Auditoria das partições
# ------------------------------------------------------------

resumo = pd.DataFrame(registros_folds)

print('\n' + '=' * 60)
print('VALIDAÇÃO CRUZADA — 5 FOLDS')
print('=' * 60)

print(resumo.to_string(index=False))

# Cada identidade deve aparecer exatamente uma vez
# como validação entre os cinco folds

ids_validacao = []

for fold in range(1, N_FOLDS + 1):
    df = pd.read_csv(
        PASTA_SPLITS / f'fold_{fold}_validacao.csv'
    )
    ids_validacao.extend(df['identidade'].tolist())

assert len(ids_validacao) == len(set(ids_validacao))
assert set(ids_validacao) == set(
    ids_desenvolvimento['identidade']
)

# Teste não pode participar de nenhum fold
assert set(ids_teste['identidade']).isdisjoint(
    set(ids_validacao)
)

resumo.to_csv(
    PASTA_SPLITS / 'resumo_folds.csv',
    index=False
)

print('\nAuditoria concluída: nenhuma identidade')
print('compartilhada entre teste e desenvolvimento.')

print('\nArquivos salvos em:', PASTA_SPLITS)
print('Bloco [6] concluído.')


In [ ]:

# ============================================================
# Bloco de código [7] — Validação simplificada dos dados
# GPU: NÃO NECESSÁRIA
# ============================================================

import pandas as pd

PASTA_SPLITS = PASTAS['splits']

df_imagens = pd.read_csv(
    PASTA_SPLITS / 'indice_imagens.csv'
)

ids_desenvolvimento = pd.read_csv(
    PASTA_SPLITS / 'identidades_desenvolvimento_70.csv'
)

ids_teste = pd.read_csv(
    PASTA_SPLITS / 'identidades_teste_30.csv'
)

# ------------------------------------------------------------
# 1. Verificar separação das identidades
# ------------------------------------------------------------

desenvolvimento = set(
    ids_desenvolvimento['identidade']
)

teste = set(
    ids_teste['identidade']
)

assert desenvolvimento.isdisjoint(teste), (
    'ERRO: identidades compartilhadas entre as partições.'
)

assert (
    desenvolvimento | teste
) == set(df_imagens['identidade']), (
    'ERRO: existem identidades fora das partições.'
)

# ------------------------------------------------------------
# 2. Verificar quantidade de imagens por identidade
# ------------------------------------------------------------

contagem = df_imagens.groupby('identidade').size()

insuficientes = contagem[contagem < 2]

assert len(insuficientes) == 0, (
    'ERRO: existem identidades com menos de 2 imagens.'
)

# ------------------------------------------------------------
# 3. Registrar partição das imagens
# ------------------------------------------------------------

df_imagens['particao'] = df_imagens[
    'identidade'
].map(
    lambda identidade:
        'desenvolvimento'
        if identidade in desenvolvimento
        else 'teste'
)

# ------------------------------------------------------------
# 4. Salvar manifesto
# ------------------------------------------------------------

destino = (
    PASTA_SPLITS / 'manifesto_imagens_validas.csv'
)

df_imagens.to_csv(destino, index=False)

print('=' * 60)
print('VALIDAÇÃO SIMPLIFICADA')
print('=' * 60)

print('Total de imagens:', len(df_imagens))
print('Total de identidades:',
      df_imagens['identidade'].nunique())

print('\nImagens por partição:')
print(df_imagens['particao'].value_counts())

print('\nIdentidades com menos de 2 imagens:',
      len(insuficientes))

print('\nSobreposição de identidades:',
      len(desenvolvimento & teste))

print('\nManifesto salvo em:', destino)

print('\nBloco [7] concluído.')


In [ ]:

# ============================================================
# Bloco de código [8] — Preparação dos dados por fold
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
import pandas as pd

PASTA_SPLITS = PASTAS['splits']

PASTA_LOCAL = Path('/content/tcc1_kfold')
PASTA_LOCAL.mkdir(parents=True, exist_ok=True)

df_imagens = pd.read_csv(
    PASTA_SPLITS / 'manifesto_imagens_validas.csv'
)

print('=' * 60)
print('PREPARAÇÃO DOS FOLDS')
print('=' * 60)

resumo_folds = []

for fold in range(1, 6):

    ids_treino = pd.read_csv(
        PASTA_SPLITS / f'fold_{fold}_treino.csv'
    )

    ids_validacao = pd.read_csv(
        PASTA_SPLITS / f'fold_{fold}_validacao.csv'
    )

    treino = df_imagens[
        df_imagens['identidade'].isin(
            ids_treino['identidade']
        )
    ].copy()

    validacao = df_imagens[
        df_imagens['identidade'].isin(
            ids_validacao['identidade']
        )
    ].copy()

    assert set(treino['identidade']).isdisjoint(
        set(validacao['identidade'])
    )

    assert (
        set(treino['identidade'])
        | set(validacao['identidade'])
    ) == set(
        df_imagens.loc[
            df_imagens['particao'] == 'desenvolvimento',
            'identidade'
        ]
    )

    treino.to_csv(
        PASTA_SPLITS / f'fold_{fold}_imagens_treino.csv',
        index=False
    )

    validacao.to_csv(
        PASTA_SPLITS / f'fold_{fold}_imagens_validacao.csv',
        index=False
    )

    resumo_folds.append({
        'fold': fold,
        'imagens_treino': len(treino),
        'imagens_validacao': len(validacao),
        'identidades_treino':
            treino['identidade'].nunique(),
        'identidades_validacao':
            validacao['identidade'].nunique()
    })

# Manifesto do teste independente
teste = df_imagens[
    df_imagens['particao'] == 'teste'
].copy()

teste.to_csv(
    PASTA_SPLITS / 'imagens_teste_30.csv',
    index=False
)

df_resumo = pd.DataFrame(resumo_folds)

print('\nResumo dos folds:')
print(df_resumo.to_string(index=False))

print('\nTeste independente:')
print('Imagens:', len(teste))
print('Identidades:', teste['identidade'].nunique())

print('\nDiretório local preparado:', PASTA_LOCAL)
print('Bloco [8] concluído.')


In [ ]:

# ============================================================
# Bloco de código [9] — Planejamento da cópia local
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
import shutil
import pandas as pd

PASTA_LOCAL = Path('/content/tcc1_kfold')
PASTA_LOCAL.mkdir(parents=True, exist_ok=True)

PASTA_DATASETS_LOCAL = PASTA_LOCAL / 'datasets'
PASTA_DATASETS_LOCAL.mkdir(
    parents=True,
    exist_ok=True
)

DOGFACENET_ORIGEM = (
    Path('/content/drive/MyDrive/datasets_extraido')
    / 'DogFaceNet-master'
    / 'after_4_bis'
)

YTBB_ORIGEM = (
    Path('/content/drive/MyDrive/datasets_extraido')
    / 'YT-BB-dog'
    / 'YT-BB-dog'
    / 'YT-BB-Dog'
    / 'YT-BB-Dog'
    / 'YT-BB-Dog'
)

ORIGENS = {
    'DogFaceNet': DOGFACENET_ORIGEM,
    'YTBB': YTBB_ORIGEM
}

print('=' * 60)
print('PLANEJAMENTO DA CÓPIA LOCAL')
print('=' * 60)

# ------------------------------------------------------------
# 1. Verificar espaço disponível
# ------------------------------------------------------------

uso = shutil.disk_usage('/content')

GB = 1024 ** 3

print('\nArmazenamento do Colab:')
print(f'Total: {uso.total / GB:.2f} GB')
print(f'Usado: {uso.used / GB:.2f} GB')
print(f'Livre: {uso.free / GB:.2f} GB')

# ------------------------------------------------------------
# 2. Verificar diretórios de origem
# ------------------------------------------------------------

print('\nDiretórios de origem:')

for nome, origem in ORIGENS.items():
    print(f'\n{nome}:')
    print('Caminho:', origem)
    print('Existe:', origem.is_dir())

    if origem.is_dir():
        subpastas = [
            p.name for p in origem.iterdir()
            if p.is_dir()
        ]

        print('Subpastas:', subpastas[:10])

# ------------------------------------------------------------
# 3. Verificar quantidade de imagens no manifesto
# ------------------------------------------------------------

df = pd.read_csv(
    PASTAS['splits'] / 'manifesto_imagens_validas.csv'
)

print('\nImagens previstas para cópia:')
print(df['dataset'].value_counts())

print('\nTotal:', len(df))

print('\nDestino local:')
print(PASTA_DATASETS_LOCAL)

print('\nBloco [9] concluído.')


In [ ]:

# ============================================================
# Bloco de código [10] — Cópia local dos datasets
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
import shutil
import time
import pandas as pd

DESTINO_BASE = Path('/content/tcc1_kfold/datasets')

DESTINO_DOGFACENET = DESTINO_BASE / 'DogFaceNet'
DESTINO_YTBB = DESTINO_BASE / 'YTBB'

COPIAS = {
    'DogFaceNet': (
        DOGFACENET_ORIGEM,
        DESTINO_DOGFACENET
    ),
    'YTBB': (
        YTBB_ORIGEM,
        DESTINO_YTBB
    )
}

print('=' * 60)
print('CÓPIA DOS DATASETS PARA O COLAB')
print('=' * 60)

inicio = time.time()

for nome, (origem, destino) in COPIAS.items():

    print(f'\nDataset: {nome}')
    print('Origem:', origem)
    print('Destino:', destino)

    if not origem.is_dir():
        raise FileNotFoundError(
            f'Diretório de origem não encontrado: {origem}'
        )

    inicio_dataset = time.time()

    # dirs_exist_ok permite retomar a cópia,
    # mas arquivos já existentes podem ser sobrescritos.
    shutil.copytree(
        origem,
        destino,
        dirs_exist_ok=True
    )

    tempo = time.time() - inicio_dataset

    print(f'Cópia concluída em {tempo / 60:.2f} minutos')

print('\n' + '=' * 60)
print('VALIDAÇÃO DOS CAMINHOS LOCAIS')
print('=' * 60)

df = pd.read_csv(
    PASTAS['splits'] / 'manifesto_imagens_validas.csv'
)

def converter_caminho_local(linha):

    caminho = Path(linha['caminho'])

    if linha['dataset'] == 'DogFaceNet':
        relativo = caminho.relative_to(DOGFACENET_ORIGEM)
        return str(DESTINO_DOGFACENET / relativo)

    elif linha['dataset'] == 'YTBB':
        relativo = caminho.relative_to(YTBB_ORIGEM)
        return str(DESTINO_YTBB / relativo)

    raise ValueError('Dataset desconhecido')


df['caminho_local'] = df.apply(
    converter_caminho_local,
    axis=1
)

# Verificação rápida de existência, sem abrir as imagens
faltantes = [
    caminho
    for caminho in df['caminho_local']
    if not Path(caminho).is_file()
]

print('Total de imagens:', len(df))
print('Arquivos não encontrados:', len(faltantes))

if faltantes:
    print('\nPrimeiros arquivos ausentes:')
    for caminho in faltantes[:10]:
        print('-', caminho)

    raise RuntimeError(
        'Cópia incompleta. Verifique os arquivos ausentes.'
    )

# Salvar manifesto com caminhos locais
df.to_csv(
    PASTAS['splits'] / 'manifesto_imagens_local.csv',
    index=False
)

tempo_total = time.time() - inicio

print('\nManifesto local salvo no Google Drive.')
print(f'Tempo total: {tempo_total / 60:.2f} minutos')
print('Bloco [10] concluído.')


In [ ]:

# ============================================================
# Bloco de código [10.1] — Diagnóstico da cópia local
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
import os

DESTINO_BASE = Path('/content/tcc1_kfold/datasets')

ESPERADOS = {
    'DogFaceNet': 8363,
    'YTBB': 27036
}

for dataset, esperado in ESPERADOS.items():

    pasta = DESTINO_BASE / dataset

    quantidade = 0
    tamanho = 0

    if pasta.exists():
        for raiz, dirs, arquivos in os.walk(pasta):
            for nome in arquivos:
                if nome.lower().endswith(
                    ('.jpg', '.jpeg', '.png', '.webp')
                ):
                    quantidade += 1

    percentual = 100 * quantidade / esperado

    print(f'\n{dataset}')
    print(f'Copiados: {quantidade:,} / {esperado:,}')
    print(f'Progresso: {percentual:.1f}%')

print('\nBloco [10.1] concluído.')


In [ ]:

# ============================================================
# Bloco de código [10.2] — Retomar cópia sem sobrescrever
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import shutil
import pandas as pd
import time

PASTA_SPLITS = PASTAS['splits']
DESTINO_BASE = Path('/content/tcc1_kfold/datasets')

df = pd.read_csv(
    PASTA_SPLITS / 'manifesto_imagens_validas.csv'
)

def caminho_destino(linha):
    origem = Path(linha['caminho'])

    if linha['dataset'] == 'DogFaceNet':
        relativo = origem.relative_to(DOGFACENET_ORIGEM)
        return DESTINO_BASE / 'DogFaceNet' / relativo

    if linha['dataset'] == 'YTBB':
        relativo = origem.relative_to(YTBB_ORIGEM)
        return DESTINO_BASE / 'YTBB' / relativo

    raise ValueError('Dataset desconhecido')

df['caminho_local'] = df.apply(
    caminho_destino,
    axis=1
).astype(str)

# Identificar somente arquivos ausentes
pendentes = [
    (Path(origem), Path(destino))
    for origem, destino in zip(
        df['caminho'],
        df['caminho_local']
    )
    if not Path(destino).is_file()
]

print('=' * 60)
print('RETOMADA DA CÓPIA LOCAL')
print('=' * 60)

print('Total de imagens:', len(df))
print('Arquivos pendentes:', len(pendentes))

def copiar_arquivo(par):
    origem, destino = par

    destino.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Arquivos já copiados não são sobrescritos
    if destino.is_file():
        return True

    shutil.copy2(origem, destino)
    return True

inicio = time.time()
erros = []

# Concorrência moderada para não sobrecarregar
# o Google Drive montado.
MAX_WORKERS = 4

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    tarefas = {
        executor.submit(copiar_arquivo, par): par
        for par in pendentes
    }

    for tarefa in tqdm(
        as_completed(tarefas),
        total=len(tarefas),
        desc='Copiando arquivos pendentes'
    ):
        try:
            tarefa.result()
        except Exception as erro:
            origem, destino = tarefas[tarefa]
            erros.append({
                'origem': str(origem),
                'destino': str(destino),
                'erro': str(erro)
            })

print('\nErros de cópia:', len(erros))

if erros:
    pd.DataFrame(erros).to_csv(
        PASTA_SPLITS / 'erros_copia_local.csv',
        index=False
    )
    print('Primeiros erros:')
    print(pd.DataFrame(erros).head(5).to_string(index=False))

# Verificar todos os caminhos locais
faltantes = [
    caminho
    for caminho in df['caminho_local']
    if not Path(caminho).is_file()
]

print('Arquivos ainda ausentes:', len(faltantes))

if not faltantes:
    destino_manifesto = (
        PASTA_SPLITS / 'manifesto_imagens_local.csv'
    )

    df.to_csv(
        destino_manifesto,
        index=False
    )

    print('\nManifesto local salvo:')
    print(destino_manifesto)
    print('Todos os arquivos estão disponíveis.')
else:
    print('\nCópia incompleta.')
    print('O bloco pode ser executado novamente.')

tempo = time.time() - inicio

print(f'\nTempo de execução: {tempo / 60:.2f} minutos')
print('Bloco [10.2] concluído.')


## 2. Arquitetura, Triplet Loss e treinamento

Pré-processamento VGG16, embeddings, mineração semi-hard, lotes P×K e otimização.

In [ ]:

# ============================================================
# Bloco de código [11] — Preparação dos triplets
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import random

SEED = 42
FOLD_ATUAL = 1

PASTA_SPLITS = PASTAS['splits']

# ------------------------------------------------------------
# 1. Carregar manifesto local e identidades do fold
# ------------------------------------------------------------

df = pd.read_csv(
    PASTA_SPLITS / 'manifesto_imagens_local.csv'
)

ids_treino = pd.read_csv(
    PASTA_SPLITS / f'fold_{FOLD_ATUAL}_treino.csv'
)

ids_validacao = pd.read_csv(
    PASTA_SPLITS / f'fold_{FOLD_ATUAL}_validacao.csv'
)

treino = df[
    df['identidade'].isin(ids_treino['identidade'])
].copy()

validacao = df[
    df['identidade'].isin(ids_validacao['identidade'])
].copy()

# ------------------------------------------------------------
# 2. Construir dicionários de identidade -> imagens
# ------------------------------------------------------------

def construir_dataset(dataframe):
    dataset = {}

    for identidade, grupo in dataframe.groupby('identidade'):
        caminhos = grupo['caminho_local'].tolist()

        if len(caminhos) >= 2:
            dataset[identidade] = caminhos

    return dataset

dataset_treino = construir_dataset(treino)
dataset_validacao = construir_dataset(validacao)

assert set(dataset_treino).isdisjoint(
    set(dataset_validacao)
)

assert len(dataset_treino) >= 2
assert len(dataset_validacao) >= 2

# ------------------------------------------------------------
# 3. Amostrador básico de triplets
# ------------------------------------------------------------

class AmostradorTriplets:
    """
    Amostrador inicial.

    A mineração semi-hard será adicionada posteriormente,
    quando o modelo de embeddings estiver disponível.
    """

    def __init__(self, dataset, seed=42):
        self.dataset = dataset
        self.ids = list(dataset.keys())
        self.rng = random.Random(seed)

    def amostrar(self):
        id_ancora = self.rng.choice(self.ids)

        ancora, positivo = self.rng.sample(
            self.dataset[id_ancora],
            2
        )

        id_negativo = self.rng.choice(self.ids)

        while id_negativo == id_ancora:
            id_negativo = self.rng.choice(self.ids)

        negativo = self.rng.choice(
            self.dataset[id_negativo]
        )

        return {
            'id_ancora': id_ancora,
            'id_negativo': id_negativo,
            'ancora': ancora,
            'positivo': positivo,
            'negativo': negativo
        }

# ------------------------------------------------------------
# 4. Testar a amostragem sem carregar imagens
# ------------------------------------------------------------

amostrador_treino = AmostradorTriplets(
    dataset_treino,
    seed=SEED
)

amostrador_validacao = AmostradorTriplets(
    dataset_validacao,
    seed=SEED + 1
)

print('=' * 60)
print(f'AMOSTRAGEM DE TRIPLETS — FOLD {FOLD_ATUAL}')
print('=' * 60)

print('Identidades de treino:', len(dataset_treino))
print('Identidades de validação:', len(dataset_validacao))

for i in range(3):
    triplet = amostrador_treino.amostrar()

    assert triplet['id_ancora'] != triplet['id_negativo']
    assert triplet['ancora'] != triplet['positivo']

    for chave in ['ancora', 'positivo', 'negativo']:
        assert Path(triplet[chave]).is_file()

    print(f'\nTriplet {i + 1}')
    print('Cão âncora:', triplet['id_ancora'])
    print('Cão negativo:', triplet['id_negativo'])
    print('Âncora:', Path(triplet['ancora']).name)
    print('Positivo:', Path(triplet['positivo']).name)
    print('Negativo:', Path(triplet['negativo']).name)

print('\nBloco [11] concluído.')


In [ ]:

# ============================================================
# Bloco de código [12] — Pré-processamento para VGG16
# GPU: NÃO NECESSÁRIA
# ============================================================

import tensorflow as tf
import numpy as np
from pathlib import Path

IMG_SIZE = (224, 224)

def carregar_imagem(caminho):
    """
    Carrega uma imagem e aplica o pré-processamento
    esperado pela VGG16 pré-treinada no ImageNet.

    Retorna um tensor float32 com formato (224, 224, 3).
    """

    caminho = tf.convert_to_tensor(caminho)

    conteudo = tf.io.read_file(caminho)

    imagem = tf.io.decode_image(
        conteudo,
        channels=3,
        expand_animations=False
    )

    imagem.set_shape([None, None, 3])

    imagem = tf.image.resize(
        imagem,
        IMG_SIZE,
        method='bilinear'
    )

    imagem = tf.cast(imagem, tf.float32)

    imagem = tf.keras.applications.vgg16.preprocess_input(
        imagem
    )

    return imagem


def carregar_triplet(triplet):
    """
    Carrega âncora, positivo e negativo.
    """

    ancora = carregar_imagem(triplet['ancora'])
    positivo = carregar_imagem(triplet['positivo'])
    negativo = carregar_imagem(triplet['negativo'])

    return ancora, positivo, negativo


# ------------------------------------------------------------
# Testar com um triplet do bloco [11]
# ------------------------------------------------------------

triplet_teste = amostrador_treino.amostrar()

ancora, positivo, negativo = carregar_triplet(
    triplet_teste
)

print('=' * 60)
print('TESTE DO PRÉ-PROCESSAMENTO')
print('=' * 60)

print('Formato âncora:', ancora.shape)
print('Formato positivo:', positivo.shape)
print('Formato negativo:', negativo.shape)

print('\nTipo dos dados:', ancora.dtype)

print('\nIntervalo dos valores após preprocess_input:')
print('Mínimo:', float(tf.reduce_min(ancora)))
print('Máximo:', float(tf.reduce_max(ancora)))

assert ancora.shape == (224, 224, 3)
assert positivo.shape == (224, 224, 3)
assert negativo.shape == (224, 224, 3)

assert bool(
    tf.reduce_all(tf.math.is_finite(ancora))
)

print('\nPré-processamento validado.')
print('Bloco [12] concluído.')


In [ ]:

# ============================================================
# Bloco de código [13] — Arquitetura VGG16 e embeddings
# GPU: NÃO NECESSÁRIA
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16

SEED = 42
DIM_EMBEDDING = 128

tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# 1. Backbone pré-treinado
# ------------------------------------------------------------

backbone = VGG16(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)

# Primeira fase: congelar todas as camadas convolucionais
backbone.trainable = False

# ------------------------------------------------------------
# 2. Cabeça de embeddings
# ------------------------------------------------------------

entrada = layers.Input(
    shape=(224, 224, 3),
    name='imagem'
)

x = backbone(
    entrada,
    training=False
)

x = layers.GlobalAveragePooling2D(
    name='global_average_pooling'
)(x)

x = layers.Dense(
    256,
    activation='relu',
    name='dense_256'
)(x)

x = layers.Dense(
    DIM_EMBEDDING,
    activation='linear',
    name='embedding_linear'
)(x)

saida = layers.UnitNormalization(
    axis=-1,
    name='embedding_l2'
)(x)

modelo_embeddings = Model(
    inputs=entrada,
    outputs=saida,
    name='vgg16_dog_reid'
)

# ------------------------------------------------------------
# 3. Verificar a arquitetura
# ------------------------------------------------------------

print('=' * 60)
print('ARQUITETURA VGG16 — DOG REIDENTIFICATION')
print('=' * 60)

print('Dimensão do embedding:', DIM_EMBEDDING)
print('Backbone congelado:', not backbone.trainable)

print('\nParâmetros totais:')
print(f'{modelo_embeddings.count_params():,}')

print('\nCamadas treináveis:')
for camada in modelo_embeddings.layers:
    if camada.trainable:
        print('-', camada.name)

# ------------------------------------------------------------
# 4. Testar inferência com uma imagem
# ------------------------------------------------------------

entrada_teste = tf.expand_dims(
    ancora,
    axis=0
)

embedding = modelo_embeddings(
    entrada_teste,
    training=False
)

norma = tf.norm(
    embedding,
    axis=-1
)

print('\nFormato do embedding:', embedding.shape)
print('Norma L2:', norma.numpy())

assert embedding.shape == (1, DIM_EMBEDDING)

assert bool(
    tf.reduce_all(
        tf.abs(norma - 1.0) < 1e-4
    )
)

print('\nArquitetura validada.')
print('Bloco [13] concluído.')


In [ ]:

# ============================================================
# Bloco de código [14] — Triplet Loss e mineração semi-hard
# GPU: NÃO NECESSÁRIA
# ============================================================

import tensorflow as tf
import numpy as np

MARGEM_TRIPLET = 0.2

# ------------------------------------------------------------
# 1. Distância euclidiana quadrática
# ------------------------------------------------------------

def distancia_quadratica(a, b):
    return tf.reduce_sum(
        tf.square(a - b),
        axis=-1
    )

# ------------------------------------------------------------
# 2. Triplet Loss
# ------------------------------------------------------------

def triplet_loss(
    ancora,
    positivo,
    negativo,
    margem=MARGEM_TRIPLET
):
    dist_ap = distancia_quadratica(
        ancora,
        positivo
    )

    dist_an = distancia_quadratica(
        ancora,
        negativo
    )

    perdas = tf.maximum(
        dist_ap - dist_an + margem,
        0.0
    )

    return tf.reduce_mean(perdas)

# ------------------------------------------------------------
# 3. Identificação de triplets semi-hard
# ------------------------------------------------------------

def mascara_semi_hard(
    ancora,
    positivo,
    negativo,
    margem=MARGEM_TRIPLET
):
    dist_ap = distancia_quadratica(
        ancora,
        positivo
    )

    dist_an = distancia_quadratica(
        ancora,
        negativo
    )

    return tf.logical_and(
        dist_ap < dist_an,
        dist_an < dist_ap + margem
    )

# ------------------------------------------------------------
# 4. Testes controlados
# ------------------------------------------------------------

# Vetores simples para validar a matemática.
# Estes exemplos não representam embeddings reais.

ancora_teste = tf.constant(
    [[0.0, 0.0]],
    dtype=tf.float32
)

positivo_teste = tf.constant(
    [[0.2, 0.0]],
    dtype=tf.float32
)

negativo_semi_hard = tf.constant(
    [[0.3, 0.0]],
    dtype=tf.float32
)

negativo_facil = tf.constant(
    [[1.0, 0.0]],
    dtype=tf.float32
)

perda_semi_hard = triplet_loss(
    ancora_teste,
    positivo_teste,
    negativo_semi_hard
)

perda_facil = triplet_loss(
    ancora_teste,
    positivo_teste,
    negativo_facil
)

mascara = mascara_semi_hard(
    ancora_teste,
    positivo_teste,
    negativo_semi_hard
)

print('=' * 60)
print('VALIDAÇÃO DA TRIPLET LOSS')
print('=' * 60)

print('Margem:', MARGEM_TRIPLET)

print('\nDistância âncora-positivo:',
      distancia_quadratica(
          ancora_teste,
          positivo_teste
      ).numpy())

print('Distância âncora-negativo semi-hard:',
      distancia_quadratica(
          ancora_teste,
          negativo_semi_hard
      ).numpy())

print('\nLoss semi-hard:', float(perda_semi_hard))
print('Loss negativo fácil:', float(perda_facil))
print('É semi-hard:', bool(mascara.numpy()[0]))

assert bool(mascara.numpy()[0])
assert float(perda_semi_hard) > 0
assert np.isclose(float(perda_facil), 0.0)

print('\nTriplet Loss validada.')
print('Bloco [14] concluído.')


In [ ]:

# ============================================================
# Bloco de código [15] — Mineração semi-hard real
# GPU: NÃO NECESSÁRIA
# ============================================================

import numpy as np
import tensorflow as tf
from pathlib import Path

NUM_CANDIDATOS = 12
MARGEM_TRIPLET = 0.2

def gerar_embedding(caminho):
    """
    Calcula o embedding de uma imagem utilizando
    o modelo VGG16 criado no bloco [13].
    """
    imagem = carregar_imagem(str(caminho))
    imagem = tf.expand_dims(imagem, axis=0)

    embedding = modelo_embeddings(
        imagem,
        training=False
    )

    return embedding.numpy()[0]


def minerar_negativo_semi_hard(
    caminho_ancora,
    caminho_positivo,
    candidatos_negativos,
    margem=MARGEM_TRIPLET
):
    """
    Seleciona um negativo semi-hard, quando disponível.

    Critério:
        d(A,P) < d(A,N) < d(A,P) + margem

    Caso contrário, utiliza o negativo mais próximo
    entre os candidatos e registra o fallback.
    """

    emb_ancora = gerar_embedding(caminho_ancora)
    emb_positivo = gerar_embedding(caminho_positivo)

    emb_negativos = np.stack([
        gerar_embedding(caminho)
        for caminho in candidatos_negativos
    ])

    dist_ap = np.sum(
        (emb_ancora - emb_positivo) ** 2
    )

    dist_an = np.sum(
        (emb_negativos - emb_ancora) ** 2,
        axis=1
    )

    mascara = (
        (dist_an > dist_ap) &
        (dist_an < dist_ap + margem)
    )

    indices_semi_hard = np.where(mascara)[0]

    if len(indices_semi_hard) > 0:
        # Negativo semi-hard mais próximo
        indice = indices_semi_hard[
            np.argmin(dist_an[indices_semi_hard])
        ]
        tipo = 'semi-hard'
    else:
        indice = int(np.argmin(dist_an))
        tipo = 'fallback'

    return {
        'negativo': candidatos_negativos[indice],
        'dist_ap': float(dist_ap),
        'dist_an': float(dist_an[indice]),
        'tipo': tipo,
        'semi_hard_disponiveis': len(indices_semi_hard)
    }


# ------------------------------------------------------------
# Testar com um triplet do fold 1
# ------------------------------------------------------------

triplet = amostrador_treino.amostrar()

# Selecionar identidades negativas distintas da âncora
ids_negativos = [
    identidade
    for identidade in dataset_treino
    if identidade != triplet['id_ancora']
]

rng = np.random.default_rng(SEED)

ids_selecionados = rng.choice(
    ids_negativos,
    size=NUM_CANDIDATOS,
    replace=False
)

candidatos = [
    str(rng.choice(dataset_treino[identidade]))
    for identidade in ids_selecionados
]

resultado = minerar_negativo_semi_hard(
    triplet['ancora'],
    triplet['positivo'],
    candidatos
)

print('=' * 60)
print('MINERAÇÃO SEMI-HARD — FOLD 1')
print('=' * 60)

print('Cão âncora:', triplet['id_ancora'])
print('Candidatos avaliados:', len(candidatos))

print('\nDistância âncora–positivo:',
      round(resultado['dist_ap'], 6))

print('Distância âncora–negativo:',
      round(resultado['dist_an'], 6))

print('Semi-hard disponíveis:',
      resultado['semi_hard_disponiveis'])

print('Tipo de seleção:', resultado['tipo'])

print('Negativo selecionado:',
      Path(resultado['negativo']).name)

assert resultado['tipo'] in ('semi-hard', 'fallback')
assert Path(resultado['negativo']).is_file()

if resultado['tipo'] == 'semi-hard':
    assert (
        resultado['dist_ap']
        < resultado['dist_an']
        < resultado['dist_ap'] + MARGEM_TRIPLET
    )

print('\nBloco [15] concluído.')


In [ ]:

# ============================================================
# Bloco de código [16] — Mineração semi-hard em lotes
# GPU: NÃO NECESSÁRIA — teste pequeno
# ============================================================

import numpy as np
import tensorflow as tf
from collections import Counter

P = 8
K = 4
MARGEM_TRIPLET = 0.2

rng_batch = np.random.default_rng(SEED + 16)

# ------------------------------------------------------------
# 1. Selecionar P identidades com pelo menos K imagens
# ------------------------------------------------------------

ids_elegiveis = [
    identidade
    for identidade, imagens in dataset_treino.items()
    if len(imagens) >= K
]

assert len(ids_elegiveis) >= P

ids_batch = rng_batch.choice(
    ids_elegiveis,
    size=P,
    replace=False
)

caminhos_batch = []
rotulos_batch = []

for indice, identidade in enumerate(ids_batch):

    imagens = rng_batch.choice(
        dataset_treino[identidade],
        size=K,
        replace=False
    )

    caminhos_batch.extend(imagens.tolist())
    rotulos_batch.extend([indice] * K)

rotulos_batch = np.array(rotulos_batch)

# ------------------------------------------------------------
# 2. Carregar as 32 imagens
# ------------------------------------------------------------

imagens_batch = tf.stack([
    carregar_imagem(caminho)
    for caminho in caminhos_batch
])

# ------------------------------------------------------------
# 3. Calcular embeddings em uma única chamada
# ------------------------------------------------------------

embeddings_batch = modelo_embeddings(
    imagens_batch,
    training=False
)

# ------------------------------------------------------------
# 4. Matriz de distâncias quadráticas
# ------------------------------------------------------------

diferencas = (
    embeddings_batch[:, None, :] -
    embeddings_batch[None, :, :]
)

distancias = tf.reduce_sum(
    tf.square(diferencas),
    axis=-1
).numpy()

# ------------------------------------------------------------
# 5. Selecionar triplets semi-hard
# ------------------------------------------------------------

triplets_selecionados = []
contagem = Counter()

N = len(caminhos_batch)

for i in range(N):

    positivos = np.where(
        rotulos_batch == rotulos_batch[i]
    )[0]

    positivos = positivos[positivos != i]

    negativos = np.where(
        rotulos_batch != rotulos_batch[i]
    )[0]

    for j in positivos:

        dist_ap = distancias[i, j]

        dist_negativos = distancias[i, negativos]

        mascara = (
            (dist_negativos > dist_ap) &
            (dist_negativos < dist_ap + MARGEM_TRIPLET)
        )

        candidatos = negativos[mascara]

        if len(candidatos) > 0:
            k = candidatos[
                np.argmin(distancias[i, candidatos])
            ]
            tipo = 'semi-hard'
        else:
            k = negativos[
                np.argmin(dist_negativos)
            ]
            tipo = 'fallback'

        triplets_selecionados.append(
            (i, j, int(k), tipo)
        )

        contagem[tipo] += 1

# ------------------------------------------------------------
# 6. Validar os triplets
# ------------------------------------------------------------

for i, j, k, tipo in triplets_selecionados:

    assert rotulos_batch[i] == rotulos_batch[j]
    assert rotulos_batch[i] != rotulos_batch[k]

    if tipo == 'semi-hard':
        assert (
            distancias[i, j]
            < distancias[i, k]
            < distancias[i, j] + MARGEM_TRIPLET
        )

total = len(triplets_selecionados)

print('=' * 60)
print('MINERAÇÃO SEMI-HARD VETORIZADA')
print('=' * 60)

print('Identidades:', P)
print('Imagens por identidade:', K)
print('Imagens no lote:', N)
print('Formato dos embeddings:', embeddings_batch.shape)

print('\nTriplets gerados:', total)
print('Semi-hard:', contagem['semi-hard'])
print('Fallback:', contagem['fallback'])

print(
    'Taxa semi-hard:',
    f"{100 * contagem['semi-hard'] / total:.2f}%"
)

print('\nPrimeiros 3 triplets:')

for i, j, k, tipo in triplets_selecionados[:3]:
    print(
        f'A={i}, P={j}, N={k}, '
        f'tipo={tipo}, '
        f'dAP={distancias[i,j]:.4f}, '
        f'dAN={distancias[i,k]:.4f}'
    )

print('\nBloco [16] concluído.')



In [ ]:

# ============================================================
# Bloco de código [17] — Validação dos gradientes
# GPU: NÃO NECESSÁRIA — teste pequeno
# ============================================================

import tensorflow as tf
import numpy as np

# ------------------------------------------------------------
# 1. Recuperar índices dos triplets selecionados
# ------------------------------------------------------------

indices_a = tf.constant(
    [t[0] for t in triplets_selecionados],
    dtype=tf.int32
)

indices_p = tf.constant(
    [t[1] for t in triplets_selecionados],
    dtype=tf.int32
)

indices_n = tf.constant(
    [t[2] for t in triplets_selecionados],
    dtype=tf.int32
)

# ------------------------------------------------------------
# 2. Calcular a perda dentro do GradientTape
# ------------------------------------------------------------

with tf.GradientTape() as tape:

    embeddings = modelo_embeddings(
        imagens_batch,
        training=True
    )

    emb_a = tf.gather(embeddings, indices_a)
    emb_p = tf.gather(embeddings, indices_p)
    emb_n = tf.gather(embeddings, indices_n)

    perda = triplet_loss(
        emb_a,
        emb_p,
        emb_n,
        margem=MARGEM_TRIPLET
    )

# ------------------------------------------------------------
# 3. Calcular gradientes das camadas treináveis
# ------------------------------------------------------------

variaveis = modelo_embeddings.trainable_variables

gradientes = tape.gradient(
    perda,
    variaveis
)

gradientes_validos = [
    g for g in gradientes
    if g is not None
]

# ------------------------------------------------------------
# 4. Validar resultados
# ------------------------------------------------------------

print('=' * 60)
print('VALIDAÇÃO DA RETROPROPAGAÇÃO')
print('=' * 60)

print('Triplets utilizados:', len(indices_a))
print('Loss:', float(perda.numpy()))

print('Variáveis treináveis:', len(variaveis))
print('Gradientes encontrados:', len(gradientes_validos))

for variavel, gradiente in zip(
    variaveis,
    gradientes
):
    if gradiente is None:
        print(variavel.name, '-> SEM GRADIENTE')
    else:
        norma = float(tf.norm(gradiente).numpy())
        print(
            variavel.name,
            '-> norma:',
            round(norma, 8)
        )

assert len(gradientes_validos) == len(variaveis), (
    'Nem todas as variáveis treináveis receberam gradientes.'
)

assert bool(
    tf.math.is_finite(perda).numpy()
)

assert all(
    bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
    for g in gradientes_validos
)

assert any(
    float(tf.norm(g).numpy()) > 0
    for g in gradientes_validos
)

print('\nRetropropagação validada.')
print('Bloco [17] concluído.')


In [ ]:

# ============================================================
# Bloco de código [18] — Configuração experimental
# GPU: NÃO NECESSÁRIA
# ============================================================

import json
import tensorflow as tf
from pathlib import Path
from datetime import datetime, timezone

FOLD_ATUAL = 1

CONFIG = {
    'experimento': 'tcc1_semihard_kfold',
    'fold': FOLD_ATUAL,
    'seed': 42,

    'dataset': {
        'desenvolvimento': 0.70,
        'teste_independente': 0.30,
        'n_folds': 5,
        'separacao_por': 'identidade'
    },

    'modelo': {
        'backbone': 'VGG16',
        'pesos_iniciais': 'imagenet',
        'input_shape': [224, 224, 3],
        'embedding_dim': 128,
        'normalizacao': 'L2',
        'backbone_congelado_fase1': True
    },

    'mineracao': {
        'tipo': 'semi-hard_in_batch',
        'P': 8,
        'K': 4,
        'batch_size': 32,
        'margem': 0.2,
        'distancia': 'euclidiana_quadratica',
        'fallback': 'negativo_mais_proximo'
    },

    'treinamento': {
        # Valores provisórios, sujeitos à revisão
        # antes da primeira época.
        'epocas_fase1': 10,
        'epocas_fase2': 10,
        'learning_rate_fase1': 1e-4,
        'learning_rate_fase2': 1e-5,
        'otimizador': 'Adam',
        'fine_tuning': 'block5',
        'early_stopping_patience': 4
    }
}

# ------------------------------------------------------------
# Diretórios exclusivos do fold
# ------------------------------------------------------------

DIR_CHECKPOINT = (
    PASTAS['checkpoints'] / f'fold_{FOLD_ATUAL}'
)

DIR_LOGS = (
    PASTAS['logs'] / f'fold_{FOLD_ATUAL}'
)

DIR_RESULTADOS = (
    PASTAS['resultados'] / f'fold_{FOLD_ATUAL}'
)

for pasta in [
    DIR_CHECKPOINT,
    DIR_LOGS,
    DIR_RESULTADOS
]:
    pasta.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# Salvar configuração
# ------------------------------------------------------------

caminho_config = (
    DIR_RESULTADOS / 'configuracao.json'
)

if caminho_config.exists():
    with open(
        caminho_config,
        'r',
        encoding='utf-8'
    ) as arquivo:
        config_anterior = json.load(arquivo)

    if config_anterior != CONFIG:
        raise RuntimeError(
            'Já existe uma configuração diferente para '
            'este fold. Revise antes de sobrescrever.'
        )
else:
    with open(
        caminho_config,
        'w',
        encoding='utf-8'
    ) as arquivo:
        json.dump(
            CONFIG,
            arquivo,
            indent=4,
            ensure_ascii=False
        )

# ------------------------------------------------------------
# Resumo
# ------------------------------------------------------------

print('=' * 60)
print('CONFIGURAÇÃO DO EXPERIMENTO')
print('=' * 60)

print('Experimento:', CONFIG['experimento'])
print('Fold:', CONFIG['fold'])
print('Backbone:', CONFIG['modelo']['backbone'])
print('Embedding:', CONFIG['modelo']['embedding_dim'])
print('Mineração:', CONFIG['mineracao']['tipo'])
print('Batch:', CONFIG['mineracao']['batch_size'])
print('Margem:', CONFIG['mineracao']['margem'])

print('\nTreinamento planejado:')
print('Fase 1:', CONFIG['treinamento']['epocas_fase1'], 'épocas')
print('Fase 2:', CONFIG['treinamento']['epocas_fase2'], 'épocas')
print('Fine-tuning:', CONFIG['treinamento']['fine_tuning'])

print('\nDiretório de checkpoints:')
print(DIR_CHECKPOINT)

print('\nConfiguração salva:')
print(caminho_config)

print('\nBloco [18] concluído.')


In [ ]:

# ============================================================
# Bloco de código [19] — Gerador de lotes P x K
# GPU: NÃO NECESSÁRIA — teste estrutural
# ============================================================

import numpy as np
import tensorflow as tf
from pathlib import Path

class GeradorLotesPK:
    """
    Seleciona P identidades e K imagens por identidade.

    Cada lote contém P*K imagens.
    Não realiza mineração nem atualização dos pesos.
    """

    def __init__(
        self,
        dataset,
        p=8,
        k=4,
        seed=42
    ):
        self.dataset = dataset
        self.p = p
        self.k = k
        self.rng = np.random.default_rng(seed)

        self.ids_elegiveis = [
            identidade
            for identidade, imagens in dataset.items()
            if len(imagens) >= k
        ]

        if len(self.ids_elegiveis) < p:
            raise ValueError(
                'Identidades insuficientes para P x K.'
            )

    def proximo_lote(self):

        ids = self.rng.choice(
            self.ids_elegiveis,
            size=self.p,
            replace=False
        )

        caminhos = []
        rotulos = []

        for indice, identidade in enumerate(ids):

            imagens = self.rng.choice(
                self.dataset[identidade],
                size=self.k,
                replace=False
            )

            caminhos.extend(imagens.tolist())
            rotulos.extend([indice] * self.k)

        return caminhos, np.array(
            rotulos,
            dtype=np.int32
        )


def carregar_lote_pk(caminhos, rotulos):
    """
    Carrega um lote P x K e devolve tensores
    prontos para a VGG16.
    """

    imagens = tf.stack([
        carregar_imagem(caminho)
        for caminho in caminhos
    ])

    labels = tf.convert_to_tensor(
        rotulos,
        dtype=tf.int32
    )

    return imagens, labels


# ------------------------------------------------------------
# Instanciar geradores separados
# ------------------------------------------------------------

gerador_treino = GeradorLotesPK(
    dataset_treino,
    p=CONFIG['mineracao']['P'],
    k=CONFIG['mineracao']['K'],
    seed=SEED
)

gerador_validacao = GeradorLotesPK(
    dataset_validacao,
    p=CONFIG['mineracao']['P'],
    k=CONFIG['mineracao']['K'],
    seed=SEED + 1000
)

# ------------------------------------------------------------
# Teste estrutural
# ------------------------------------------------------------

caminhos_teste, rotulos_teste = (
    gerador_treino.proximo_lote()
)

assert len(caminhos_teste) == 32
assert len(np.unique(rotulos_teste)) == 8

contagens = np.unique(
    rotulos_teste,
    return_counts=True
)[1]

assert np.all(contagens == 4)

# Testar somente duas imagens, evitando leitura
# desnecessária de todo o lote.
imagens_teste = tf.stack([
    carregar_imagem(caminhos_teste[0]),
    carregar_imagem(caminhos_teste[1])
])

assert imagens_teste.shape == (2, 224, 224, 3)

print('=' * 60)
print('GERADOR DE LOTES P x K')
print('=' * 60)

print('P:', gerador_treino.p)
print('K:', gerador_treino.k)
print('Imagens por lote:', len(caminhos_teste))

print('\nIdentidades elegíveis:')
print('Treino:', len(gerador_treino.ids_elegiveis))
print('Validação:', len(gerador_validacao.ids_elegiveis))

print('\nDistribuição das identidades no lote:')
print(np.unique(
    rotulos_teste,
    return_counts=True
)[1])

print('\nTeste de carregamento:', imagens_teste.shape)

print('\nBloco [19] concluído.')


In [ ]:

# ============================================================
# Bloco de código [20] — Passo de treinamento semi-hard
# GPU: NÃO NECESSÁRIA — apenas definição das funções
# ============================================================

import numpy as np
import tensorflow as tf

MARGEM_TRIPLET = CONFIG['mineracao']['margem']

# ------------------------------------------------------------
# 1. Seleção dos índices dos triplets
# ------------------------------------------------------------

def selecionar_triplets_semihard(embeddings, rotulos, margem=0.2):
    """
    Seleciona negativos semi-hard dentro do lote.

    Se não houver negativo semi-hard, utiliza o
    negativo mais próximo (fallback).

    A seleção é realizada sem gradientes.
    """

    emb = tf.stop_gradient(embeddings)

    distancias = tf.reduce_sum(
        tf.square(
            emb[:, None, :] - emb[None, :, :]
        ),
        axis=-1
    ).numpy()

    labels = np.asarray(rotulos)

    indices_a = []
    indices_p = []
    indices_n = []

    tipos = {
        'semi-hard': 0,
        'fallback': 0
    }

    n = len(labels)

    for i in range(n):

        positivos = np.where(
            labels == labels[i]
        )[0]

        positivos = positivos[positivos != i]

        negativos = np.where(
            labels != labels[i]
        )[0]

        for j in positivos:

            dist_ap = distancias[i, j]
            dist_an = distancias[i, negativos]

            mascara = (
                (dist_an > dist_ap) &
                (dist_an < dist_ap + margem)
            )

            candidatos = negativos[mascara]

            if len(candidatos) > 0:

                k = candidatos[
                    np.argmin(distancias[i, candidatos])
                ]

                tipos['semi-hard'] += 1

            else:

                k = negativos[
                    np.argmin(dist_an)
                ]

                tipos['fallback'] += 1

            indices_a.append(i)
            indices_p.append(j)
            indices_n.append(int(k))

    return (
        tf.constant(indices_a, dtype=tf.int32),
        tf.constant(indices_p, dtype=tf.int32),
        tf.constant(indices_n, dtype=tf.int32),
        tipos
    )


# ------------------------------------------------------------
# 2. Otimizador da fase 1
# ------------------------------------------------------------

otimizador_fase1 = tf.keras.optimizers.Adam(
    learning_rate=CONFIG['treinamento']['learning_rate_fase1']
)


# ------------------------------------------------------------
# 3. Passo de treinamento
# ------------------------------------------------------------

def passo_treinamento(imagens, rotulos):
    """
    Executa um passo completo:
    1. Forward pass
    2. Mineração semi-hard
    3. Triplet Loss
    4. Retropropagação
    5. Atualização dos pesos
    """

    with tf.GradientTape() as tape:

        embeddings = modelo_embeddings(
            imagens,
            training=True
        )

        idx_a, idx_p, idx_n, tipos = (
            selecionar_triplets_semihard(
                embeddings,
                rotulos,
                margem=MARGEM_TRIPLET
            )
        )

        emb_a = tf.gather(embeddings, idx_a)
        emb_p = tf.gather(embeddings, idx_p)
        emb_n = tf.gather(embeddings, idx_n)

        perda = triplet_loss(
            emb_a,
            emb_p,
            emb_n,
            margem=MARGEM_TRIPLET
        )

    variaveis = modelo_embeddings.trainable_variables

    gradientes = tape.gradient(
        perda,
        variaveis
    )

    pares_validos = [
        (gradiente, variavel)
        for gradiente, variavel in zip(
            gradientes,
            variaveis
        )
        if gradiente is not None
    ]

    if not pares_validos:
        raise RuntimeError(
            'Nenhum gradiente foi calculado.'
        )

    otimizador_fase1.apply_gradients(
        pares_validos
    )

    return {
        'loss': float(perda.numpy()),
        'semi-hard': tipos['semi-hard'],
        'fallback': tipos['fallback'],
        'triplets': len(idx_a)
    }


# ------------------------------------------------------------
# 4. Passo de validação (sem atualização)
# ------------------------------------------------------------

def passo_validacao(imagens, rotulos):

    embeddings = modelo_embeddings(
        imagens,
        training=False
    )

    idx_a, idx_p, idx_n, tipos = (
        selecionar_triplets_semihard(
            embeddings,
            rotulos,
            margem=MARGEM_TRIPLET
        )
    )

    perda = triplet_loss(
        tf.gather(embeddings, idx_a),
        tf.gather(embeddings, idx_p),
        tf.gather(embeddings, idx_n),
        margem=MARGEM_TRIPLET
    )

    return {
        'loss': float(perda.numpy()),
        'semi-hard': tipos['semi-hard'],
        'fallback': tipos['fallback'],
        'triplets': len(idx_a)
    }


# ------------------------------------------------------------
# 5. Verificação das funções
# ------------------------------------------------------------

print('=' * 60)
print('PASSOS DE TREINAMENTO E VALIDAÇÃO')
print('=' * 60)

print('Modelo:', modelo_embeddings.name)
print('Otimizador:', type(otimizador_fase1).__name__)
print('Learning rate:', float(
    otimizador_fase1.learning_rate.numpy()
))
print('Margem:', MARGEM_TRIPLET)

print('\nFunções disponíveis:')
print('- selecionar_triplets_semihard')
print('- passo_treinamento')
print('- passo_validacao')

print('\nNenhum peso foi atualizado.')
print('Bloco [20] concluído.')


In [ ]:

# ============================================================
# Bloco de código [21] — Teste de um passo de treinamento
# GPU: NÃO NECESSÁRIA — teste de apenas um lote
# ============================================================

import numpy as np
import tensorflow as tf
import time

print('=' * 60)
print('TESTE CONTROLADO — UM PASSO DE TREINAMENTO')
print('=' * 60)

# ------------------------------------------------------------
# 1. Preparar um lote P x K
# ------------------------------------------------------------

caminhos, rotulos = gerador_treino.proximo_lote()

imagens, labels = carregar_lote_pk(
    caminhos,
    rotulos
)

print('Formato do lote:', imagens.shape)

# ------------------------------------------------------------
# 2. Salvar pesos antes do treinamento
# ------------------------------------------------------------

pesos_antes = [
    np.array(peso, copy=True)
    for peso in modelo_embeddings.get_weights()
]

# Criar otimizador temporário para o teste.
# Assim, não modificamos o estado do otimizador
# principal configurado no bloco [20].
otimizador_original = otimizador_fase1

otimizador_teste = tf.keras.optimizers.Adam(
    learning_rate=CONFIG['treinamento']['learning_rate_fase1']
)

otimizador_fase1 = otimizador_teste

# ------------------------------------------------------------
# 3. Executar um passo e restaurar o modelo
# ------------------------------------------------------------

inicio = time.time()
resultado = None
alteracoes = []

try:
    resultado = passo_treinamento(
        imagens,
        labels
    )

    tempo = time.time() - inicio

    pesos_depois = modelo_embeddings.get_weights()

    alteracoes = [
        float(np.max(np.abs(depois - antes)))
        for antes, depois in zip(
            pesos_antes,
            pesos_depois
        )
    ]

finally:
    # Restaurar os pesos iniciais, mesmo em caso
    # de erro durante o teste.
    modelo_embeddings.set_weights(pesos_antes)
    otimizador_fase1 = otimizador_original

# ------------------------------------------------------------
# 4. Validar o resultado
# ------------------------------------------------------------

print('\nResultado do passo:')
print('Loss:', resultado['loss'])
print('Triplets:', resultado['triplets'])
print('Semi-hard:', resultado['semi-hard'])
print('Fallback:', resultado['fallback'])

print('\nTempo:', round(tempo, 2), 'segundos')

print('\nMaior alteração absoluta nos pesos:')
print(max(alteracoes))

assert np.isfinite(resultado['loss'])
assert resultado['triplets'] == 96
assert max(alteracoes) > 0

# Verificar restauração
pesos_restaurados = modelo_embeddings.get_weights()

assert all(
    np.array_equal(antes, restaurado)
    for antes, restaurado in zip(
        pesos_antes,
        pesos_restaurados
    )
)

print('\nPesos iniciais restaurados.')
print('Teste de treinamento validado.')
print('Bloco [21] concluído.')


## 3. Treinamento e retomadas — Fold 1

Contém checkpoints, diagnósticos e rotinas históricas de recuperação após interrupções do Colab. **Não executar todos os blocos de retomada sem revisar o estado da sessão.**

In [ ]:

# ============================================================
# Bloco de código [22] — Diagnóstico do ambiente GPU
# GPU: OBRIGATÓRIA — ativar T4 antes do treinamento
# ============================================================

import tensorflow as tf
from pathlib import Path

print('=' * 60)
print('DIAGNÓSTICO DO AMBIENTE')
print('=' * 60)

gpus = tf.config.list_physical_devices('GPU')

print('TensorFlow:', tf.__version__)
print('GPUs disponíveis:', len(gpus))

for gpu in gpus:
    print('Dispositivo:', gpu)

print('\nVerificação dos datasets locais:')

pastas = {
    'DogFaceNet': Path('/content/tcc1_kfold/datasets/DogFaceNet'),
    'YTBB': Path('/content/tcc1_kfold/datasets/YTBB')
}

for nome, pasta in pastas.items():
    print(nome, '->', 'OK' if pasta.exists() else 'AUSENTE')

print('\nVerificação do manifesto:')
manifesto = (
    PASTAS['splits'] / 'manifesto_imagens_local.csv'
)

print('Manifesto:', 'OK' if manifesto.exists() else 'AUSENTE')

if not gpus:
    print('\nATENÇÃO: GPU não detectada.')
    print('Não inicie o treinamento completo.')
else:
    print('\nGPU detectada. Ambiente preparado.')

print('\nBloco [22] concluído.')


In [ ]:

# ============================================================
# Bloco de código [23] — Backup dos datasets locais
# GPU: NÃO NECESSÁRIA
# ============================================================

import tarfile
from pathlib import Path
import os
import time

origem = Path('/content/tcc1_kfold/datasets')

destino = (
    BASE / 'experimentos' / 'tcc1_kfold' /
    'datasets_backup.tar'
)

temporario = destino.with_suffix('.tar.tmp')

assert origem.exists(), 'Datasets locais não encontrados.'

print('=' * 60)
print('BACKUP DOS DATASETS LOCAIS')
print('=' * 60)

print('Origem:', origem)
print('Destino:', destino)

if destino.exists():
    print('\nBackup já existe.')
    print('Tamanho:', round(
        destino.stat().st_size / (1024**3), 2
    ), 'GB')
    print('Nenhum arquivo foi sobrescrito.')

else:
    inicio = time.time()

    print('\nCriando backup...')

    with tarfile.open(temporario, 'w') as tar:
        tar.add(
            origem,
            arcname='datasets',
            recursive=True
        )

    os.replace(temporario, destino)

    tempo = time.time() - inicio

    print('\nBackup concluído.')
    print('Tamanho:', round(
        destino.stat().st_size / (1024**3), 2
    ), 'GB')
    print('Tempo:', round(tempo / 60, 2), 'minutos')

print('\nBloco [23] concluído.')


In [ ]:

# ============================================================
# Bloco de código [24] — Restaurar datasets na GPU
# GPU: OBRIGATÓRIA — T4
# ============================================================

from google.colab import drive
from pathlib import Path
import tensorflow as tf
import tarfile
import time

# ------------------------------------------------------------
# 1. Montar Google Drive
# ------------------------------------------------------------

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/TCC_Caes')

BACKUP = (
    BASE / 'experimentos' /
    'tcc1_kfold' / 'datasets_backup.tar'
)

DESTINO = Path('/content/tcc1_kfold')
DATASETS = DESTINO / 'datasets'

# ------------------------------------------------------------
# 2. Verificar GPU
# ------------------------------------------------------------

gpus = tf.config.list_physical_devices('GPU')

print('=' * 60)
print('RESTAURAÇÃO DO AMBIENTE GPU')
print('=' * 60)

print('TensorFlow:', tf.__version__)
print('GPUs detectadas:', len(gpus))

if not gpus:
    raise RuntimeError(
        'GPU não detectada. Ative a T4 antes de continuar.'
    )

for gpu in gpus:
    print('GPU:', gpu)

# ------------------------------------------------------------
# 3. Restaurar datasets
# ------------------------------------------------------------

assert BACKUP.exists(), (
    f'Backup não encontrado: {BACKUP}'
)

DESTINO.mkdir(parents=True, exist_ok=True)

inicio = time.time()

if DATASETS.exists():
    print('\nDatasets locais já existem.')
    print('Verificando arquivos...')
else:
    print('\nExtraindo backup...')

    with tarfile.open(BACKUP, 'r') as tar:
        tar.extractall(
            path=DESTINO,
            filter='data'
        )

    print('Extração concluída.')

# ------------------------------------------------------------
# 4. Conferir quantidade de imagens
# ------------------------------------------------------------

arquivos = [
    caminho
    for caminho in DATASETS.rglob('*')
    if caminho.is_file()
    and caminho.suffix.lower()
    in {'.jpg', '.jpeg', '.png'}
]

quantidade = len(arquivos)

print('\nImagens encontradas:', quantidade)
print('Imagens esperadas:', 35399)

assert quantidade == 35399, (
    'Quantidade de imagens diferente da esperada.'
)

print(
    'Tempo total:',
    round(time.time() - inicio, 2),
    'segundos'
)

print('\nGPU e datasets disponíveis.')
print('Bloco [24] concluído.')


In [ ]:

# ============================================================
# Bloco de código [25] — Reconstruir ambiente do fold 1
# GPU: ATIVADA — T4
# ============================================================

import json
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# ------------------------------------------------------------
# 1. Configuração e reprodutibilidade
# ------------------------------------------------------------

BASE = Path('/content/drive/MyDrive/TCC_Caes')
EXPERIMENTO = BASE / 'experimentos' / 'tcc1_kfold'

PASTAS = {
    nome: EXPERIMENTO / nome
    for nome in [
        'datasets',
        'splits',
        'checkpoints',
        'modelos',
        'logs',
        'resultados'
    ]
}

FOLD_ATUAL = 1

with open(
    PASTAS['resultados']
    / f'fold_{FOLD_ATUAL}'
    / 'configuracao.json',
    'r',
    encoding='utf-8'
) as arquivo:
    CONFIG = json.load(arquivo)

SEED = CONFIG['seed']

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# 2. Recuperar manifesto e identidades
# ------------------------------------------------------------

manifesto = pd.read_csv(
    PASTAS['splits'] / 'manifesto_imagens_local.csv'
)

ids_treino = set(
    pd.read_csv(
        PASTAS['splits']
        / f'fold_{FOLD_ATUAL}_treino.csv'
    )['identidade'].astype(str)
)

ids_validacao = set(
    pd.read_csv(
        PASTAS['splits']
        / f'fold_{FOLD_ATUAL}_validacao.csv'
    )['identidade'].astype(str)
)

ids_teste = set(
    pd.read_csv(
        PASTAS['splits']
        / 'identidades_teste_30.csv'
    )['identidade'].astype(str)
)

assert ids_treino.isdisjoint(ids_validacao)
assert ids_treino.isdisjoint(ids_teste)
assert ids_validacao.isdisjoint(ids_teste)

manifesto['identidade'] = (
    manifesto['identidade'].astype(str)
)

treino = manifesto[
    manifesto['identidade'].isin(ids_treino)
].copy()

validacao = manifesto[
    manifesto['identidade'].isin(ids_validacao)
].copy()

dataset_treino = (
    treino.groupby('identidade')['caminho_local']
    .apply(list)
    .to_dict()
)

dataset_validacao = (
    validacao.groupby('identidade')['caminho_local']
    .apply(list)
    .to_dict()
)

assert set(dataset_treino) == ids_treino
assert set(dataset_validacao) == ids_validacao

# ------------------------------------------------------------
# 3. Reconstruir modelo de embeddings
# ------------------------------------------------------------

tf.keras.backend.clear_session()

backbone = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

backbone.trainable = False

entrada = layers.Input(
    shape=(224, 224, 3),
    name='imagem'
)

x = backbone(entrada)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dense(128, activation='linear')(x)

saida = layers.UnitNormalization(
    axis=-1,
    name='embedding_normalizado'
)(x)

modelo_embeddings = Model(
    inputs=entrada,
    outputs=saida,
    name='vgg16_dog_reid'
)

DIM_EMBEDDING = 128
MARGEM_TRIPLET = CONFIG['mineracao']['margem']

# ------------------------------------------------------------
# 4. Restaurar função de carregamento
# ------------------------------------------------------------

def carregar_imagem(caminho):
    imagem = tf.io.read_file(str(caminho))
    imagem = tf.io.decode_image(
        imagem,
        channels=3,
        expand_animations=False
    )
    imagem.set_shape([None, None, 3])

    imagem = tf.image.resize(
        imagem,
        [224, 224]
    )

    imagem = tf.cast(imagem, tf.float32)

    return preprocess_input(imagem)

# ------------------------------------------------------------
# 5. Verificações
# ------------------------------------------------------------

print('=' * 60)
print('AMBIENTE RECONSTRUÍDO — FOLD 1')
print('=' * 60)

print('GPU:', tf.config.list_physical_devices('GPU'))
print('Identidades de treino:', len(dataset_treino))
print('Identidades de validação:', len(dataset_validacao))
print('Imagens de treino:', len(treino))
print('Imagens de validação:', len(validacao))

print('\nModelo:', modelo_embeddings.name)
print('Parâmetros totais:', modelo_embeddings.count_params())
print('Variáveis treináveis:', len(
    modelo_embeddings.trainable_variables
))

assert len(modelo_embeddings.trainable_variables) == 4
assert modelo_embeddings.output_shape == (None, 128)

print('\nSeparação de identidades: OK')
print('Modelo reconstruído: OK')
print('Bloco [25] concluído.')


In [ ]:

# ============================================================
# Bloco de código [26] — Reconstruir treinamento semi-hard
# GPU: ATIVADA — T4
# ============================================================

import numpy as np
import tensorflow as tf

# ------------------------------------------------------------
# 1. Gerador P x K
# ------------------------------------------------------------

class GeradorLotesPK:

    def __init__(self, dataset, p=8, k=4, seed=42):

        self.dataset = dataset
        self.p = p
        self.k = k
        self.rng = np.random.default_rng(seed)

        self.ids_elegiveis = [
            identidade
            for identidade, imagens in dataset.items()
            if len(imagens) >= k
        ]

        if len(self.ids_elegiveis) < p:
            raise ValueError(
                'Identidades insuficientes para P x K.'
            )

    def proximo_lote(self):

        ids = self.rng.choice(
            self.ids_elegiveis,
            size=self.p,
            replace=False
        )

        caminhos = []
        rotulos = []

        for indice, identidade in enumerate(ids):

            imagens = self.rng.choice(
                self.dataset[identidade],
                size=self.k,
                replace=False
            )

            caminhos.extend(imagens.tolist())
            rotulos.extend([indice] * self.k)

        return caminhos, np.asarray(
            rotulos, dtype=np.int32
        )


def carregar_lote_pk(caminhos, rotulos):

    imagens = tf.stack([
        carregar_imagem(caminho)
        for caminho in caminhos
    ])

    labels = tf.convert_to_tensor(
        rotulos, dtype=tf.int32
    )

    return imagens, labels


# ------------------------------------------------------------
# 2. Triplet Loss
# ------------------------------------------------------------

def triplet_loss(ancora, positivo, negativo, margem=0.2):

    dist_ap = tf.reduce_sum(
        tf.square(ancora - positivo),
        axis=-1
    )

    dist_an = tf.reduce_sum(
        tf.square(ancora - negativo),
        axis=-1
    )

    return tf.reduce_mean(
        tf.maximum(
            dist_ap - dist_an + margem,
            0.0
        )
    )


# ------------------------------------------------------------
# 3. Mineração semi-hard
# ------------------------------------------------------------

def selecionar_triplets_semihard(
    embeddings, rotulos, margem=0.2
):

    emb = tf.stop_gradient(embeddings)

    distancias = tf.reduce_sum(
        tf.square(
            emb[:, None, :] - emb[None, :, :]
        ),
        axis=-1
    ).numpy()

    labels = np.asarray(rotulos)

    indices_a = []
    indices_p = []
    indices_n = []

    tipos = {
        'semi-hard': 0,
        'fallback': 0
    }

    for i in range(len(labels)):

        positivos = np.where(
            labels == labels[i]
        )[0]

        positivos = positivos[positivos != i]

        negativos = np.where(
            labels != labels[i]
        )[0]

        for j in positivos:

            dist_ap = distancias[i, j]
            dist_an = distancias[i, negativos]

            mascara = (
                (dist_an > dist_ap) &
                (dist_an < dist_ap + margem)
            )

            candidatos = negativos[mascara]

            if len(candidatos) > 0:

                k = candidatos[
                    np.argmin(distancias[i, candidatos])
                ]

                tipos['semi-hard'] += 1

            else:

                k = negativos[
                    np.argmin(dist_an)
                ]

                tipos['fallback'] += 1

            indices_a.append(i)
            indices_p.append(j)
            indices_n.append(int(k))

    return (
        tf.constant(indices_a, dtype=tf.int32),
        tf.constant(indices_p, dtype=tf.int32),
        tf.constant(indices_n, dtype=tf.int32),
        tipos
    )


# ------------------------------------------------------------
# 4. Otimizador da fase 1
# ------------------------------------------------------------

otimizador_fase1 = tf.keras.optimizers.Adam(
    learning_rate=CONFIG['treinamento']['learning_rate_fase1']
)


# ------------------------------------------------------------
# 5. Passo de treinamento
# ------------------------------------------------------------

def passo_treinamento(imagens, rotulos):

    with tf.GradientTape() as tape:

        embeddings = modelo_embeddings(
            imagens,
            training=True
        )

        idx_a, idx_p, idx_n, tipos = (
            selecionar_triplets_semihard(
                embeddings,
                rotulos,
                margem=MARGEM_TRIPLET
            )
        )

        perda = triplet_loss(
            tf.gather(embeddings, idx_a),
            tf.gather(embeddings, idx_p),
            tf.gather(embeddings, idx_n),
            margem=MARGEM_TRIPLET
        )

    variaveis = modelo_embeddings.trainable_variables

    gradientes = tape.gradient(
        perda,
        variaveis
    )

    pares = [
        (g, v)
        for g, v in zip(gradientes, variaveis)
        if g is not None
    ]

    if not pares:
        raise RuntimeError(
            'Nenhum gradiente calculado.'
        )

    otimizador_fase1.apply_gradients(pares)

    return {
        'loss': float(perda.numpy()),
        'semi-hard': tipos['semi-hard'],
        'fallback': tipos['fallback'],
        'triplets': int(tf.size(idx_a))
    }


# ------------------------------------------------------------
# 6. Passo de validação
# ------------------------------------------------------------

def passo_validacao(imagens, rotulos):

    embeddings = modelo_embeddings(
        imagens,
        training=False
    )

    idx_a, idx_p, idx_n, tipos = (
        selecionar_triplets_semihard(
            embeddings,
            rotulos,
            margem=MARGEM_TRIPLET
        )
    )

    perda = triplet_loss(
        tf.gather(embeddings, idx_a),
        tf.gather(embeddings, idx_p),
        tf.gather(embeddings, idx_n),
        margem=MARGEM_TRIPLET
    )

    return {
        'loss': float(perda.numpy()),
        'semi-hard': tipos['semi-hard'],
        'fallback': tipos['fallback'],
        'triplets': int(tf.size(idx_a))
    }


# ------------------------------------------------------------
# 7. Instanciar geradores
# ------------------------------------------------------------

gerador_treino = GeradorLotesPK(
    dataset_treino,
    p=CONFIG['mineracao']['P'],
    k=CONFIG['mineracao']['K'],
    seed=SEED
)

gerador_validacao = GeradorLotesPK(
    dataset_validacao,
    p=CONFIG['mineracao']['P'],
    k=CONFIG['mineracao']['K'],
    seed=SEED + 1000
)

# ------------------------------------------------------------
# 8. Verificações
# ------------------------------------------------------------

print('=' * 60)
print('TREINAMENTO SEMI-HARD RECONSTRUÍDO')
print('=' * 60)

print('P:', gerador_treino.p)
print('K:', gerador_treino.k)
print('Batch:', gerador_treino.p * gerador_treino.k)

print('\nIdentidades elegíveis:')
print('Treino:', len(gerador_treino.ids_elegiveis))
print('Validação:', len(gerador_validacao.ids_elegiveis))

print('\nMargem:', MARGEM_TRIPLET)
print('Learning rate:', float(
    otimizador_fase1.learning_rate.numpy()
))

print('\nFunções reconstruídas.')
print('Nenhum peso atualizado.')
print('Bloco [26] concluído.')


In [ ]:

# ============================================================
# Bloco de código [27] — Planejamento das épocas
# GPU: NÃO NECESSÁRIA
# ============================================================

import json
import math
import numpy as np
import pandas as pd

from pathlib import Path

# ------------------------------------------------------------
# 1. Diretórios do fold
# ------------------------------------------------------------

DIR_CHECKPOINT = (
    PASTAS['checkpoints'] / f'fold_{FOLD_ATUAL}'
)

DIR_LOGS = (
    PASTAS['logs'] / f'fold_{FOLD_ATUAL}'
)

DIR_RESULTADOS = (
    PASTAS['resultados'] / f'fold_{FOLD_ATUAL}'
)

for pasta in [
    DIR_CHECKPOINT,
    DIR_LOGS,
    DIR_RESULTADOS
]:
    pasta.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# 2. Calcular orçamento de lotes por época
# ------------------------------------------------------------

P = CONFIG['mineracao']['P']
K = CONFIG['mineracao']['K']
BATCH_SIZE = P * K

imagens_elegiveis_treino = sum(
    len(dataset_treino[identidade])
    for identidade in gerador_treino.ids_elegiveis
)

imagens_elegiveis_validacao = sum(
    len(dataset_validacao[identidade])
    for identidade in gerador_validacao.ids_elegiveis
)

PASSOS_TREINO = math.ceil(
    imagens_elegiveis_treino / BATCH_SIZE
)

PASSOS_VALIDACAO = math.ceil(
    imagens_elegiveis_validacao / BATCH_SIZE
)

# ------------------------------------------------------------
# 3. Preparar lotes fixos de validação
# ------------------------------------------------------------

CAMINHO_LOTES_VALIDACAO = (
    DIR_RESULTADOS / 'lotes_validacao.json'
)

if CAMINHO_LOTES_VALIDACAO.exists():

    with open(
        CAMINHO_LOTES_VALIDACAO,
        'r',
        encoding='utf-8'
    ) as arquivo:
        lotes_validacao = json.load(arquivo)

    print('Lotes de validação recuperados do Drive.')

else:

    lotes_validacao = []

    for _ in range(PASSOS_VALIDACAO):

        caminhos, rotulos = (
            gerador_validacao.proximo_lote()
        )

        lotes_validacao.append({
            'caminhos': list(caminhos),
            'rotulos': rotulos.tolist()
        })

    with open(
        CAMINHO_LOTES_VALIDACAO,
        'w',
        encoding='utf-8'
    ) as arquivo:
        json.dump(
            lotes_validacao,
            arquivo,
            indent=2,
            ensure_ascii=False
        )

    print('Lotes de validação gerados e salvos.')

# ------------------------------------------------------------
# 4. Verificar estrutura
# ------------------------------------------------------------

assert len(lotes_validacao) == PASSOS_VALIDACAO

for lote in lotes_validacao:

    assert len(lote['caminhos']) == BATCH_SIZE
    assert len(lote['rotulos']) == BATCH_SIZE

    contagens = np.unique(
        lote['rotulos'],
        return_counts=True
    )[1]

    assert len(contagens) == P
    assert np.all(contagens == K)

# Garantir que a validação usa apenas suas identidades.
caminhos_validacao_permitidos = set(
    validacao['caminho_local'].astype(str)
)

assert all(
    caminho in caminhos_validacao_permitidos
    for lote in lotes_validacao
    for caminho in lote['caminhos']
)

# ------------------------------------------------------------
# 5. Salvar planejamento
# ------------------------------------------------------------

PLANEJAMENTO = {
    'fold': FOLD_ATUAL,
    'batch_size': BATCH_SIZE,
    'passos_treino': PASSOS_TREINO,
    'passos_validacao': PASSOS_VALIDACAO,
    'imagens_elegiveis_treino': imagens_elegiveis_treino,
    'imagens_elegiveis_validacao': imagens_elegiveis_validacao,
    'epocas_fase1': CONFIG['treinamento']['epocas_fase1'],
    'epocas_fase2': CONFIG['treinamento']['epocas_fase2'],
    'validacao_fixa': True
}

CAMINHO_PLANEJAMENTO = (
    DIR_RESULTADOS / 'planejamento.json'
)

if CAMINHO_PLANEJAMENTO.exists():

    with open(
        CAMINHO_PLANEJAMENTO,
        'r',
        encoding='utf-8'
    ) as arquivo:
        planejamento_anterior = json.load(arquivo)

    assert planejamento_anterior == PLANEJAMENTO, (
        'Planejamento existente diferente. '
        'Revise antes de prosseguir.'
    )

else:

    with open(
        CAMINHO_PLANEJAMENTO,
        'w',
        encoding='utf-8'
    ) as arquivo:
        json.dump(
            PLANEJAMENTO,
            arquivo,
            indent=2,
            ensure_ascii=False
        )

# ------------------------------------------------------------
# 6. Resumo
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('PLANEJAMENTO DO TREINAMENTO — FOLD 1')
print('=' * 60)

print('Imagens elegíveis de treino:', imagens_elegiveis_treino)
print('Imagens elegíveis de validação:', imagens_elegiveis_validacao)

print('\nBatch size:', BATCH_SIZE)
print('Passos de treino por época:', PASSOS_TREINO)
print('Passos de validação por época:', PASSOS_VALIDACAO)

print('\nÉpocas fase 1:', PLANEJAMENTO['epocas_fase1'])
print('Épocas fase 2:', PLANEJAMENTO['epocas_fase2'])

print('\nLotes fixos de validação:', len(lotes_validacao))
print('Planejamento salvo:', CAMINHO_PLANEJAMENTO)

print('\nBloco [27] concluído.')


In [ ]:

# ============================================================
# Bloco de código [28] — Controle e checkpoints
# GPU: ATIVADA — sem treinamento
# ============================================================

import os
import json
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path

# ------------------------------------------------------------
# 1. Arquivos de acompanhamento
# ------------------------------------------------------------

ARQUIVO_HISTORICO = (
    DIR_RESULTADOS / 'historico_treinamento.csv'
)

ARQUIVO_ESTADO = (
    DIR_RESULTADOS / 'estado_treinamento.json'
)

ARQUIVO_MELHOR_MODELO = (
    DIR_CHECKPOINT / 'melhor_modelo.weights.h5'
)

ARQUIVO_ULTIMO_MODELO = (
    DIR_CHECKPOINT / 'ultimo_modelo.weights.h5'
)

# Checkpoint TensorFlow: modelo + otimizador
CHECKPOINT_TF = tf.train.Checkpoint(
    modelo=modelo_embeddings,
    otimizador=otimizador_fase1
)

GERENCIADOR_CHECKPOINT = tf.train.CheckpointManager(
    CHECKPOINT_TF,
    directory=str(DIR_CHECKPOINT / 'estado_completo'),
    max_to_keep=3
)

# ------------------------------------------------------------
# 2. Histórico
# ------------------------------------------------------------

COLUNAS_HISTORICO = [
    'fold',
    'fase',
    'epoca',
    'loss_treino',
    'loss_validacao',
    'semi_hard_treino',
    'fallback_treino',
    'semi_hard_validacao',
    'fallback_validacao',
    'tempo_segundos'
]

if ARQUIVO_HISTORICO.exists():

    historico = pd.read_csv(ARQUIVO_HISTORICO)

    print('Histórico existente recuperado.')

else:

    historico = pd.DataFrame(
        columns=COLUNAS_HISTORICO
    )

    print('Histórico inicializado.')

# ------------------------------------------------------------
# 3. Estado do treinamento
# ------------------------------------------------------------

if ARQUIVO_ESTADO.exists():

    with open(
        ARQUIVO_ESTADO,
        'r',
        encoding='utf-8'
    ) as arquivo:
        estado_treinamento = json.load(arquivo)

    print('Estado anterior encontrado.')

else:

    estado_treinamento = {
        'fold': FOLD_ATUAL,
        'fase': 1,
        'ultima_epoca_concluida': 0,
        'melhor_loss_validacao': None,
        'epoca_melhor_modelo': None,
        'epocas_sem_melhoria': 0,
        'status': 'preparado'
    }

    with open(
        ARQUIVO_ESTADO,
        'w',
        encoding='utf-8'
    ) as arquivo:
        json.dump(
            estado_treinamento,
            arquivo,
            indent=2,
            ensure_ascii=False
        )

    print('Estado inicial criado.')

# ------------------------------------------------------------
# 4. Funções auxiliares
# ------------------------------------------------------------

def salvar_estado(estado):

    temporario = ARQUIVO_ESTADO.with_suffix('.json.tmp')

    with open(
        temporario,
        'w',
        encoding='utf-8'
    ) as arquivo:
        json.dump(
            estado,
            arquivo,
            indent=2,
            ensure_ascii=False
        )

    os.replace(temporario, ARQUIVO_ESTADO)


def registrar_epoca(registro):

    global historico

    novo = pd.DataFrame([registro])

    historico = pd.concat(
        [historico, novo],
        ignore_index=True
    )

    historico.to_csv(
        ARQUIVO_HISTORICO,
        index=False
    )


def salvar_checkpoint(
    loss_validacao,
    fase,
    epoca
):

    global estado_treinamento

    melhor_anterior = (
        estado_treinamento['melhor_loss_validacao']
    )

    melhorou = (
        melhor_anterior is None
        or loss_validacao < melhor_anterior
    )

    if melhorou:

        modelo_embeddings.save_weights(
            str(ARQUIVO_MELHOR_MODELO)
        )

        estado_treinamento[
            'melhor_loss_validacao'
        ] = float(loss_validacao)

        estado_treinamento[
            'epoca_melhor_modelo'
        ] = int(epoca)

        estado_treinamento[
            'epocas_sem_melhoria'
        ] = 0

    else:

        estado_treinamento[
            'epocas_sem_melhoria'
        ] += 1

    modelo_embeddings.save_weights(
        str(ARQUIVO_ULTIMO_MODELO)
    )

    GERENCIADOR_CHECKPOINT.save(
        checkpoint_number=epoca
    )

    estado_treinamento['fase'] = int(fase)

    estado_treinamento[
        'ultima_epoca_concluida'
    ] = int(epoca)

    estado_treinamento['status'] = 'em_andamento'

    salvar_estado(estado_treinamento)

    return melhorou


# ------------------------------------------------------------
# 5. Verificação
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('CONTROLE DE TREINAMENTO')
print('=' * 60)

print('Fold:', estado_treinamento['fold'])
print('Fase:', estado_treinamento['fase'])
print(
    'Última época:',
    estado_treinamento['ultima_epoca_concluida']
)

print(
    'Melhor loss:',
    estado_treinamento['melhor_loss_validacao']
)

print('\nHistórico:', ARQUIVO_HISTORICO)
print('Checkpoints:', DIR_CHECKPOINT)

print('\nNenhum peso atualizado.')
print('Bloco [28] concluído.')


In [ ]:

# ============================================================
# Bloco de código [29] — Benchmark na GPU T4
# GPU: OBRIGATÓRIA
# ============================================================

import time
import numpy as np
import tensorflow as tf

print('=' * 60)
print('BENCHMARK — TREINAMENTO NA GPU T4')
print('=' * 60)

assert tf.config.list_physical_devices('GPU'), (
    'GPU não detectada.'
)

# ------------------------------------------------------------
# 1. Criar modelo e otimizador temporários
# ------------------------------------------------------------

modelo_oficial = modelo_embeddings
otimizador_oficial = otimizador_fase1

modelo_teste = tf.keras.models.clone_model(
    modelo_oficial
)

modelo_teste.set_weights(
    modelo_oficial.get_weights()
)

otimizador_teste = tf.keras.optimizers.Adam(
    learning_rate=CONFIG['treinamento']['learning_rate_fase1']
)

# As funções existentes usam estas variáveis globais.
modelo_embeddings = modelo_teste
otimizador_fase1 = otimizador_teste

# ------------------------------------------------------------
# 2. Executar benchmark
# ------------------------------------------------------------

TEMPOS_TREINO = []
TEMPOS_VALIDACAO = []

try:

    print('\nExecutando 5 lotes de treinamento...')

    for passo in range(5):

        inicio = time.perf_counter()

        caminhos, rotulos = (
            gerador_treino.proximo_lote()
        )

        imagens, labels = carregar_lote_pk(
            caminhos,
            rotulos
        )

        resultado = passo_treinamento(
            imagens,
            labels
        )

        tempo = time.perf_counter() - inicio
        TEMPOS_TREINO.append(tempo)

        print(
            f'Passo {passo + 1}/5 | '
            f'Loss: {resultado["loss"]:.5f} | '
            f'Tempo: {tempo:.2f}s'
        )

    print('\nExecutando 2 lotes de validação...')

    for passo in range(2):

        inicio = time.perf_counter()

        lote = lotes_validacao[passo]

        imagens, labels = carregar_lote_pk(
            lote['caminhos'],
            lote['rotulos']
        )

        resultado = passo_validacao(
            imagens,
            labels
        )

        tempo = time.perf_counter() - inicio
        TEMPOS_VALIDACAO.append(tempo)

        print(
            f'Validação {passo + 1}/2 | '
            f'Loss: {resultado["loss"]:.5f} | '
            f'Tempo: {tempo:.2f}s'
        )

finally:

    # Restaurar referências oficiais.
    modelo_embeddings = modelo_oficial
    otimizador_fase1 = otimizador_oficial

# ------------------------------------------------------------
# 3. Estimativa de duração
# ------------------------------------------------------------

# Desconsiderar primeiro passo (aquecimento).
tempo_medio_treino = np.mean(
    TEMPOS_TREINO[1:]
)

tempo_medio_validacao = np.mean(
    TEMPOS_VALIDACAO
)

estimativa_epoca = (
    PASSOS_TREINO * tempo_medio_treino
    + PASSOS_VALIDACAO * tempo_medio_validacao
)

print('\n' + '=' * 60)
print('RESULTADO DO BENCHMARK')
print('=' * 60)

print(
    'Tempo médio de treinamento:',
    round(tempo_medio_treino, 2),
    's/lote'
)

print(
    'Tempo médio de validação:',
    round(tempo_medio_validacao, 2),
    's/lote'
)

print(
    'Estimativa por época:',
    round(estimativa_epoca / 60, 2),
    'minutos'
)

print(
    'Estimativa fase 1 (10 épocas):',
    round(estimativa_epoca * 10 / 3600, 2),
    'horas'
)

print(
    'Estimativa total (20 épocas):',
    round(estimativa_epoca * 20 / 3600, 2),
    'horas'
)

print('\nModelo oficial preservado.')
print('Bloco [29] concluído.')


In [ ]:

# ============================================================
# Bloco de código [30] — Primeira época do fold 1
# GPU: OBRIGATÓRIA — T4
# ============================================================

import time
import numpy as np
import tensorflow as tf

# ------------------------------------------------------------
# 1. Proteção contra execução duplicada
# ------------------------------------------------------------

if (
    estado_treinamento['ultima_epoca_concluida'] != 0
    or estado_treinamento['fase'] != 1
):
    raise RuntimeError(
        'O treinamento já foi iniciado. '
        'Não execute novamente este bloco.'
    )

assert tf.config.list_physical_devices('GPU'), (
    'GPU não detectada.'
)

# ------------------------------------------------------------
# 2. Reinicializar gerador após benchmark
# ------------------------------------------------------------

gerador_treino = GeradorLotesPK(
    dataset_treino,
    p=CONFIG['mineracao']['P'],
    k=CONFIG['mineracao']['K'],
    seed=SEED
)

# ------------------------------------------------------------
# 3. Executar uma época
# ------------------------------------------------------------

EPOCA = 1

perdas_treino = []
perdas_validacao = []

semi_treino = 0
fallback_treino = 0

semi_validacao = 0
fallback_validacao = 0

inicio_epoca = time.perf_counter()

print('=' * 60)
print('TREINAMENTO — FOLD 1 — FASE 1 — ÉPOCA 1')
print('=' * 60)

print('Passos de treino:', PASSOS_TREINO)
print('Passos de validação:', PASSOS_VALIDACAO)

print('\nIniciando treinamento...')

for passo in range(PASSOS_TREINO):

    caminhos, rotulos = gerador_treino.proximo_lote()

    imagens, labels = carregar_lote_pk(
        caminhos,
        rotulos
    )

    resultado = passo_treinamento(
        imagens,
        labels
    )

    if not np.isfinite(resultado['loss']):
        raise RuntimeError(
            f'Loss inválida no passo {passo + 1}.'
        )

    perdas_treino.append(resultado['loss'])

    semi_treino += resultado['semi-hard']
    fallback_treino += resultado['fallback']

    if (
        (passo + 1) % 50 == 0
        or passo + 1 == PASSOS_TREINO
    ):

        print(
            f'Treino {passo + 1}/{PASSOS_TREINO} | '
            f'Loss média: {np.mean(perdas_treino):.5f}'
        )

# ------------------------------------------------------------
# 4. Validação com lotes fixos
# ------------------------------------------------------------

print('\nIniciando validação...')

for passo, lote in enumerate(lotes_validacao):

    imagens, labels = carregar_lote_pk(
        lote['caminhos'],
        lote['rotulos']
    )

    resultado = passo_validacao(
        imagens,
        labels
    )

    if not np.isfinite(resultado['loss']):
        raise RuntimeError(
            f'Loss de validação inválida no passo {passo + 1}.'
        )

    perdas_validacao.append(resultado['loss'])

    semi_validacao += resultado['semi-hard']
    fallback_validacao += resultado['fallback']

    if (
        (passo + 1) % 50 == 0
        or passo + 1 == PASSOS_VALIDACAO
    ):

        print(
            f'Validação {passo + 1}/{PASSOS_VALIDACAO} | '
            f'Loss média: {np.mean(perdas_validacao):.5f}'
        )

# ------------------------------------------------------------
# 5. Registrar resultados
# ------------------------------------------------------------

tempo_epoca = time.perf_counter() - inicio_epoca

loss_treino = float(np.mean(perdas_treino))
loss_validacao = float(np.mean(perdas_validacao))

registro = {
    'fold': FOLD_ATUAL,
    'fase': 1,
    'epoca': EPOCA,
    'loss_treino': loss_treino,
    'loss_validacao': loss_validacao,
    'semi_hard_treino': semi_treino,
    'fallback_treino': fallback_treino,
    'semi_hard_validacao': semi_validacao,
    'fallback_validacao': fallback_validacao,
    'tempo_segundos': tempo_epoca
}

# Salvar checkpoint antes de registrar o histórico.
# Assim, evitamos registrar uma época sem pesos salvos.
melhorou = salvar_checkpoint(
    loss_validacao=loss_validacao,
    fase=1,
    epoca=EPOCA
)

registrar_epoca(registro)

# ------------------------------------------------------------
# 6. Resumo
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('RESULTADO — ÉPOCA 1')
print('=' * 60)

print('Loss treino:', round(loss_treino, 6))
print('Loss validação:', round(loss_validacao, 6))

print('\nSemi-hard treino:', semi_treino)
print('Fallback treino:', fallback_treino)

print('\nSemi-hard validação:', semi_validacao)
print('Fallback validação:', fallback_validacao)

print(
    '\nTempo:',
    round(tempo_epoca / 60, 2),
    'minutos'
)

print('Melhor modelo atualizado:', melhorou)
print('Checkpoint salvo:', GERENCIADOR_CHECKPOINT.latest_checkpoint)

print('\nBloco [30] concluído.')


In [ ]:

# ============================================================
# Bloco de código [31] — Época 2 do fold 1
# GPU: OBRIGATÓRIA — T4
# ============================================================

import time
import json
import numpy as np
import tensorflow as tf

# ------------------------------------------------------------
# 1. Verificar estado persistido
# ------------------------------------------------------------

with open(
    ARQUIVO_ESTADO,
    'r',
    encoding='utf-8'
) as arquivo:
    estado_treinamento = json.load(arquivo)

assert estado_treinamento['fold'] == 1
assert estado_treinamento['fase'] == 1
assert estado_treinamento['ultima_epoca_concluida'] == 1, (
    'A época 1 precisa estar concluída e a época 2 '
    'ainda não pode ter sido executada.'
)

assert GERENCIADOR_CHECKPOINT.latest_checkpoint, (
    'Checkpoint não encontrado.'
)

assert tf.config.list_physical_devices('GPU'), (
    'GPU não detectada.'
)

# ------------------------------------------------------------
# 2. Preparar acumuladores
# ------------------------------------------------------------

EPOCA = 2

perdas_treino = []
perdas_validacao = []

semi_treino = 0
fallback_treino = 0
semi_validacao = 0
fallback_validacao = 0

inicio_epoca = time.perf_counter()

print('=' * 60)
print('TREINAMENTO — FOLD 1 — FASE 1 — ÉPOCA 2')
print('=' * 60)

# ------------------------------------------------------------
# 3. Treinamento
# ------------------------------------------------------------

for passo in range(PASSOS_TREINO):

    caminhos, rotulos = gerador_treino.proximo_lote()

    imagens, labels = carregar_lote_pk(
        caminhos,
        rotulos
    )

    resultado = passo_treinamento(
        imagens,
        labels
    )

    if not np.isfinite(resultado['loss']):
        raise RuntimeError(
            f'Loss inválida no passo {passo + 1}.'
        )

    perdas_treino.append(resultado['loss'])

    semi_treino += resultado['semi-hard']
    fallback_treino += resultado['fallback']

    if (
        (passo + 1) % 50 == 0
        or passo + 1 == PASSOS_TREINO
    ):
        print(
            f'Treino {passo + 1}/{PASSOS_TREINO} | '
            f'Loss média: {np.mean(perdas_treino):.5f}'
        )

# ------------------------------------------------------------
# 4. Validação
# ------------------------------------------------------------

print('\nIniciando validação...')

for passo, lote in enumerate(lotes_validacao):

    imagens, labels = carregar_lote_pk(
        lote['caminhos'],
        lote['rotulos']
    )

    resultado = passo_validacao(
        imagens,
        labels
    )

    if not np.isfinite(resultado['loss']):
        raise RuntimeError(
            f'Loss inválida na validação {passo + 1}.'
        )

    perdas_validacao.append(resultado['loss'])

    semi_validacao += resultado['semi-hard']
    fallback_validacao += resultado['fallback']

    if (
        (passo + 1) % 50 == 0
        or passo + 1 == PASSOS_VALIDACAO
    ):
        print(
            f'Validação {passo + 1}/{PASSOS_VALIDACAO} | '
            f'Loss média: {np.mean(perdas_validacao):.5f}'
        )

# ------------------------------------------------------------
# 5. Checkpoint e histórico
# ------------------------------------------------------------

tempo_epoca = time.perf_counter() - inicio_epoca

loss_treino = float(np.mean(perdas_treino))
loss_validacao = float(np.mean(perdas_validacao))

registro = {
    'fold': FOLD_ATUAL,
    'fase': 1,
    'epoca': EPOCA,
    'loss_treino': loss_treino,
    'loss_validacao': loss_validacao,
    'semi_hard_treino': semi_treino,
    'fallback_treino': fallback_treino,
    'semi_hard_validacao': semi_validacao,
    'fallback_validacao': fallback_validacao,
    'tempo_segundos': tempo_epoca
}

melhorou = salvar_checkpoint(
    loss_validacao=loss_validacao,
    fase=1,
    epoca=EPOCA
)

registrar_epoca(registro)

# ------------------------------------------------------------
# 6. Resultado
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('RESULTADO — ÉPOCA 2')
print('=' * 60)

print('Loss treino:', round(loss_treino, 6))
print('Loss validação:', round(loss_validacao, 6))

print('\nSemi-hard treino:', semi_treino)
print('Fallback treino:', fallback_treino)

print('\nSemi-hard validação:', semi_validacao)
print('Fallback validação:', fallback_validacao)

print(
    '\nTempo:',
    round(tempo_epoca / 60, 2),
    'minutos'
)

print('Melhor modelo atualizado:', melhorou)
print(
    'Checkpoint:',
    GERENCIADOR_CHECKPOINT.latest_checkpoint
)

print('\nBloco [31] concluído.')


In [ ]:

# ============================================================
# Bloco de código [32] — Fase 1, épocas restantes
# GPU: OBRIGATÓRIA — T4
# ============================================================

import json
import time
import numpy as np
import pandas as pd
import tensorflow as tf

# ------------------------------------------------------------
# 1. Conferir estado e continuidade
# ------------------------------------------------------------

with open(
    ARQUIVO_ESTADO,
    'r',
    encoding='utf-8'
) as arquivo:
    estado_treinamento = json.load(arquivo)

assert estado_treinamento['fold'] == FOLD_ATUAL
assert estado_treinamento['fase'] == 1
assert tf.config.list_physical_devices('GPU'), (
    'GPU não detectada.'
)

ultima_epoca = int(
    estado_treinamento['ultima_epoca_concluida']
)

assert 2 <= ultima_epoca <= 10, (
    'Estado inesperado. Verifique o checkpoint.'
)

# Confere se o checkpoint em memória corresponde ao último
# checkpoint salvo. Não restaura automaticamente a sessão.
assert GERENCIADOR_CHECKPOINT.latest_checkpoint is not None

PACIENCIA = CONFIG['treinamento']['early_stopping_patience']
MAX_EPOCAS = CONFIG['treinamento']['epocas_fase1']

# ------------------------------------------------------------
# 2. Função de execução de uma época
# ------------------------------------------------------------

def executar_epoca_fase1(epoca):

    perdas_treino = []
    perdas_validacao = []

    semi_treino = 0
    fallback_treino = 0
    semi_validacao = 0
    fallback_validacao = 0

    inicio = time.perf_counter()

    print('\n' + '=' * 60)
    print(f'FOLD {FOLD_ATUAL} — FASE 1 — ÉPOCA {epoca}')
    print('=' * 60)

    for passo in range(PASSOS_TREINO):

        caminhos, rotulos = gerador_treino.proximo_lote()

        imagens, labels = carregar_lote_pk(
            caminhos,
            rotulos
        )

        resultado = passo_treinamento(
            imagens,
            labels
        )

        perda = float(resultado['loss'])

        if not np.isfinite(perda):
            raise RuntimeError(
                f'Loss inválida no treino, '
                f'época {epoca}, passo {passo + 1}.'
            )

        perdas_treino.append(perda)

        semi_treino += int(resultado['semi-hard'])
        fallback_treino += int(resultado['fallback'])

        if (passo + 1) % 100 == 0:
            print(
                f'Treino {passo + 1}/{PASSOS_TREINO} | '
                f'Loss: {np.mean(perdas_treino):.5f}'
            )

    print('Validando...')

    for lote in lotes_validacao:

        imagens, labels = carregar_lote_pk(
            lote['caminhos'],
            lote['rotulos']
        )

        resultado = passo_validacao(
            imagens,
            labels
        )

        perda = float(resultado['loss'])

        if not np.isfinite(perda):
            raise RuntimeError(
                f'Loss inválida na validação da época {epoca}.'
            )

        perdas_validacao.append(perda)

        semi_validacao += int(resultado['semi-hard'])
        fallback_validacao += int(resultado['fallback'])

    tempo = time.perf_counter() - inicio

    registro = {
        'fold': FOLD_ATUAL,
        'fase': 1,
        'epoca': epoca,
        'loss_treino': float(np.mean(perdas_treino)),
        'loss_validacao': float(np.mean(perdas_validacao)),
        'semi_hard_treino': semi_treino,
        'fallback_treino': fallback_treino,
        'semi_hard_validacao': semi_validacao,
        'fallback_validacao': fallback_validacao,
        'tempo_segundos': tempo
    }

    # Persistir checkpoint antes de atualizar histórico.
    melhorou = salvar_checkpoint(
        loss_validacao=registro['loss_validacao'],
        fase=1,
        epoca=epoca
    )

    # Evita o FutureWarning de concatenação.
    global historico

    historico = pd.read_csv(ARQUIVO_HISTORICO)

    if (historico['epoca'] == epoca).any():
        raise RuntimeError(
            f'Época {epoca} já registrada no histórico.'
        )

    historico = pd.concat(
        [historico, pd.DataFrame([registro])],
        ignore_index=True
    )

    historico.to_csv(
        ARQUIVO_HISTORICO,
        index=False
    )

    print(
        f'\nÉpoca {epoca} concluída | '
        f'Treino: {registro["loss_treino"]:.6f} | '
        f'Validação: {registro["loss_validacao"]:.6f}'
    )

    print(
        f'Tempo: {tempo / 60:.2f} min | '
        f'Melhorou: {melhorou}'
    )

    print(
        'Épocas sem melhoria:',
        estado_treinamento['epocas_sem_melhoria']
    )

    return registro

# ------------------------------------------------------------
# 3. Executar épocas restantes
# ------------------------------------------------------------

if ultima_epoca >= MAX_EPOCAS:
    print('Fase 1 já atingiu o máximo de épocas.')

elif estado_treinamento['epocas_sem_melhoria'] >= PACIENCIA:
    print('Early stopping já atingido.')

else:

    for epoca in range(
        ultima_epoca + 1,
        MAX_EPOCAS + 1
    ):

        executar_epoca_fase1(epoca)

        if (
            estado_treinamento['epocas_sem_melhoria']
            >= PACIENCIA
        ):
            print('\nEARLY STOPPING ATIVADO.')
            break

# ------------------------------------------------------------
# 4. Resumo final
# ------------------------------------------------------------

historico_final = pd.read_csv(
    ARQUIVO_HISTORICO
)

historico_fase1 = historico_final[
    historico_final['fase'] == 1
].copy()

print('\n' + '=' * 60)
print('RESUMO DA FASE 1')
print('=' * 60)

print(
    historico_fase1[
        ['epoca', 'loss_treino', 'loss_validacao']
    ].to_string(index=False)
)

print(
    '\nMelhor loss de validação:',
    estado_treinamento['melhor_loss_validacao']
)

print(
    'Melhor época:',
    estado_treinamento['epoca_melhor_modelo']
)

print(
    'Última época:',
    estado_treinamento['ultima_epoca_concluida']
)

print(
    '\nCheckpoint mais recente:',
    GERENCIADOR_CHECKPOINT.latest_checkpoint
)

print('\nBloco [32] concluído.')


In [ ]:

# ============================================================
# Bloco de código [33] — Diagnóstico do dataset
# GPU: NÃO NECESSÁRIA
# ============================================================

import os
import json
import pandas as pd

from pathlib import Path

print('=' * 60)
print('DIAGNÓSTICO DE INTEGRIDADE DO DATASET')
print('=' * 60)

# ------------------------------------------------------------
# 1. Confirmar último checkpoint concluído
# ------------------------------------------------------------

with open(
    ARQUIVO_ESTADO,
    'r',
    encoding='utf-8'
) as arquivo:
    estado_atual = json.load(arquivo)

print(
    'Última época concluída:',
    estado_atual['ultima_epoca_concluida']
)

print(
    'Checkpoint disponível:',
    GERENCIADOR_CHECKPOINT.latest_checkpoint
)

assert estado_atual['ultima_epoca_concluida'] == 3, (
    'Estado diferente do esperado. '
    'Verifique antes de continuar.'
)

# ------------------------------------------------------------
# 2. Inspecionar arquivos locais
# ------------------------------------------------------------

caminhos = sorted(
    set(
        str(caminho)
        for imagens in [
            dataset_treino,
            dataset_validacao
        ]
        for lista in imagens.values()
        for caminho in lista
    )
)

problemas = []

total_ausentes = 0
total_vazios = 0

for caminho in caminhos:

    arquivo = Path(caminho)

    if not arquivo.exists():

        problemas.append({
            'caminho_local': caminho,
            'problema': 'ausente',
            'tamanho_bytes': None
        })

        total_ausentes += 1

    elif arquivo.stat().st_size == 0:

        problemas.append({
            'caminho_local': caminho,
            'problema': 'vazio',
            'tamanho_bytes': 0
        })

        total_vazios += 1

# ------------------------------------------------------------
# 3. Relatório
# ------------------------------------------------------------

df_problemas = pd.DataFrame(
    problemas,
    columns=[
        'caminho_local',
        'problema',
        'tamanho_bytes'
    ]
)

CAMINHO_RELATORIO = (
    DIR_RESULTADOS / 'diagnostico_arquivos.csv'
)

df_problemas.to_csv(
    CAMINHO_RELATORIO,
    index=False
)

print('\nArquivos inspecionados:', len(caminhos))
print('Arquivos ausentes:', total_ausentes)
print('Arquivos vazios:', total_vazios)

print('\nArquivos problemáticos:')

if df_problemas.empty:

    print(
        'Nenhum arquivo ausente ou vazio encontrado.'
    )

else:

    print(
        df_problemas.to_string(
            index=False,
            max_rows=30
        )
    )

print('\nRelatório:', CAMINHO_RELATORIO)

print('\nBloco [33] concluído.')


In [ ]:

# ============================================================
# Bloco de código [34] — Recuperar imagem vazia
# GPU: NÃO NECESSÁRIA
# ============================================================

import os
import shutil
import pandas as pd
import tensorflow as tf

from pathlib import Path

print('=' * 60)
print('RECUPERAÇÃO DE IMAGEM — YTBB')
print('=' * 60)

ARQUIVO_PROBLEMATICO = Path(
    '/content/tcc1_kfold/datasets/YTBB/train/1051/1051_7.jpg'
)

# ------------------------------------------------------------
# 1. Consultar manifesto
# ------------------------------------------------------------

manifesto_local = pd.read_csv(
    PASTAS['splits'] / 'manifesto_imagens_local.csv'
)

registro = manifesto_local[
    manifesto_local['caminho_local'].astype(str)
    == str(ARQUIVO_PROBLEMATICO)
]

assert len(registro) == 1, (
    'Arquivo não identificado de forma única no manifesto.'
)

print('\nRegistro encontrado:')
print(registro.to_string(index=False))

# ------------------------------------------------------------
# 2. Localizar a origem no Drive
# ------------------------------------------------------------

# Verificar quais colunas contêm caminhos.
colunas_caminho = [
    coluna
    for coluna in registro.columns
    if (
        'caminho' in coluna.lower()
        or 'path' in coluna.lower()
    )
    and coluna != 'caminho_local'
]

candidatos = []

for coluna in colunas_caminho:

    valor = registro.iloc[0][coluna]

    if pd.isna(valor):
        continue

    caminho = Path(str(valor))

    if caminho.is_file():
        candidatos.append((coluna, caminho))

print('\nOrigens existentes encontradas:')

for coluna, caminho in candidatos:
    print(f'{coluna}: {caminho}')

if not candidatos:
    print(
        '\nNenhuma origem existente foi encontrada '
        'nas colunas do manifesto.'
    )
    print(
        'Não houve alteração no arquivo local.'
    )

else:

    # --------------------------------------------------------
    # 3. Validar origem antes de copiar
    # --------------------------------------------------------

    coluna_origem, origem = candidatos[0]

    tamanho_origem = origem.stat().st_size

    assert tamanho_origem > 0, (
        'O arquivo de origem também está vazio.'
    )

    conteudo = tf.io.read_file(str(origem))

    imagem = tf.io.decode_image(
        conteudo,
        channels=3,
        expand_animations=False
    )

    print('\nOrigem validada:')
    print('Arquivo:', origem)
    print('Tamanho:', tamanho_origem, 'bytes')
    print('Dimensões:', imagem.shape)

    # --------------------------------------------------------
    # 4. Copiar e validar destino
    # --------------------------------------------------------

    temporario = ARQUIVO_PROBLEMATICO.with_suffix(
        '.jpg.tmp'
    )

    shutil.copy2(
        origem,
        temporario
    )

    # Validar o temporário antes da substituição.
    imagem_teste = tf.io.decode_image(
        tf.io.read_file(str(temporario)),
        channels=3,
        expand_animations=False
    )

    assert imagem_teste.shape.rank == 3

    os.replace(
        temporario,
        ARQUIVO_PROBLEMATICO
    )

    assert ARQUIVO_PROBLEMATICO.stat().st_size > 0

    print('\nImagem recuperada com sucesso.')
    print(
        'Tamanho final:',
        ARQUIVO_PROBLEMATICO.stat().st_size,
        'bytes'
    )

print('\nBloco [34] concluído.')


In [ ]:

# ============================================================
# Bloco de código [35] — Restaurar checkpoint da época 3
# GPU: ATIVADA — sem treinamento
# ============================================================

import json
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path

print('=' * 60)
print('RESTAURAÇÃO DO CHECKPOINT — FOLD 1')
print('=' * 60)

# ------------------------------------------------------------
# 1. Verificar estado persistido
# ------------------------------------------------------------

with open(
    ARQUIVO_ESTADO,
    'r',
    encoding='utf-8'
) as arquivo:
    estado_treinamento = json.load(arquivo)

assert estado_treinamento['fold'] == FOLD_ATUAL
assert estado_treinamento['fase'] == 1

assert (
    estado_treinamento['ultima_epoca_concluida'] == 3
), 'O estado salvo não corresponde à época 3.'

# ------------------------------------------------------------
# 2. Confirmar imagem recuperada
# ------------------------------------------------------------

arquivo_recuperado = Path(
    '/content/tcc1_kfold/datasets/YTBB/train/1051/1051_7.jpg'
)

assert arquivo_recuperado.exists()
assert arquivo_recuperado.stat().st_size > 0

imagem_teste = tf.io.decode_image(
    tf.io.read_file(str(arquivo_recuperado)),
    channels=3,
    expand_animations=False
)

assert imagem_teste.shape.rank == 3

print('Imagem recuperada: OK')

# ------------------------------------------------------------
# 3. Localizar checkpoint específico da época 3
# ------------------------------------------------------------

CAMINHO_CKPT3 = (
    DIR_CHECKPOINT / 'estado_completo' / 'ckpt-3'
)

assert Path(str(CAMINHO_CKPT3) + '.index').exists(), (
    'Checkpoint ckpt-3 não encontrado.'
)

# ------------------------------------------------------------
# 4. Restaurar modelo e otimizador
# ------------------------------------------------------------

status = CHECKPOINT_TF.restore(
    str(CAMINHO_CKPT3)
)

# Verifica se os objetos existentes foram correspondidos.
status.assert_existing_objects_matched()

print('Modelo e otimizador restaurados do ckpt-3.')

# ------------------------------------------------------------
# 5. Reconstruir gerador de treinamento
# ------------------------------------------------------------

# A sequência não será idêntica à execução interrompida.
# A semente distinta identifica esta retomada.

SEED_RETOMADA = SEED + 3004

gerador_treino = GeradorLotesPK(
    dataset_treino,
    p=CONFIG['mineracao']['P'],
    k=CONFIG['mineracao']['K'],
    seed=SEED_RETOMADA
)

print('Gerador reinicializado.')
print('Semente da retomada:', SEED_RETOMADA)

# ------------------------------------------------------------
# 6. Conferir histórico
# ------------------------------------------------------------

historico = pd.read_csv(
    ARQUIVO_HISTORICO
)

historico_fase1 = historico[
    historico['fase'] == 1
].copy()

assert set(
    historico_fase1['epoca'].astype(int)
) == {1, 2, 3}, (
    'Histórico inconsistente com o checkpoint.'
)

print('\nHistórico preservado:')

print(
    historico_fase1[
        ['epoca', 'loss_treino', 'loss_validacao']
    ].to_string(index=False)
)

print('\n' + '=' * 60)
print('RESTAURAÇÃO CONCLUÍDA')
print('=' * 60)

print('Última época válida: 3')
print('Próxima época: 4')
print('Pesos parciais da época 4 descartados.')
print('Dataset local corrigido.')

print('\nBloco [35] concluído.')


In [ ]:

# ============================================================
# Bloco de código [36] — Retomar fase 1
# GPU: OBRIGATÓRIA — T4
# ============================================================

import json
import pandas as pd
import tensorflow as tf

print('=' * 60)
print('RETOMADA DO TREINAMENTO — FOLD 1')
print('=' * 60)

# ------------------------------------------------------------
# 1. Conferir GPU e checkpoint
# ------------------------------------------------------------

assert tf.config.list_physical_devices('GPU'), (
    'GPU não detectada.'
)

with open(
    ARQUIVO_ESTADO,
    'r',
    encoding='utf-8'
) as arquivo:
    estado_treinamento = json.load(arquivo)

assert estado_treinamento['fold'] == FOLD_ATUAL
assert estado_treinamento['fase'] == 1

ultima_epoca = int(
    estado_treinamento['ultima_epoca_concluida']
)

assert 3 <= ultima_epoca <= 10, (
    'Estado inesperado para a retomada.'
)

historico_atual = pd.read_csv(
    ARQUIVO_HISTORICO
)

epocas_registradas = set(
    historico_atual.loc[
        historico_atual['fase'] == 1,
        'epoca'
    ].astype(int)
)

assert epocas_registradas == set(
    range(1, ultima_epoca + 1)
), 'Histórico inconsistente com o estado salvo.'

# ------------------------------------------------------------
# 2. Configuração
# ------------------------------------------------------------

MAX_EPOCAS = CONFIG['treinamento']['epocas_fase1']

PACIENCIA = CONFIG[
    'treinamento'
]['early_stopping_patience']

print('Última época concluída:', ultima_epoca)
print('Próxima época:', ultima_epoca + 1)
print('Máximo de épocas:', MAX_EPOCAS)
print('Paciência:', PACIENCIA)

# ------------------------------------------------------------
# 3. Retomar treinamento
# ------------------------------------------------------------

if ultima_epoca >= MAX_EPOCAS:

    print('\nTodas as épocas já foram concluídas.')

elif (
    estado_treinamento['epocas_sem_melhoria']
    >= PACIENCIA
):

    print('\nEarly stopping já atingido.')

else:

    for epoca in range(
        ultima_epoca + 1,
        MAX_EPOCAS + 1
    ):

        executar_epoca_fase1(epoca)

        if (
            estado_treinamento['epocas_sem_melhoria']
            >= PACIENCIA
        ):

            print('\nEARLY STOPPING ATIVADO.')
            break

# ------------------------------------------------------------
# 4. Resumo
# ------------------------------------------------------------

historico_final = pd.read_csv(
    ARQUIVO_HISTORICO
)

fase1 = historico_final[
    historico_final['fase'] == 1
].sort_values('epoca')

print('\n' + '=' * 60)
print('RESUMO ATUALIZADO — FASE 1')
print('=' * 60)

print(
    fase1[
        ['epoca', 'loss_treino', 'loss_validacao']
    ].to_string(index=False)
)

print(
    '\nMelhor loss:',
    estado_treinamento['melhor_loss_validacao']
)

print(
    'Melhor época:',
    estado_treinamento['epoca_melhor_modelo']
)

print(
    'Última época:',
    estado_treinamento['ultima_epoca_concluida']
)

print(
    '\nCheckpoint:',
    GERENCIADOR_CHECKPOINT.latest_checkpoint
)

print('\nBloco [36] concluído.')


In [ ]:

# ============================================================
# Bloco de código [37] — Preparar fine-tuning
# GPU: ATIVADA — sem treinamento
# ============================================================

import json
import shutil
import tensorflow as tf

from pathlib import Path

print('=' * 60)
print('PREPARAÇÃO DO FINE-TUNING — FOLD 1')
print('=' * 60)

# ------------------------------------------------------------
# 1. Verificar conclusão da fase 1
# ------------------------------------------------------------

with open(
    ARQUIVO_ESTADO,
    'r',
    encoding='utf-8'
) as arquivo:
    estado_treinamento = json.load(arquivo)

assert estado_treinamento['fold'] == FOLD_ATUAL
assert estado_treinamento['fase'] == 1
assert estado_treinamento['ultima_epoca_concluida'] == 10
assert estado_treinamento['epoca_melhor_modelo'] == 9

assert ARQUIVO_MELHOR_MODELO.exists(), (
    'Melhor modelo da fase 1 não encontrado.'
)

# ------------------------------------------------------------
# 2. Preservar melhor modelo da fase 1
# ------------------------------------------------------------

ARQUIVO_MELHOR_FASE1 = (
    DIR_CHECKPOINT / 'melhor_fase1.weights.h5'
)

if not ARQUIVO_MELHOR_FASE1.exists():

    shutil.copy2(
        ARQUIVO_MELHOR_MODELO,
        ARQUIVO_MELHOR_FASE1
    )

print(
    'Melhor modelo da fase 1 preservado:',
    ARQUIVO_MELHOR_FASE1
)

# ------------------------------------------------------------
# 3. Carregar pesos da melhor época
# ------------------------------------------------------------

modelo_embeddings.load_weights(
    str(ARQUIVO_MELHOR_FASE1)
)

print('Pesos da época 9 carregados.')

# ------------------------------------------------------------
# 4. Identificar backbone VGG16
# ------------------------------------------------------------

backbone = next(
    (
        camada
        for camada in modelo_embeddings.layers
        if isinstance(camada, tf.keras.Model)
        and 'vgg16' in camada.name.lower()
    ),
    None
)

assert backbone is not None, (
    'Backbone VGG16 não encontrado.'
)

print('\nBackbone:', backbone.name)

# ------------------------------------------------------------
# 5. Descongelar somente o bloco 5
# ------------------------------------------------------------

backbone.trainable = True

for camada in backbone.layers:

    camada.trainable = (
        camada.name.startswith('block5_')
    )

camadas_liberadas = [
    camada.name
    for camada in backbone.layers
    if camada.trainable
]

print('\nCamadas liberadas:')

for nome in camadas_liberadas:
    print(' -', nome)

assert camadas_liberadas == [
    'block5_conv1',
    'block5_conv2',
    'block5_conv3',
    'block5_pool'
], (
    'Camadas liberadas diferentes do esperado.'
)

# ------------------------------------------------------------
# 6. Novo otimizador para fase 2
# ------------------------------------------------------------

otimizador_fase2 = tf.keras.optimizers.Adam(
    learning_rate=CONFIG[
        'treinamento'
    ]['learning_rate_fase2']
)

# As funções de treinamento usam esta variável.
otimizador_fase1 = otimizador_fase2

# Recriar checkpoint com o novo otimizador.
CHECKPOINT_TF = tf.train.Checkpoint(
    modelo=modelo_embeddings,
    otimizador=otimizador_fase2
)

GERENCIADOR_CHECKPOINT = tf.train.CheckpointManager(
    CHECKPOINT_TF,
    directory=str(
        DIR_CHECKPOINT / 'estado_completo_fase2'
    ),
    max_to_keep=3
)

# ------------------------------------------------------------
# 7. Conferir parâmetros treináveis
# ------------------------------------------------------------

parametros_treinaveis = sum(
    int(tf.size(variavel))
    for variavel in modelo_embeddings.trainable_weights
)

print('\n' + '=' * 60)
print('CONFIGURAÇÃO DA FASE 2')
print('=' * 60)

print(
    'Learning rate:',
    float(otimizador_fase2.learning_rate.numpy())
)

print(
    'Parâmetros treináveis:',
    f'{parametros_treinaveis:,}'
)

print(
    'Variáveis treináveis:',
    len(modelo_embeddings.trainable_variables)
)

print(
    'Épocas planejadas:',
    CONFIG['treinamento']['epocas_fase2']
)

print('\nNenhum passo de treinamento executado.')
print('Bloco [37] concluído.')


In [ ]:

# ============================================================
# Bloco de código [38] — Benchmark do fine-tuning
# GPU: OBRIGATÓRIA — T4
# ============================================================

import time
import numpy as np
import tensorflow as tf

print('=' * 60)
print('BENCHMARK — FINE-TUNING VGG16')
print('=' * 60)

assert tf.config.list_physical_devices('GPU'), (
    'GPU não detectada.'
)

# ------------------------------------------------------------
# 1. Preservar referências oficiais
# ------------------------------------------------------------

modelo_oficial = modelo_embeddings
otimizador_oficial = otimizador_fase1

# ------------------------------------------------------------
# 2. Criar cópia temporária do modelo
# ------------------------------------------------------------

modelo_teste = tf.keras.models.clone_model(
    modelo_oficial
)

modelo_teste.set_weights(
    modelo_oficial.get_weights()
)

# Garantir que a cópia tenha o mesmo conjunto
# de camadas treináveis da fase 2.
backbone_teste = next(
    camada
    for camada in modelo_teste.layers
    if isinstance(camada, tf.keras.Model)
    and 'vgg16' in camada.name.lower()
)

backbone_teste.trainable = True

for camada in backbone_teste.layers:
    camada.trainable = (
        camada.name.startswith('block5_')
    )

assert len(modelo_teste.trainable_variables) == 10

otimizador_teste = tf.keras.optimizers.Adam(
    learning_rate=CONFIG[
        'treinamento'
    ]['learning_rate_fase2']
)

# ------------------------------------------------------------
# 3. Benchmark temporário
# ------------------------------------------------------------

TEMPOS_TREINO_FASE2 = []
TEMPOS_VALIDACAO_FASE2 = []

try:

    modelo_embeddings = modelo_teste
    otimizador_fase1 = otimizador_teste

    print('\nExecutando 3 lotes de treinamento...')

    for passo in range(3):

        inicio = time.perf_counter()

        caminhos, rotulos = (
            gerador_treino.proximo_lote()
        )

        imagens, labels = carregar_lote_pk(
            caminhos,
            rotulos
        )

        resultado = passo_treinamento(
            imagens,
            labels
        )

        tempo = time.perf_counter() - inicio

        TEMPOS_TREINO_FASE2.append(tempo)

        print(
            f'Treino {passo + 1}/3 | '
            f'Loss: {resultado["loss"]:.5f} | '
            f'Tempo: {tempo:.2f}s'
        )

    print('\nExecutando 2 lotes de validação...')

    for passo in range(2):

        inicio = time.perf_counter()

        lote = lotes_validacao[passo]

        imagens, labels = carregar_lote_pk(
            lote['caminhos'],
            lote['rotulos']
        )

        resultado = passo_validacao(
            imagens,
            labels
        )

        tempo = time.perf_counter() - inicio

        TEMPOS_VALIDACAO_FASE2.append(tempo)

        print(
            f'Validação {passo + 1}/2 | '
            f'Loss: {resultado["loss"]:.5f} | '
            f'Tempo: {tempo:.2f}s'
        )

finally:

    # Restaurar modelo e otimizador oficiais.
    modelo_embeddings = modelo_oficial
    otimizador_fase1 = otimizador_oficial

# ------------------------------------------------------------
# 4. Estimativas
# ------------------------------------------------------------

# Excluir primeiro passo, que inclui aquecimento.
tempo_treino = float(
    np.mean(TEMPOS_TREINO_FASE2[1:])
)

tempo_validacao = float(
    np.mean(TEMPOS_VALIDACAO_FASE2)
)

tempo_epoca = (
    PASSOS_TREINO * tempo_treino
    + PASSOS_VALIDACAO * tempo_validacao
)

print('\n' + '=' * 60)
print('RESULTADOS — FASE 2')
print('=' * 60)

print(
    'Treinamento:',
    round(tempo_treino, 2),
    's/lote'
)

print(
    'Validação:',
    round(tempo_validacao, 2),
    's/lote'
)

print(
    'Estimativa por época:',
    round(tempo_epoca / 60, 2),
    'minutos'
)

print(
    'Estimativa para 10 épocas:',
    round(tempo_epoca * 10 / 3600, 2),
    'horas'
)

print('\nModelo oficial preservado.')
print('Bloco [38] concluído.')


In [ ]:

# ============================================================
# Bloco de código [39] — Inicializar controle da fase 2
# GPU: ATIVADA — sem treinamento
# ============================================================

import json
import inspect
import tensorflow as tf
from pathlib import Path

print('=' * 60)
print('INICIALIZAÇÃO — FASE 2 / FOLD 1')
print('=' * 60)

# ------------------------------------------------------------
# 1. Validar modelo e otimizador
# ------------------------------------------------------------

assert len(modelo_embeddings.trainable_variables) == 10, (
    'Esperadas 10 variáveis treináveis.'
)

assert otimizador_fase1 is otimizador_fase2, (
    'Referência do otimizador inconsistente.'
)

print(
    'Variáveis treináveis:',
    len(modelo_embeddings.trainable_variables)
)

print(
    'Learning rate:',
    float(otimizador_fase2.learning_rate.numpy())
)

# ------------------------------------------------------------
# 2. Inspecionar funções de treinamento
# ------------------------------------------------------------

print('\nFunção de treinamento:')

try:
    codigo_passo = inspect.getsource(
        passo_treinamento.python_function
        if hasattr(passo_treinamento, 'python_function')
        else passo_treinamento
    )

    print(codigo_passo[:3000])

except (OSError, TypeError):

    print(
        'Código-fonte indisponível para inspeção.'
    )

# ------------------------------------------------------------
# 3. Diretórios e arquivos independentes
# ------------------------------------------------------------

DIR_FASE2 = DIR_RESULTADOS / 'fase2'
DIR_FASE2.mkdir(
    parents=True,
    exist_ok=True
)

ARQUIVO_ESTADO_FASE2 = (
    DIR_FASE2 / 'estado_treinamento.json'
)

ARQUIVO_HISTORICO_FASE2 = (
    DIR_FASE2 / 'historico_treinamento.csv'
)

ARQUIVO_MELHOR_FASE2 = (
    DIR_CHECKPOINT / 'melhor_fase2.weights.h5'
)

ARQUIVO_ULTIMO_FASE2 = (
    DIR_CHECKPOINT / 'ultimo_fase2.weights.h5'
)

# ------------------------------------------------------------
# 4. Criar estado inicial
# ------------------------------------------------------------

assert not ARQUIVO_ESTADO_FASE2.exists(), (
    'Já existe um estado da fase 2. '
    'Não reinicializar sem verificar a retomada.'
)

estado_fase2 = {
    'fold': FOLD_ATUAL,
    'fase': 2,
    'ultima_epoca_concluida': 0,
    'melhor_loss_validacao': None,
    'epoca_melhor_modelo': None,
    'epocas_sem_melhoria': 0,
    'status': 'preparado',
    'epoca_origem_fase1': 9,
    'seed_gerador': SEED + 2000
}

with open(
    ARQUIVO_ESTADO_FASE2,
    'w',
    encoding='utf-8'
) as arquivo:

    json.dump(
        estado_fase2,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 5. Reinicializar gerador P x K
# ------------------------------------------------------------

gerador_treino = GeradorLotesPK(
    dataset_treino,
    p=CONFIG['mineracao']['P'],
    k=CONFIG['mineracao']['K'],
    seed=estado_fase2['seed_gerador']
)

print('\nGerador P x K reinicializado.')
print('Semente:', estado_fase2['seed_gerador'])

# ------------------------------------------------------------
# 6. Conferência final
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('FASE 2 PREPARADA')
print('=' * 60)

print('Fold:', FOLD_ATUAL)
print('Época inicial:', 1)
print('Máximo de épocas:', CONFIG['treinamento']['epocas_fase2'])
print('Melhor época da fase 1 utilizada: 9')
print('Estado:', ARQUIVO_ESTADO_FASE2)
print('Histórico:', ARQUIVO_HISTORICO_FASE2)
print(
    'Checkpoints:',
    DIR_CHECKPOINT / 'estado_completo_fase2'
)

print('\nNenhum treinamento executado.')
print('Bloco [39] concluído.')


In [ ]:

# ============================================================
# Bloco de código [40] — Fase 2, época 1
# GPU: OBRIGATÓRIA — T4
# ============================================================

import json
import time
import os
import pandas as pd
import tensorflow as tf

print('=' * 60)
print('FOLD 1 — FASE 2 — ÉPOCA 1')
print('=' * 60)

assert tf.config.list_physical_devices('GPU')
assert len(modelo_embeddings.trainable_variables) == 10

assert not ARQUIVO_HISTORICO_FASE2.exists(), (
    'Já existe histórico da fase 2. '
    'Não executar novamente sem verificar.'
)

with open(
    ARQUIVO_ESTADO_FASE2,
    'r',
    encoding='utf-8'
) as arquivo:
    estado_fase2 = json.load(arquivo)

assert estado_fase2['ultima_epoca_concluida'] == 0

# ------------------------------------------------------------
# 1. Treinamento
# ------------------------------------------------------------

inicio_epoca = time.perf_counter()

perdas_treino = []
total_semihard = 0
total_fallback = 0

for passo in range(PASSOS_TREINO):

    caminhos, rotulos = (
        gerador_treino.proximo_lote()
    )

    imagens, labels = carregar_lote_pk(
        caminhos,
        rotulos
    )

    resultado = passo_treinamento(
        imagens,
        labels
    )

    perdas_treino.append(
        resultado['loss']
    )

    total_semihard += int(
        resultado['semi-hard']
    )

    total_fallback += int(
        resultado['fallback']
    )

    if (passo + 1) % 100 == 0:

        print(
            f'Treino {passo + 1}/{PASSOS_TREINO} | '
            f'Loss: {sum(perdas_treino) / len(perdas_treino):.5f}'
        )

# ------------------------------------------------------------
# 2. Validação
# ------------------------------------------------------------

print('\nValidando...')

perdas_validacao = []

for lote in lotes_validacao:

    imagens, labels = carregar_lote_pk(
        lote['caminhos'],
        lote['rotulos']
    )

    resultado = passo_validacao(
        imagens,
        labels
    )

    perdas_validacao.append(
        resultado['loss']
    )

loss_treino = float(
    sum(perdas_treino) / len(perdas_treino)
)

loss_validacao = float(
    sum(perdas_validacao) / len(perdas_validacao)
)

tempo_minutos = (
    time.perf_counter() - inicio_epoca
) / 60

# ------------------------------------------------------------
# 3. Salvar checkpoints
# ------------------------------------------------------------

# Na primeira época, este é o melhor resultado da fase 2.
modelo_embeddings.save_weights(
    str(ARQUIVO_MELHOR_FASE2)
)

modelo_embeddings.save_weights(
    str(ARQUIVO_ULTIMO_FASE2)
)

caminho_checkpoint = (
    GERENCIADOR_CHECKPOINT.save(
        checkpoint_number=1
    )
)

# ------------------------------------------------------------
# 4. Persistir histórico
# ------------------------------------------------------------

registro = {
    'fold': FOLD_ATUAL,
    'fase': 2,
    'epoca': 1,
    'loss_treino': loss_treino,
    'loss_validacao': loss_validacao,
    'semi_hard': total_semihard,
    'fallback': total_fallback,
    'tempo_minutos': tempo_minutos,
    'melhorou': True
}

pd.DataFrame(
    [registro]
).to_csv(
    ARQUIVO_HISTORICO_FASE2,
    index=False
)

# ------------------------------------------------------------
# 5. Atualizar estado
# ------------------------------------------------------------

estado_fase2.update({
    'ultima_epoca_concluida': 1,
    'melhor_loss_validacao': loss_validacao,
    'epoca_melhor_modelo': 1,
    'epocas_sem_melhoria': 0,
    'status': 'em_andamento'
})

arquivo_temporario = Path(
    str(ARQUIVO_ESTADO_FASE2) + '.tmp'
)

with open(
    arquivo_temporario,
    'w',
    encoding='utf-8'
) as arquivo:

    json.dump(
        estado_fase2,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

os.replace(
    arquivo_temporario,
    ARQUIVO_ESTADO_FASE2
)

# ------------------------------------------------------------
# 6. Resumo
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('RESULTADO — FASE 2 / ÉPOCA 1')
print('=' * 60)

print(f'Loss treino: {loss_treino:.6f}')
print(f'Loss validação: {loss_validacao:.6f}')
print(f'Tempo: {tempo_minutos:.2f} min')

print('Triplets semi-hard:', total_semihard)
print('Triplets fallback:', total_fallback)

print('Checkpoint:', caminho_checkpoint)
print('Melhor modelo:', ARQUIVO_MELHOR_FASE2)
print('Histórico:', ARQUIVO_HISTORICO_FASE2)

print('\nBloco [40] concluído.')


In [ ]:

# ============================================================
# Bloco de código [41] — Fase 2, épocas 2 a 10
# GPU: OBRIGATÓRIA — T4
# ============================================================

import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

print('=' * 60)
print('CONTINUIDADE DO FINE-TUNING — FOLD 1')
print('=' * 60)

assert tf.config.list_physical_devices('GPU')
assert len(modelo_embeddings.trainable_variables) == 10

# ------------------------------------------------------------
# 1. Recuperar estado persistido
# ------------------------------------------------------------

with open(
    ARQUIVO_ESTADO_FASE2,
    'r',
    encoding='utf-8'
) as arquivo:
    estado_fase2 = json.load(arquivo)

historico_fase2 = pd.read_csv(
    ARQUIVO_HISTORICO_FASE2
)

ultima_epoca = int(
    estado_fase2['ultima_epoca_concluida']
)

assert estado_fase2['fold'] == FOLD_ATUAL
assert estado_fase2['fase'] == 2

assert set(
    historico_fase2['epoca'].astype(int)
) == set(range(1, ultima_epoca + 1)), (
    'Histórico inconsistente com o estado salvo.'
)

# A sessão atual deve conter o modelo da última época.
# Para retomada após desconexão, restaurar também o
# checkpoint e o gerador antes de executar este bloco.

assert ultima_epoca == 1, (
    'Este bloco foi preparado para continuar da época 1. '
    'Não repetir sem verificar a retomada.'
)

MAX_EPOCAS = int(
    CONFIG['treinamento']['epocas_fase2']
)

PACIENCIA = int(
    CONFIG['treinamento']['early_stopping_patience']
)

print('Última época:', ultima_epoca)
print('Próxima época:', ultima_epoca + 1)
print('Máximo:', MAX_EPOCAS)
print('Paciência:', PACIENCIA)

# ------------------------------------------------------------
# 2. Função de execução de uma época
# ------------------------------------------------------------

def executar_epoca_fase2(epoca):

    global estado_fase2

    print('\n' + '=' * 60)
    print(f'FOLD 1 — FASE 2 — ÉPOCA {epoca}')
    print('=' * 60)

    inicio = time.perf_counter()

    perdas_treino = []
    semi_hard_total = 0
    fallback_total = 0

    for passo in range(PASSOS_TREINO):

        caminhos, rotulos = (
            gerador_treino.proximo_lote()
        )

        imagens, labels = carregar_lote_pk(
            caminhos,
            rotulos
        )

        resultado = passo_treinamento(
            imagens,
            labels
        )

        perdas_treino.append(
            float(resultado['loss'])
        )

        semi_hard_total += int(
            resultado['semi-hard']
        )

        fallback_total += int(
            resultado['fallback']
        )

        if (passo + 1) % 100 == 0:

            print(
                f'Treino {passo + 1}/{PASSOS_TREINO} | '
                f'Loss: {np.mean(perdas_treino):.5f}'
            )

    print('Validando...')

    perdas_validacao = []

    for lote in lotes_validacao:

        imagens, labels = carregar_lote_pk(
            lote['caminhos'],
            lote['rotulos']
        )

        resultado = passo_validacao(
            imagens,
            labels
        )

        perdas_validacao.append(
            float(resultado['loss'])
        )

    loss_treino = float(
        np.mean(perdas_treino)
    )

    loss_validacao = float(
        np.mean(perdas_validacao)
    )

    tempo_minutos = (
        time.perf_counter() - inicio
    ) / 60

    # --------------------------------------------------------
    # 3. Avaliar melhoria
    # --------------------------------------------------------

    melhor_anterior = float(
        estado_fase2['melhor_loss_validacao']
    )

    melhorou = (
        loss_validacao < melhor_anterior
    )

    if melhorou:

        estado_fase2['melhor_loss_validacao'] = (
            loss_validacao
        )

        estado_fase2['epoca_melhor_modelo'] = epoca

        estado_fase2['epocas_sem_melhoria'] = 0

        modelo_embeddings.save_weights(
            str(ARQUIVO_MELHOR_FASE2)
        )

    else:

        estado_fase2['epocas_sem_melhoria'] += 1

    # --------------------------------------------------------
    # 4. Salvar último modelo e checkpoint completo
    # --------------------------------------------------------

    modelo_embeddings.save_weights(
        str(ARQUIVO_ULTIMO_FASE2)
    )

    checkpoint = GERENCIADOR_CHECKPOINT.save(
        checkpoint_number=epoca
    )

    # --------------------------------------------------------
    # 5. Registrar histórico
    # --------------------------------------------------------

    registro = {
        'fold': FOLD_ATUAL,
        'fase': 2,
        'epoca': epoca,
        'loss_treino': loss_treino,
        'loss_validacao': loss_validacao,
        'semi_hard': semi_hard_total,
        'fallback': fallback_total,
        'tempo_minutos': tempo_minutos,
        'melhorou': melhorou
    }

    historico_atual = pd.read_csv(
        ARQUIVO_HISTORICO_FASE2
    )

    historico_atual = pd.concat(
        [
            historico_atual,
            pd.DataFrame([registro])
        ],
        ignore_index=True
    )

    historico_atual.to_csv(
        ARQUIVO_HISTORICO_FASE2,
        index=False
    )

    # --------------------------------------------------------
    # 6. Atualizar estado persistido
    # --------------------------------------------------------

    estado_fase2['ultima_epoca_concluida'] = epoca

    estado_fase2['status'] = (
        'early_stopping'
        if estado_fase2['epocas_sem_melhoria'] >= PACIENCIA
        else 'em_andamento'
    )

    temporario = Path(
        str(ARQUIVO_ESTADO_FASE2) + '.tmp'
    )

    with open(
        temporario,
        'w',
        encoding='utf-8'
    ) as arquivo:

        json.dump(
            estado_fase2,
            arquivo,
            indent=4,
            ensure_ascii=False
        )

    os.replace(
        temporario,
        ARQUIVO_ESTADO_FASE2
    )

    print(
        f'\nÉpoca {epoca} concluída | '
        f'Treino: {loss_treino:.6f} | '
        f'Validação: {loss_validacao:.6f}'
    )

    print(
        f'Tempo: {tempo_minutos:.2f} min | '
        f'Melhorou: {melhorou}'
    )

    print(
        'Épocas sem melhoria:',
        estado_fase2['epocas_sem_melhoria']
    )

    print('Checkpoint:', checkpoint)

# ------------------------------------------------------------
# 7. Executar épocas restantes
# ------------------------------------------------------------

for epoca in range(
    ultima_epoca + 1,
    MAX_EPOCAS + 1
):

    executar_epoca_fase2(epoca)

    if (
        estado_fase2['epocas_sem_melhoria']
        >= PACIENCIA
    ):

        print('\nEARLY STOPPING ATIVADO.')
        break

# ------------------------------------------------------------
# 8. Resumo
# ------------------------------------------------------------

historico_final = pd.read_csv(
    ARQUIVO_HISTORICO_FASE2
)

print('\n' + '=' * 60)
print('RESUMO — FASE 2')
print('=' * 60)

print(
    historico_final[
        ['epoca', 'loss_treino', 'loss_validacao']
    ].to_string(index=False)
)

print(
    '\nMelhor época:',
    estado_fase2['epoca_melhor_modelo']
)

print(
    'Melhor loss:',
    estado_fase2['melhor_loss_validacao']
)

print(
    'Última época:',
    estado_fase2['ultima_epoca_concluida']
)

print('\nBloco [41] concluído.')


In [ ]:

# ============================================================
# Bloco de código [42] — Consolidar melhor modelo
# GPU: ATIVADA — sem treinamento
# ============================================================

import json
import shutil
import hashlib
import pandas as pd
import tensorflow as tf

from pathlib import Path

print('=' * 60)
print('CONSOLIDAÇÃO DO MODELO — FOLD 1')
print('=' * 60)

# ------------------------------------------------------------
# 1. Conferir histórico da fase 2
# ------------------------------------------------------------

with open(
    ARQUIVO_ESTADO_FASE2,
    'r',
    encoding='utf-8'
) as arquivo:
    estado_fase2 = json.load(arquivo)

historico_fase2 = pd.read_csv(
    ARQUIVO_HISTORICO_FASE2
)

assert estado_fase2['ultima_epoca_concluida'] == 10
assert estado_fase2['epoca_melhor_modelo'] == 9

melhor_registro = historico_fase2.loc[
    historico_fase2['loss_validacao'].idxmin()
]

assert int(melhor_registro['epoca']) == 9

print('Melhor época:', int(melhor_registro['epoca']))
print(
    'Melhor loss:',
    float(melhor_registro['loss_validacao'])
)

# ------------------------------------------------------------
# 2. Preservar modelo final
# ------------------------------------------------------------

ARQUIVO_MODELO_FINAL_FOLD1 = (
    DIR_CHECKPOINT / 'modelo_final_fold1.weights.h5'
)

assert ARQUIVO_MELHOR_FASE2.exists()

shutil.copy2(
    ARQUIVO_MELHOR_FASE2,
    ARQUIVO_MODELO_FINAL_FOLD1
)

# ------------------------------------------------------------
# 3. Calcular hash SHA-256
# ------------------------------------------------------------

def calcular_sha256(caminho):

    hash_arquivo = hashlib.sha256()

    with open(caminho, 'rb') as arquivo:

        for bloco in iter(
            lambda: arquivo.read(1024 * 1024),
            b''
        ):
            hash_arquivo.update(bloco)

    return hash_arquivo.hexdigest()

hash_origem = calcular_sha256(
    ARQUIVO_MELHOR_FASE2
)

hash_destino = calcular_sha256(
    ARQUIVO_MODELO_FINAL_FOLD1
)

assert hash_origem == hash_destino, (
    'Falha na integridade da cópia.'
)

print('\nCópia verificada por SHA-256.')

# ------------------------------------------------------------
# 4. Carregar pesos finais
# ------------------------------------------------------------

modelo_embeddings.load_weights(
    str(ARQUIVO_MODELO_FINAL_FOLD1)
)

# ------------------------------------------------------------
# 5. Verificar inferência
# ------------------------------------------------------------

lote = lotes_validacao[0]

imagens, labels = carregar_lote_pk(
    lote['caminhos'],
    lote['rotulos']
)

embeddings = modelo_embeddings(
    imagens,
    training=False
)

normas = tf.norm(
    embeddings,
    axis=1
).numpy()

assert bool(
    tf.reduce_all(
        tf.math.is_finite(embeddings)
    ).numpy()
), 'Embeddings contêm valores não finitos.'

assert embeddings.shape[1] == 128

assert all(
    abs(float(norma) - 1.0) < 1e-3
    for norma in normas
), 'Embeddings não estão normalizados.'

print('\nTeste de inferência: OK')
print('Dimensão dos embeddings:', embeddings.shape)
print(
    'Norma mínima:',
    round(float(normas.min()), 6)
)
print(
    'Norma máxima:',
    round(float(normas.max()), 6)
)

# ------------------------------------------------------------
# 6. Registrar metadados finais
# ------------------------------------------------------------

metadados = {
    'fold': FOLD_ATUAL,
    'modelo': 'VGG16',
    'embedding_dim': 128,
    'loss': 'triplet_semihard',
    'margem': float(MARGEM_TRIPLET),
    'fase1_melhor_epoca': 9,
    'fase2_melhor_epoca': 9,
    'fase2_melhor_loss_validacao': float(
        melhor_registro['loss_validacao']
    ),
    'arquivo_pesos': str(
        ARQUIVO_MODELO_FINAL_FOLD1
    ),
    'sha256': hash_destino
}

ARQUIVO_METADADOS_FOLD1 = (
    DIR_RESULTADOS / 'modelo_final_fold1.json'
)

with open(
    ARQUIVO_METADADOS_FOLD1,
    'w',
    encoding='utf-8'
) as arquivo:

    json.dump(
        metadados,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

print('\n' + '=' * 60)
print('MODELO FINAL CONSOLIDADO')
print('=' * 60)

print('Pesos:', ARQUIVO_MODELO_FINAL_FOLD1)
print('Metadados:', ARQUIVO_METADADOS_FOLD1)
print('SHA-256:', hash_destino)

print('\nBloco [42] concluído.')


In [ ]:

# ============================================================
# Bloco de código [43] — Backup completo do fold 1
# GPU: NÃO NECESSÁRIA
# ============================================================

import os
import json
import tarfile
import hashlib
from pathlib import Path
from datetime import datetime, timezone

print('=' * 60)
print('BACKUP DE SEGURANÇA — FOLD 1')
print('=' * 60)

# ------------------------------------------------------------
# 1. Definir caminhos
# ------------------------------------------------------------

RAIZ_EXPERIMENTO = Path(
    '/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold'
)

DIR_BACKUP = RAIZ_EXPERIMENTO / 'backups'
DIR_BACKUP.mkdir(parents=True, exist_ok=True)

ARQUIVO_BACKUP = (
    DIR_BACKUP / 'fold_1_completo.tar.gz'
)

ARQUIVO_TEMPORARIO = (
    DIR_BACKUP / 'fold_1_completo.tar.gz.tmp'
)

ARQUIVO_MANIFESTO = (
    DIR_BACKUP / 'fold_1_backup_manifesto.json'
)

# ------------------------------------------------------------
# 2. Verificar arquivos essenciais
# ------------------------------------------------------------

arquivos_essenciais = [
    ARQUIVO_MODELO_FINAL_FOLD1,
    ARQUIVO_METADADOS_FOLD1,
    ARQUIVO_MELHOR_FASE1,
    ARQUIVO_MELHOR_FASE2,
    ARQUIVO_HISTORICO,
    ARQUIVO_HISTORICO_FASE2,
    ARQUIVO_ESTADO,
    ARQUIVO_ESTADO_FASE2
]

for arquivo in arquivos_essenciais:

    assert arquivo.is_file(), (
        f'Arquivo ausente: {arquivo}'
    )

    assert arquivo.stat().st_size > 0, (
        f'Arquivo vazio: {arquivo}'
    )

print('Arquivos essenciais: OK')

# ------------------------------------------------------------
# 3. Selecionar arquivos do experimento
# ------------------------------------------------------------

# Preservar resultados e checkpoints.
# Incluir também arquivos de configuração e splits,
# quando estiverem dentro da raiz do experimento.

diretorios_backup = [
    RAIZ_EXPERIMENTO / 'resultados' / 'fold_1',
    RAIZ_EXPERIMENTO / 'checkpoints' / 'fold_1',
    RAIZ_EXPERIMENTO / 'splits'
]

arquivos_backup = set()

for diretorio in diretorios_backup:

    if diretorio.is_dir():

        for arquivo in diretorio.rglob('*'):

            if arquivo.is_file():
                arquivos_backup.add(arquivo)

# Incluir arquivos essenciais mesmo se estiverem
# fora dos diretórios acima.
for arquivo in arquivos_essenciais:
    arquivos_backup.add(Path(arquivo))

arquivos_backup = sorted(arquivos_backup)

print(
    'Arquivos selecionados:',
    len(arquivos_backup)
)

# ------------------------------------------------------------
# 4. Criar pacote compactado
# ------------------------------------------------------------

with tarfile.open(
    ARQUIVO_TEMPORARIO,
    mode='w:gz'
) as pacote:

    for indice, arquivo in enumerate(
        arquivos_backup,
        start=1
    ):

        nome_interno = arquivo.relative_to(
            RAIZ_EXPERIMENTO
        )

        pacote.add(
            arquivo,
            arcname=str(nome_interno),
            recursive=False
        )

        if indice % 25 == 0:
            print(
                f'Compactados: '
                f'{indice}/{len(arquivos_backup)}'
            )

# ------------------------------------------------------------
# 5. Verificar conteúdo do pacote
# ------------------------------------------------------------

with tarfile.open(
    ARQUIVO_TEMPORARIO,
    mode='r:gz'
) as pacote:

    membros = pacote.getmembers()

    nomes = {
        membro.name
        for membro in membros
    }

    for arquivo in arquivos_essenciais:

        relativo = str(
            arquivo.relative_to(
                RAIZ_EXPERIMENTO
            )
        )

        assert relativo in nomes, (
            f'Arquivo não incluído: {relativo}'
        )

    # Ler todos os arquivos compactados para
    # verificar a integridade do conteúdo.
    for membro in membros:

        if membro.isfile():

            extraido = pacote.extractfile(membro)

            assert extraido is not None

            while extraido.read(1024 * 1024):
                pass

# ------------------------------------------------------------
# 6. Finalizar backup
# ------------------------------------------------------------

os.replace(
    ARQUIVO_TEMPORARIO,
    ARQUIVO_BACKUP
)

def sha256_arquivo(caminho):

    resumo = hashlib.sha256()

    with open(caminho, 'rb') as arquivo:

        for bloco in iter(
            lambda: arquivo.read(1024 * 1024),
            b''
        ):
            resumo.update(bloco)

    return resumo.hexdigest()

hash_backup = sha256_arquivo(
    ARQUIVO_BACKUP
)

manifesto = {
    'experimento': 'tcc1_kfold',
    'fold': 1,
    'data_utc': datetime.now(
        timezone.utc
    ).isoformat(),
    'arquivo_backup': str(ARQUIVO_BACKUP),
    'quantidade_arquivos': len(arquivos_backup),
    'tamanho_bytes': ARQUIVO_BACKUP.stat().st_size,
    'sha256': hash_backup,
    'modelo_final_sha256': sha256_arquivo(
        ARQUIVO_MODELO_FINAL_FOLD1
    ),
    'fase1_epocas': 10,
    'fase2_epocas': 10,
    'fase2_melhor_epoca': 9
}

with open(
    ARQUIVO_MANIFESTO,
    'w',
    encoding='utf-8'
) as arquivo:

    json.dump(
        manifesto,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 7. Resumo
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('BACKUP CONCLUÍDO')
print('=' * 60)

print('Arquivo:', ARQUIVO_BACKUP)

print(
    'Tamanho:',
    round(
        ARQUIVO_BACKUP.stat().st_size
        / (1024 ** 2),
        2
    ),
    'MB'
)

print(
    'Arquivos preservados:',
    len(arquivos_backup)
)

print('SHA-256:', hash_backup)
print('Manifesto:', ARQUIVO_MANIFESTO)

print('\nBloco [43] concluído.')


## 4. Avaliação inicial — Fold 1

Extração de embeddings, recuperação e verificação, análises de erros e simulação de múltiplas imagens de referência.

In [ ]:

# ============================================================
# Bloco [44] — Recuperar ambiente do TCC1
# Nova sessão do Google Colab
# ============================================================

from google.colab import drive
from pathlib import Path

import json
import hashlib
import tensorflow as tf

print("=" * 60)
print("RECUPERAÇÃO DO EXPERIMENTO — FOLD 1")
print("=" * 60)

# 1. Reconectar Google Drive
drive.mount("/content/drive")

# 2. Definir diretórios
BASE = Path("/content/drive/MyDrive/TCC_Caes")
EXPERIMENTO = BASE / "experimentos/tcc1_kfold"

DIR_RESULTADOS = EXPERIMENTO / "resultados/fold_1"
DIR_CHECKPOINT = EXPERIMENTO / "checkpoints/fold_1"
DIR_BACKUP = EXPERIMENTO / "backups"

ARQUIVO_MODELO_FINAL_FOLD1 = (
    DIR_CHECKPOINT / "modelo_final_fold1.weights.h5"
)

ARQUIVO_METADADOS_FOLD1 = (
    DIR_RESULTADOS / "modelo_final_fold1.json"
)

ARQUIVO_BACKUP = (
    DIR_BACKUP / "fold_1_completo.tar.gz"
)

ARQUIVO_MANIFESTO = (
    DIR_BACKUP / "fold_1_backup_manifesto.json"
)

# 3. Verificar arquivos essenciais
arquivos = {
    "Modelo final": ARQUIVO_MODELO_FINAL_FOLD1,
    "Metadados": ARQUIVO_METADADOS_FOLD1,
    "Backup": ARQUIVO_BACKUP,
    "Manifesto": ARQUIVO_MANIFESTO,
    "Histórico fase 1": (
        DIR_RESULTADOS / "historico_treinamento.csv"
    ),
    "Histórico fase 2": (
        DIR_RESULTADOS / "fase2/historico_treinamento.csv"
    )
}

print("\nARQUIVOS PERSISTIDOS")

for nome, caminho in arquivos.items():
    existe = caminho.is_file()
    tamanho = (
        caminho.stat().st_size
        if existe else 0
    )

    print(
        f'{"OK" if existe and tamanho > 0 else "FALHA"} '
        f'| {nome} | {tamanho:,} bytes'
    )

    assert existe and tamanho > 0, (
        f"Arquivo ausente ou vazio: {caminho}"
    )

# 4. Verificar integridade do modelo
def calcular_sha256(caminho):
    resumo = hashlib.sha256()

    with open(caminho, "rb") as arquivo:
        for bloco in iter(
            lambda: arquivo.read(1024 * 1024),
            b""
        ):
            resumo.update(bloco)

    return resumo.hexdigest()

with open(
    ARQUIVO_METADADOS_FOLD1,
    encoding="utf-8"
) as arquivo:
    metadados = json.load(arquivo)

hash_atual = calcular_sha256(
    ARQUIVO_MODELO_FINAL_FOLD1
)

assert hash_atual == metadados["sha256"], (
    "Falha de integridade dos pesos!"
)

print("\nIntegridade SHA-256 do modelo: OK")

# 5. Verificar GPU
gpus = tf.config.list_physical_devices("GPU")

print("\nGPU detectada:", gpus)

if gpus:
    print("Ambiente preparado para inferência.")
else:
    print("Sem GPU. A recuperação ainda é possível.")

# 6. Verificar dados do modelo
print("\nMODELO SALVO")
print("Fold:", metadados["fold"])
print("Arquitetura:", metadados["modelo"])
print("Dimensão embedding:", metadados["embedding_dim"])
print("Melhor época fase 2:",
      metadados["fase2_melhor_epoca"])
print("Loss validação:",
      metadados["fase2_melhor_loss_validacao"])

print("\nBloco [44] concluído.")


In [ ]:

# ============================================================
# Bloco [45] — Reconstruir modelo do fold 1
# GPU: ATIVADA — sem treinamento
# ============================================================

import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, Model

print("=" * 60)
print("RECONSTRUÇÃO DO MODELO — FOLD 1")
print("=" * 60)

tf.keras.backend.clear_session()

# ------------------------------------------------------------
# 1. Reconstruir backbone VGG16
# ------------------------------------------------------------

entrada = layers.Input(
    shape=(224, 224, 3),
    name="entrada_imagem"
)

backbone = tf.keras.applications.VGG16(
    include_top=False,
    weights=None,
    input_shape=(224, 224, 3)
)

backbone.trainable = False

# Preservar o backbone como submodelo.
x = backbone(entrada)

# ------------------------------------------------------------
# 2. Reconstruir cabeça de embeddings
# ------------------------------------------------------------

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(
    256,
    activation="relu"
)(x)

x = layers.Dense(
    128,
    activation="linear"
)(x)

saida = layers.UnitNormalization(
    axis=-1
)(x)

modelo_embeddings = Model(
    inputs=entrada,
    outputs=saida,
    name="modelo_embeddings"
)

# ------------------------------------------------------------
# 3. Carregar pesos finais
# ------------------------------------------------------------

modelo_embeddings.load_weights(
    str(ARQUIVO_MODELO_FINAL_FOLD1)
)

print("\nPesos carregados com sucesso.")

# ------------------------------------------------------------
# 4. Verificar arquitetura
# ------------------------------------------------------------

print("\nArquitetura reconstruída:")
modelo_embeddings.summary()

assert modelo_embeddings.output_shape == (
    None, 128
)

# ------------------------------------------------------------
# 5. Teste de inferência
# ------------------------------------------------------------

# Entrada sintética somente para validar a arquitetura.
# Não representa uma imagem real do dataset.

imagem_teste = tf.zeros(
    (1, 224, 224, 3),
    dtype=tf.float32
)

embedding = modelo_embeddings(
    imagem_teste,
    training=False
)

norma = tf.norm(
    embedding,
    axis=1
).numpy()

assert embedding.shape == (1, 128)

assert np.all(
    np.isfinite(embedding.numpy())
), "Embedding contém valores inválidos."

print("\nTESTE DE INFERÊNCIA")
print("Dimensão:", embedding.shape)
print("Norma L2:", norma)

print("\nModelo pronto para extração de embeddings.")
print("Bloco [45] concluído.")


In [ ]:

# ============================================================
# Bloco [46] — Diagnóstico dos dados de validação
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
import json
import csv

print("=" * 60)
print("RECUPERAÇÃO DA VALIDAÇÃO — FOLD 1")
print("=" * 60)

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

LOCAL = Path("/content/tcc1_kfold")

# ------------------------------------------------------------
# 1. Verificar diretórios de imagens
# ------------------------------------------------------------

print("\nDIRETÓRIOS DE DATASET")

candidatos_dataset = [
    LOCAL / "datasets",
    RAIZ / "datasets",
    RAIZ / "dados",
]

for caminho in candidatos_dataset:
    print(
        "OK" if caminho.exists() else "AUSENTE",
        "|",
        caminho
    )

# ------------------------------------------------------------
# 2. Localizar arquivos de divisão e validação
# ------------------------------------------------------------

print("\nARQUIVOS DE DIVISÃO / VALIDAÇÃO")

extensoes = {".json", ".csv", ".jsonl", ".txt"}

arquivos_encontrados = []

for raiz_busca in [RAIZ, LOCAL]:

    if not raiz_busca.exists():
        continue

    for caminho in raiz_busca.rglob("*"):

        if not caminho.is_file():
            continue

        if caminho.suffix.lower() not in extensoes:
            continue

        nome = caminho.name.lower()

        if any(
            termo in nome
            for termo in [
                "split",
                "fold",
                "valid",
                "identidade",
                "dataset",
                "config"
            ]
        ):
            arquivos_encontrados.append(caminho)

arquivos_encontrados = sorted(
    set(arquivos_encontrados)
)

for caminho in arquivos_encontrados[:60]:
    print(caminho)

print(
    "\nTotal de arquivos candidatos:",
    len(arquivos_encontrados)
)

# ------------------------------------------------------------
# 3. Examinar estrutura dos JSONs relevantes
# ------------------------------------------------------------

print("\nESTRUTURA DOS ARQUIVOS JSON")

jsons_prioritarios = [
    caminho
    for caminho in arquivos_encontrados
    if caminho.suffix.lower() == ".json"
    and any(
        termo in caminho.name.lower()
        for termo in ["split", "fold", "valid"]
    )
]

for caminho in jsons_prioritarios[:12]:

    print("\nArquivo:", caminho)

    try:
        with open(
            caminho,
            "r",
            encoding="utf-8"
        ) as arquivo:
            dados = json.load(arquivo)

        print("Tipo:", type(dados).__name__)

        if isinstance(dados, dict):

            print(
                "Chaves:",
                list(dados.keys())[:20]
            )

            for chave, valor in list(
                dados.items()
            )[:8]:

                if isinstance(valor, (list, dict)):
                    print(
                        f"  {chave}: "
                        f"{type(valor).__name__}, "
                        f"{len(valor)} elementos"
                    )
                else:
                    print(
                        f"  {chave}: "
                        f"{str(valor)[:120]}"
                    )

        elif isinstance(dados, list):

            print("Elementos:", len(dados))

            if dados:
                primeiro = dados[0]

                if isinstance(primeiro, dict):
                    print(
                        "Chaves do primeiro elemento:",
                        list(primeiro.keys())
                    )
                else:
                    print(
                        "Primeiro elemento:",
                        str(primeiro)[:180]
                    )

    except Exception as erro:
        print(
            "Não foi possível ler:",
            type(erro).__name__,
            str(erro)[:150]
        )

# ------------------------------------------------------------
# 4. Verificar lista fixa de validação
# ------------------------------------------------------------

arquivo_lotes = (
    RAIZ
    / "resultados/fold_1/lotes_validacao.json"
)

print("\nLOTES FIXOS DE VALIDAÇÃO")

if arquivo_lotes.is_file():

    with open(
        arquivo_lotes,
        "r",
        encoding="utf-8"
    ) as arquivo:
        lotes_validacao = json.load(arquivo)

    print(
        "Quantidade de lotes:",
        len(lotes_validacao)
    )

    if lotes_validacao:

        print(
            "Chaves do primeiro lote:",
            list(lotes_validacao[0].keys())
        )

        print(
            "Quantidade de imagens no primeiro lote:",
            len(
                lotes_validacao[0].get(
                    "caminhos", []
                )
            )
        )

        print(
            "Exemplo de caminho:",
            lotes_validacao[0].get(
                "caminhos", ["N/D"]
            )[0]
        )

else:
    print("Arquivo de lotes não encontrado.")

print("\nBloco [46] concluído.")


In [ ]:

# ============================================================
# Bloco [47] — Localizar dataset e validar splits
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
import pandas as pd
import json

print("=" * 60)
print("LOCALIZAÇÃO DO DATASET — FOLD 1")
print("=" * 60)

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

SPLITS = RAIZ / "splits"

# ------------------------------------------------------------
# 1. Procurar arquivos compactados do dataset
# ------------------------------------------------------------

print("\nBACKUPS DE DATASET")

pastas_busca = [
    Path("/content/drive/MyDrive/TCC_Caes"),
    Path("/content")
]

extensoes = (
    ".tar.gz",
    ".tgz",
    ".tar",
    ".zip"
)

backups_encontrados = []

for pasta in pastas_busca:

    if not pasta.exists():
        continue

    for arquivo in pasta.rglob("*"):

        if not arquivo.is_file():
            continue

        if arquivo.name.lower().endswith(extensoes):

            # Evitar duplicar resultados.
            if arquivo not in backups_encontrados:
                backups_encontrados.append(arquivo)

for arquivo in sorted(backups_encontrados):

    tamanho_gb = (
        arquivo.stat().st_size / (1024 ** 3)
    )

    print(
        f"{tamanho_gb:.3f} GB | {arquivo}"
    )

print(
    "\nArquivos compactados encontrados:",
    len(backups_encontrados)
)

# ------------------------------------------------------------
# 2. Examinar CSVs de divisão do fold 1
# ------------------------------------------------------------

arquivos_csv = {
    "Treino (imagens)":
        SPLITS / "fold_1_imagens_treino.csv",

    "Validação (imagens)":
        SPLITS / "fold_1_imagens_validacao.csv",

    "Treino (identidades)":
        SPLITS / "fold_1_treino.csv",

    "Validação (identidades)":
        SPLITS / "fold_1_validacao.csv",

    "Teste independente":
        SPLITS / "identidades_teste_30.csv"
}

tabelas = {}

print("\n" + "=" * 60)
print("ESTRUTURA DOS SPLITS")
print("=" * 60)

for nome, arquivo in arquivos_csv.items():

    df = pd.read_csv(arquivo)

    tabelas[nome] = df

    print(f"\n{nome}")
    print("Linhas:", len(df))
    print("Colunas:", list(df.columns))
    print(df.head(2).to_string(index=False))

# ------------------------------------------------------------
# 3. Verificar lotes fixos de validação
# ------------------------------------------------------------

arquivo_lotes = (
    RAIZ
    / "resultados/fold_1/lotes_validacao.json"
)

with open(
    arquivo_lotes,
    encoding="utf-8"
) as arquivo:
    lotes = json.load(arquivo)

caminhos_lotes = [
    caminho
    for lote in lotes
    for caminho in lote["caminhos"]
]

print("\n" + "=" * 60)
print("LOTES FIXOS")
print("=" * 60)

print("Lotes:", len(lotes))
print("Amostras totais:", len(caminhos_lotes))
print("Imagens únicas:", len(set(caminhos_lotes)))

# ------------------------------------------------------------
# 4. Verificar configuração de pré-processamento
# ------------------------------------------------------------

arquivo_config = RAIZ / "config.json"

if arquivo_config.is_file():

    with open(
        arquivo_config,
        encoding="utf-8"
    ) as arquivo:
        config_salva = json.load(arquivo)

    print("\nCONFIGURAÇÃO ORIGINAL")
    print(
        json.dumps(
            config_salva,
            indent=2,
            ensure_ascii=False
        )[:5000]
    )

print("\nBloco [47] concluído.")


In [ ]:

# ============================================================
# Bloco [48] — Restaurar dataset e verificar validação
# GPU: NÃO NECESSÁRIA
# ============================================================

import tarfile
import shutil
import pandas as pd
from pathlib import Path
from time import perf_counter

print("=" * 60)
print("RESTAURAÇÃO DO DATASET — FOLD 1")
print("=" * 60)

inicio = perf_counter()

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

ARQUIVO_DATASET = RAIZ / "datasets_backup.tar"

DESTINO_LOCAL = Path(
    "/content/tcc1_kfold"
)

DIR_DATASET = DESTINO_LOCAL / "datasets"

DESTINO_LOCAL.mkdir(
    parents=True,
    exist_ok=True
)

assert ARQUIVO_DATASET.is_file(), (
    "Backup do dataset não encontrado."
)

# ------------------------------------------------------------
# 1. Inspecionar estrutura do backup
# ------------------------------------------------------------

print("\nINSPEÇÃO DO BACKUP")

with tarfile.open(
    ARQUIVO_DATASET,
    "r"
) as pacote:

    membros = pacote.getmembers()

    arquivos_imagem = [
        membro
        for membro in membros
        if membro.isfile()
        and membro.name.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ]

    print(
        "Arquivos de imagem:",
        len(arquivos_imagem)
    )

    print("\nExemplos de caminhos:")

    for membro in membros[:8]:
        print(" ", membro.name)

    # --------------------------------------------------------
    # 2. Validar caminhos antes da extração
    # --------------------------------------------------------

    raiz_segura = DESTINO_LOCAL.resolve()

    for membro in membros:

        destino = (
            DESTINO_LOCAL / membro.name
        ).resolve()

        assert destino.is_relative_to(
            raiz_segura
        ), (
            f"Caminho inseguro: {membro.name}"
        )

        assert not (
            membro.issym() or membro.islnk()
        ), (
            f"Link não permitido: {membro.name}"
        )

        assert (
            membro.isfile() or membro.isdir()
        ), (
            f"Tipo não permitido: {membro.name}"
        )

    # --------------------------------------------------------
    # 3. Extrair arquivos
    # --------------------------------------------------------

    print("\nExtraindo dataset...")

    for membro in membros:

        destino = DESTINO_LOCAL / membro.name

        if membro.isdir():

            destino.mkdir(
                parents=True,
                exist_ok=True
            )

        elif membro.isfile():

            destino.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            with pacote.extractfile(membro) as origem:

                assert origem is not None

                with open(destino, "wb") as saida:
                    shutil.copyfileobj(
                        origem,
                        saida
                    )

print("\nExtração concluída.")

# ------------------------------------------------------------
# 4. Verificar imagens de validação
# ------------------------------------------------------------

CSV_VALIDACAO = (
    RAIZ
    / "splits/fold_1_imagens_validacao.csv"
)

df_validacao = pd.read_csv(
    CSV_VALIDACAO
)

PREFIXO_ORIGINAL = (
    "/content/drive/MyDrive/datasets_extraido/"
)

def converter_caminho(caminho):

    caminho = str(caminho)

    assert caminho.startswith(
        PREFIXO_ORIGINAL
    ), (
        f"Prefixo inesperado: {caminho}"
    )

    relativo = caminho[
        len(PREFIXO_ORIGINAL):
    ]

    # Estrutura local utilizada no treinamento:
    # DogFaceNet-master/after_4_bis/... -> DogFaceNet/...
    # YT-BB-dog/... -> YTBB/...
    if relativo.startswith(
        "DogFaceNet-master/after_4_bis/"
    ):

        relativo = (
            "DogFaceNet/"
            + relativo.split(
                "DogFaceNet-master/after_4_bis/",
                1
            )[1]
        )

    elif relativo.startswith("YT-BB-dog/"):

        relativo = (
            "YTBB/"
            + relativo.split(
                "YT-BB-dog/",
                1
            )[1]
        )

    return str(
        DIR_DATASET / relativo
    )

df_validacao["caminho_local"] = (
    df_validacao["caminho"].apply(
        converter_caminho
    )
)

df_validacao["arquivo_existe"] = (
    df_validacao["caminho_local"].apply(
        lambda p: Path(p).is_file()
    )
)

encontradas = int(
    df_validacao["arquivo_existe"].sum()
)

ausentes = len(df_validacao) - encontradas

print("\n" + "=" * 60)
print("VERIFICAÇÃO DA VALIDAÇÃO")
print("=" * 60)

print(
    "Imagens esperadas:",
    len(df_validacao)
)

print(
    "Imagens encontradas:",
    encontradas
)

print(
    "Imagens ausentes:",
    ausentes
)

print(
    "Identidades:",
    df_validacao["identidade"].nunique()
)

if ausentes > 0:

    print("\nExemplos de arquivos ausentes:")

    print(
        df_validacao.loc[
            ~df_validacao["arquivo_existe"],
            ["dataset", "identidade", "caminho_local"]
        ].head(10).to_string(index=False)
    )

print(
    "\nTempo total:",
    round(
        perf_counter() - inicio,
        2
    ),
    "segundos"
)

print("\nBloco [48] concluído.")


In [ ]:

# ============================================================
# Bloco [49] — Corrigir caminhos e validar dataset
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 60)
print("VALIDAÇÃO DOS CAMINHOS — FOLD 1")
print("=" * 60)

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_DATASET = Path(
    "/content/tcc1_kfold/datasets"
)

SPLITS = RAIZ / "splits"

# ------------------------------------------------------------
# 1. Carregar divisões originais
# ------------------------------------------------------------

df_validacao = pd.read_csv(
    SPLITS / "fold_1_imagens_validacao.csv"
)

df_treino = pd.read_csv(
    SPLITS / "fold_1_imagens_treino.csv"
)

df_teste = pd.read_csv(
    SPLITS / "identidades_teste_30.csv"
)

# ------------------------------------------------------------
# 2. Reconstruir caminhos locais
# ------------------------------------------------------------

def caminho_local(linha):

    dataset = linha["dataset"]
    original = str(linha["caminho"])

    if dataset == "DogFaceNet":

        marcador = (
            "DogFaceNet-master/after_4_bis/"
        )

        assert marcador in original, (
            f"Caminho inesperado: {original}"
        )

        relativo = original.split(
            marcador, 1
        )[1]

        return str(
            DIR_DATASET / "DogFaceNet" / relativo
        )

    elif dataset == "YTBB":

        # O dataset YTBB foi reorganizado
        # durante a preparação do treinamento.
        #
        # Preservar apenas:
        # train/identidade/imagem
        # test/identidade/imagem

        partes = Path(original).parts

        candidatos = [
            i for i, parte in enumerate(partes)
            if parte in ("train", "test")
        ]

        assert candidatos, (
            f"Divisão train/test não encontrada: {original}"
        )

        indice = candidatos[-1]

        relativo = Path(
            *partes[indice:]
        )

        return str(
            DIR_DATASET / "YTBB" / relativo
        )

    else:

        raise ValueError(
            f"Dataset desconhecido: {dataset}"
        )

for df in [df_treino, df_validacao]:

    df["caminho_local"] = df.apply(
        caminho_local,
        axis=1
    )

# ------------------------------------------------------------
# 3. Verificar arquivos de validação
# ------------------------------------------------------------

df_validacao["existe"] = (
    df_validacao["caminho_local"].apply(
        lambda p: Path(p).is_file()
    )
)

df_validacao["tamanho"] = (
    df_validacao["caminho_local"].apply(
        lambda p: (
            Path(p).stat().st_size
            if Path(p).is_file()
            else 0
        )
    )
)

encontradas = int(
    df_validacao["existe"].sum()
)

vazias = int(
    (
        df_validacao["existe"]
        & (df_validacao["tamanho"] == 0)
    ).sum()
)

print("\nARQUIVOS DE VALIDAÇÃO")
print("Esperados:", len(df_validacao))
print("Encontrados:", encontradas)
print("Ausentes:", len(df_validacao) - encontradas)
print("Arquivos vazios:", vazias)

print("\nPOR DATASET")

print(
    df_validacao.groupby("dataset").agg(
        imagens=("caminho_local", "count"),
        encontradas=("existe", "sum")
    )
)

# ------------------------------------------------------------
# 4. Verificar separação por identidade
# ------------------------------------------------------------

ids_treino = set(
    df_treino["identidade"]
)

ids_validacao = set(
    df_validacao["identidade"]
)

ids_teste = set(
    df_teste["identidade"]
)

intersecao_tv = (
    ids_treino & ids_validacao
)

intersecao_tt = (
    ids_treino & ids_teste
)

intersecao_vt = (
    ids_validacao & ids_teste
)

print("\nSEPARAÇÃO POR IDENTIDADE")
print(
    "Treino x validação:",
    len(intersecao_tv)
)

print(
    "Treino x teste:",
    len(intersecao_tt)
)

print(
    "Validação x teste:",
    len(intersecao_vt)
)

# ------------------------------------------------------------
# 5. Verificar duplicações
# ------------------------------------------------------------

duplicadas = int(
    df_validacao["caminho_local"].duplicated().sum()
)

print("\nDUPLICAÇÕES")
print("Caminhos duplicados:", duplicadas)

# ------------------------------------------------------------
# 6. Validação final
# ------------------------------------------------------------

assert encontradas == 4848, (
    "Nem todas as imagens foram encontradas."
)

assert vazias == 0, (
    "Existem imagens vazias na validação."
)

assert duplicadas == 0, (
    "Existem caminhos duplicados."
)

assert len(intersecao_tv) == 0
assert len(intersecao_tt) == 0
assert len(intersecao_vt) == 0

print("\n" + "=" * 60)
print("VALIDAÇÃO CONCLUÍDA COM SUCESSO")
print("=" * 60)

print(
    "Imagens prontas para inferência:",
    len(df_validacao)
)

print(
    "Identidades:",
    len(ids_validacao)
)

print("\nBloco [49] concluído.")


In [ ]:

# ============================================================
# Bloco [50] — Extração de embeddings de validação
# GPU: SIM — inferência
# ============================================================

import json
import hashlib
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone

print("=" * 60)
print("EXTRAÇÃO DE EMBEDDINGS — FOLD 1")
print("=" * 60)

# ------------------------------------------------------------
# 1. Configuração
# ------------------------------------------------------------

DIR_AVALIACAO = (
    RAIZ / "resultados/fold_1/avaliacao"
)

DIR_AVALIACAO.mkdir(
    parents=True,
    exist_ok=True
)

ARQUIVO_EMBEDDINGS = (
    DIR_AVALIACAO / "embeddings_validacao.npy"
)

ARQUIVO_METADATA = (
    DIR_AVALIACAO / "metadados_validacao.csv"
)

ARQUIVO_CONFIG_AVALIACAO = (
    DIR_AVALIACAO / "configuracao_extracao.json"
)

BATCH_SIZE = 32
TAMANHO_IMAGEM = (224, 224)

assert len(df_validacao) == 4848
assert df_validacao["identidade"].nunique() == 577

assert all(
    Path(p).is_file()
    for p in df_validacao["caminho_local"]
)

assert modelo_embeddings.output_shape[-1] == 128

print("Imagens:", len(df_validacao))
print("Identidades:", df_validacao["identidade"].nunique())
print("Batch size:", BATCH_SIZE)

print(
    "GPU:",
    tf.config.list_physical_devices("GPU")
)

# ------------------------------------------------------------
# 2. Preparar carregamento das imagens
# ------------------------------------------------------------

def carregar_imagem_avaliacao(caminho):

    conteudo = tf.io.read_file(caminho)

    imagem = tf.io.decode_image(
        conteudo,
        channels=3,
        expand_animations=False
    )

    imagem.set_shape([None, None, 3])

    imagem = tf.image.resize(
        imagem,
        TAMANHO_IMAGEM
    )

    imagem = tf.cast(
        imagem,
        tf.float32
    )

    imagem = (
        tf.keras.applications.vgg16.preprocess_input(
            imagem
        )
    )

    return imagem

caminhos = (
    df_validacao["caminho_local"]
    .astype(str)
    .tolist()
)

dataset = tf.data.Dataset.from_tensor_slices(
    caminhos
)

dataset = dataset.map(
    carregar_imagem_avaliacao,
    num_parallel_calls=tf.data.AUTOTUNE
)

dataset = dataset.batch(
    BATCH_SIZE
)

dataset = dataset.prefetch(
    tf.data.AUTOTUNE
)

# ------------------------------------------------------------
# 3. Extrair embeddings
# ------------------------------------------------------------

print("\nIniciando inferência...")

inicio = perf_counter()

embeddings_lotes = []

total_lotes = int(
    np.ceil(len(caminhos) / BATCH_SIZE)
)

for indice, imagens in enumerate(dataset, start=1):

    vetores = modelo_embeddings(
        imagens,
        training=False
    )

    vetores = vetores.numpy().astype(
        np.float32
    )

    embeddings_lotes.append(vetores)

    if (
        indice % 25 == 0
        or indice == total_lotes
    ):

        print(
            f"Lotes processados: "
            f"{indice}/{total_lotes}"
        )

embeddings_validacao = np.concatenate(
    embeddings_lotes,
    axis=0
)

tempo_total = (
    perf_counter() - inicio
)

# ------------------------------------------------------------
# 4. Validar embeddings
# ------------------------------------------------------------

assert embeddings_validacao.shape == (
    4848, 128
)

assert np.all(
    np.isfinite(embeddings_validacao)
), "Embeddings inválidos."

normas = np.linalg.norm(
    embeddings_validacao,
    axis=1
)

assert np.allclose(
    normas,
    1.0,
    atol=1e-4
), "Normas L2 inconsistentes."

print("\nVALIDAÇÃO DOS EMBEDDINGS")

print(
    "Dimensão:",
    embeddings_validacao.shape
)

print(
    "Norma mínima:",
    float(normas.min())
)

print(
    "Norma máxima:",
    float(normas.max())
)

# ------------------------------------------------------------
# 5. Salvar no Google Drive
# ------------------------------------------------------------

np.save(
    ARQUIVO_EMBEDDINGS,
    embeddings_validacao
)

metadata = df_validacao[
    [
        "dataset",
        "identidade",
        "identidade_local",
        "divisao_original",
        "caminho",
        "caminho_local",
        "particao"
    ]
].copy()

metadata.insert(
    0,
    "indice_embedding",
    np.arange(len(metadata))
)

metadata.to_csv(
    ARQUIVO_METADATA,
    index=False
)

configuracao = {
    "experimento": "TCC1_VGG16_Triplet_KFold",
    "fold": 1,
    "data_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "quantidade_imagens": len(metadata),
    "quantidade_identidades": int(
        metadata["identidade"].nunique()
    ),
    "dimensao_embedding": 128,
    "batch_size": BATCH_SIZE,
    "tamanho_imagem": [224, 224],
    "preprocessamento": (
        "tf.keras.applications.vgg16.preprocess_input"
    ),
    "modelo_pesos": str(
        ARQUIVO_MODELO_FINAL_FOLD1
    ),
    "modelo_sha256": hash_atual
}

with open(
    ARQUIVO_CONFIG_AVALIACAO,
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        configuracao,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 6. Verificar arquivos persistidos
# ------------------------------------------------------------

embeddings_recarregados = np.load(
    ARQUIVO_EMBEDDINGS,
    mmap_mode="r"
)

metadata_recarregada = pd.read_csv(
    ARQUIVO_METADATA
)

assert embeddings_recarregados.shape == (
    4848, 128
)

assert len(metadata_recarregada) == 4848

assert (
    metadata_recarregada["identidade"].nunique()
    == 577
)

print("\n" + "=" * 60)
print("EXTRAÇÃO CONCLUÍDA")
print("=" * 60)

print(
    "Tempo:",
    round(tempo_total, 2),
    "segundos"
)

print(
    "Imagens por segundo:",
    round(
        len(metadata) / tempo_total,
        2
    )
)

print("Embeddings:", ARQUIVO_EMBEDDINGS)
print("Metadados:", ARQUIVO_METADATA)
print("Configuração:", ARQUIVO_CONFIG_AVALIACAO)

print("\nBloco [50] concluído.")


In [ ]:

# ============================================================
# Bloco [51] — Auditoria do pré-processamento
# GPU: NÃO NECESSÁRIA
# ============================================================

import json
import numpy as np
import tensorflow as tf
from pathlib import Path

print("=" * 60)
print("AUDITORIA DO PRÉ-PROCESSAMENTO — FOLD 1")
print("=" * 60)

# ------------------------------------------------------------
# 1. Inspecionar configurações persistidas
# ------------------------------------------------------------

arquivos_configuracao = [
    RAIZ / "config.json",
    RAIZ / "resultados/fold_1/configuracao.json",
    DIR_AVALIACAO / "configuracao_extracao.json"
]

for caminho in arquivos_configuracao:

    print("\nARQUIVO:", caminho.name)

    if caminho.is_file():

        with open(
            caminho,
            encoding="utf-8"
        ) as arquivo:
            dados = json.load(arquivo)

        print(
            json.dumps(
                dados,
                indent=2,
                ensure_ascii=False
            )[:6000]
        )

    else:
        print("Não encontrado.")

# ------------------------------------------------------------
# 2. Localizar possíveis registros do pipeline
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("ARQUIVOS DE CÓDIGO DO EXPERIMENTO")
print("=" * 60)

extensoes_codigo = {".py", ".ipynb"}

arquivos_codigo = []

for pasta in [
    Path("/content/tcc1_kfold"),
    RAIZ
]:

    if pasta.exists():

        for caminho in pasta.rglob("*"):

            if (
                caminho.is_file()
                and caminho.suffix.lower()
                in extensoes_codigo
            ):
                arquivos_codigo.append(caminho)

for caminho in sorted(set(arquivos_codigo))[:40]:
    print(caminho)

print(
    "Total:",
    len(set(arquivos_codigo))
)

# ------------------------------------------------------------
# 3. Comparar transformações em uma imagem real
# ------------------------------------------------------------

caminho_teste = (
    df_validacao.iloc[0]["caminho_local"]
)

conteudo = tf.io.read_file(caminho_teste)

imagem = tf.io.decode_image(
    conteudo,
    channels=3,
    expand_animations=False
)

imagem = tf.image.resize(
    imagem,
    (224, 224)
)

imagem = tf.cast(
    imagem,
    tf.float32
)

# Opção A: escala [0, 1]
normalizacao_01 = imagem / 255.0

# Opção B: VGG16 padrão
normalizacao_vgg16 = (
    tf.keras.applications.vgg16.preprocess_input(
        tf.identity(imagem)
    )
)

# Opção C: escala [-1, 1]
normalizacao_m11 = (
    imagem / 127.5 - 1.0
)

print("\n" + "=" * 60)
print("COMPARAÇÃO DAS TRANSFORMAÇÕES")
print("=" * 60)

transformacoes = {
    "Pixels RGB [0,255]": imagem,
    "Normalização [0,1]": normalizacao_01,
    "VGG16 preprocess_input": normalizacao_vgg16,
    "Normalização [-1,1]": normalizacao_m11
}

for nome, tensor in transformacoes.items():

    valores = tensor.numpy()

    print(
        f"{nome}: "
        f"mín={valores.min():.4f}, "
        f"máx={valores.max():.4f}, "
        f"média={valores.mean():.4f}"
    )

print("\nBloco [51] concluído.")


In [ ]:

# ============================================================
# Bloco [52] — Diagnóstico de pré-processamento
# GPU: SIM — inferência, sem treinamento
# ============================================================

import json
import numpy as np
import tensorflow as tf
from pathlib import Path
from time import perf_counter

print("=" * 60)
print("DIAGNÓSTICO DE PRÉ-PROCESSAMENTO — FOLD 1")
print("=" * 60)

ARQUIVO_LOTES = (
    RAIZ / "resultados/fold_1/lotes_validacao.json"
)

with open(
    ARQUIVO_LOTES,
    encoding="utf-8"
) as arquivo:
    lotes = json.load(arquivo)

assert len(lotes) == 150

# ------------------------------------------------------------
# 1. Reconstruir caminhos locais dos lotes
# ------------------------------------------------------------

PREFIXO_LOCAL = (
    "/content/tcc1_kfold/datasets/"
)

def resolver_caminho_lote(caminho):

    caminho = str(caminho)

    assert caminho.startswith(PREFIXO_LOCAL), (
        f"Caminho inesperado: {caminho}"
    )

    assert Path(caminho).is_file(), (
        f"Imagem ausente: {caminho}"
    )

    return caminho

# ------------------------------------------------------------
# 2. Carregar e preparar imagens
# ------------------------------------------------------------

def carregar_lote(caminhos, metodo):

    imagens = []

    for caminho in caminhos:

        conteudo = tf.io.read_file(caminho)

        imagem = tf.io.decode_image(
            conteudo,
            channels=3,
            expand_animations=False
        )

        imagem.set_shape([None, None, 3])

        imagem = tf.image.resize(
            imagem,
            (224, 224)
        )

        imagem = tf.cast(
            imagem,
            tf.float32
        )

        if metodo == "vgg16":

            imagem = (
                tf.keras.applications.vgg16.preprocess_input(
                    imagem
                )
            )

        elif metodo == "zero_um":

            imagem = imagem / 255.0

        elif metodo == "menos_um_um":

            imagem = imagem / 127.5 - 1.0

        else:
            raise ValueError(metodo)

        imagens.append(imagem)

    return tf.stack(imagens)

# ------------------------------------------------------------
# 3. Triplet Loss com mineração semi-hard
# ------------------------------------------------------------

def avaliar_triplet(embeddings, rotulos, margem=0.2):

    embeddings = tf.cast(
        embeddings,
        tf.float32
    )

    rotulos = tf.convert_to_tensor(rotulos)

    # Distância euclidiana quadrática
    produto = tf.matmul(
        embeddings,
        embeddings,
        transpose_b=True
    )

    quadrados = tf.reduce_sum(
        tf.square(embeddings),
        axis=1,
        keepdims=True
    )

    distancias = tf.maximum(
        quadrados
        - 2.0 * produto
        + tf.transpose(quadrados),
        0.0
    )

    rotulos_iguais = tf.equal(
        rotulos[:, None],
        rotulos[None, :]
    )

    n = tf.shape(rotulos)[0]

    diagonal = tf.eye(
        n,
        dtype=tf.bool
    )

    positivos = (
        rotulos_iguais & ~diagonal
    )

    negativos = ~rotulos_iguais

    perdas = []

    for ancora in range(int(n.numpy())):

        indices_positivos = tf.where(
            positivos[ancora]
        )[:, 0].numpy()

        indices_negativos = tf.where(
            negativos[ancora]
        )[:, 0].numpy()

        dist_negativas = tf.gather(
            distancias[ancora],
            indices_negativos
        ).numpy()

        for positivo in indices_positivos:

            d_ap = float(
                distancias[
                    ancora,
                    positivo
                ].numpy()
            )

            # Semi-hard: d_ap < d_an < d_ap + margem
            mascara_semihard = (
                (dist_negativas > d_ap)
                & (
                    dist_negativas
                    < d_ap + margem
                )
            )

            if mascara_semihard.any():

                d_an = float(
                    dist_negativas[
                        mascara_semihard
                    ].min()
                )

            else:

                # Fallback: negativo mais próximo
                d_an = float(
                    dist_negativas.min()
                )

            perdas.append(
                max(
                    d_ap - d_an + margem,
                    0.0
                )
            )

    return float(
        np.mean(perdas)
    )

# ------------------------------------------------------------
# 4. Comparar métodos
# ------------------------------------------------------------

metodos = [
    "vgg16",
    "zero_um",
    "menos_um_um"
]

resultados = {}

inicio = perf_counter()

for metodo in metodos:

    print(f"\nAvaliando: {metodo}")

    perdas = []

    for indice, lote in enumerate(lotes, start=1):

        caminhos = [
            resolver_caminho_lote(p)
            for p in lote["caminhos"]
        ]

        imagens = carregar_lote(
            caminhos,
            metodo
        )

        embeddings = modelo_embeddings(
            imagens,
            training=False
        )

        loss = avaliar_triplet(
            embeddings,
            lote["rotulos"],
            margem=0.2
        )

        perdas.append(loss)

        if indice % 50 == 0:
            print(
                f"  Lotes: {indice}/{len(lotes)}"
            )

    resultados[metodo] = float(
        np.mean(perdas)
    )

# ------------------------------------------------------------
# 5. Apresentar comparação
# ------------------------------------------------------------

loss_referencia = 0.0133142266312885

print("\n" + "=" * 60)
print("RESULTADOS COMPARATIVOS")
print("=" * 60)

print(
    "Loss original de validação:",
    f"{loss_referencia:.8f}"
)

for metodo, loss in resultados.items():

    diferenca = abs(
        loss - loss_referencia
    )

    print(
        f"{metodo:15s} "
        f"| loss={loss:.8f} "
        f"| diferença={diferenca:.8f}"
    )

print(
    "\nTempo total:",
    round(
        perf_counter() - inicio,
        2
    ),
    "segundos"
)

print("\nBloco [52] concluído.")


In [ ]:

# ============================================================
# Bloco [53] — Avaliação de verificação por pares
# GPU: NÃO NECESSÁRIA
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from datetime import datetime, timezone

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 60)
print("AVALIAÇÃO DE VERIFICAÇÃO — FOLD 1")
print("=" * 60)

SEED_AVALIACAO = 42
rng = np.random.default_rng(SEED_AVALIACAO)

DIR_AVALIACAO = (
    RAIZ / "resultados/fold_1/avaliacao"
)

# ------------------------------------------------------------
# 1. Carregar embeddings e metadados
# ------------------------------------------------------------

embeddings = np.load(
    DIR_AVALIACAO / "embeddings_validacao.npy"
)

metadata = pd.read_csv(
    DIR_AVALIACAO / "metadados_validacao.csv"
)

assert embeddings.shape == (4848, 128)
assert len(metadata) == 4848

assert np.array_equal(
    metadata["indice_embedding"].to_numpy(),
    np.arange(len(metadata))
)

identidades = (
    metadata["identidade"]
    .astype(str)
    .to_numpy()
)

print("Imagens:", len(embeddings))
print("Identidades:", len(np.unique(identidades)))

# ------------------------------------------------------------
# 2. Construir todos os pares positivos
# ------------------------------------------------------------

grupos = metadata.groupby(
    "identidade"
).indices

pares_positivos = []

for indices in grupos.values():

    if len(indices) < 2:
        continue

    pares_positivos.extend(
        combinations(
            indices.tolist(),
            2
        )
    )

pares_positivos = np.asarray(
    pares_positivos,
    dtype=np.int32
).reshape(-1, 2)

n_positivos = len(pares_positivos)

print("\nPares positivos:", n_positivos)

assert n_positivos > 0

# ------------------------------------------------------------
# 3. Amostrar pares negativos únicos
# ------------------------------------------------------------

# Amostragem sem reposição a partir de todos os pares
# possíveis entre identidades distintas.

n_imagens = len(identidades)

indices_superiores = np.triu_indices(
    n_imagens,
    k=1
)

mascara_negativos = (
    identidades[indices_superiores[0]]
    != identidades[indices_superiores[1]]
)

posicoes_negativas = np.flatnonzero(
    mascara_negativos
)

assert len(posicoes_negativas) >= n_positivos

selecionados = rng.choice(
    posicoes_negativas,
    size=n_positivos,
    replace=False
)

pares_negativos = np.column_stack([
    indices_superiores[0][selecionados],
    indices_superiores[1][selecionados]
]).astype(np.int32)

print("Pares negativos:", len(pares_negativos))

# ------------------------------------------------------------
# 4. Calcular distâncias
# ------------------------------------------------------------

def calcular_distancias(pares):

    diferencas = (
        embeddings[pares[:, 0]]
        - embeddings[pares[:, 1]]
    )

    return np.linalg.norm(
        diferencas,
        axis=1
    )

dist_positivos = calcular_distancias(
    pares_positivos
)

dist_negativos = calcular_distancias(
    pares_negativos
)

y_true = np.concatenate([
    np.ones(n_positivos, dtype=np.int8),
    np.zeros(n_positivos, dtype=np.int8)
])

distancias = np.concatenate([
    dist_positivos,
    dist_negativos
])

# Distância menor significa maior similaridade.
scores = -distancias

# ------------------------------------------------------------
# 5. ROC-AUC e EER
# ------------------------------------------------------------

auc = roc_auc_score(
    y_true,
    scores
)

fpr, tpr, thresholds = roc_curve(
    y_true,
    scores
)

fnr = 1.0 - tpr

indice_eer = int(
    np.argmin(
        np.abs(fpr - fnr)
    )
)

eer = float(
    (fpr[indice_eer] + fnr[indice_eer])
    / 2.0
)

limiar_distancia = float(
    -thresholds[indice_eer]
)

# ------------------------------------------------------------
# 6. Métricas no limiar EER
# ------------------------------------------------------------

y_pred = (
    distancias <= limiar_distancia
).astype(np.int8)

acuracia = accuracy_score(
    y_true,
    y_pred
)

precisao = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

matriz = confusion_matrix(
    y_true,
    y_pred
)

# ------------------------------------------------------------
# 7. Salvar pares e resultados
# ------------------------------------------------------------

pares = np.vstack([
    pares_positivos,
    pares_negativos
])

df_pares = pd.DataFrame({
    "indice_a": pares[:, 0],
    "indice_b": pares[:, 1],
    "mesmo_cao": y_true,
    "distancia_euclidiana": distancias
})

ARQUIVO_PARES = (
    DIR_AVALIACAO / "pares_verificacao.csv"
)

df_pares.to_csv(
    ARQUIVO_PARES,
    index=False
)

resultados = {
    "experimento": "TCC1_VGG16_Triplet_KFold",
    "fold": 1,
    "data_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "seed_avaliacao": SEED_AVALIACAO,
    "imagens": len(embeddings),
    "identidades": len(np.unique(identidades)),
    "pares_positivos": int(n_positivos),
    "pares_negativos": int(len(pares_negativos)),
    "roc_auc": float(auc),
    "eer_aproximado": eer,
    "limiar_distancia_eer": limiar_distancia,
    "acuracia": float(acuracia),
    "precisao": float(precisao),
    "recall": float(recall),
    "f1_score": float(f1),
    "matriz_confusao": matriz.tolist(),
    "observacao": (
        "Métricas de desenvolvimento no fold 1. "
        "Pares negativos amostrados sem reposição. "
        "Limiar selecionado no próprio conjunto de validação."
    )
}

ARQUIVO_RESULTADOS = (
    DIR_AVALIACAO / "metricas_verificacao.json"
)

with open(
    ARQUIVO_RESULTADOS,
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        resultados,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 8. Apresentar resultados
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RESULTADOS — FOLD 1")
print("=" * 60)

print(f"ROC-AUC:       {auc:.6f}")
print(f"EER aprox.:    {eer:.6f}")
print(f"Limiar EER:    {limiar_distancia:.6f}")
print(f"Acurácia:      {acuracia:.6f}")
print(f"Precisão:      {precisao:.6f}")
print(f"Recall:        {recall:.6f}")
print(f"F1-score:      {f1:.6f}")

print("\nMatriz de confusão:")
print(matriz)

print("\nDISTÂNCIAS")
print(
    "Mesmo cão — média:",
    round(float(dist_positivos.mean()), 6)
)
print(
    "Cães diferentes — média:",
    round(float(dist_negativos.mean()), 6)
)

print("\nArquivos salvos:")
print(ARQUIVO_PARES)
print(ARQUIVO_RESULTADOS)

print("\nBloco [53] concluído.")


In [ ]:

# ============================================================
# Bloco [54] — Avaliação de recuperação de imagens
# GPU: NÃO NECESSÁRIA
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

print("=" * 60)
print("AVALIAÇÃO DE RECUPERAÇÃO — FOLD 1")
print("=" * 60)

DIR_AVALIACAO = (
    RAIZ / "resultados/fold_1/avaliacao"
)

embeddings = np.load(
    DIR_AVALIACAO / "embeddings_validacao.npy"
).astype(np.float32)

metadata = pd.read_csv(
    DIR_AVALIACAO / "metadados_validacao.csv"
)

assert embeddings.shape == (4848, 128)
assert len(metadata) == len(embeddings)

assert np.array_equal(
    metadata["indice_embedding"].to_numpy(),
    np.arange(len(metadata))
)

identidades = (
    metadata["identidade"]
    .astype(str)
    .to_numpy()
)

n = len(embeddings)

# ------------------------------------------------------------
# 1. Distâncias euclidianas quadráticas
# ------------------------------------------------------------

# Embeddings L2-normalizados:
# ||a-b||² = 2 - 2(a·b)

similaridades = embeddings @ embeddings.T

distancias = np.maximum(
    2.0 - 2.0 * similaridades,
    0.0
)

# Excluir a própria imagem da busca.
np.fill_diagonal(
    distancias,
    np.inf
)

# ------------------------------------------------------------
# 2. Ordenar candidatos
# ------------------------------------------------------------

ranking = np.argsort(
    distancias,
    axis=1
)

identidades_ranking = (
    identidades[ranking]
)

relevantes = (
    identidades_ranking
    == identidades[:, None]
)

# Cada consulta precisa ter ao menos uma
# outra fotografia da mesma identidade.

total_relevantes = relevantes.sum(
    axis=1
)

consultas_validas = (
    total_relevantes > 0
)

assert consultas_validas.any()

print("Imagens:", n)
print(
    "Consultas válidas:",
    int(consultas_validas.sum())
)

print(
    "Consultas sem outra imagem do mesmo cão:",
    int((~consultas_validas).sum())
)

# ------------------------------------------------------------
# 3. Hit@K e Recall@K tradicional
# ------------------------------------------------------------

valores_k = [1, 5, 10]

resultados_k = {}

for k in valores_k:

    acertos = relevantes[:, :k].sum(
        axis=1
    )

    hit_k = np.mean(
        (acertos > 0)[consultas_validas]
    )

    recall_k = np.mean(
        (
            acertos[consultas_validas]
            / total_relevantes[consultas_validas]
        )
    )

    resultados_k[f"hit_at_{k}"] = float(
        hit_k
    )

    resultados_k[f"recall_at_{k}"] = float(
        recall_k
    )

# ------------------------------------------------------------
# 4. Average Precision e mAP
# ------------------------------------------------------------

posicoes = np.arange(
    1,
    n + 1,
    dtype=np.float32
)

acertos_acumulados = np.cumsum(
    relevantes,
    axis=1
)

precisao_por_posicao = (
    acertos_acumulados
    / posicoes[None, :]
)

average_precision = (
    (
        precisao_por_posicao
        * relevantes
    ).sum(axis=1)
    / np.maximum(total_relevantes, 1)
)

map_score = float(
    average_precision[consultas_validas].mean()
)

# ------------------------------------------------------------
# 5. Mean Reciprocal Rank
# ------------------------------------------------------------

primeira_posicao = (
    np.argmax(relevantes, axis=1) + 1
)

reciprocal_rank = np.zeros(
    n,
    dtype=np.float32
)

reciprocal_rank[consultas_validas] = (
    1.0
    / primeira_posicao[consultas_validas]
)

mrr = float(
    reciprocal_rank[consultas_validas].mean()
)

# ------------------------------------------------------------
# 6. Salvar métricas por consulta
# ------------------------------------------------------------

df_consultas = metadata[
    [
        "indice_embedding",
        "dataset",
        "identidade",
        "caminho_local"
    ]
].copy()

df_consultas["total_relevantes"] = (
    total_relevantes
)

df_consultas["consulta_valida"] = (
    consultas_validas
)

df_consultas["average_precision"] = (
    average_precision
)

df_consultas["primeira_posicao_relevante"] = (
    np.where(
        consultas_validas,
        primeira_posicao,
        np.nan
    )
)

for k in valores_k:

    df_consultas[f"hit_at_{k}"] = (
        relevantes[:, :k].any(axis=1)
    ).astype(int)

ARQUIVO_CONSULTAS = (
    DIR_AVALIACAO / "metricas_por_consulta.csv"
)

df_consultas.to_csv(
    ARQUIVO_CONSULTAS,
    index=False
)

# ------------------------------------------------------------
# 7. Salvar resultados agregados
# ------------------------------------------------------------

resultados = {
    "experimento": "TCC1_VGG16_Triplet_KFold",
    "fold": 1,
    "data_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "protocolo": (
        "image_to_image_leave_one_out"
    ),
    "imagens": n,
    "identidades": int(
        len(np.unique(identidades))
    ),
    "consultas_validas": int(
        consultas_validas.sum()
    ),
    "map": map_score,
    "mrr": mrr,
    **resultados_k
}

ARQUIVO_RESULTADOS = (
    DIR_AVALIACAO / "metricas_recuperacao.json"
)

with open(
    ARQUIVO_RESULTADOS,
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        resultados,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 8. Apresentar resultados
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RESULTADOS DE RECUPERAÇÃO — FOLD 1")
print("=" * 60)

for k in valores_k:

    print(
        f"Hit@{k}: "
        f"{resultados_k[f'hit_at_{k}']:.4%}"
    )

    print(
        f"Recall@{k} tradicional: "
        f"{resultados_k[f'recall_at_{k}']:.4%}"
    )

print(f"\nmAP: {map_score:.4%}")
print(f"MRR: {mrr:.4%}")

print("\nArquivos salvos:")
print(ARQUIVO_CONSULTAS)
print(ARQUIVO_RESULTADOS)

print("\nBloco [54] concluído.")


In [ ]:

# ============================================================
# Bloco [55] — Análise de recuperação por dataset
# GPU: NÃO NECESSÁRIA
# ============================================================

import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone

print("=" * 60)
print("ANÁLISE DE RECUPERAÇÃO POR DATASET — FOLD 1")
print("=" * 60)

DIR_AVALIACAO = (
    RAIZ / "resultados/fold_1/avaliacao"
)

ARQUIVO_CONSULTAS = (
    DIR_AVALIACAO / "metricas_por_consulta.csv"
)

df_consultas = pd.read_csv(
    ARQUIVO_CONSULTAS
)

assert len(df_consultas) == 4848

assert df_consultas["consulta_valida"].all()

# ------------------------------------------------------------
# 1. Calcular MRR individual
# ------------------------------------------------------------

df_consultas["reciprocal_rank"] = (
    1.0
    / df_consultas["primeira_posicao_relevante"]
)

# ------------------------------------------------------------
# 2. Agregar por dataset
# ------------------------------------------------------------

resumo = (
    df_consultas.groupby("dataset")
    .agg(
        imagens=("indice_embedding", "count"),
        identidades=("identidade", "nunique"),
        hit_1=("hit_at_1", "mean"),
        hit_5=("hit_at_5", "mean"),
        hit_10=("hit_at_10", "mean"),
        map=("average_precision", "mean"),
        mrr=("reciprocal_rank", "mean")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 3. Analisar casos difíceis
# ------------------------------------------------------------

casos_dificeis = (
    df_consultas.sort_values(
        [
            "average_precision",
            "primeira_posicao_relevante"
        ],
        ascending=[True, False]
    )
    .head(30)
    .copy()
)

ARQUIVO_CASOS_DIFICEIS = (
    DIR_AVALIACAO / "casos_dificeis_fold1.csv"
)

casos_dificeis.to_csv(
    ARQUIVO_CASOS_DIFICEIS,
    index=False
)

# ------------------------------------------------------------
# 4. Persistir resumo
# ------------------------------------------------------------

ARQUIVO_RESUMO = (
    DIR_AVALIACAO / "metricas_por_dataset.csv"
)

resumo.to_csv(
    ARQUIVO_RESUMO,
    index=False
)

resultado_json = {
    "experimento": "TCC1_VGG16_Triplet_KFold",
    "fold": 1,
    "data_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "protocolo": "image_to_image_leave_one_out",
    "galeria": "global_4848_imagens",
    "resultados": resumo.to_dict(
        orient="records"
    )
}

with open(
    DIR_AVALIACAO / "metricas_por_dataset.json",
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        resultado_json,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 5. Apresentar resultados
# ------------------------------------------------------------

print("\nRESULTADOS POR DATASET\n")

for _, linha in resumo.iterrows():

    print("-" * 50)
    print("Dataset:", linha["dataset"])
    print("Imagens:", int(linha["imagens"]))
    print("Identidades:", int(linha["identidades"]))

    print(f"Hit@1:  {linha['hit_1']:.4%}")
    print(f"Hit@5:  {linha['hit_5']:.4%}")
    print(f"Hit@10: {linha['hit_10']:.4%}")
    print(f"mAP:    {linha['map']:.4%}")
    print(f"MRR:    {linha['mrr']:.4%}")

print("\n" + "=" * 60)
print("CASOS MAIS DIFÍCEIS")
print("=" * 60)

print(
    casos_dificeis[
        [
            "dataset",
            "identidade",
            "average_precision",
            "primeira_posicao_relevante"
        ]
    ].head(15).to_string(index=False)
)

print("\nArquivos salvos:")
print(ARQUIVO_RESUMO)
print(ARQUIVO_CASOS_DIFICEIS)

print("\nBloco [55] concluído.")


In [ ]:

# ============================================================
# Bloco [56] — Inspeção visual dos casos difíceis
# GPU: NÃO NECESSÁRIA
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from pathlib import Path

print("=" * 60)
print("INSPEÇÃO VISUAL DOS CASOS DIFÍCEIS — FOLD 1")
print("=" * 60)

DIR_AVALIACAO = (
    RAIZ / "resultados/fold_1/avaliacao"
)

embeddings = np.load(
    DIR_AVALIACAO / "embeddings_validacao.npy"
).astype(np.float32)

metadata = pd.read_csv(
    DIR_AVALIACAO / "metadados_validacao.csv"
)

casos = pd.read_csv(
    DIR_AVALIACAO / "casos_dificeis_fold1.csv"
)

assert len(embeddings) == len(metadata)
assert len(casos) >= 6

assert np.array_equal(
    metadata["indice_embedding"].to_numpy(),
    np.arange(len(metadata))
)

identidades = (
    metadata["identidade"].astype(str).to_numpy()
)

# ------------------------------------------------------------
# 1. Selecionar seis consultas difíceis
# ------------------------------------------------------------

selecionados = casos.head(6)

# ------------------------------------------------------------
# 2. Função de carregamento para visualização
# ------------------------------------------------------------

def abrir_imagem(caminho):

    with Image.open(caminho) as imagem:
        imagem = ImageOps.exif_transpose(imagem)
        return imagem.convert("RGB").copy()

# ------------------------------------------------------------
# 3. Encontrar resultados do ranking
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=6,
    ncols=3,
    figsize=(12, 22)
)

for linha, (_, caso) in enumerate(
    selecionados.iterrows()
):

    indice = int(
        caso["indice_embedding"]
    )

    consulta = embeddings[indice]

    # Distância euclidiana quadrática
    distancias = np.maximum(
        2.0 - 2.0 * (embeddings @ consulta),
        0.0
    )

    distancias[indice] = np.inf

    ranking = np.argsort(
        distancias
    )

    indice_top1 = int(
        ranking[0]
    )

    mascara_corretos = (
        identidades[ranking]
        == identidades[indice]
    )

    assert mascara_corretos.any()

    posicao_correta = int(
        np.flatnonzero(
            mascara_corretos
        )[0]
    )

    indice_correto = int(
        ranking[posicao_correta]
    )

    indices_exibir = [
        indice,
        indice_top1,
        indice_correto
    ]

    titulos = [
        "Consulta",
        "Primeiro resultado",
        f"Primeiro correto (posição {posicao_correta + 1})"
    ]

    for coluna, indice_imagem in enumerate(
        indices_exibir
    ):

        ax = axes[linha, coluna]

        imagem = abrir_imagem(
            metadata.iloc[indice_imagem][
                "caminho_local"
            ]
        )

        ax.imshow(imagem)
        ax.set_xticks([])
        ax.set_yticks([])

        correto = (
            identidades[indice_imagem]
            == identidades[indice]
        )

        cor = (
            "green"
            if correto
            else "red"
        )

        for borda in ax.spines.values():
            borda.set_edgecolor(cor)
            borda.set_linewidth(3)

        ax.set_title(
            titulos[coluna],
            fontsize=10
        )

    axes[linha, 0].set_ylabel(
        f"{identidades[indice]}\n"
        f"AP={caso['average_precision']:.4f}",
        fontsize=9
    )

plt.suptitle(
    "Fold 1 — Consultas com menor Average Precision",
    fontsize=15
)

plt.tight_layout(
    rect=[0, 0, 1, 0.98]
)

ARQUIVO_FIGURA = (
    DIR_AVALIACAO / "casos_dificeis_visual.png"
)

plt.savefig(
    ARQUIVO_FIGURA,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

print("\nFigura salva em:")
print(ARQUIVO_FIGURA)

print("\nBloco [56] concluído.")


In [ ]:

# ============================================================
# Bloco [57] — Inspeção dos primeiros resultados incorretos
# GPU: NÃO NECESSÁRIA
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

print("=" * 60)
print("ANÁLISE DOS FALSOS POSITIVOS — FOLD 1")
print("=" * 60)

DIR_AVALIACAO = (
    RAIZ / "resultados/fold_1/avaliacao"
)

embeddings = np.load(
    DIR_AVALIACAO / "embeddings_validacao.npy"
).astype(np.float32)

metadata = pd.read_csv(
    DIR_AVALIACAO / "metadados_validacao.csv"
)

casos = pd.read_csv(
    DIR_AVALIACAO / "casos_dificeis_fold1.csv"
)

assert np.array_equal(
    metadata["indice_embedding"].to_numpy(),
    np.arange(len(metadata))
)

identidades = (
    metadata["identidade"].astype(str).to_numpy()
)

# Selecionar seis consultas difíceis
selecionados = casos.head(6)

def abrir_imagem(caminho):
    with Image.open(caminho) as img:
        return ImageOps.exif_transpose(
            img
        ).convert("RGB").copy()

fig, axes = plt.subplots(
    6, 6,
    figsize=(17, 19)
)

registros = []

for linha, (_, caso) in enumerate(
    selecionados.iterrows()
):

    indice = int(
        caso["indice_embedding"]
    )

    consulta = embeddings[indice]

    distancias = np.maximum(
        2.0 - 2.0 * (
            embeddings @ consulta
        ),
        0.0
    )

    distancias[indice] = np.inf

    ranking = np.argsort(
        distancias
    )[:5]

    indices = [indice] + ranking.tolist()

    for coluna, idx in enumerate(indices):

        ax = axes[linha, coluna]

        img = abrir_imagem(
            metadata.iloc[idx]["caminho_local"]
        )

        ax.imshow(img)
        ax.set_xticks([])
        ax.set_yticks([])

        correto = (
            identidades[idx]
            == identidades[indice]
        )

        cor = "green" if correto else "red"

        for borda in ax.spines.values():
            borda.set_color(cor)
            borda.set_linewidth(3)

        if coluna == 0:
            ax.set_title(
                "Consulta",
                fontsize=11
            )
        else:
            ax.set_title(
                f"Top {coluna}\n"
                f"d={np.sqrt(distancias[idx]):.3f}",
                fontsize=10
            )

            registros.append({
                "consulta": int(indice),
                "posicao": int(coluna),
                "indice_resultado": int(idx),
                "identidade_consulta": identidades[indice],
                "identidade_resultado": identidades[idx],
                "correto": bool(correto),
                "distancia": float(
                    np.sqrt(distancias[idx])
                )
            })

    axes[linha, 0].set_ylabel(
        f"{identidades[indice]}\n"
        f"AP={caso['average_precision']:.4f}",
        fontsize=9
    )

plt.suptitle(
    "Fold 1 — Cinco primeiros resultados das consultas difíceis",
    fontsize=15
)

plt.tight_layout(
    rect=[0, 0, 1, 0.98]
)

ARQUIVO_FIGURA = (
    DIR_AVALIACAO / "falsos_positivos_top5.png"
)

plt.savefig(
    ARQUIVO_FIGURA,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

df_resultados = pd.DataFrame(registros)

ARQUIVO_CSV = (
    DIR_AVALIACAO / "falsos_positivos_top5.csv"
)

df_resultados.to_csv(
    ARQUIVO_CSV,
    index=False
)

print("\nRESUMO")
print(
    "Resultados corretos no Top 5:",
    int(df_resultados["correto"].sum()),
    "de",
    len(df_resultados)
)

print("\nFigura:", ARQUIVO_FIGURA)
print("CSV:", ARQUIVO_CSV)

print("\nBloco [57] concluído.")


In [ ]:

# ============================================================
# Bloco [58] — Auditoria de erros e identidades
# GPU: NÃO NECESSÁRIA
# ============================================================

import numpy as np
import pandas as pd
import json
from datetime import datetime, timezone

print("=" * 60)
print("AUDITORIA DE ERROS — FOLD 1")
print("=" * 60)

DIR_AVALIACAO = (
    RAIZ / "resultados/fold_1/avaliacao"
)

df = pd.read_csv(
    DIR_AVALIACAO / "metricas_por_consulta.csv"
)

assert len(df) == 4848

# ------------------------------------------------------------
# 1. Identificar falhas de Top 1
# ------------------------------------------------------------

df["erro_top1"] = (
    df["hit_at_1"] == 0
)

print("\nERROS DE TOP 1")

print(
    "Total de erros:",
    int(df["erro_top1"].sum())
)

print(
    "Taxa de erro:",
    f"{df['erro_top1'].mean():.4%}"
)

# ------------------------------------------------------------
# 2. Erros por dataset
# ------------------------------------------------------------

resumo_dataset = (
    df.groupby("dataset")
    .agg(
        consultas=("erro_top1", "count"),
        erros=("erro_top1", "sum"),
        taxa_erro=("erro_top1", "mean"),
        map=("average_precision", "mean")
    )
    .reset_index()
)

print("\nERROS POR DATASET")
print(
    resumo_dataset.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 3. Quantidade de imagens por identidade
# ------------------------------------------------------------

quantidade_por_identidade = (
    df.groupby("identidade")
    .size()
    .rename("imagens_identidade")
)

df = df.join(
    quantidade_por_identidade,
    on="identidade"
)

# ------------------------------------------------------------
# 4. Erros conforme tamanho da identidade
# ------------------------------------------------------------

df["faixa_imagens"] = pd.cut(
    df["imagens_identidade"],
    bins=[0, 3, 5, 10, 20, float("inf")],
    labels=[
        "2-3",
        "4-5",
        "6-10",
        "11-20",
        "21+"
    ]
)

resumo_faixas = (
    df.groupby(
        "faixa_imagens",
        observed=True
    )
    .agg(
        consultas=("erro_top1", "count"),
        erros=("erro_top1", "sum"),
        taxa_erro=("erro_top1", "mean"),
        map=("average_precision", "mean")
    )
    .reset_index()
)

print("\nERROS POR QUANTIDADE DE IMAGENS")
print(
    resumo_faixas.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 5. Identidades com maior taxa de erro
# ------------------------------------------------------------

resumo_identidades = (
    df.groupby(
        ["dataset", "identidade"]
    )
    .agg(
        imagens=("erro_top1", "count"),
        erros=("erro_top1", "sum"),
        taxa_erro=("erro_top1", "mean"),
        map=("average_precision", "mean")
    )
    .reset_index()
)

resumo_identidades = (
    resumo_identidades.sort_values(
        ["erros", "taxa_erro", "map"],
        ascending=[False, False, True]
    )
)

print("\n15 IDENTIDADES COM MAIS ERROS")
print(
    resumo_identidades.head(15).to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 6. Salvar auditoria
# ------------------------------------------------------------

df.to_csv(
    DIR_AVALIACAO / "auditoria_consultas.csv",
    index=False
)

resumo_dataset.to_csv(
    DIR_AVALIACAO / "auditoria_erros_dataset.csv",
    index=False
)

resumo_faixas.to_csv(
    DIR_AVALIACAO / "auditoria_erros_faixas.csv",
    index=False
)

resumo_identidades.to_csv(
    DIR_AVALIACAO / "auditoria_erros_identidades.csv",
    index=False
)

print("\n" + "=" * 60)
print("AUDITORIA CONCLUÍDA")
print("=" * 60)

print(
    "Identidades com pelo menos um erro:",
    int(
        (resumo_identidades["erros"] > 0).sum()
    )
)

print(
    "Identidades sem erros:",
    int(
        (resumo_identidades["erros"] == 0).sum()
    )
)

print("\nBloco [58] concluído.")


In [ ]:

# ============================================================
# Bloco [59] — Simulação de cadastro com 1 a 4 fotos
# GPU: NÃO NECESSÁRIA
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 60)
print("SIMULAÇÃO DE CADASTRO — FOLD 1")
print("=" * 60)

DIR_AVALIACAO = (
    RAIZ / "resultados/fold_1/avaliacao"
)

embeddings = np.load(
    DIR_AVALIACAO / "embeddings_validacao.npy"
).astype(np.float32)

metadata = pd.read_csv(
    DIR_AVALIACAO / "metadados_validacao.csv"
)

assert embeddings.shape == (4848, 128)

assert np.array_equal(
    metadata["indice_embedding"].to_numpy(),
    np.arange(len(metadata))
)

# ------------------------------------------------------------
# 1. Selecionar identidades elegíveis
# ------------------------------------------------------------

grupos = metadata.groupby(
    "identidade"
).indices

grupos_elegiveis = {
    identidade: np.asarray(indices, dtype=int)
    for identidade, indices in grupos.items()
    if len(indices) >= 5
}

identidades_elegiveis = sorted(
    grupos_elegiveis.keys()
)

print(
    "Identidades elegíveis:",
    len(identidades_elegiveis)
)

assert len(identidades_elegiveis) > 0

# ------------------------------------------------------------
# 2. Executar repetições
# ------------------------------------------------------------

REPETICOES = 5
SEED = 42

rng = np.random.default_rng(SEED)

registros = []

for repeticao in range(REPETICOES):

    consultas = []
    galerias_por_identidade = {}

    for identidade in identidades_elegiveis:

        indices = grupos_elegiveis[
            identidade
        ]

        selecionados = rng.choice(
            indices,
            size=5,
            replace=False
        )

        # Uma consulta fixa e quatro imagens
        # de cadastro ordenadas.
        consultas.append(
            (identidade, int(selecionados[0]))
        )

        galerias_por_identidade[
            identidade
        ] = selecionados[1:].tolist()

    # --------------------------------------------------------
    # 3. Avaliar galerias progressivas
    # --------------------------------------------------------

    for fotos_cadastro in [1, 2, 3, 4]:

        indices_galeria = []
        rotulos_galeria = []

        for identidade in identidades_elegiveis:

            indices = (
                galerias_por_identidade[identidade]
                [:fotos_cadastro]
            )

            indices_galeria.extend(indices)

            rotulos_galeria.extend(
                [identidade] * len(indices)
            )

        indices_galeria = np.asarray(
            indices_galeria,
            dtype=int
        )

        rotulos_galeria = np.asarray(
            rotulos_galeria
        )

        indices_consulta = np.asarray(
            [indice for _, indice in consultas],
            dtype=int
        )

        rotulos_consulta = np.asarray(
            [identidade for identidade, _ in consultas]
        )

        vetores_consulta = embeddings[
            indices_consulta
        ]

        vetores_galeria = embeddings[
            indices_galeria
        ]

        # Produto interno equivalente à ordenação
        # por distância euclidiana para vetores L2.
        similaridades = (
            vetores_consulta
            @ vetores_galeria.T
        )

        ranking = np.argsort(
            -similaridades,
            axis=1
        )

        rotulos_ranking = (
            rotulos_galeria[ranking]
        )

        acertos = (
            rotulos_ranking
            == rotulos_consulta[:, None]
        )

        # Hit@K: ao menos uma imagem correta no Top K.
        hit1 = float(
            acertos[:, :1].any(axis=1).mean()
        )

        hit5 = float(
            acertos[:, :5].any(axis=1).mean()
        )

        hit10 = float(
            acertos[:, :10].any(axis=1).mean()
        )

        registros.append({
            "repeticao": repeticao + 1,
            "fotos_cadastro": fotos_cadastro,
            "identidades": len(identidades_elegiveis),
            "consultas": len(indices_consulta),
            "imagens_galeria": len(indices_galeria),
            "hit_1": hit1,
            "hit_5": hit5,
            "hit_10": hit10
        })

# ------------------------------------------------------------
# 4. Consolidar resultados
# ------------------------------------------------------------

df_resultados = pd.DataFrame(
    registros
)

resumo = (
    df_resultados.groupby(
        "fotos_cadastro"
    )
    .agg(
        hit1_medio=("hit_1", "mean"),
        hit1_desvio=("hit_1", "std"),
        hit5_medio=("hit_5", "mean"),
        hit10_medio=("hit_10", "mean"),
        imagens_galeria=("imagens_galeria", "first")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. Persistir resultados
# ------------------------------------------------------------

ARQUIVO_REPETICOES = (
    DIR_AVALIACAO / "simulacao_cadastro_repeticoes.csv"
)

ARQUIVO_RESUMO = (
    DIR_AVALIACAO / "simulacao_cadastro_resumo.csv"
)

df_resultados.to_csv(
    ARQUIVO_REPETICOES,
    index=False
)

resumo.to_csv(
    ARQUIVO_RESUMO,
    index=False
)

# ------------------------------------------------------------
# 6. Apresentar resultados
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RESULTADOS — SIMULAÇÃO DE CADASTRO")
print("=" * 60)

for _, linha in resumo.iterrows():

    print(
        f"\n{int(linha['fotos_cadastro'])} "
        f"foto(s) por cão"
    )

    print(
        "Imagens na galeria:",
        int(linha["imagens_galeria"])
    )

    print(
        f"Hit@1: "
        f"{linha['hit1_medio']:.4%}"
    )

    print(
        f"Desvio-padrão Hit@1: "
        f"{linha['hit1_desvio']:.4%}"
    )

    print(
        f"Hit@5: "
        f"{linha['hit5_medio']:.4%}"
    )

    print(
        f"Hit@10: "
        f"{linha['hit10_medio']:.4%}"
    )

print("\nArquivos salvos:")
print(ARQUIVO_REPETICOES)
print(ARQUIVO_RESUMO)

print("\nBloco [59] concluído.")


In [ ]:

# ============================================================
# Bloco [60] — Consolidação do fold 1
# GPU: NÃO NECESSÁRIA
# ============================================================

import json
import hashlib
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

print("=" * 60)
print("CONSOLIDAÇÃO FINAL — FOLD 1")
print("=" * 60)

DIR_AVALIACAO = (
    RAIZ / "resultados/fold_1/avaliacao"
)

DIR_CONSOLIDACAO = (
    RAIZ / "resultados/fold_1/consolidacao"
)

DIR_CONSOLIDACAO.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Carregar resultados
# ------------------------------------------------------------

def ler_json(caminho):
    with open(caminho, encoding="utf-8") as f:
        return json.load(f)

verificacao = ler_json(
    DIR_AVALIACAO / "metricas_verificacao.json"
)

recuperacao = ler_json(
    DIR_AVALIACAO / "metricas_recuperacao.json"
)

simulacao = pd.read_csv(
    DIR_AVALIACAO / "simulacao_cadastro_resumo.csv"
)

por_dataset = pd.read_csv(
    DIR_AVALIACAO / "metricas_por_dataset.csv"
)

auditoria = pd.read_csv(
    DIR_AVALIACAO / "auditoria_erros_dataset.csv"
)

# ------------------------------------------------------------
# 2. Consolidar métricas principais
# ------------------------------------------------------------

consolidado = {
    "experimento": "TCC1_VGG16_Triplet_KFold",
    "fold": 1,
    "data_consolidacao_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "modelo": {
        "backbone": "VGG16",
        "embedding_dim": 128,
        "normalizacao": "L2",
        "loss": "Triplet Loss semi-hard",
        "margem": 0.2,
        "preprocessamento": (
            "tf.keras.applications.vgg16.preprocess_input"
        ),
        "melhor_loss_validacao": 0.013314226631288572
    },
    "validacao": {
        "imagens": 4848,
        "identidades": 577,
        "separacao_por_identidade": True
    },
    "verificacao": {
        "pares_positivos": verificacao["pares_positivos"],
        "pares_negativos": verificacao["pares_negativos"],
        "roc_auc": verificacao["roc_auc"],
        "eer_aproximado": verificacao["eer_aproximado"],
        "limiar_eer": verificacao["limiar_distancia_eer"],
        "acuracia": verificacao["acuracia"],
        "precisao": verificacao["precisao"],
        "recall": verificacao["recall"],
        "f1_score": verificacao["f1_score"]
    },
    "recuperacao": {
        "protocolo": "image_to_image_leave_one_out",
        "hit_at_1": recuperacao["hit_at_1"],
        "hit_at_5": recuperacao["hit_at_5"],
        "hit_at_10": recuperacao["hit_at_10"],
        "map": recuperacao["map"],
        "mrr": recuperacao["mrr"]
    },
    "por_dataset": por_dataset.to_dict(
        orient="records"
    ),
    "auditoria_erros": auditoria.to_dict(
        orient="records"
    ),
    "simulacao_cadastro": {
        "identidades_elegiveis": 513,
        "repeticoes": 5,
        "resultados": simulacao.to_dict(
            orient="records"
        )
    },
    "limitacoes": [
        "Resultados restritos ao fold 1.",
        "Teste independente de 30% ainda não avaliado.",
        "Limiar de verificação escolhido na validação.",
        "Pares negativos amostrados e balanceados.",
        "Recuperação avaliada com correspondência presente.",
        "Simulação de cadastro restrita a identidades com pelo menos cinco imagens.",
        "Possível correlação entre imagens provenientes de vídeos."
    ]
}

ARQUIVO_CONSOLIDADO = (
    DIR_CONSOLIDACAO / "resultados_fold1.json"
)

with open(
    ARQUIVO_CONSOLIDADO,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        consolidado,
        f,
        indent=4,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 3. Manifesto de integridade
# ------------------------------------------------------------

arquivos_principais = {
    "pesos_modelo": ARQUIVO_MODELO_FINAL_FOLD1,
    "embeddings": (
        DIR_AVALIACAO / "embeddings_validacao.npy"
    ),
    "metadados": (
        DIR_AVALIACAO / "metadados_validacao.csv"
    ),
    "metricas_verificacao": (
        DIR_AVALIACAO / "metricas_verificacao.json"
    ),
    "metricas_recuperacao": (
        DIR_AVALIACAO / "metricas_recuperacao.json"
    ),
    "simulacao_cadastro": (
        DIR_AVALIACAO / "simulacao_cadastro_resumo.csv"
    ),
    "consolidado": ARQUIVO_CONSOLIDADO
}

def calcular_sha256(caminho):

    hash_obj = hashlib.sha256()

    with open(caminho, "rb") as f:
        for bloco in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            hash_obj.update(bloco)

    return hash_obj.hexdigest()

manifesto = {}

for nome, caminho in arquivos_principais.items():

    caminho = Path(caminho)

    assert caminho.is_file(), (
        f"Arquivo ausente: {caminho}"
    )

    manifesto[nome] = {
        "caminho": str(caminho),
        "tamanho_bytes": caminho.stat().st_size,
        "sha256": calcular_sha256(caminho)
    }

ARQUIVO_MANIFESTO_FOLD1 = (
    DIR_CONSOLIDACAO / "manifesto_integridade.json"
)

with open(
    ARQUIVO_MANIFESTO_FOLD1,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifesto,
        f,
        indent=4,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 4. Exibir resumo
# ------------------------------------------------------------

print("\nMÉTRICAS PRINCIPAIS")
print(
    f"ROC-AUC: {verificacao['roc_auc']:.4%}"
)
print(
    f"EER: {verificacao['eer_aproximado']:.4%}"
)
print(
    f"Hit@1: {recuperacao['hit_at_1']:.4%}"
)
print(
    f"Hit@5: {recuperacao['hit_at_5']:.4%}"
)
print(
    f"mAP: {recuperacao['map']:.4%}"
)

print("\nSIMULAÇÃO DE CADASTRO")

for _, linha in simulacao.iterrows():
    print(
        f"{int(linha['fotos_cadastro'])} foto(s): "
        f"Hit@1={linha['hit1_medio']:.4%}"
    )

print("\nARQUIVOS VERIFICADOS")
for nome in manifesto:
    print("OK:", nome)

print("\nConsolidado:", ARQUIVO_CONSOLIDADO)
print("Manifesto:", ARQUIVO_MANIFESTO_FOLD1)

print("\nBloco [60] concluído.")


## 5. Treinamento e avaliação — Fold 2

Inclui inicialização independente, treinamento e recuperação de ambiente.

In [ ]:

# ============================================================
# Bloco [61] — Preparação e auditoria do fold 2
# GPU: NÃO NECESSÁRIA
# ============================================================

from pathlib import Path
import pandas as pd
import json

print("=" * 60)
print("PREPARAÇÃO DO FOLD 2")
print("=" * 60)

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_DATASET = Path(
    "/content/tcc1_kfold/datasets"
)

DIR_SPLITS = RAIZ / "splits"

DIR_RESULTADOS_FOLD2 = (
    RAIZ / "resultados/fold_2"
)

DIR_CHECKPOINT_FOLD2 = (
    RAIZ / "checkpoints/fold_2"
)

# ------------------------------------------------------------
# 1. Localizar arquivos do fold 2
# ------------------------------------------------------------

print("\nARQUIVOS DE SPLIT DISPONÍVEIS")

for arquivo in sorted(DIR_SPLITS.glob("*fold_2*")):
    print(arquivo.name)

ARQUIVO_TREINO = (
    DIR_SPLITS / "fold_2_imagens_treino.csv"
)

ARQUIVO_VALIDACAO = (
    DIR_SPLITS / "fold_2_imagens_validacao.csv"
)

ARQUIVO_TESTE = (
    DIR_SPLITS / "identidades_teste_30.csv"
)

for arquivo in [
    ARQUIVO_TREINO,
    ARQUIVO_VALIDACAO,
    ARQUIVO_TESTE
]:
    assert arquivo.is_file(), (
        f"Arquivo ausente: {arquivo}"
    )

# ------------------------------------------------------------
# 2. Carregar divisões
# ------------------------------------------------------------

df_treino_f2 = pd.read_csv(
    ARQUIVO_TREINO
)

df_validacao_f2 = pd.read_csv(
    ARQUIVO_VALIDACAO
)

df_teste_f2 = pd.read_csv(
    ARQUIVO_TESTE
)

colunas_necessarias = {
    "dataset",
    "identidade",
    "caminho"
}

assert colunas_necessarias.issubset(
    df_treino_f2.columns
)

assert colunas_necessarias.issubset(
    df_validacao_f2.columns
)

# ------------------------------------------------------------
# 3. Reconstruir caminhos locais
# ------------------------------------------------------------

def caminho_local_fold2(linha):

    dataset = linha["dataset"]
    original = str(linha["caminho"])

    if dataset == "DogFaceNet":

        marcador = (
            "DogFaceNet-master/after_4_bis/"
        )

        assert marcador in original, original

        relativo = original.split(
            marcador, 1
        )[1]

        return str(
            DIR_DATASET / "DogFaceNet" / relativo
        )

    if dataset == "YTBB":

        partes = Path(original).parts

        candidatos = [
            i for i, parte in enumerate(partes)
            if parte in ("train", "test")
        ]

        assert candidatos, original

        relativo = Path(
            *partes[candidatos[-1]:]
        )

        return str(
            DIR_DATASET / "YTBB" / relativo
        )

    raise ValueError(
        f"Dataset desconhecido: {dataset}"
    )

for df in [
    df_treino_f2,
    df_validacao_f2
]:

    df["caminho_local"] = df.apply(
        caminho_local_fold2,
        axis=1
    )

    df["existe"] = df["caminho_local"].map(
        lambda p: Path(p).is_file()
    )

# ------------------------------------------------------------
# 4. Verificar separação por identidade
# ------------------------------------------------------------

ids_treino = set(
    df_treino_f2["identidade"].astype(str)
)

ids_validacao = set(
    df_validacao_f2["identidade"].astype(str)
)

ids_teste = set(
    df_teste_f2["identidade"].astype(str)
)

print("\nDISTRIBUIÇÃO")

print(
    "Treino:",
    len(df_treino_f2),
    "imagens /",
    len(ids_treino),
    "identidades"
)

print(
    "Validação:",
    len(df_validacao_f2),
    "imagens /",
    len(ids_validacao),
    "identidades"
)

print(
    "Teste independente:",
    len(ids_teste),
    "identidades"
)

print("\nARQUIVOS AUSENTES")

ausentes_treino = int(
    (~df_treino_f2["existe"]).sum()
)

ausentes_validacao = int(
    (~df_validacao_f2["existe"]).sum()
)

print("Treino:", ausentes_treino)
print("Validação:", ausentes_validacao)

print("\nSOBREPOSIÇÃO DE IDENTIDADES")

print(
    "Treino x validação:",
    len(ids_treino & ids_validacao)
)

print(
    "Treino x teste:",
    len(ids_treino & ids_teste)
)

print(
    "Validação x teste:",
    len(ids_validacao & ids_teste)
)

assert ausentes_treino == 0
assert ausentes_validacao == 0

assert not (ids_treino & ids_validacao)
assert not (ids_treino & ids_teste)
assert not (ids_validacao & ids_teste)

assert not df_treino_f2[
    "caminho_local"
].duplicated().any()

assert not df_validacao_f2[
    "caminho_local"
].duplicated().any()

# ------------------------------------------------------------
# 5. Verificar checkpoints existentes
# ------------------------------------------------------------

print("\nCHECKPOINTS EXISTENTES")

arquivos_existentes = []

for pasta in [
    DIR_RESULTADOS_FOLD2,
    DIR_CHECKPOINT_FOLD2
]:

    if pasta.exists():

        arquivos_existentes.extend(
            p for p in pasta.rglob("*")
            if p.is_file()
        )

if arquivos_existentes:

    for arquivo in sorted(
        arquivos_existentes
    )[:30]:
        print(arquivo)

else:
    print("Nenhum artefato anterior encontrado.")

# ------------------------------------------------------------
# 6. Resumo final
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("AUDITORIA DO FOLD 2 CONCLUÍDA")
print("=" * 60)

print(
    "Dataset disponível:",
    DIR_DATASET.exists()
)

print(
    "Identidades de desenvolvimento:",
    len(ids_treino | ids_validacao)
)

print(
    "Artefatos anteriores:",
    len(arquivos_existentes)
)

print("\nBloco [61] concluído.")


In [ ]:

# ============================================================
# Bloco [62] — Inicialização independente do modelo — Fold 2
# GPU: RECOMENDADA
# ============================================================

import os
import json
import random
import numpy as np
import tensorflow as tf

from pathlib import Path
from datetime import datetime, timezone

print("=" * 60)
print("INICIALIZAÇÃO DO MODELO — FOLD 2")
print("=" * 60)

# ------------------------------------------------------------
# 1. Configuração de reprodutibilidade
# ------------------------------------------------------------

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print(
    "GPU:",
    tf.config.list_physical_devices("GPU")
)

# ------------------------------------------------------------
# 2. Construir backbone VGG16
# ------------------------------------------------------------

backbone_f2 = tf.keras.applications.VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

backbone_f2.trainable = False

# ------------------------------------------------------------
# 3. Construir rede de embeddings
# ------------------------------------------------------------

entradas_f2 = tf.keras.Input(
    shape=(224, 224, 3),
    name="imagem"
)

x = backbone_f2(
    entradas_f2,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = tf.keras.layers.Dense(
    256,
    activation="relu",
    name="dense_256"
)(x)

x = tf.keras.layers.Dense(
    128,
    activation=None,
    name="dense_128"
)(x)

saidas_f2 = tf.keras.layers.UnitNormalization(
    axis=-1,
    name="normalizacao_l2"
)(x)

modelo_embeddings_f2 = tf.keras.Model(
    inputs=entradas_f2,
    outputs=saidas_f2,
    name="modelo_embeddings"
)

# ------------------------------------------------------------
# 4. Verificar arquitetura
# ------------------------------------------------------------

print("\nARQUITETURA")
modelo_embeddings_f2.summary()

assert modelo_embeddings_f2.output_shape == (
    None, 128
)

assert backbone_f2.trainable is False

# ------------------------------------------------------------
# 5. Comparar com a arquitetura do fold 1
# ------------------------------------------------------------

print("\nCOMPARAÇÃO COM O FOLD 1")

assert (
    modelo_embeddings_f2.count_params()
    == modelo_embeddings.count_params()
), "Quantidade de parâmetros diferente."

assert (
    len(modelo_embeddings_f2.layers)
    == len(modelo_embeddings.layers)
), "Quantidade de camadas diferente."

print(
    "Parâmetros fold 1:",
    modelo_embeddings.count_params()
)

print(
    "Parâmetros fold 2:",
    modelo_embeddings_f2.count_params()
)

# ------------------------------------------------------------
# 6. Testar saída e normalização
# ------------------------------------------------------------

entrada_teste = tf.zeros(
    (1, 224, 224, 3),
    dtype=tf.float32
)

saida_teste = modelo_embeddings_f2(
    entrada_teste,
    training=False
).numpy()

norma = float(
    np.linalg.norm(saida_teste[0])
)

assert saida_teste.shape == (1, 128)

assert np.isfinite(saida_teste).all()

assert np.isclose(
    norma,
    1.0,
    atol=1e-4
)

print("\nTESTE DE INFERÊNCIA")
print("Formato:", saida_teste.shape)
print("Norma L2:", norma)

# ------------------------------------------------------------
# 7. Confirmar independência dos pesos
# ------------------------------------------------------------

# Compara os pesos iniciais das camadas Dense.
# Eles devem diferir dos pesos treinados no fold 1.

dense_f1 = modelo_embeddings.get_layer(
    "dense_128"
).get_weights()[0]

dense_f2 = modelo_embeddings_f2.get_layer(
    "dense_128"
).get_weights()[0]

diferenca = float(
    np.max(
        np.abs(dense_f1 - dense_f2)
    )
)

assert diferenca > 0, (
    "Os pesos Dense são idênticos."
)

print("\nINDEPENDÊNCIA DOS FOLDS")
print(
    "Maior diferença entre pesos Dense:",
    diferenca
)

print(
    "Pesos do fold 1 reutilizados: NÃO"
)

# ------------------------------------------------------------
# 8. Persistir configuração inicial
# ------------------------------------------------------------

DIR_RESULTADOS_FOLD2.mkdir(
    parents=True,
    exist_ok=True
)

DIR_CHECKPOINT_FOLD2.mkdir(
    parents=True,
    exist_ok=True
)

configuracao_f2 = {
    "experimento": "tcc1_semihard_kfold",
    "fold": 2,
    "seed": SEED,
    "data_inicializacao_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "modelo": {
        "backbone": "VGG16",
        "pesos_iniciais": "imagenet",
        "input_shape": [224, 224, 3],
        "embedding_dim": 128,
        "normalizacao": "L2",
        "backbone_congelado_fase1": True
    },
    "mineracao": {
        "tipo": "semi-hard_in_batch",
        "P": 8,
        "K": 4,
        "batch_size": 32,
        "margem": 0.2,
        "distancia": "euclidiana_quadratica",
        "fallback": "negativo_mais_proximo"
    },
    "treinamento": {
        "epocas_fase1": 10,
        "epocas_fase2": 10,
        "learning_rate_fase1": 1e-4,
        "learning_rate_fase2": 1e-5,
        "otimizador": "Adam",
        "fine_tuning": "block5",
        "early_stopping_patience": 4
    },
    "preprocessamento": (
        "tf.keras.applications.vgg16.preprocess_input"
    ),
    "divisao": {
        "identidades_treino": len(ids_treino),
        "identidades_validacao": len(ids_validacao),
        "imagens_treino": len(df_treino_f2),
        "imagens_validacao": len(df_validacao_f2)
    }
}

ARQUIVO_CONFIG_FOLD2 = (
    DIR_RESULTADOS_FOLD2 / "configuracao.json"
)

with open(
    ARQUIVO_CONFIG_FOLD2,
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        configuracao_f2,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

print("\n" + "=" * 60)
print("MODELO DO FOLD 2 INICIALIZADO")
print("=" * 60)

print("Configuração:", ARQUIVO_CONFIG_FOLD2)
print("Backbone congelado:", not backbone_f2.trainable)
print("Dimensão dos embeddings:", 128)

print("\nBloco [62] concluído.")


In [ ]:
# ============================================================
# Bloco [63] — Correção da validação do fold 2
# GPU: NÃO NECESSÁRIA
# ============================================================

import json
import numpy as np
import tensorflow as tf
from pathlib import Path
from datetime import datetime, timezone

print("=" * 60)
print("VALIDAÇÃO CORRIGIDA — FOLD 2")
print("=" * 60)

# ------------------------------------------------------------
# 1. Localizar camadas Dense independentemente dos nomes
# ------------------------------------------------------------

def obter_camadas_dense(modelo):
    return [
        camada
        for camada in modelo.layers
        if isinstance(camada, tf.keras.layers.Dense)
    ]

dense_f1 = obter_camadas_dense(modelo_embeddings)
dense_f2 = obter_camadas_dense(modelo_embeddings_f2)

assert len(dense_f1) == 2
assert len(dense_f2) == 2

for camada1, camada2 in zip(dense_f1, dense_f2):
    assert camada1.units == camada2.units
    assert camada1.activation.__name__ == camada2.activation.__name__

print("\nCAMADAS DENSE")

for camada1, camada2 in zip(dense_f1, dense_f2):
    print(
        f"Fold 1: {camada1.name} | "
        f"Fold 2: {camada2.name} | "
        f"Unidades: {camada1.units}"
    )

# ------------------------------------------------------------
# 2. Confirmar independência das camadas Dense
# ------------------------------------------------------------

for camada1, camada2 in zip(dense_f1, dense_f2):

    pesos1 = camada1.get_weights()[0]
    pesos2 = camada2.get_weights()[0]

    diferenca = float(
        np.max(np.abs(pesos1 - pesos2))
    )

    assert diferenca > 0

    print(
        f"Diferença máxima ({camada2.name}): "
        f"{diferenca:.8f}"
    )

# ------------------------------------------------------------
# 3. Confirmar integridade do backbone ImageNet
# ------------------------------------------------------------

backbone_f1 = modelo_embeddings.get_layer("vgg16")
backbone_f2 = modelo_embeddings_f2.get_layer("vgg16")

# O primeiro bloco permaneceu congelado no fold 1.
# Seus pesos devem corresponder aos pesos ImageNet.
pesos_backbone_f1 = (
    backbone_f1.get_layer("block1_conv1").get_weights()[0]
)

pesos_backbone_f2 = (
    backbone_f2.get_layer("block1_conv1").get_weights()[0]
)

diferenca_backbone = float(
    np.max(
        np.abs(
            pesos_backbone_f1 - pesos_backbone_f2
        )
    )
)

assert np.allclose(
    pesos_backbone_f1,
    pesos_backbone_f2,
    atol=1e-7
)

print("\nBACKBONE")
print(
    "Diferença máxima no block1_conv1:",
    diferenca_backbone
)
print(
    "Backbone fold 2 congelado:",
    not backbone_f2.trainable
)

assert not backbone_f2.trainable

# ------------------------------------------------------------
# 4. Confirmar arquitetura
# ------------------------------------------------------------

assert (
    modelo_embeddings.count_params()
    == modelo_embeddings_f2.count_params()
)

assert modelo_embeddings_f2.output_shape == (
    None, 128
)

print("\nARQUITETURA")
print(
    "Parâmetros:",
    modelo_embeddings_f2.count_params()
)
print(
    "Dimensão do embedding:",
    modelo_embeddings_f2.output_shape[-1]
)

# ------------------------------------------------------------
# 5. Salvar configuração do fold 2
# ------------------------------------------------------------

DIR_RESULTADOS_FOLD2.mkdir(
    parents=True,
    exist_ok=True
)

DIR_CHECKPOINT_FOLD2.mkdir(
    parents=True,
    exist_ok=True
)

configuracao_f2 = {
    "experimento": "TCC1_VGG16_Triplet_KFold",
    "fold": 2,
    "seed": 42,
    "data_inicializacao_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "modelo": {
        "backbone": "VGG16",
        "pesos_iniciais": "imagenet",
        "input_shape": [224, 224, 3],
        "embedding_dim": 128,
        "normalizacao": "L2",
        "parametros": modelo_embeddings_f2.count_params(),
        "backbone_congelado_fase1": True
    },
    "mineracao": {
        "tipo": "semi-hard_in_batch",
        "P": 8,
        "K": 4,
        "batch_size": 32,
        "margem": 0.2,
        "distancia": "euclidiana_quadratica",
        "fallback": "negativo_mais_proximo"
    },
    "treinamento": {
        "epocas_fase1": 10,
        "epocas_fase2": 10,
        "learning_rate_fase1": 1e-4,
        "learning_rate_fase2": 1e-5,
        "otimizador": "Adam",
        "fine_tuning": "block5",
        "early_stopping_patience": 4
    },
    "preprocessamento": (
        "tf.keras.applications.vgg16.preprocess_input"
    ),
    "divisao": {
        "identidades_treino": len(ids_treino),
        "identidades_validacao": len(ids_validacao),
        "imagens_treino": len(df_treino_f2),
        "imagens_validacao": len(df_validacao_f2)
    }
}

ARQUIVO_CONFIG_FOLD2 = (
    DIR_RESULTADOS_FOLD2 / "configuracao.json"
)

with open(
    ARQUIVO_CONFIG_FOLD2,
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        configuracao_f2,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

print("\n" + "=" * 60)
print("INICIALIZAÇÃO DO FOLD 2 VALIDADA")
print("=" * 60)

print("Pesos treinados do fold 1 reutilizados: NÃO")
print("Configuração salva:", ARQUIVO_CONFIG_FOLD2)
print("Bloco [63] concluído.")

In [ ]:

# ============================================================
# Bloco [64] — Preparação dos lotes P=8, K=4 — Fold 2
# GPU: NÃO NECESSÁRIA
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 60)
print("PREPARAÇÃO DOS LOTES — FOLD 2")
print("=" * 60)

P = 8
K = 4
BATCH_SIZE = P * K
SEED = 42
LOTES_VALIDACAO = 150

assert BATCH_SIZE == 32

# ------------------------------------------------------------
# 1. Conferir distribuição das imagens
# ------------------------------------------------------------

def analisar_identidades(df, nome):

    contagens = df.groupby(
        "identidade"
    ).size()

    print(f"\n{nome.upper()}")
    print("Identidades:", len(contagens))
    print("Imagens:", len(df))
    print("Mínimo:", int(contagens.min()))
    print("Máximo:", int(contagens.max()))
    print("Média:", round(float(contagens.mean()), 2))

    print(
        "Identidades com menos de 4 imagens:",
        int((contagens < K).sum())
    )

    print(
        "Identidades com pelo menos 4 imagens:",
        int((contagens >= K).sum())
    )

    return contagens

contagens_treino = analisar_identidades(
    df_treino_f2,
    "Treinamento"
)

contagens_validacao = analisar_identidades(
    df_validacao_f2,
    "Validação"
)

# ------------------------------------------------------------
# 2. Criar índice por identidade
# ------------------------------------------------------------

def construir_indice(df):

    indice = {}

    for identidade, grupo in df.groupby(
        "identidade",
        sort=True
    ):

        caminhos = sorted(
            grupo["caminho_local"].astype(str).tolist()
        )

        assert len(caminhos) == len(set(caminhos))

        indice[str(identidade)] = caminhos

    return indice

indice_treino_f2 = construir_indice(
    df_treino_f2
)

indice_validacao_f2 = construir_indice(
    df_validacao_f2
)

# ------------------------------------------------------------
# 3. Gerar lotes P×K
# ------------------------------------------------------------

def gerar_lotes_pk(
    indice,
    quantidade_lotes,
    seed,
    p=8,
    k=4
):
    rng = np.random.default_rng(seed)

    identidades = sorted(indice.keys())

    assert len(identidades) >= p

    lotes = []

    for numero_lote in range(quantidade_lotes):

        escolhidas = rng.choice(
            identidades,
            size=p,
            replace=False
        )

        caminhos_lote = []
        rotulos_lote = []

        for identidade in escolhidas:

            imagens = indice[identidade]

            # Reposição somente quando houver
            # menos de K imagens disponíveis.
            selecionadas = rng.choice(
                imagens,
                size=k,
                replace=(len(imagens) < k)
            )

            caminhos_lote.extend(
                str(caminho)
                for caminho in selecionadas
            )

            rotulos_lote.extend(
                [str(identidade)] * k
            )

        assert len(caminhos_lote) == p * k
        assert len(rotulos_lote) == p * k

        lotes.append({
            "lote": numero_lote + 1,
            "caminhos": caminhos_lote,
            "identidades": rotulos_lote
        })

    return lotes

lotes_validacao_f2 = gerar_lotes_pk(
    indice=indice_validacao_f2,
    quantidade_lotes=LOTES_VALIDACAO,
    seed=SEED,
    p=P,
    k=K
)

# ------------------------------------------------------------
# 4. Auditar lotes gerados
# ------------------------------------------------------------

total_aparicoes = sum(
    len(lote["caminhos"])
    for lote in lotes_validacao_f2
)

imagens_distintas = len({
    caminho
    for lote in lotes_validacao_f2
    for caminho in lote["caminhos"]
})

identidades_distintas = len({
    identidade
    for lote in lotes_validacao_f2
    for identidade in lote["identidades"]
})

lotes_com_repeticao = sum(
    len(set(lote["caminhos"])) < len(lote["caminhos"])
    for lote in lotes_validacao_f2
)

print("\nAUDITORIA DA VALIDAÇÃO")

print("Lotes:", len(lotes_validacao_f2))
print("Aparições de imagens:", total_aparicoes)
print("Imagens distintas:", imagens_distintas)
print("Identidades distintas:", identidades_distintas)
print(
    "Lotes com imagens repetidas internamente:",
    lotes_com_repeticao
)

assert total_aparicoes == (
    LOTES_VALIDACAO * BATCH_SIZE
)

# ------------------------------------------------------------
# 5. Salvar lotes e protocolo
# ------------------------------------------------------------

DIR_RESULTADOS_FOLD2.mkdir(
    parents=True,
    exist_ok=True
)

ARQUIVO_LOTES_FOLD2 = (
    DIR_RESULTADOS_FOLD2 / "lotes_validacao.json"
)

with open(
    ARQUIVO_LOTES_FOLD2,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        lotes_validacao_f2,
        f,
        indent=2,
        ensure_ascii=False
    )

protocolo = {
    "fold": 2,
    "seed": SEED,
    "P": P,
    "K": K,
    "batch_size": BATCH_SIZE,
    "lotes_validacao": LOTES_VALIDACAO,
    "aparicoes_validacao": total_aparicoes,
    "imagens_distintas_validacao": imagens_distintas,
    "identidades_distintas_validacao": identidades_distintas,
    "reposicao_quando_imagens_insuficientes": True,
    "amostragem_identidades": "uniforme_sem_reposicao_por_lote",
    "amostragem_imagens": "sem_reposicao_se_possivel",
    "validacao_fixa": True
}

ARQUIVO_PROTOCOLO_FOLD2 = (
    DIR_RESULTADOS_FOLD2 / "protocolo_amostragem.json"
)

with open(
    ARQUIVO_PROTOCOLO_FOLD2,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        protocolo,
        f,
        indent=4,
        ensure_ascii=False
    )

print("\nARQUIVOS SALVOS")
print(ARQUIVO_LOTES_FOLD2)
print(ARQUIVO_PROTOCOLO_FOLD2)

print("\n" + "=" * 60)
print("PREPARAÇÃO DOS LOTES CONCLUÍDA")
print("=" * 60)
print("Bloco [64] concluído.")


In [ ]:

# ============================================================
# Bloco [65] — Pipeline de treinamento — Fold 2
# GPU: ATIVA
# ============================================================

import json
import numpy as np
import tensorflow as tf
from pathlib import Path

print("=" * 60)
print("PIPELINE DE TREINAMENTO — FOLD 2")
print("=" * 60)

P = 8
K = 4
BATCH_SIZE = 32
MARGEM = 0.2

# ------------------------------------------------------------
# 1. Comparar formato da validação com o fold 1
# ------------------------------------------------------------

arquivo_lotes_f1 = (
    RAIZ / "resultados/fold_1/lotes_validacao.json"
)

with open(
    arquivo_lotes_f1,
    encoding="utf-8"
) as f:
    lotes_f1 = json.load(f)

with open(
    ARQUIVO_LOTES_FOLD2,
    encoding="utf-8"
) as f:
    lotes_f2 = json.load(f)

print("Lotes fold 1:", len(lotes_f1))
print("Lotes fold 2:", len(lotes_f2))

print(
    "Formato fold 1:",
    type(lotes_f1[0]).__name__
)

print(
    "Formato fold 2:",
    type(lotes_f2[0]).__name__
)

if isinstance(lotes_f1[0], dict):
    print(
        "Chaves fold 1:",
        list(lotes_f1[0].keys())
    )

assert len(lotes_f1) == len(lotes_f2) == 150

# ------------------------------------------------------------
# 2. Pré-processamento VGG16
# ------------------------------------------------------------

def carregar_imagem(caminho):

    imagem = tf.io.read_file(caminho)

    imagem = tf.io.decode_image(
        imagem,
        channels=3,
        expand_animations=False
    )

    imagem.set_shape((None, None, 3))

    imagem = tf.image.resize(
        imagem,
        (224, 224)
    )

    imagem = tf.cast(
        imagem,
        tf.float32
    )

    imagem = tf.keras.applications.vgg16.preprocess_input(
        imagem
    )

    return imagem

# ------------------------------------------------------------
# 3. Preparar lotes fixos de validação
# ------------------------------------------------------------

caminhos_validacao = []
rotulos_validacao = []

for lote in lotes_f2:

    caminhos_validacao.extend(
        lote["caminhos"]
    )

    rotulos_validacao.extend(
        lote["identidades"]
    )

# Codificar identidades em inteiros
identidades_validacao = sorted(
    set(rotulos_validacao)
)

mapa_validacao = {
    identidade: i
    for i, identidade in enumerate(
        identidades_validacao
    )
}

rotulos_validacao_int = np.asarray(
    [
        mapa_validacao[identidade]
        for identidade in rotulos_validacao
    ],
    dtype=np.int32
)

caminhos_validacao = np.asarray(
    caminhos_validacao,
    dtype=str
)

def preparar_exemplo(caminho, rotulo):

    imagem = carregar_imagem(caminho)

    return imagem, rotulo

dataset_validacao_f2 = (
    tf.data.Dataset.from_tensor_slices(
        (
            caminhos_validacao,
            rotulos_validacao_int
        )
    )
    .map(
        preparar_exemplo,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

# ------------------------------------------------------------
# 4. Função Triplet Loss semi-hard
# ------------------------------------------------------------

@tf.function
def triplet_loss_semihard(embeddings, rotulos):

    rotulos = tf.reshape(rotulos, [-1])

    # Distâncias euclidianas quadráticas
    produto = tf.matmul(
        embeddings,
        embeddings,
        transpose_b=True
    )

    normas = tf.reduce_sum(
        tf.square(embeddings),
        axis=1,
        keepdims=True
    )

    distancias = (
        normas
        - 2.0 * produto
        + tf.transpose(normas)
    )

    distancias = tf.maximum(
        distancias,
        0.0
    )

    n = tf.shape(distancias)[0]

    mesmos = tf.equal(
        tf.expand_dims(rotulos, 1),
        tf.expand_dims(rotulos, 0)
    )

    diagonal = tf.eye(
        n,
        dtype=tf.bool
    )

    positivos = tf.logical_and(
        mesmos,
        tf.logical_not(diagonal)
    )

    negativos = tf.logical_not(mesmos)

    # Cada par âncora-positivo é avaliado.
    d_ap = tf.expand_dims(
        distancias,
        axis=2
    )

    d_an = tf.expand_dims(
        distancias,
        axis=1
    )

    mascara_negativos = tf.expand_dims(
        negativos,
        axis=1
    )

    # Semi-hard:
    # negativo mais distante que o positivo,
    # mas dentro da margem.
    candidatos_semihard = tf.logical_and(
        mascara_negativos,
        tf.logical_and(
            d_an > d_ap,
            d_an < d_ap + MARGEM
        )
    )

    infinito = tf.constant(
        np.inf,
        dtype=tf.float32
    )

    distancias_semihard = tf.where(
        candidatos_semihard,
        d_an,
        infinito
    )

    melhor_semihard = tf.reduce_min(
        distancias_semihard,
        axis=2
    )

    # Fallback: negativo mais próximo.
    distancias_negativos = tf.where(
        negativos,
        distancias,
        infinito
    )

    negativo_mais_proximo = tf.reduce_min(
        distancias_negativos,
        axis=1
    )

    negativo_mais_proximo = tf.expand_dims(
        negativo_mais_proximo,
        axis=1
    )

    d_negativo = tf.where(
        tf.math.is_finite(melhor_semihard),
        melhor_semihard,
        negativo_mais_proximo
    )

    perdas = tf.maximum(
        distancias + MARGEM - d_negativo,
        0.0
    )

    perdas = tf.where(
        positivos,
        perdas,
        0.0
    )

    total_positivos = tf.reduce_sum(
        tf.cast(positivos, tf.float32)
    )

    return tf.reduce_sum(perdas) / tf.maximum(
        total_positivos,
        1.0
    )

# ------------------------------------------------------------
# 5. Teste da função de perda
# ------------------------------------------------------------

imagens_teste, rotulos_teste = next(
    iter(dataset_validacao_f2)
)

embeddings_teste = modelo_embeddings_f2(
    imagens_teste,
    training=False
)

perda_teste = triplet_loss_semihard(
    embeddings_teste,
    rotulos_teste
)

assert tf.math.is_finite(perda_teste)

print("\nTESTE DO PIPELINE")
print("Imagens:", imagens_teste.shape)
print("Embeddings:", embeddings_teste.shape)
print("Loss inicial:", float(perda_teste))

print("\n" + "=" * 60)
print("PIPELINE PRONTO PARA TREINAMENTO")
print("=" * 60)
print("Bloco [65] concluído.")


In [ ]:

# ============================================================
# Bloco [66] — Treinamento Fold 2 — Fase 1
# GPU: ATIVA
# ============================================================

import json
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

print("=" * 60)
print("TREINAMENTO FOLD 2 — FASE 1")
print("=" * 60)

SEED = 42
EPOCAS = 10
P = 8
K = 4
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

# Quantidade de lotes de treinamento por época.
# Aproximadamente uma passagem pelo conjunto.
PASSOS_POR_EPOCA = int(
    np.ceil(len(df_treino_f2) / BATCH_SIZE)
)

print("Épocas:", EPOCAS)
print("Passos por época:", PASSOS_POR_EPOCA)
print("Lotes de validação:", 150)
print("Learning rate:", LEARNING_RATE)

# ------------------------------------------------------------
# 1. Garantir backbone congelado
# ------------------------------------------------------------

backbone_f2.trainable = False

otimizador_f2 = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

# ------------------------------------------------------------
# 2. Gerador de lotes P×K para treinamento
# ------------------------------------------------------------

identidades_treino_f2 = sorted(
    indice_treino_f2.keys()
)

mapa_treino_f2 = {
    identidade: i
    for i, identidade in enumerate(
        identidades_treino_f2
    )
}

def gerador_treino_f2(epoca):

    rng = np.random.default_rng(
        SEED + epoca
    )

    for _ in range(PASSOS_POR_EPOCA):

        escolhidas = rng.choice(
            identidades_treino_f2,
            size=P,
            replace=False
        )

        caminhos = []
        rotulos = []

        for identidade in escolhidas:

            imagens = indice_treino_f2[
                identidade
            ]

            selecionadas = rng.choice(
                imagens,
                size=K,
                replace=(len(imagens) < K)
            )

            caminhos.extend(
                str(c) for c in selecionadas
            )

            rotulos.extend(
                [mapa_treino_f2[identidade]] * K
            )

        yield (
            np.asarray(caminhos, dtype=str),
            np.asarray(rotulos, dtype=np.int32)
        )

def criar_dataset_treino(epoca):

    def gerador_imagens():

        for caminhos, rotulos in gerador_treino_f2(
            epoca
        ):
            for caminho, rotulo in zip(
                caminhos,
                rotulos
            ):
                yield caminho, rotulo

    ds = tf.data.Dataset.from_generator(
        gerador_imagens,
        output_signature=(
            tf.TensorSpec(
                shape=(),
                dtype=tf.string
            ),
            tf.TensorSpec(
                shape=(),
                dtype=tf.int32
            )
        )
    )

    return (
        ds.map(
            preparar_exemplo,
            num_parallel_calls=tf.data.AUTOTUNE
        )
        .batch(
            BATCH_SIZE,
            drop_remainder=True
        )
        .prefetch(tf.data.AUTOTUNE)
    )

# ------------------------------------------------------------
# 3. Funções de treinamento e validação
# ------------------------------------------------------------

@tf.function
def passo_treino_f2(imagens, rotulos):

    with tf.GradientTape() as tape:

        embeddings = modelo_embeddings_f2(
            imagens,
            training=True
        )

        perda = triplet_loss_semihard(
            embeddings,
            rotulos
        )

    variaveis = (
        modelo_embeddings_f2.trainable_variables
    )

    gradientes = tape.gradient(
        perda,
        variaveis
    )

    pares_validos = [
        (g, v)
        for g, v in zip(gradientes, variaveis)
        if g is not None
    ]

    otimizador_f2.apply_gradients(
        pares_validos
    )

    return perda

@tf.function
def passo_validacao_f2(imagens, rotulos):

    embeddings = modelo_embeddings_f2(
        imagens,
        training=False
    )

    return triplet_loss_semihard(
        embeddings,
        rotulos
    )

# ------------------------------------------------------------
# 4. Caminhos dos resultados
# ------------------------------------------------------------

ARQUIVO_CHECKPOINT_FASE1 = (
    DIR_CHECKPOINT_FOLD2
    / "melhor_fase1.weights.h5"
)

ARQUIVO_HISTORICO_FASE1 = (
    DIR_RESULTADOS_FOLD2
    / "historico_fase1.csv"
)

assert not ARQUIVO_CHECKPOINT_FASE1.exists(), (
    "Já existe checkpoint da fase 1. "
    "Interrompido para evitar sobrescrita."
)

assert not ARQUIVO_HISTORICO_FASE1.exists(), (
    "Já existe histórico da fase 1. "
    "Interrompido para evitar sobrescrita."
)

# ------------------------------------------------------------
# 5. Treinamento
# ------------------------------------------------------------

melhor_loss = float("inf")
historico = []

inicio_total = time.time()

for epoca in range(1, EPOCAS + 1):

    inicio_epoca = time.time()

    dataset_treino = criar_dataset_treino(
        epoca
    )

    perdas_treino = []

    for imagens, rotulos in dataset_treino:

        perda = passo_treino_f2(
            imagens,
            rotulos
        )

        perdas_treino.append(
            float(perda.numpy())
        )

    # Validação fixa: 150 lotes
    perdas_validacao = []

    for imagens, rotulos in dataset_validacao_f2:

        perda = passo_validacao_f2(
            imagens,
            rotulos
        )

        perdas_validacao.append(
            float(perda.numpy())
        )

    loss_treino = float(
        np.mean(perdas_treino)
    )

    loss_validacao = float(
        np.mean(perdas_validacao)
    )

    assert np.isfinite(loss_treino)
    assert np.isfinite(loss_validacao)

    melhorou = (
        loss_validacao < melhor_loss
    )

    if melhorou:

        melhor_loss = loss_validacao

        modelo_embeddings_f2.save_weights(
            ARQUIVO_CHECKPOINT_FASE1
        )

    duracao = time.time() - inicio_epoca

    registro = {
        "epoca": epoca,
        "loss_treino": loss_treino,
        "loss_validacao": loss_validacao,
        "melhor_checkpoint": melhorou,
        "duracao_segundos": duracao
    }

    historico.append(registro)

    pd.DataFrame(historico).to_csv(
        ARQUIVO_HISTORICO_FASE1,
        index=False
    )

    print(
        f"Época {epoca:02d}/{EPOCAS} | "
        f"Treino: {loss_treino:.6f} | "
        f"Val: {loss_validacao:.6f} | "
        f"Tempo: {duracao:.1f}s"
        + (" | MELHOR" if melhorou else ""),
        flush=True
    )

# ------------------------------------------------------------
# 6. Restaurar melhor checkpoint
# ------------------------------------------------------------

modelo_embeddings_f2.load_weights(
    ARQUIVO_CHECKPOINT_FASE1
)

tempo_total = (
    time.time() - inicio_total
)

print("\n" + "=" * 60)
print("FASE 1 CONCLUÍDA")
print("=" * 60)

print(
    "Melhor loss de validação:",
    melhor_loss
)

print(
    "Tempo total (min):",
    round(tempo_total / 60, 2)
)

print(
    "Checkpoint:",
    ARQUIVO_CHECKPOINT_FASE1
)

print(
    "Histórico:",
    ARQUIVO_HISTORICO_FASE1
)

print("\nBloco [66] concluído.")


In [ ]:

# ============================================================
# Bloco [67] — Corrigir leitura e retomar fase 1
# GPU: ATIVA
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

print("=" * 60)
print("RECUPERAÇÃO DO TREINAMENTO — FOLD 2")
print("=" * 60)

# ------------------------------------------------------------
# 1. Verificar imagens de treinamento
# ------------------------------------------------------------

arquivos_invalidos = []

for identidade, caminhos in indice_treino_f2.items():

    for caminho in caminhos:

        arquivo = Path(caminho)

        if (
            not arquivo.is_file()
            or arquivo.stat().st_size == 0
        ):
            arquivos_invalidos.append(
                (identidade, caminho)
            )

print(
    "Arquivos ausentes ou vazios:",
    len(arquivos_invalidos)
)

for identidade, caminho in arquivos_invalidos[:20]:
    print(identidade, caminho)

# ------------------------------------------------------------
# 2. Remover arquivos inválidos do índice
# ------------------------------------------------------------

indice_treino_f2 = {
    identidade: [
        caminho
        for caminho in caminhos
        if (
            Path(caminho).is_file()
            and Path(caminho).stat().st_size > 0
        )
    ]
    for identidade, caminhos in indice_treino_f2.items()
}

indice_treino_f2 = {
    identidade: caminhos
    for identidade, caminhos in indice_treino_f2.items()
    if len(caminhos) >= 2
}

identidades_treino_f2 = sorted(
    indice_treino_f2.keys()
)

assert len(identidades_treino_f2) >= P

print(
    "Identidades disponíveis:",
    len(identidades_treino_f2)
)

# ------------------------------------------------------------
# 3. Confirmar checkpoint e histórico
# ------------------------------------------------------------

assert ARQUIVO_CHECKPOINT_FASE1.exists()
assert ARQUIVO_HISTORICO_FASE1.exists()

historico_df = pd.read_csv(
    ARQUIVO_HISTORICO_FASE1
)

print("\nHISTÓRICO EXISTENTE")
print(
    historico_df[
        ["epoca", "loss_treino", "loss_validacao"]
    ].to_string(index=False)
)

ultima_epoca = int(
    historico_df["epoca"].max()
)

assert ultima_epoca == 1, (
    "O histórico não termina na época 1. "
    "Verifique antes de continuar."
)

# Preservar os pesos e o estado do Adam em memória.
# NÃO chamar load_weights aqui.

melhor_loss = float(
    historico_df["loss_validacao"].min()
)

historico = historico_df.to_dict(
    orient="records"
)

print("\nRetomando a partir da época 2.")
print("Melhor loss atual:", melhor_loss)

# ------------------------------------------------------------
# 4. Continuar treinamento
# ------------------------------------------------------------

inicio_retomada = time.time()

for epoca in range(2, EPOCAS + 1):

    inicio_epoca = time.time()

    dataset_treino = criar_dataset_treino(
        epoca
    )

    perdas_treino = []

    for imagens, rotulos in dataset_treino:

        perda = passo_treino_f2(
            imagens,
            rotulos
        )

        perdas_treino.append(
            float(perda.numpy())
        )

    perdas_validacao = []

    for imagens, rotulos in dataset_validacao_f2:

        perda = passo_validacao_f2(
            imagens,
            rotulos
        )

        perdas_validacao.append(
            float(perda.numpy())
        )

    loss_treino = float(
        np.mean(perdas_treino)
    )

    loss_validacao = float(
        np.mean(perdas_validacao)
    )

    assert np.isfinite(loss_treino)
    assert np.isfinite(loss_validacao)

    melhorou = (
        loss_validacao < melhor_loss
    )

    if melhorou:

        melhor_loss = loss_validacao

        modelo_embeddings_f2.save_weights(
            ARQUIVO_CHECKPOINT_FASE1
        )

    duracao = time.time() - inicio_epoca

    registro = {
        "epoca": epoca,
        "loss_treino": loss_treino,
        "loss_validacao": loss_validacao,
        "melhor_checkpoint": melhorou,
        "duracao_segundos": duracao
    }

    historico.append(registro)

    pd.DataFrame(historico).to_csv(
        ARQUIVO_HISTORICO_FASE1,
        index=False
    )

    print(
        f"Época {epoca:02d}/{EPOCAS} | "
        f"Treino: {loss_treino:.6f} | "
        f"Val: {loss_validacao:.6f} | "
        f"Tempo: {duracao:.1f}s"
        + (" | MELHOR" if melhorou else ""),
        flush=True
    )

# ------------------------------------------------------------
# 5. Restaurar melhor checkpoint
# ------------------------------------------------------------

modelo_embeddings_f2.load_weights(
    ARQUIVO_CHECKPOINT_FASE1
)

print("\n" + "=" * 60)
print("FASE 1 CONCLUÍDA")
print("=" * 60)

print("Melhor loss:", melhor_loss)

print(
    "Tempo de retomada (min):",
    round(
        (time.time() - inicio_retomada) / 60,
        2
    )
)

print(
    "Checkpoint:",
    ARQUIVO_CHECKPOINT_FASE1
)

print("\nBloco [67] concluído.")


In [ ]:

# ============================================================
# BLOCO [68] — RECUPERAÇÃO DO AMBIENTE — FOLD 2
# TCC 1 — Reidentificação de cães
# ============================================================

import os
import json
import tarfile
import hashlib
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from google.colab import drive

print("=" * 65)
print("RECUPERAÇÃO DO AMBIENTE — FOLD 2")
print("=" * 65)

# ------------------------------------------------------------
# 1. Google Drive e GPU
# ------------------------------------------------------------

drive.mount("/content/drive")

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

assert RAIZ.exists(), "Diretório do experimento não encontrado."

GPUS = tf.config.list_physical_devices("GPU")

print("TensorFlow:", tf.__version__)
print("GPU:", GPUS)

assert GPUS, "GPU não detectada. Verifique o ambiente Colab."

tf.keras.utils.set_random_seed(42)

# ------------------------------------------------------------
# 2. Restaurar datasets locais
# ------------------------------------------------------------

DIR_DATASETS = Path("/content/tcc1_kfold/datasets")

ARQUIVO_BACKUP = RAIZ / "datasets_backup.tar"

if not DIR_DATASETS.exists() or not any(
    DIR_DATASETS.iterdir()
):

    print("\nRestaurando datasets...")

    assert ARQUIVO_BACKUP.exists(), (
        "Backup dos datasets não encontrado."
    )

    DIR_DATASETS.mkdir(
        parents=True,
        exist_ok=True
    )

    with tarfile.open(
        ARQUIVO_BACKUP,
        "r"
    ) as tar:

        destino = Path("/content/tcc1_kfold").resolve()

        for membro in tar.getmembers():

            caminho_destino = (
                destino / membro.name
            ).resolve()

            assert (
                caminho_destino == destino
                or destino in caminho_destino.parents
            ), "Caminho inseguro no arquivo TAR."

        tar.extractall(
            path=destino,
            filter="data"
        )

    print("Datasets restaurados.")

else:
    print("\nDatasets locais encontrados.")

# ------------------------------------------------------------
# 3. Recuperar divisões do fold 2
# ------------------------------------------------------------

DIR_SPLITS = RAIZ / "splits"

df_treino_f2 = pd.read_csv(
    DIR_SPLITS / "fold_2_imagens_treino.csv"
)

df_validacao_f2 = pd.read_csv(
    DIR_SPLITS / "fold_2_imagens_validacao.csv"
)

df_teste_f2 = pd.read_csv(
    DIR_SPLITS / "identidades_teste_30.csv"
)

def resolver_caminho_local(caminho):

    caminho = str(caminho).replace("\\", "/")

    marcador = "/datasets/"

    if marcador in caminho:
        relativo = caminho.split(
            marcador,
            1
        )[1]
        return str(DIR_DATASETS / relativo)

    if caminho.startswith("datasets/"):
        return str(
            DIR_DATASETS / caminho[len("datasets/"):]
        )

    if Path(caminho).is_absolute():
        return caminho

    return str(DIR_DATASETS / caminho)

for df in [df_treino_f2, df_validacao_f2]:

    df["caminho_local"] = df["caminho"].map(
        resolver_caminho_local
    )

print("\nDIVISÕES RECUPERADAS")
print("Treino:", len(df_treino_f2))
print("Validação:", len(df_validacao_f2))
print("Identidades de teste:", len(df_teste_f2))

assert len(df_treino_f2) == 19662
assert len(df_validacao_f2) == 5209
assert len(df_teste_f2) == 1235

# ------------------------------------------------------------
# 4. Recuperar índices de amostragem
# ------------------------------------------------------------

def construir_indice(df):

    indice = {}

    for identidade, grupo in df.groupby("identidade"):

        indice[str(identidade)] = sorted(
            grupo["caminho_local"]
            .astype(str)
            .tolist()
        )

    return indice

indice_treino_f2 = construir_indice(
    df_treino_f2
)

indice_validacao_f2 = construir_indice(
    df_validacao_f2
)

# ------------------------------------------------------------
# 5. Detectar imagens vazias ou ausentes
# ------------------------------------------------------------

invalidos = []

for identidade, caminhos in indice_treino_f2.items():

    for caminho in caminhos:

        arquivo = Path(caminho)

        if (
            not arquivo.is_file()
            or arquivo.stat().st_size == 0
        ):
            invalidos.append(
                (identidade, caminho)
            )

print("\nArquivos ausentes ou vazios:", len(invalidos))

for identidade, caminho in invalidos[:10]:
    print("Inválido:", identidade, caminho)

# Não alteramos o conjunto original neste bloco.
# O tratamento será realizado no pipeline.

# ------------------------------------------------------------
# 6. Reconstruir VGG16 e embeddings
# ------------------------------------------------------------

tf.keras.backend.clear_session()

entradas_f2 = tf.keras.Input(
    shape=(224, 224, 3),
    name="entrada_imagem"
)

backbone_f2 = tf.keras.applications.VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

backbone_f2.trainable = False

x = backbone_f2(
    entradas_f2,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dense(
    256,
    activation="relu",
    name="dense_256"
)(x)

x = tf.keras.layers.Dense(
    128,
    activation="linear",
    name="dense_128"
)(x)

saidas_f2 = tf.keras.layers.UnitNormalization(
    axis=-1,
    name="normalizacao_l2"
)(x)

modelo_embeddings_f2 = tf.keras.Model(
    entradas_f2,
    saidas_f2,
    name="modelo_embeddings_fold2"
)

# ------------------------------------------------------------
# 7. Diretórios de saída
# ------------------------------------------------------------

DIR_RESULTADOS_FOLD2 = (
    RAIZ / "resultados/fold_2"
)

DIR_CHECKPOINT_FOLD2 = (
    RAIZ / "checkpoints/fold_2"
)

DIR_RESULTADOS_FOLD2.mkdir(
    parents=True,
    exist_ok=True
)

DIR_CHECKPOINT_FOLD2.mkdir(
    parents=True,
    exist_ok=True
)

ARQUIVO_CHECKPOINT_FASE1 = (
    DIR_CHECKPOINT_FOLD2
    / "melhor_fase1.weights.h5"
)

ARQUIVO_HISTORICO_FASE1 = (
    DIR_RESULTADOS_FOLD2
    / "historico_fase1.csv"
)

ARQUIVO_LOTES_FOLD2 = (
    DIR_RESULTADOS_FOLD2
    / "lotes_validacao.json"
)

assert ARQUIVO_LOTES_FOLD2.exists()

print("\nCHECKPOINT ANTERIOR")
print(
    "Existe:",
    ARQUIVO_CHECKPOINT_FASE1.exists()
)

print(
    "Histórico existe:",
    ARQUIVO_HISTORICO_FASE1.exists()
)

# Não carregar o checkpoint anterior:
# reiniciaremos a fase 1 para manter o protocolo.

# ------------------------------------------------------------
# 8. Testar modelo e GPU
# ------------------------------------------------------------

entrada_teste = tf.zeros(
    (1, 224, 224, 3),
    dtype=tf.float32
)

saida_teste = modelo_embeddings_f2(
    entrada_teste,
    training=False
)

assert saida_teste.shape == (1, 128)

print("\nMODELO")
print("Parâmetros:", modelo_embeddings_f2.count_params())
print("Saída:", saida_teste.shape)
print("Backbone congelado:", not backbone_f2.trainable)

print("\n" + "=" * 65)
print("AMBIENTE RECUPERADO — FOLD 2")
print("=" * 65)
print("Bloco [68] concluído.")


In [ ]:

# ============================================================
# BLOCO [69] — FOLD 2 / FASE 1 — TREINAMENTO
# VGG16 congelada | Adam 1e-4 | Triplet semi-hard
# ============================================================

import time
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

print("=" * 65)
print("TREINAMENTO — FOLD 2 — FASE 1")
print("=" * 65)

SEED = 42
P, K = 8, 4
BATCH_SIZE = P * K
MARGEM = 0.2
EPOCAS = 10
PASSOS_POR_EPOCA = int(np.ceil(len(df_treino_f2) / BATCH_SIZE))

# ------------------------------------------------------------
# 1. Preservar checkpoint e histórico anteriores
# ------------------------------------------------------------

for origem in [ARQUIVO_CHECKPOINT_FASE1, ARQUIVO_HISTORICO_FASE1]:
    if origem.exists():
        destino = origem.with_name(
            origem.stem + "_tentativa_anterior" + origem.suffix
        )
        if not destino.exists():
            shutil.copy2(origem, destino)
        origem.unlink()

# ------------------------------------------------------------
# 2. Índices e lotes fixos
# ------------------------------------------------------------

with open(ARQUIVO_LOTES_FOLD2, encoding="utf-8") as f:
    lotes_validacao_f2 = json.load(f)

ids_treino_f2 = sorted(indice_treino_f2)
mapa_treino_f2 = {identidade: i for i, identidade in enumerate(ids_treino_f2)}

ids_val_f2 = sorted({
    str(identidade)
    for lote in lotes_validacao_f2
    for identidade in lote["identidades"]
})
mapa_val_f2 = {identidade: i for i, identidade in enumerate(ids_val_f2)}

# ------------------------------------------------------------
# 3. Leitura robusta de imagens
# ------------------------------------------------------------

def carregar_imagem_segura(caminho):
    conteudo = tf.io.read_file(caminho)
    tf.debugging.assert_positive(
        tf.strings.length(conteudo),
        message="Arquivo de imagem vazio"
    )
    imagem = tf.io.decode_image(
        conteudo, channels=3, expand_animations=False
    )
    imagem.set_shape([None, None, 3])
    imagem = tf.image.resize(imagem, [224, 224])
    imagem = tf.keras.applications.vgg16.preprocess_input(
        tf.cast(imagem, tf.float32)
    )
    return imagem

def preparar_exemplo_f2(caminho, rotulo):
    return carregar_imagem_segura(caminho), rotulo

def montar_dataset(caminhos, rotulos):
    return (
        tf.data.Dataset.from_tensor_slices((
            tf.constant(caminhos, dtype=tf.string),
            tf.constant(rotulos, dtype=tf.int32)
        ))
        .map(preparar_exemplo_f2, num_parallel_calls=4)
        .batch(BATCH_SIZE, drop_remainder=True)
        .prefetch(2)
    )

caminhos_val = []
rotulos_val = []

for lote in lotes_validacao_f2:
    caminhos_val.extend(lote["caminhos"])
    rotulos_val.extend([
        mapa_val_f2[str(identidade)]
        for identidade in lote["identidades"]
    ])

dataset_validacao_f2 = montar_dataset(caminhos_val, rotulos_val)

def criar_dataset_treino_f2(epoca):
    rng = np.random.default_rng(SEED + epoca)
    caminhos, rotulos = [], []

    for _ in range(PASSOS_POR_EPOCA):
        escolhidas = rng.choice(ids_treino_f2, size=P, replace=False)

        for identidade in escolhidas:
            fotos = indice_treino_f2[identidade]
            selecionadas = rng.choice(
                fotos, size=K, replace=len(fotos) < K
            )
            caminhos.extend(map(str, selecionadas))
            rotulos.extend([mapa_treino_f2[identidade]] * K)

    return montar_dataset(caminhos, rotulos)

# ------------------------------------------------------------
# 4. Triplet Loss semi-hard
# ------------------------------------------------------------

@tf.function
def triplet_loss_semihard_f2(embeddings, rotulos):
    rotulos = tf.reshape(rotulos, [-1])
    produto = tf.matmul(embeddings, embeddings, transpose_b=True)
    normas = tf.reduce_sum(tf.square(embeddings), axis=1, keepdims=True)
    dist = tf.maximum(normas - 2.0 * produto + tf.transpose(normas), 0.0)

    n = tf.shape(dist)[0]
    iguais = tf.equal(rotulos[:, None], rotulos[None, :])
    positivos = tf.logical_and(iguais, ~tf.eye(n, dtype=tf.bool))
    negativos = ~iguais

    d_ap = dist[:, :, None]
    d_an = dist[:, None, :]

    candidatos = (
        negativos[:, None, :]
        & (d_an > d_ap)
        & (d_an < d_ap + MARGEM)
    )

    inf = tf.constant(float("inf"), dtype=dist.dtype)
    melhor_semi = tf.reduce_min(
        tf.where(candidatos, d_an, inf), axis=2
    )
    mais_proximo = tf.reduce_min(
        tf.where(negativos, dist, inf), axis=1
    )[:, None]

    d_neg = tf.where(
        tf.math.is_finite(melhor_semi), melhor_semi, mais_proximo
    )
    perdas = tf.where(
        positivos,
        tf.maximum(dist + MARGEM - d_neg, 0.0),
        0.0
    )
    return tf.reduce_sum(perdas) / tf.maximum(
        tf.reduce_sum(tf.cast(positivos, tf.float32)), 1.0
    )

# ------------------------------------------------------------
# 5. Otimizador e passos
# ------------------------------------------------------------

backbone_f2.trainable = False
otimizador_f2 = tf.keras.optimizers.Adam(learning_rate=1e-4)

@tf.function
def passo_treino_f2(imagens, rotulos):
    with tf.GradientTape() as tape:
        embeddings = modelo_embeddings_f2(imagens, training=True)
        loss = triplet_loss_semihard_f2(embeddings, rotulos)

    variaveis = modelo_embeddings_f2.trainable_variables
    gradientes = tape.gradient(loss, variaveis)
    otimizador_f2.apply_gradients([
        (g, v) for g, v in zip(gradientes, variaveis)
        if g is not None
    ])
    return loss

@tf.function
def passo_validacao_f2(imagens, rotulos):
    embeddings = modelo_embeddings_f2(imagens, training=False)
    return triplet_loss_semihard_f2(embeddings, rotulos)

# ------------------------------------------------------------
# 6. Executar treinamento
# ------------------------------------------------------------

historico_f2 = []
melhor_loss = float("inf")

print("Épocas:", EPOCAS)
print("Passos por época:", PASSOS_POR_EPOCA)
print("Lotes de validação:", len(lotes_validacao_f2))
print("GPU:", tf.config.list_physical_devices("GPU"))
print("-" * 65, flush=True)

for epoca in range(1, EPOCAS + 1):
    inicio = time.time()

    perdas_treino = []
    for imagens, rotulos in criar_dataset_treino_f2(epoca):
        perdas_treino.append(float(
            passo_treino_f2(imagens, rotulos).numpy()
        ))

    perdas_val = []
    for imagens, rotulos in dataset_validacao_f2:
        perdas_val.append(float(
            passo_validacao_f2(imagens, rotulos).numpy()
        ))

    loss_treino = float(np.mean(perdas_treino))
    loss_val = float(np.mean(perdas_val))

    assert np.isfinite(loss_treino) and np.isfinite(loss_val)

    melhorou = loss_val < melhor_loss
    if melhorou:
        melhor_loss = loss_val
        modelo_embeddings_f2.save_weights(ARQUIVO_CHECKPOINT_FASE1)

    historico_f2.append({
        "epoca": epoca,
        "loss_treino": loss_treino,
        "loss_validacao": loss_val,
        "melhor_checkpoint": melhorou,
        "duracao_segundos": time.time() - inicio
    })

    pd.DataFrame(historico_f2).to_csv(
        ARQUIVO_HISTORICO_FASE1, index=False
    )

    print(
        f"Época {epoca:02d}/{EPOCAS} | "
        f"Treino: {loss_treino:.6f} | "
        f"Val: {loss_val:.6f} | "
        f"Tempo: {time.time()-inicio:.1f}s"
        + (" | MELHOR" if melhorou else ""),
        flush=True
    )

modelo_embeddings_f2.load_weights(ARQUIVO_CHECKPOINT_FASE1)

print("\nFASE 1 CONCLUÍDA")
print("Melhor loss:", melhor_loss)
print("Checkpoint:", ARQUIVO_CHECKPOINT_FASE1)
print("Bloco [69] concluído.")


In [ ]:

# ============================================================
# BLOCO [70] — FOLD 2 / FASE 2 — FINE-TUNING
# VGG16 block5 | Adam 1e-5 | Triplet semi-hard
# ============================================================

import time
import numpy as np
import pandas as pd
import tensorflow as tf

print("=" * 65)
print("FINE-TUNING — FOLD 2 — FASE 2")
print("=" * 65)

# ------------------------------------------------------------
# 1. Recuperar melhor checkpoint da fase 1
# ------------------------------------------------------------

assert ARQUIVO_CHECKPOINT_FASE1.exists()

modelo_embeddings_f2.load_weights(
    ARQUIVO_CHECKPOINT_FASE1
)

# ------------------------------------------------------------
# 2. Descongelar somente o bloco 5 da VGG16
# ------------------------------------------------------------

backbone_f2.trainable = True

for camada in backbone_f2.layers:
    camada.trainable = camada.name.startswith("block5_")

camadas_treinaveis = [
    camada.name
    for camada in backbone_f2.layers
    if camada.trainable
]

print("Camadas treináveis do backbone:")
for nome in camadas_treinaveis:
    print(" -", nome)

assert camadas_treinaveis == [
    "block5_conv1",
    "block5_conv2",
    "block5_conv3",
    "block5_pool"
]

# ------------------------------------------------------------
# 3. Novo otimizador para fine-tuning
# ------------------------------------------------------------

otimizador_f2_fase2 = tf.keras.optimizers.Adam(
    learning_rate=1e-5
)

@tf.function
def passo_treino_f2_fase2(imagens, rotulos):

    with tf.GradientTape() as tape:
        embeddings = modelo_embeddings_f2(
            imagens,
            training=True
        )

        loss = triplet_loss_semihard_f2(
            embeddings,
            rotulos
        )

    variaveis = modelo_embeddings_f2.trainable_variables

    gradientes = tape.gradient(
        loss,
        variaveis
    )

    otimizador_f2_fase2.apply_gradients([
        (g, v)
        for g, v in zip(gradientes, variaveis)
        if g is not None
    ])

    return loss

# ------------------------------------------------------------
# 4. Arquivos de saída
# ------------------------------------------------------------

ARQUIVO_CHECKPOINT_FASE2 = (
    DIR_CHECKPOINT_FOLD2
    / "melhor_fase2.weights.h5"
)

ARQUIVO_HISTORICO_FASE2 = (
    DIR_RESULTADOS_FOLD2
    / "historico_fase2.csv"
)

ARQUIVO_MODELO_FINAL_F2 = (
    DIR_CHECKPOINT_FOLD2
    / "modelo_final_fold2.weights.h5"
)

for arquivo in [
    ARQUIVO_CHECKPOINT_FASE2,
    ARQUIVO_HISTORICO_FASE2,
    ARQUIVO_MODELO_FINAL_F2
]:
    assert not arquivo.exists(), (
        f"Arquivo já existe: {arquivo}. "
        "Interrompido para evitar sobrescrita."
    )

# ------------------------------------------------------------
# 5. Treinamento
# ------------------------------------------------------------

EPOCAS_FASE2 = 10
PACIENCIA = 4

melhor_loss_fase2 = float("inf")
epocas_sem_melhora = 0
historico_fase2 = []

print("\nÉpocas máximas:", EPOCAS_FASE2)
print("Learning rate:", 1e-5)
print("Early stopping — paciência:", PACIENCIA)
print("Passos por época:", PASSOS_POR_EPOCA)
print("-" * 65, flush=True)

for epoca in range(1, EPOCAS_FASE2 + 1):

    inicio = time.time()

    perdas_treino = []

    for imagens, rotulos in criar_dataset_treino_f2(
        10 + epoca
    ):
        perda = passo_treino_f2_fase2(
            imagens,
            rotulos
        )

        perdas_treino.append(
            float(perda.numpy())
        )

    perdas_val = []

    for imagens, rotulos in dataset_validacao_f2:

        perda = passo_validacao_f2(
            imagens,
            rotulos
        )

        perdas_val.append(
            float(perda.numpy())
        )

    loss_treino = float(np.mean(perdas_treino))
    loss_val = float(np.mean(perdas_val))

    assert np.isfinite(loss_treino)
    assert np.isfinite(loss_val)

    melhorou = loss_val < melhor_loss_fase2

    if melhorou:
        melhor_loss_fase2 = loss_val
        epocas_sem_melhora = 0

        modelo_embeddings_f2.save_weights(
            ARQUIVO_CHECKPOINT_FASE2
        )

    else:
        epocas_sem_melhora += 1

    duracao = time.time() - inicio

    historico_fase2.append({
        "epoca": epoca,
        "loss_treino": loss_treino,
        "loss_validacao": loss_val,
        "melhor_checkpoint": melhorou,
        "duracao_segundos": duracao
    })

    pd.DataFrame(historico_fase2).to_csv(
        ARQUIVO_HISTORICO_FASE2,
        index=False
    )

    print(
        f"Época {epoca:02d}/{EPOCAS_FASE2} | "
        f"Treino: {loss_treino:.6f} | "
        f"Val: {loss_val:.6f} | "
        f"Tempo: {duracao:.1f}s"
        + (" | MELHOR" if melhorou else ""),
        flush=True
    )

    if epocas_sem_melhora >= PACIENCIA:
        print(
            "\nEarly stopping acionado "
            f"após {epoca} épocas."
        )
        break

# ------------------------------------------------------------
# 6. Restaurar melhor modelo e salvar versão final
# ------------------------------------------------------------

modelo_embeddings_f2.load_weights(
    ARQUIVO_CHECKPOINT_FASE2
)

modelo_embeddings_f2.save_weights(
    ARQUIVO_MODELO_FINAL_F2
)

melhor_epoca = int(
    pd.DataFrame(historico_fase2)
    .loc[
        lambda df: df["loss_validacao"].idxmin(),
        "epoca"
    ]
)

print("\n" + "=" * 65)
print("FINE-TUNING CONCLUÍDO — FOLD 2")
print("=" * 65)

print("Melhor época:", melhor_epoca)
print("Melhor loss:", melhor_loss_fase2)
print("Modelo final:", ARQUIVO_MODELO_FINAL_F2)
print("Histórico:", ARQUIVO_HISTORICO_FASE2)
print("Bloco [70] concluído.")


In [ ]:

# ============================================================
# BLOCO [71] — EXTRAÇÃO DE EMBEDDINGS — FOLD 2
# TCC 1 — Avaliação do conjunto de validação
# ============================================================

import time
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

print("=" * 65)
print("EXTRAÇÃO DE EMBEDDINGS — FOLD 2")
print("=" * 65)

DIR_AVALIACAO_F2 = DIR_RESULTADOS_FOLD2 / "avaliacao"
DIR_AVALIACAO_F2.mkdir(parents=True, exist_ok=True)

ARQUIVO_EMBEDDINGS_F2 = (
    DIR_AVALIACAO_F2 / "embeddings_validacao.npy"
)
ARQUIVO_METADADOS_F2 = (
    DIR_AVALIACAO_F2 / "metadados_validacao.csv"
)

assert ARQUIVO_MODELO_FINAL_F2.exists(), (
    "Modelo final do fold 2 não encontrado."
)

# ------------------------------------------------------------
# 1. Garantir pesos finais do fold 2
# ------------------------------------------------------------

modelo_embeddings_f2.load_weights(
    ARQUIVO_MODELO_FINAL_F2
)

# ------------------------------------------------------------
# 2. Preparar imagens e metadados
# ------------------------------------------------------------

df_avaliacao_f2 = df_validacao_f2.copy()

df_avaliacao_f2 = df_avaliacao_f2.sort_values(
    ["dataset", "identidade", "caminho_local"]
).reset_index(drop=True)

caminhos = df_avaliacao_f2[
    "caminho_local"
].astype(str).tolist()

assert len(caminhos) == 5209
assert len(set(caminhos)) == len(caminhos), (
    "Existem caminhos duplicados na validação."
)

# ------------------------------------------------------------
# 3. Dataset de inferência
# ------------------------------------------------------------

def carregar_para_inferencia(caminho):

    conteudo = tf.io.read_file(caminho)

    imagem = tf.io.decode_image(
        conteudo,
        channels=3,
        expand_animations=False
    )

    imagem.set_shape([None, None, 3])

    imagem = tf.image.resize(
        imagem,
        [224, 224]
    )

    imagem = tf.keras.applications.vgg16.preprocess_input(
        tf.cast(imagem, tf.float32)
    )

    return imagem

dataset_inferencia_f2 = (
    tf.data.Dataset.from_tensor_slices(caminhos)
    .map(
        carregar_para_inferencia,
        num_parallel_calls=4
    )
    .batch(32)
    .prefetch(2)
)

# ------------------------------------------------------------
# 4. Extração dos embeddings
# ------------------------------------------------------------

inicio = time.time()
embeddings_partes = []

for numero, imagens in enumerate(
    dataset_inferencia_f2,
    start=1
):

    embeddings = modelo_embeddings_f2(
        imagens,
        training=False
    )

    embeddings_partes.append(
        embeddings.numpy()
    )

    if numero % 25 == 0:
        print(
            f"Lotes processados: {numero}",
            flush=True
        )

embeddings_f2 = np.concatenate(
    embeddings_partes,
    axis=0
).astype(np.float32)

# ------------------------------------------------------------
# 5. Verificações essenciais
# ------------------------------------------------------------

assert embeddings_f2.shape == (5209, 128)
assert np.isfinite(embeddings_f2).all()

normas = np.linalg.norm(
    embeddings_f2,
    axis=1
)

assert np.allclose(
    normas,
    1.0,
    atol=1e-4
), "Embeddings não estão normalizados."

# ------------------------------------------------------------
# 6. Persistência
# ------------------------------------------------------------

np.save(
    ARQUIVO_EMBEDDINGS_F2,
    embeddings_f2
)

df_avaliacao_f2.to_csv(
    ARQUIVO_METADADOS_F2,
    index=False
)

resumo = {
    "fold": 2,
    "imagens": int(len(embeddings_f2)),
    "dimensoes": int(embeddings_f2.shape[1]),
    "identidades": int(
        df_avaliacao_f2["identidade"].nunique()
    ),
    "norma_minima": float(normas.min()),
    "norma_maxima": float(normas.max()),
    "tempo_segundos": float(time.time() - inicio)
}

with open(
    DIR_AVALIACAO_F2 / "resumo_extracao.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(resumo, f, indent=2)

print("\n" + "=" * 65)
print("EXTRAÇÃO CONCLUÍDA — FOLD 2")
print("=" * 65)

print("Embeddings:", embeddings_f2.shape)
print("Identidades:", resumo["identidades"])
print("Normas:", resumo["norma_minima"], resumo["norma_maxima"])
print("Tempo (min):", round(resumo["tempo_segundos"] / 60, 2))
print("Arquivo:", ARQUIVO_EMBEDDINGS_F2)
print("Bloco [71] concluído.")


In [ ]:

# ============================================================
# BLOCO [72] — AVALIAÇÃO COMPLETA — FOLD 2
# TCC 1 — Verificação e recuperação por embeddings
# ============================================================

import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 65)
print("AVALIAÇÃO COMPLETA — FOLD 2")
print("=" * 65)

SEED = 42
rng = np.random.default_rng(SEED)

# ------------------------------------------------------------
# 1. Carregar embeddings e metadados
# ------------------------------------------------------------

emb = np.load(ARQUIVO_EMBEDDINGS_F2)

meta = pd.read_csv(
    ARQUIVO_METADADOS_F2
)

assert len(emb) == len(meta) == 5209
assert emb.shape[1] == 128

# Identidade global: dataset + identidade
identidades = (
    meta["dataset"].astype(str)
    + "::"
    + meta["identidade"].astype(str)
).to_numpy()

n = len(emb)

print("Imagens:", n)
print("Identidades:", len(np.unique(identidades)))

# ------------------------------------------------------------
# 2. Pares balanceados para verificação
# ------------------------------------------------------------

grupos = {
    identidade: np.where(
        identidades == identidade
    )[0]
    for identidade in np.unique(identidades)
}

pares_positivos = []

for indices in grupos.values():
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            pares_positivos.append(
                (indices[i], indices[j])
            )

pares_positivos = np.asarray(
    pares_positivos,
    dtype=np.int32
)

quantidade_positivos = len(pares_positivos)

pares_negativos = set()

while len(pares_negativos) < quantidade_positivos:

    i = int(rng.integers(n))
    j = int(rng.integers(n))

    if i == j or identidades[i] == identidades[j]:
        continue

    pares_negativos.add(
        (min(i, j), max(i, j))
    )

pares_negativos = np.asarray(
    sorted(pares_negativos),
    dtype=np.int32
)

# ------------------------------------------------------------
# 3. Distâncias e métricas de verificação
# ------------------------------------------------------------

def distancias_pares(pares):

    diferenca = (
        emb[pares[:, 0]]
        - emb[pares[:, 1]]
    )

    return np.linalg.norm(
        diferenca,
        axis=1
    )

dist_pos = distancias_pares(
    pares_positivos
)

dist_neg = distancias_pares(
    pares_negativos
)

y_true = np.concatenate([
    np.ones(len(dist_pos), dtype=np.int32),
    np.zeros(len(dist_neg), dtype=np.int32)
])

distancias = np.concatenate([
    dist_pos,
    dist_neg
])

scores = -distancias

auc = roc_auc_score(
    y_true,
    scores
)

fpr, tpr, thresholds = roc_curve(
    y_true,
    scores
)

fnr = 1 - tpr

indice_eer = np.argmin(
    np.abs(fpr - fnr)
)

eer = float(
    (fpr[indice_eer] + fnr[indice_eer]) / 2
)

limiar_distancia = float(
    -thresholds[indice_eer]
)

y_pred = (
    distancias <= limiar_distancia
).astype(np.int32)

precision = precision_score(
    y_true, y_pred,
    zero_division=0
)

recall = recall_score(
    y_true, y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true, y_pred,
    zero_division=0
)

matriz = confusion_matrix(
    y_true, y_pred
)

# ------------------------------------------------------------
# 4. Recuperação global leave-one-out
# ------------------------------------------------------------

# Similaridade de cosseno: embeddings L2-normalizados
# Equivale à ordenação por distância euclidiana.

similaridades = emb @ emb.T

np.fill_diagonal(
    similaridades,
    -np.inf
)

ordem = np.argsort(
    -similaridades,
    axis=1
)

acertos = (
    identidades[ordem]
    == identidades[:, None]
)

hit1 = float(
    np.mean(acertos[:, 0])
)

hit5 = float(
    np.mean(np.any(acertos[:, :5], axis=1))
)

hit10 = float(
    np.mean(np.any(acertos[:, :10], axis=1))
)

# ------------------------------------------------------------
# 5. Average Precision e Reciprocal Rank
# ------------------------------------------------------------

aps = []
rrs = []

for linha in acertos:

    posicoes = np.flatnonzero(linha)

    if len(posicoes) == 0:
        aps.append(0.0)
        rrs.append(0.0)
        continue

    precisao_acumulada = (
        np.arange(1, len(posicoes) + 1)
        / (posicoes + 1)
    )

    aps.append(
        float(np.mean(precisao_acumulada))
    )

    rrs.append(
        float(1.0 / (posicoes[0] + 1))
    )

map_score = float(np.mean(aps))
mrr = float(np.mean(rrs))

# ------------------------------------------------------------
# 6. Métricas por dataset
# ------------------------------------------------------------

metricas_dataset = {}

for dataset in sorted(meta["dataset"].unique()):

    mascara = (
        meta["dataset"].to_numpy() == dataset
    )

    metricas_dataset[str(dataset)] = {
        "imagens": int(mascara.sum()),
        "hit1": float(
            np.mean(acertos[mascara, 0])
        ),
        "hit5": float(
            np.mean(
                np.any(acertos[mascara, :5], axis=1)
            )
        ),
        "hit10": float(
            np.mean(
                np.any(acertos[mascara, :10], axis=1)
            )
        )
    }

# ------------------------------------------------------------
# 7. Salvar resultados
# ------------------------------------------------------------

resultados_f2 = {
    "fold": 2,
    "imagens_validacao": int(n),
    "identidades_validacao": int(
        len(np.unique(identidades))
    ),
    "verificacao": {
        "pares_positivos": int(len(dist_pos)),
        "pares_negativos": int(len(dist_neg)),
        "roc_auc": float(auc),
        "eer_aproximado": eer,
        "limiar_distancia_eer": limiar_distancia,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "matriz_confusao": matriz.tolist()
    },
    "recuperacao": {
        "hit1": hit1,
        "hit5": hit5,
        "hit10": hit10,
        "map": map_score,
        "mrr": mrr
    },
    "por_dataset": metricas_dataset
}

ARQUIVO_RESULTADOS_F2 = (
    DIR_AVALIACAO_F2
    / "metricas_fold2.json"
)

with open(
    ARQUIVO_RESULTADOS_F2,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        resultados_f2,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 8. Exibir resultados
# ------------------------------------------------------------

print("\nVERIFICAÇÃO")
print("Pares positivos:", len(dist_pos))
print("Pares negativos:", len(dist_neg))
print(f"ROC-AUC: {auc:.6f}")
print(f"EER aproximado: {eer:.6f}")
print(f"Limiar EER: {limiar_distancia:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall: {recall:.6f}")
print(f"F1: {f1:.6f}")

print("\nRECUPERAÇÃO GLOBAL")
print(f"Hit@1: {hit1:.2%}")
print(f"Hit@5: {hit5:.2%}")
print(f"Hit@10: {hit10:.2%}")
print(f"mAP: {map_score:.2%}")
print(f"MRR: {mrr:.2%}")

print("\nPOR DATASET")

for dataset, dados in metricas_dataset.items():
    print(
        f"{dataset}: "
        f"Hit@1={dados['hit1']:.2%}, "
        f"Hit@5={dados['hit5']:.2%}, "
        f"Hit@10={dados['hit10']:.2%}"
    )

print("\nArquivo:", ARQUIVO_RESULTADOS_F2)
print("Bloco [72] concluído.")


## 6. Treinamento e avaliação — Fold 3

Inclui continuação de sessão, extração de embeddings e avaliação quantitativa.

In [ ]:

# ============================================================
# BLOCO [73] — PREPARAÇÃO DO FOLD 3
# TCC 1 — Validação cruzada por identidade
# ============================================================

import json
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

print("=" * 65)
print("PREPARAÇÃO DO FOLD 3")
print("=" * 65)

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# 1. Carregar divisões
# ------------------------------------------------------------

ARQUIVO_TREINO_F3 = (
    RAIZ / "splits/fold_3_imagens_treino.csv"
)

ARQUIVO_VALIDACAO_F3 = (
    RAIZ / "splits/fold_3_imagens_validacao.csv"
)

assert ARQUIVO_TREINO_F3.exists()
assert ARQUIVO_VALIDACAO_F3.exists()

df_treino_f3 = pd.read_csv(ARQUIVO_TREINO_F3)
df_validacao_f3 = pd.read_csv(ARQUIVO_VALIDACAO_F3)

for df in [df_treino_f3, df_validacao_f3]:
    df["caminho_local"] = df["caminho"].map(
        resolver_caminho_local
    )

# ------------------------------------------------------------
# 2. Separação das identidades
# ------------------------------------------------------------

ids_treino = set(
    df_treino_f3["identidade"].astype(str)
)

ids_validacao = set(
    df_validacao_f3["identidade"].astype(str)
)

ids_teste = set(
    df_teste_f2["identidade"].astype(str)
)

assert not (ids_treino & ids_validacao)
assert not (ids_treino & ids_teste)
assert not (ids_validacao & ids_teste)

print("Imagens de treino:", len(df_treino_f3))
print("Imagens de validação:", len(df_validacao_f3))
print("Identidades de treino:", len(ids_treino))
print("Identidades de validação:", len(ids_validacao))

# ------------------------------------------------------------
# 3. Criar índices de amostragem
# ------------------------------------------------------------

indice_treino_f3 = construir_indice(
    df_treino_f3
)

indice_validacao_f3 = construir_indice(
    df_validacao_f3
)

# ------------------------------------------------------------
# 4. Inicializar modelo do zero (ImageNet)
# ------------------------------------------------------------

entradas_f3 = tf.keras.Input(
    shape=(224, 224, 3),
    name="entrada_imagem"
)

backbone_f3 = tf.keras.applications.VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

backbone_f3.trainable = False

x = backbone_f3(
    entradas_f3,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dense(
    256,
    activation="relu",
    name="dense_256"
)(x)

x = tf.keras.layers.Dense(
    128,
    activation="linear",
    name="dense_128"
)(x)

saidas_f3 = tf.keras.layers.UnitNormalization(
    axis=-1,
    name="normalizacao_l2"
)(x)

modelo_embeddings_f3 = tf.keras.Model(
    entradas_f3,
    saidas_f3,
    name="modelo_embeddings_fold3"
)

# ------------------------------------------------------------
# 5. Diretórios do fold 3
# ------------------------------------------------------------

DIR_RESULTADOS_FOLD3 = (
    RAIZ / "resultados/fold_3"
)

DIR_CHECKPOINT_FOLD3 = (
    RAIZ / "checkpoints/fold_3"
)

DIR_RESULTADOS_FOLD3.mkdir(
    parents=True,
    exist_ok=True
)

DIR_CHECKPOINT_FOLD3.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 6. Confirmação
# ------------------------------------------------------------

assert modelo_embeddings_f3.output_shape == (
    None, 128
)

print("\nMODELO FOLD 3")
print("Parâmetros:", modelo_embeddings_f3.count_params())
print("Backbone congelado:", not backbone_f3.trainable)
print("GPU:", tf.config.list_physical_devices("GPU"))

print("\n" + "=" * 65)
print("FOLD 3 PREPARADO")
print("=" * 65)
print("Bloco [73] concluído.")


In [ ]:

# ============================================================
# BLOCO [74] — TREINAMENTO COMPLETO — FOLD 3
# TCC 1 | VGG16 | Triplet Loss semi-hard | P=8, K=4
# ============================================================

import json
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

print("=" * 65)
print("TREINAMENTO COMPLETO — FOLD 3")
print("=" * 65)

SEED = 42
P, K = 8, 4
BATCH_SIZE = 32
MARGEM = 0.2
EPOCAS_FASE1 = 10
EPOCAS_FASE2 = 10
PACIENCIA = 4
PASSOS_POR_EPOCA = int(np.ceil(len(df_treino_f3) / BATCH_SIZE))

# ------------------------------------------------------------
# 1. Índices e rótulos
# ------------------------------------------------------------

ids_treino_f3 = sorted(indice_treino_f3)
mapa_treino_f3 = {
    identidade: i
    for i, identidade in enumerate(ids_treino_f3)
}

# Validação fixa, seguindo o fold 2
arquivo_lotes_f3 = DIR_RESULTADOS_FOLD3 / "lotes_validacao.json"

if arquivo_lotes_f3.exists():
    with open(arquivo_lotes_f3, encoding="utf-8") as f:
        lotes_val_f3 = json.load(f)
else:
    rng_val = np.random.default_rng(SEED + 300)
    ids_val = sorted(indice_validacao_f3)
    lotes_val_f3 = []

    for numero in range(150):
        escolhidas = rng_val.choice(ids_val, size=P, replace=False)
        caminhos, identidades = [], []

        for identidade in escolhidas:
            fotos = indice_validacao_f3[identidade]
            selecionadas = rng_val.choice(
                fotos, size=K, replace=len(fotos) < K
            )
            caminhos.extend(map(str, selecionadas))
            identidades.extend([identidade] * K)

        lotes_val_f3.append({
            "lote": numero + 1,
            "caminhos": caminhos,
            "identidades": identidades
        })

    with open(arquivo_lotes_f3, "w", encoding="utf-8") as f:
        json.dump(lotes_val_f3, f, ensure_ascii=False)

ids_val_f3 = sorted(indice_validacao_f3)
mapa_val_f3 = {identidade: i for i, identidade in enumerate(ids_val_f3)}

# ------------------------------------------------------------
# 2. Pré-processamento VGG16
# ------------------------------------------------------------

def preparar_imagem_f3(caminho, rotulo):
    conteudo = tf.io.read_file(caminho)
    imagem = tf.io.decode_image(
        conteudo, channels=3, expand_animations=False
    )
    imagem.set_shape([None, None, 3])
    imagem = tf.image.resize(imagem, [224, 224])
    imagem = tf.keras.applications.vgg16.preprocess_input(
        tf.cast(imagem, tf.float32)
    )
    return imagem, rotulo

def dataset_f3(caminhos, rotulos):
    return (
        tf.data.Dataset.from_tensor_slices((
            tf.constant(caminhos, dtype=tf.string),
            tf.constant(rotulos, dtype=tf.int32)
        ))
        .map(preparar_imagem_f3, num_parallel_calls=4)
        .batch(BATCH_SIZE, drop_remainder=True)
        .prefetch(2)
    )

caminhos_val_f3 = []
rotulos_val_f3 = []

for lote in lotes_val_f3:
    caminhos_val_f3.extend(lote["caminhos"])
    rotulos_val_f3.extend([
        mapa_val_f3[identidade]
        for identidade in lote["identidades"]
    ])

dataset_validacao_f3 = dataset_f3(caminhos_val_f3, rotulos_val_f3)

def criar_dataset_treino_f3(epoca):
    rng = np.random.default_rng(SEED + epoca)
    caminhos, rotulos = [], []

    for _ in range(PASSOS_POR_EPOCA):
        escolhidas = rng.choice(ids_treino_f3, size=P, replace=False)

        for identidade in escolhidas:
            fotos = indice_treino_f3[identidade]
            selecionadas = rng.choice(
                fotos, size=K, replace=len(fotos) < K
            )
            caminhos.extend(map(str, selecionadas))
            rotulos.extend([mapa_treino_f3[identidade]] * K)

    return dataset_f3(caminhos, rotulos)

# ------------------------------------------------------------
# 3. Triplet Loss semi-hard
# ------------------------------------------------------------

@tf.function
def triplet_loss_f3(embeddings, rotulos):
    rotulos = tf.reshape(rotulos, [-1])
    produto = tf.matmul(embeddings, embeddings, transpose_b=True)
    normas = tf.reduce_sum(tf.square(embeddings), axis=1, keepdims=True)
    dist = tf.maximum(normas - 2 * produto + tf.transpose(normas), 0.0)

    n = tf.shape(dist)[0]
    iguais = tf.equal(rotulos[:, None], rotulos[None, :])
    positivos = iguais & ~tf.eye(n, dtype=tf.bool)
    negativos = ~iguais

    d_ap = dist[:, :, None]
    d_an = dist[:, None, :]

    candidatos = (
        negativos[:, None, :]
        & (d_an > d_ap)
        & (d_an < d_ap + MARGEM)
    )

    inf = tf.constant(float("inf"), dtype=dist.dtype)
    melhor_semi = tf.reduce_min(
        tf.where(candidatos, d_an, inf), axis=2
    )
    negativo_proximo = tf.reduce_min(
        tf.where(negativos, dist, inf), axis=1
    )[:, None]

    d_neg = tf.where(
        tf.math.is_finite(melhor_semi),
        melhor_semi,
        negativo_proximo
    )

    perdas = tf.where(
        positivos,
        tf.maximum(dist + MARGEM - d_neg, 0.0),
        0.0
    )

    return tf.reduce_sum(perdas) / tf.maximum(
        tf.reduce_sum(tf.cast(positivos, tf.float32)),
        1.0
    )

# ------------------------------------------------------------
# 4. Função de treinamento por fase
# ------------------------------------------------------------

def treinar_fase_f3(fase, epocas, learning_rate, paciencia=None):

    otimizador = tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    @tf.function
    def passo_treino(imagens, rotulos):
        with tf.GradientTape() as tape:
            embeddings = modelo_embeddings_f3(
                imagens, training=True
            )
            perda = triplet_loss_f3(embeddings, rotulos)

        variaveis = modelo_embeddings_f3.trainable_variables
        gradientes = tape.gradient(perda, variaveis)

        otimizador.apply_gradients([
            (g, v)
            for g, v in zip(gradientes, variaveis)
            if g is not None
        ])
        return perda

    @tf.function
    def passo_val(imagens, rotulos):
        embeddings = modelo_embeddings_f3(
            imagens, training=False
        )
        return triplet_loss_f3(embeddings, rotulos)

    checkpoint = (
        DIR_CHECKPOINT_FOLD3 / f"melhor_fase{fase}.weights.h5"
    )
    arquivo_historico = (
        DIR_RESULTADOS_FOLD3 / f"historico_fase{fase}.csv"
    )

    assert not checkpoint.exists(), f"Checkpoint já existe: {checkpoint}"
    assert not arquivo_historico.exists(), f"Histórico já existe: {arquivo_historico}"

    melhor = float("inf")
    sem_melhora = 0
    historico = []

    print(f"\nFASE {fase} | LR={learning_rate} | Épocas={epocas}", flush=True)

    for epoca in range(1, epocas + 1):
        inicio = time.time()
        perdas_treino = []

        for imagens, rotulos in criar_dataset_treino_f3(
            epoca + (10 if fase == 2 else 0)
        ):
            perdas_treino.append(float(
                passo_treino(imagens, rotulos).numpy()
            ))

        perdas_val = []
        for imagens, rotulos in dataset_validacao_f3:
            perdas_val.append(float(
                passo_val(imagens, rotulos).numpy()
            ))

        lt = float(np.mean(perdas_treino))
        lv = float(np.mean(perdas_val))
        assert np.isfinite(lt) and np.isfinite(lv)

        melhorou = lv < melhor
        if melhorou:
            melhor = lv
            sem_melhora = 0
            modelo_embeddings_f3.save_weights(checkpoint)
        else:
            sem_melhora += 1

        historico.append({
            "epoca": epoca,
            "loss_treino": lt,
            "loss_validacao": lv,
            "melhor_checkpoint": melhorou,
            "duracao_segundos": time.time() - inicio
        })
        pd.DataFrame(historico).to_csv(
            arquivo_historico, index=False
        )

        print(
            f"Época {epoca:02d}/{epocas} | "
            f"Treino: {lt:.6f} | Val: {lv:.6f} | "
            f"Tempo: {time.time()-inicio:.1f}s"
            + (" | MELHOR" if melhorou else ""),
            flush=True
        )

        if paciencia and sem_melhora >= paciencia:
            print(f"Early stopping na época {epoca}.")
            break

    modelo_embeddings_f3.load_weights(checkpoint)
    return melhor

# ------------------------------------------------------------
# 5. Fase 1 — VGG16 congelada
# ------------------------------------------------------------

backbone_f3.trainable = False

melhor_fase1_f3 = treinar_fase_f3(
    fase=1,
    epocas=EPOCAS_FASE1,
    learning_rate=1e-4
)

# ------------------------------------------------------------
# 6. Fase 2 — Fine-tuning block5
# ------------------------------------------------------------

backbone_f3.trainable = True

for camada in backbone_f3.layers:
    camada.trainable = camada.name.startswith("block5_")

melhor_fase2_f3 = treinar_fase_f3(
    fase=2,
    epocas=EPOCAS_FASE2,
    learning_rate=1e-5,
    paciencia=PACIENCIA
)

# ------------------------------------------------------------
# 7. Salvar modelo final
# ------------------------------------------------------------

ARQUIVO_MODELO_FINAL_F3 = (
    DIR_CHECKPOINT_FOLD3 / "modelo_final_fold3.weights.h5"
)

assert not ARQUIVO_MODELO_FINAL_F3.exists()

modelo_embeddings_f3.save_weights(
    ARQUIVO_MODELO_FINAL_F3
)

print("\n" + "=" * 65)
print("TREINAMENTO FOLD 3 CONCLUÍDO")
print("=" * 65)
print("Melhor loss fase 1:", melhor_fase1_f3)
print("Melhor loss fase 2:", melhor_fase2_f3)
print("Modelo final:", ARQUIVO_MODELO_FINAL_F3)
print("Bloco [74] concluído.")


In [ ]:

# ============================================================
# BLOCO [75] — RECUPERAÇÃO E CONTINUAÇÃO DO FOLD 3
# Fase 1: épocas 5–10 | Fase 2: até 10 épocas
# ============================================================

import os
import json
import time
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_DADOS = Path("/content/tcc1_kfold/datasets")
DIR_RESULTADOS_FOLD3 = RAIZ / "resultados/fold_3"
DIR_CHECKPOINT_FOLD3 = RAIZ / "checkpoints/fold_3"

DIR_RESULTADOS_FOLD3.mkdir(parents=True, exist_ok=True)
DIR_CHECKPOINT_FOLD3.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Restaurar datasets locais
# ------------------------------------------------------------

backup_dados = RAIZ / "datasets_backup.tar"

if not DIR_DADOS.exists() or not any(DIR_DADOS.rglob("*.*")):
    print("Restaurando datasets...", flush=True)

    with tarfile.open(backup_dados, "r") as tar:
        tar.extractall("/content/tcc1_kfold", filter="data")

# ------------------------------------------------------------
# 2. Carregar divisões e reconstruir caminhos
# ------------------------------------------------------------

df_treino_f3 = pd.read_csv(
    RAIZ / "splits/fold_3_imagens_treino.csv"
)
df_validacao_f3 = pd.read_csv(
    RAIZ / "splits/fold_3_imagens_validacao.csv"
)

def localizar_imagem_f3(caminho):
    caminho = str(caminho)
    p = Path(caminho)

    if p.exists():
        return str(p)

    partes = p.parts

    for marcador in ("DogFaceNet", "YTBB-Dog"):
        if marcador in partes:
            indice = partes.index(marcador)
            candidato = DIR_DADOS.joinpath(*partes[indice:])
            if candidato.exists():
                return str(candidato)

    raise FileNotFoundError(caminho)

for df in (df_treino_f3, df_validacao_f3):
    df["caminho_local"] = df["caminho"].map(
        localizar_imagem_f3
    )

def construir_indice_f3(df):
    return {
        str(identidade): grupo["caminho_local"].tolist()
        for identidade, grupo in df.groupby("identidade")
    }

indice_treino_f3 = construir_indice_f3(df_treino_f3)
indice_validacao_f3 = construir_indice_f3(df_validacao_f3)

# ------------------------------------------------------------
# 3. Reconstruir modelo VGG16
# ------------------------------------------------------------

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

entrada = tf.keras.Input(shape=(224, 224, 3))

backbone_f3 = tf.keras.applications.VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
backbone_f3.trainable = False

x = backbone_f3(entrada, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation="relu", name="dense_256")(x)
x = tf.keras.layers.Dense(128, name="dense_128")(x)
saida = tf.keras.layers.UnitNormalization(
    axis=-1, name="normalizacao_l2"
)(x)

modelo_embeddings_f3 = tf.keras.Model(
    entrada, saida, name="modelo_embeddings_fold3"
)

checkpoint_fase1 = (
    DIR_CHECKPOINT_FOLD3 / "melhor_fase1.weights.h5"
)

assert checkpoint_fase1.exists()

modelo_embeddings_f3.load_weights(checkpoint_fase1)

# ------------------------------------------------------------
# 4. Reconstituir lotes e função de perda
# ------------------------------------------------------------

SEED, P, K, MARGEM = 42, 8, 4, 0.2
BATCH_SIZE = 32
PASSOS_POR_EPOCA = int(
    np.ceil(len(df_treino_f3) / BATCH_SIZE)
)

ids_treino = sorted(indice_treino_f3)
mapa_treino = {v: i for i, v in enumerate(ids_treino)}

ids_val = sorted(indice_validacao_f3)
mapa_val = {v: i for i, v in enumerate(ids_val)}

with open(
    DIR_RESULTADOS_FOLD3 / "lotes_validacao.json",
    encoding="utf-8"
) as f:
    lotes_val = json.load(f)

def preparar_imagem(caminho, rotulo):
    dados = tf.io.read_file(caminho)
    imagem = tf.io.decode_image(
        dados, channels=3, expand_animations=False
    )
    imagem.set_shape([None, None, 3])
    imagem = tf.image.resize(imagem, [224, 224])
    imagem = tf.keras.applications.vgg16.preprocess_input(
        tf.cast(imagem, tf.float32)
    )
    return imagem, rotulo

def montar_dataset(caminhos, rotulos):
    return (
        tf.data.Dataset.from_tensor_slices((
            tf.constant(caminhos, dtype=tf.string),
            tf.constant(rotulos, dtype=tf.int32)
        ))
        .map(preparar_imagem, num_parallel_calls=4)
        .batch(32, drop_remainder=True)
        .prefetch(2)
    )

caminhos_val, rotulos_val = [], []

for lote in lotes_val:
    caminhos_val.extend(lote["caminhos"])
    rotulos_val.extend([
        mapa_val[str(v)] for v in lote["identidades"]
    ])

dataset_val = montar_dataset(caminhos_val, rotulos_val)

def dataset_treino(epoca):
    rng = np.random.default_rng(SEED + epoca)
    caminhos, rotulos = [], []

    for _ in range(PASSOS_POR_EPOCA):
        escolhidas = rng.choice(ids_treino, size=P, replace=False)

        for identidade in escolhidas:
            fotos = indice_treino_f3[identidade]
            escolhidas_fotos = rng.choice(
                fotos, size=K, replace=len(fotos) < K
            )
            caminhos.extend(map(str, escolhidas_fotos))
            rotulos.extend([mapa_treino[identidade]] * K)

    return montar_dataset(caminhos, rotulos)

@tf.function
def triplet_loss(embeddings, rotulos):
    rotulos = tf.reshape(rotulos, [-1])
    produto = tf.matmul(embeddings, embeddings, transpose_b=True)
    normas = tf.reduce_sum(
        tf.square(embeddings), axis=1, keepdims=True
    )
    dist = tf.maximum(
        normas - 2 * produto + tf.transpose(normas), 0.0
    )

    n = tf.shape(dist)[0]
    iguais = tf.equal(rotulos[:, None], rotulos[None, :])
    positivos = iguais & ~tf.eye(n, dtype=tf.bool)
    negativos = ~iguais

    d_ap = dist[:, :, None]
    d_an = dist[:, None, :]

    candidatos = (
        negativos[:, None, :]
        & (d_an > d_ap)
        & (d_an < d_ap + MARGEM)
    )

    infinito = tf.constant(float("inf"), dtype=dist.dtype)
    semi = tf.reduce_min(
        tf.where(candidatos, d_an, infinito), axis=2
    )
    proximo = tf.reduce_min(
        tf.where(negativos, dist, infinito), axis=1
    )[:, None]

    d_neg = tf.where(tf.math.is_finite(semi), semi, proximo)

    perdas = tf.where(
        positivos,
        tf.maximum(dist + MARGEM - d_neg, 0.0),
        0.0
    )

    return tf.reduce_sum(perdas) / tf.maximum(
        tf.reduce_sum(tf.cast(positivos, tf.float32)), 1.0
    )

# ------------------------------------------------------------
# 5. Função de treinamento com estado persistente
# ------------------------------------------------------------

def executar_fase(fase, inicio_epoca, fim_epoca, lr, paciencia=None):

    otimizador = tf.keras.optimizers.Adam(learning_rate=lr)

    @tf.function
    def passo_treino(imagens, rotulos):
        with tf.GradientTape() as tape:
            emb = modelo_embeddings_f3(imagens, training=True)
            perda = triplet_loss(emb, rotulos)

        variaveis = modelo_embeddings_f3.trainable_variables
        grads = tape.gradient(perda, variaveis)
        otimizador.apply_gradients([
            (g, v) for g, v in zip(grads, variaveis)
            if g is not None
        ])
        return perda

    @tf.function
    def passo_val(imagens, rotulos):
        emb = modelo_embeddings_f3(imagens, training=False)
        return triplet_loss(emb, rotulos)

    arquivo_historico = (
        DIR_RESULTADOS_FOLD3 / f"historico_fase{fase}.csv"
    )
    arquivo_melhor = (
        DIR_CHECKPOINT_FOLD3 / f"melhor_fase{fase}.weights.h5"
    )

    if arquivo_historico.exists():
        historico = pd.read_csv(arquivo_historico).to_dict("records")
    else:
        historico = []

    melhor = min(
        (float(h["loss_validacao"]) for h in historico),
        default=float("inf")
    )
    sem_melhora = 0

    estado = tf.train.Checkpoint(
        model=modelo_embeddings_f3,
        optimizer=otimizador
    )
    gestor = tf.train.CheckpointManager(
        estado,
        str(DIR_CHECKPOINT_FOLD3 / f"estado_fase{fase}"),
        max_to_keep=2
    )

    if gestor.latest_checkpoint:
        estado.restore(gestor.latest_checkpoint)
        print("Estado do otimizador recuperado:", gestor.latest_checkpoint)

    for epoca in range(inicio_epoca, fim_epoca + 1):
        inicio = time.time()
        perdas_treino = []

        indice_amostragem = epoca + (10 if fase == 2 else 0)

        for imagens, rotulos in dataset_treino(indice_amostragem):
            perdas_treino.append(float(
                passo_treino(imagens, rotulos).numpy()
            ))

        perdas_val = [
            float(passo_val(imagens, rotulos).numpy())
            for imagens, rotulos in dataset_val
        ]

        lt = float(np.mean(perdas_treino))
        lv = float(np.mean(perdas_val))

        assert np.isfinite(lt) and np.isfinite(lv)

        melhorou = lv < melhor
        if melhorou:
            melhor = lv
            sem_melhora = 0
            modelo_embeddings_f3.save_weights(arquivo_melhor)
        else:
            sem_melhora += 1

        historico.append({
            "epoca": epoca,
            "loss_treino": lt,
            "loss_validacao": lv,
            "melhor_checkpoint": melhorou,
            "duracao_segundos": time.time() - inicio
        })

        pd.DataFrame(historico).to_csv(
            arquivo_historico, index=False
        )
        gestor.save(checkpoint_number=epoca)

        print(
            f"Fase {fase} | Época {epoca:02d}/{fim_epoca} | "
            f"Treino: {lt:.6f} | Val: {lv:.6f} | "
            f"Tempo: {time.time()-inicio:.1f}s"
            + (" | MELHOR" if melhorou else ""),
            flush=True
        )

        if paciencia and sem_melhora >= paciencia:
            print("Early stopping acionado.")
            break

    modelo_embeddings_f3.load_weights(arquivo_melhor)
    return melhor


# ============================================================
# BLOCO [75] — RETOMADA SEGURA DO FOLD 3
# Executar apenas a fase 1
# ============================================================

import re

FASE = 1
TOTAL_EPOCAS = 10

arquivo_historico = (
    DIR_RESULTADOS_FOLD3 / "historico_fase1.csv"
)

diretorio_estado = (
    DIR_CHECKPOINT_FOLD3 / "estado_fase1"
)

arquivo_melhor = (
    DIR_CHECKPOINT_FOLD3 / "melhor_fase1.weights.h5"
)

# 1. Verificar histórico
assert arquivo_historico.exists(), (
    "Histórico da fase 1 não encontrado."
)

historico = pd.read_csv(arquivo_historico)

assert not historico.empty, (
    "Histórico vazio. Verifique o Drive."
)

epoca_historico = int(historico["epoca"].max())

# 2. Verificar checkpoint completo
ultimo_checkpoint = tf.train.latest_checkpoint(
    str(diretorio_estado)
)

assert ultimo_checkpoint is not None, (
    "Checkpoint completo não encontrado."
)

resultado = re.search(
    r"ckpt-(\d+)$",
    ultimo_checkpoint
)

assert resultado is not None, (
    "Não foi possível identificar a época do checkpoint."
)

epoca_checkpoint = int(resultado.group(1))

print("Última época no histórico:", epoca_historico)
print("Última época no checkpoint:", epoca_checkpoint)
print("Checkpoint:", ultimo_checkpoint)

# Não continuar se histórico e checkpoint divergirem.
assert epoca_historico == epoca_checkpoint, (
    "Histórico e checkpoint divergentes. "
    "É necessário reconciliar os arquivos antes de treinar."
)

assert arquivo_melhor.exists(), (
    "Pesos do melhor modelo não encontrados."
)

# 3. Retomar
proxima_epoca = epoca_checkpoint + 1

if proxima_epoca <= TOTAL_EPOCAS:

    print(
        f"\nRETOMANDO FOLD 3 — FASE 1 "
        f"— ÉPOCA {proxima_epoca}/{TOTAL_EPOCAS}",
        flush=True
    )

    backbone_f3.trainable = False

    melhor_fase1 = executar_fase(
        fase=1,
        inicio_epoca=proxima_epoca,
        fim_epoca=TOTAL_EPOCAS,
        lr=1e-4
    )

    print("\nFase 1 concluída.")

else:

    print("\nFase 1 já possui 10 épocas registradas.")

    modelo_embeddings_f3.load_weights(
        str(arquivo_melhor)
    )

    print("Melhores pesos da fase 1 carregados.")

print("\nPronto para preparar o fine-tuning.")



In [ ]:
import tensorflow as tf
import pandas as pd
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/TCC_Caes/"
    "experimentos/tcc1_kfold"
)

historico = base / "resultados/fold_3/historico_fase1.csv"
estado = base / "checkpoints/fold_3/estado_fase1"

print("Checkpoint:", tf.train.latest_checkpoint(str(estado)))

if historico.exists():
    df = pd.read_csv(historico)
    print(df.tail(5).to_string(index=False))
else:
    print("Histórico não encontrado neste caminho.")

In [ ]:
import tensorflow as tf
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/TCC_Caes/"
    "experimentos/tcc1_kfold/checkpoints/fold_3"
)

print(
    "Último checkpoint:",
    tf.train.latest_checkpoint(str(base / "estado_fase1"))
)

print(
    "Melhores pesos disponíveis:",
    (base / "melhor_fase1.weights.h5").exists()
)

In [ ]:

# ============================================================
# FOLD 3 — FASE 2: FINE-TUNING COM PROGRESSO
# ============================================================

import re
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.auto import tqdm

TOTAL_EPOCAS = 10
PACIENCIA = 4
LR = 1e-5

# ------------------------------------------------------------
# 1. Verificações iniciais
# ------------------------------------------------------------

assert tf.config.list_physical_devices("GPU"), (
    "GPU não detectada. Treinamento cancelado."
)

arquivo_f1 = DIR_RESULTADOS_FOLD3 / "historico_fase1.csv"
historico_f1 = pd.read_csv(arquivo_f1)

assert int(historico_f1["epoca"].max()) == 10, (
    "A fase 1 não está concluída."
)

arquivo_historico_f2 = (
    DIR_RESULTADOS_FOLD3 / "historico_fase2.csv"
)

diretorio_estado_f2 = (
    DIR_CHECKPOINT_FOLD3 / "estado_fase2"
)

arquivo_melhor_f2 = (
    DIR_CHECKPOINT_FOLD3 / "melhor_fase2.weights.h5"
)

ultimo_ckpt = tf.train.latest_checkpoint(
    str(diretorio_estado_f2)
)

if arquivo_historico_f2.exists():
    df = pd.read_csv(arquivo_historico_f2)
    ultima_epoca = (
        int(df["epoca"].max()) if not df.empty else 0
    )
else:
    df = pd.DataFrame()
    ultima_epoca = 0

if ultimo_ckpt:
    m = re.search(r"ckpt-(\d+)$", ultimo_ckpt)
    assert m is not None
    assert int(m.group(1)) == ultima_epoca, (
        "Checkpoint e histórico divergentes."
    )
else:
    assert ultima_epoca == 0, (
        "Histórico encontrado sem checkpoint."
    )

# ------------------------------------------------------------
# 2. Configurar o modelo
# ------------------------------------------------------------

if ultimo_ckpt is None:
    modelo_embeddings_f3.load_weights(
        str(
            DIR_CHECKPOINT_FOLD3 /
            "melhor_fase1.weights.h5"
        )
    )
    print("Pesos da fase 1 carregados.", flush=True)
else:
    print(
        "Retomada da fase 2:",
        ultimo_ckpt,
        flush=True
    )

backbone_f3.trainable = True

for camada in backbone_f3.layers:
    camada.trainable = (
        camada.name.startswith("block5_")
    )

print("\nConfiguração do fine-tuning")
print("Fold: 3")
print("Fase: 2")
print("Learning rate:", LR)
print("Épocas concluídas:", ultima_epoca)
print("Épocas restantes:", TOTAL_EPOCAS - ultima_epoca)

# ------------------------------------------------------------
# 3. Função de treinamento com barras de progresso
# ------------------------------------------------------------

def executar_fase(
    fase,
    inicio_epoca,
    fim_epoca,
    lr,
    paciencia=None
):

    otimizador = tf.keras.optimizers.Adam(
        learning_rate=lr
    )

    @tf.function
    def passo_treino(imagens, rotulos):

        with tf.GradientTape() as tape:
            emb = modelo_embeddings_f3(
                imagens, training=True
            )
            perda = triplet_loss(emb, rotulos)

        variaveis = (
            modelo_embeddings_f3.trainable_variables
        )

        grads = tape.gradient(
            perda, variaveis
        )

        pares = [
            (g, v)
            for g, v in zip(grads, variaveis)
            if g is not None
        ]

        otimizador.apply_gradients(pares)

        return perda

    @tf.function
    def passo_val(imagens, rotulos):

        emb = modelo_embeddings_f3(
            imagens, training=False
        )

        return triplet_loss(emb, rotulos)

    arquivo_historico = (
        DIR_RESULTADOS_FOLD3 /
        f"historico_fase{fase}.csv"
    )

    arquivo_melhor = (
        DIR_CHECKPOINT_FOLD3 /
        f"melhor_fase{fase}.weights.h5"
    )

    if arquivo_historico.exists():
        historico = pd.read_csv(
            arquivo_historico
        ).to_dict("records")
    else:
        historico = []

    melhor = min(
        (
            float(h["loss_validacao"])
            for h in historico
        ),
        default=float("inf")
    )

    # Recuperar a sequência de épocas sem melhoria.
    sem_melhora = 0

    for h in reversed(historico):
        if bool(h["melhor_checkpoint"]):
            break
        sem_melhora += 1

    estado = tf.train.Checkpoint(
        model=modelo_embeddings_f3,
        optimizer=otimizador
    )

    gestor = tf.train.CheckpointManager(
        estado,
        str(
            DIR_CHECKPOINT_FOLD3 /
            f"estado_fase{fase}"
        ),
        max_to_keep=2
    )

    if gestor.latest_checkpoint:

        estado.restore(
            gestor.latest_checkpoint
        )

        print(
            "Checkpoint restaurado:",
            gestor.latest_checkpoint,
            flush=True
        )

    # Barra principal: progresso das épocas.
    barra_epocas = tqdm(
        total=fim_epoca,
        initial=inicio_epoca - 1,
        desc=f"Fold 3 | Fase {fase}",
        unit="época"
    )

    if paciencia and sem_melhora >= paciencia:
        print(
            "Early stopping já atingido "
            "no histórico anterior."
        )
        barra_epocas.close()
        modelo_embeddings_f3.load_weights(
            str(arquivo_melhor)
        )
        return melhor

    for epoca in range(
        inicio_epoca,
        fim_epoca + 1
    ):

        inicio = time.time()

        perdas_treino = []
        perdas_val = []

        indice_amostragem = (
            epoca + (10 if fase == 2 else 0)
        )

        print(
            f"\nÉpoca {epoca}/{fim_epoca} "
            f"| Restantes: {fim_epoca-epoca}",
            flush=True
        )

        # --------------------------------------------
        # Treinamento
        # --------------------------------------------

        barra_treino = tqdm(
            dataset_treino(indice_amostragem),
            desc=f"Época {epoca} | Treino",
            unit="batch",
            leave=False
        )

        for imagens, rotulos in barra_treino:

            perda = float(
                passo_treino(
                    imagens, rotulos
                ).numpy()
            )

            perdas_treino.append(perda)

            barra_treino.set_postfix(
                loss=f"{np.mean(perdas_treino):.6f}"
            )

        barra_treino.close()

        # --------------------------------------------
        # Validação
        # --------------------------------------------

        barra_val = tqdm(
            dataset_val,
            desc=f"Época {epoca} | Validação",
            unit="batch",
            leave=False
        )

        for imagens, rotulos in barra_val:

            perda = float(
                passo_val(
                    imagens, rotulos
                ).numpy()
            )

            perdas_val.append(perda)

            barra_val.set_postfix(
                loss=f"{np.mean(perdas_val):.6f}"
            )

        barra_val.close()

        assert perdas_treino and perdas_val, (
            "Treinamento ou validação sem batches."
        )

        lt = float(np.mean(perdas_treino))
        lv = float(np.mean(perdas_val))

        assert np.isfinite(lt) and np.isfinite(lv)

        # --------------------------------------------
        # Melhor modelo
        # --------------------------------------------

        melhorou = lv < melhor

        if melhorou:

            melhor = lv
            sem_melhora = 0

            modelo_embeddings_f3.save_weights(
                str(arquivo_melhor)
            )

        else:
            sem_melhora += 1

        duracao = time.time() - inicio

        # --------------------------------------------
        # Persistência
        # --------------------------------------------

        historico.append({
            "epoca": epoca,
            "loss_treino": lt,
            "loss_validacao": lv,
            "melhor_checkpoint": melhorou,
            "duracao_segundos": duracao
        })

        pd.DataFrame(historico).to_csv(
            arquivo_historico,
            index=False
        )

        gestor.save(
            checkpoint_number=epoca
        )

        barra_epocas.update(1)

        barra_epocas.set_postfix(
            treino=f"{lt:.6f}",
            val=f"{lv:.6f}",
            melhor=f"{melhor:.6f}"
        )

        print(
            f"Época {epoca:02d}/{fim_epoca} | "
            f"Treino: {lt:.6f} | "
            f"Val: {lv:.6f} | "
            f"Tempo: {duracao/60:.1f} min | "
            f"Restam {fim_epoca-epoca} épocas"
            + (" | MELHOR" if melhorou else ""),
            flush=True
        )

        if paciencia and sem_melhora >= paciencia:
            print(
                "\nEarly stopping acionado.",
                flush=True
            )
            break

    barra_epocas.close()

    # Restaurar os melhores pesos ao encerrar.
    modelo_embeddings_f3.load_weights(
        str(arquivo_melhor)
    )

    return melhor


# ------------------------------------------------------------
# 4. Executar fase 2
# ------------------------------------------------------------

proxima_epoca = ultima_epoca + 1

if proxima_epoca <= TOTAL_EPOCAS:

    melhor_fase2 = executar_fase(
        fase=2,
        inicio_epoca=proxima_epoca,
        fim_epoca=TOTAL_EPOCAS,
        lr=LR,
        paciencia=PACIENCIA
    )

    print(
        "\nExecução da fase 2 encerrada.",
        flush=True
    )

else:

    assert arquivo_melhor_f2.exists(), (
        "Melhores pesos da fase 2 não encontrados."
    )

    modelo_embeddings_f3.load_weights(
        str(arquivo_melhor_f2)
    )

    print("Fase 2 já possui 10 épocas.")


# ------------------------------------------------------------
# 5. Salvar modelo final
# ------------------------------------------------------------

assert arquivo_melhor_f2.exists(), (
    "Melhores pesos da fase 2 não encontrados."
)

modelo_embeddings_f3.load_weights(
    str(arquivo_melhor_f2)
)

arquivo_final = (
    DIR_CHECKPOINT_FOLD3 /
    "modelo_final_fold3.weights.h5"
)

modelo_embeddings_f3.save_weights(
    str(arquivo_final)
)

print("\n" + "=" * 55)
print("FOLD 3 — FINE-TUNING ENCERRADO")
print("=" * 55)
print("Modelo salvo:", arquivo_final)
print("Melhor loss de validação:", melhor_fase2 if proxima_epoca <= TOTAL_EPOCAS else "Ver histórico")


In [ ]:
from pathlib import Path
import pandas as pd
import tensorflow as tf

raiz = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

pasta_resultados = raiz / "resultados/fold_3"
pasta_ckpt = raiz / "checkpoints/fold_3"

for fase in [1, 2]:
    historico = pasta_resultados / f"historico_fase{fase}.csv"

    print(f"\n=== FOLD 3 — FASE {fase} ===")

    if historico.exists():
        df = pd.read_csv(historico)
        print(df.tail(10).to_string(index=False))
        print("Melhor loss:", df["loss_validacao"].min())
    else:
        print("Histórico não encontrado.")

    estado = pasta_ckpt / f"estado_fase{fase}"
    print(
        "Último checkpoint:",
        tf.train.latest_checkpoint(str(estado))
    )

arquivo_final = pasta_ckpt / "modelo_final_fold3.weights.h5"

print("\n=== MODELO FINAL ===")
print("Arquivo existe:", arquivo_final.exists())

if arquivo_final.exists():
    print("Tamanho:", round(arquivo_final.stat().st_size / 1024**2, 2), "MB")

In [ ]:

# ============================================================
# FOLD 3 — EXTRAÇÃO DE EMBEDDINGS DA VALIDAÇÃO
# Equivalente ao bloco [71] do Fold 2
# ============================================================

import time
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.auto import tqdm

DIR_AVALIACAO_F3 = DIR_RESULTADOS_FOLD3 / "avaliacao"
DIR_AVALIACAO_F3.mkdir(parents=True, exist_ok=True)

ARQUIVO_EMBEDDINGS_F3 = (
    DIR_AVALIACAO_F3 / "embeddings_validacao.npy"
)

ARQUIVO_METADADOS_F3 = (
    DIR_AVALIACAO_F3 / "metadados_validacao.csv"
)

ARQUIVO_MODELO_FINAL_F3 = (
    DIR_CHECKPOINT_FOLD3 /
    "modelo_final_fold3.weights.h5"
)

assert ARQUIVO_MODELO_FINAL_F3.exists(), (
    "Modelo final do Fold 3 não encontrado."
)

# Carregar o melhor modelo do Fold 3
modelo_embeddings_f3.load_weights(
    str(ARQUIVO_MODELO_FINAL_F3)
)

# Preparar o conjunto de validação
df_avaliacao_f3 = (
    df_validacao_f3.copy()
    .sort_values(
        ["dataset", "identidade", "caminho_local"]
    )
    .reset_index(drop=True)
)

caminhos = (
    df_avaliacao_f3["caminho_local"]
    .astype(str)
    .tolist()
)

assert len(caminhos) > 0
assert len(set(caminhos)) == len(caminhos)

def carregar_para_inferencia_f3(caminho):
    conteudo = tf.io.read_file(caminho)

    imagem = tf.io.decode_image(
        conteudo,
        channels=3,
        expand_animations=False
    )

    imagem.set_shape([None, None, 3])

    imagem = tf.image.resize(
        imagem, [224, 224]
    )

    return tf.keras.applications.vgg16.preprocess_input(
        tf.cast(imagem, tf.float32)
    )

dataset_inferencia_f3 = (
    tf.data.Dataset.from_tensor_slices(caminhos)
    .map(
        carregar_para_inferencia_f3,
        num_parallel_calls=4
    )
    .batch(32)
    .prefetch(2)
)

total_lotes = (len(caminhos) + 31) // 32

print("Imagens:", len(caminhos))
print("Lotes previstos:", total_lotes)
print("Iniciando extração...\n", flush=True)

inicio = time.time()
partes = []

for imagens in tqdm(
    dataset_inferencia_f3,
    total=total_lotes,
    desc="Fold 3 | Extração",
    unit="lote"
):
    embeddings = modelo_embeddings_f3(
        imagens,
        training=False
    )
    partes.append(embeddings.numpy())

embeddings_f3 = np.concatenate(
    partes, axis=0
).astype(np.float32)

assert embeddings_f3.shape == (
    len(df_avaliacao_f3), 128
)

assert np.isfinite(embeddings_f3).all()

normas = np.linalg.norm(
    embeddings_f3, axis=1
)

assert np.allclose(
    normas, 1.0, atol=1e-4
), "Embeddings não estão normalizados."

np.save(
    ARQUIVO_EMBEDDINGS_F3,
    embeddings_f3
)

df_avaliacao_f3.to_csv(
    ARQUIVO_METADADOS_F3,
    index=False
)

resumo = {
    "fold": 3,
    "imagens": int(len(embeddings_f3)),
    "dimensoes": int(embeddings_f3.shape[1]),
    "identidades": int(
        (
            df_avaliacao_f3["dataset"].astype(str)
            + "::"
            + df_avaliacao_f3["identidade"].astype(str)
        ).nunique()
    ),
    "norma_minima": float(normas.min()),
    "norma_maxima": float(normas.max()),
    "tempo_segundos": float(time.time() - inicio)
}

with open(
    DIR_AVALIACAO_F3 / "resumo_extracao.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(resumo, f, indent=2)

print("\n=== EXTRAÇÃO CONCLUÍDA — FOLD 3 ===")
print("Formato:", embeddings_f3.shape)
print("Identidades:", resumo["identidades"])
print("Tempo (min):", round(
    resumo["tempo_segundos"] / 60, 2
))
print("Arquivo:", ARQUIVO_EMBEDDINGS_F3)


In [ ]:

# ============================================================
# FOLD 3 — AVALIAÇÃO QUANTITATIVA COMPLETA
# Mesmo protocolo do Fold 2 — Bloco [72]
# ============================================================

import json
import time
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

inicio = time.time()
SEED = 42
rng = np.random.default_rng(SEED)

DIR_AVALIACAO_F3 = (
    DIR_RESULTADOS_FOLD3 / "avaliacao"
)

ARQUIVO_EMBEDDINGS_F3 = (
    DIR_AVALIACAO_F3 / "embeddings_validacao.npy"
)

ARQUIVO_METADADOS_F3 = (
    DIR_AVALIACAO_F3 / "metadados_validacao.csv"
)

# ------------------------------------------------------------
# 1. Carregar embeddings e metadados
# ------------------------------------------------------------

print("1/5 — Carregando embeddings...", flush=True)

emb = np.load(ARQUIVO_EMBEDDINGS_F3)
meta = pd.read_csv(ARQUIVO_METADADOS_F3)

assert len(emb) == len(meta) == 4978
assert emb.shape[1] == 128
assert np.isfinite(emb).all()

identidades = (
    meta["dataset"].astype(str)
    + "::"
    + meta["identidade"].astype(str)
).to_numpy()

n = len(emb)

print("Imagens:", n)
print("Identidades:", len(np.unique(identidades)))

# ------------------------------------------------------------
# 2. Construir pares balanceados
# ------------------------------------------------------------

print("\n2/5 — Construindo pares...", flush=True)

grupos = {
    identidade: np.where(
        identidades == identidade
    )[0]
    for identidade in np.unique(identidades)
}

pares_positivos = []

for indices in tqdm(
    grupos.values(),
    total=len(grupos),
    desc="Pares positivos"
):
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            pares_positivos.append(
                (indices[i], indices[j])
            )

pares_positivos = np.asarray(
    pares_positivos,
    dtype=np.int32
).reshape(-1, 2)

quantidade_positivos = len(pares_positivos)

assert quantidade_positivos > 0


# Construção vetorizada de pares negativos únicos
print(
    "Pares positivos:",
    quantidade_positivos,
    flush=True
)

pares_negativos = np.empty((0, 2), dtype=np.int32)

while len(pares_negativos) < quantidade_positivos:
    faltantes = quantidade_positivos - len(pares_negativos)
    tamanho_lote = max(10000, faltantes * 2)

    candidatos = rng.integers(
        0, n, size=(tamanho_lote, 2)
    ).astype(np.int32)

    candidatos.sort(axis=1)

    mascara = (
        (candidatos[:, 0] != candidatos[:, 1])
        & (
            identidades[candidatos[:, 0]]
            != identidades[candidatos[:, 1]]
        )
    )

    candidatos = candidatos[mascara]

    pares_negativos = np.unique(
        np.concatenate(
            [pares_negativos, candidatos],
            axis=0
        ),
        axis=0
    )

    if len(pares_negativos) > quantidade_positivos:
        pares_negativos = pares_negativos[
            :quantidade_positivos
        ]

    print(
        f"Pares negativos: "
        f"{len(pares_negativos)}/{quantidade_positivos}",
        flush=True
    )

assert len(pares_negativos) == quantidade_positivos
print("Pares balanceados construídos.", flush=True)


# ------------------------------------------------------------
# 3. Métricas de verificação
# ------------------------------------------------------------

print("\n3/5 — Calculando verificação...", flush=True)

def distancias_pares(pares):
    diferenca = (
        emb[pares[:, 0]] - emb[pares[:, 1]]
    )
    return np.linalg.norm(
        diferenca,
        axis=1
    )

dist_pos = distancias_pares(pares_positivos)
dist_neg = distancias_pares(pares_negativos)

y_true = np.concatenate([
    np.ones(len(dist_pos), dtype=np.int32),
    np.zeros(len(dist_neg), dtype=np.int32)
])

distancias = np.concatenate([
    dist_pos, dist_neg
])

scores = -distancias

auc = roc_auc_score(y_true, scores)

fpr, tpr, thresholds = roc_curve(
    y_true, scores
)

fnr = 1 - tpr
indice_eer = np.argmin(np.abs(fpr - fnr))

eer = float(
    (fpr[indice_eer] + fnr[indice_eer]) / 2
)

limiar_distancia = float(
    -thresholds[indice_eer]
)

y_pred = (
    distancias <= limiar_distancia
).astype(np.int32)

precision = precision_score(
    y_true, y_pred, zero_division=0
)

recall = recall_score(
    y_true, y_pred, zero_division=0
)

f1 = f1_score(
    y_true, y_pred, zero_division=0
)

matriz = confusion_matrix(
    y_true, y_pred
)

# ------------------------------------------------------------
# 4. Recuperação global leave-one-out
# ------------------------------------------------------------

print("\n4/5 — Calculando recuperação...", flush=True)

# Embeddings L2-normalizados:
# cosseno e distância euclidiana produzem
# a mesma ordenação dos vizinhos.

similaridades = emb @ emb.T

np.fill_diagonal(
    similaridades, -np.inf
)

print("Ordenando vizinhos...", flush=True)

ordem = np.argsort(
    -similaridades,
    axis=1
)

del similaridades

acertos = (
    identidades[ordem]
    == identidades[:, None]
)

del ordem

hit1 = float(
    np.mean(acertos[:, 0])
)

hit5 = float(
    np.mean(np.any(acertos[:, :5], axis=1))
)

hit10 = float(
    np.mean(np.any(acertos[:, :10], axis=1))
)

aps = []
rrs = []

for linha in tqdm(
    acertos,
    total=n,
    desc="mAP e MRR",
    unit="imagem"
):

    posicoes = np.flatnonzero(linha)

    if len(posicoes) == 0:
        aps.append(0.0)
        rrs.append(0.0)
        continue

    precisao_acumulada = (
        np.arange(1, len(posicoes) + 1)
        / (posicoes + 1)
    )

    aps.append(
        float(np.mean(precisao_acumulada))
    )

    rrs.append(
        float(1.0 / (posicoes[0] + 1))
    )

map_score = float(np.mean(aps))
mrr = float(np.mean(rrs))

# ------------------------------------------------------------
# 5. Métricas por dataset
# ------------------------------------------------------------

metricas_dataset = {}

for dataset in sorted(meta["dataset"].unique()):

    mascara = (
        meta["dataset"].to_numpy() == dataset
    )

    metricas_dataset[str(dataset)] = {
        "imagens": int(mascara.sum()),
        "hit1": float(
            np.mean(acertos[mascara, 0])
        ),
        "hit5": float(
            np.mean(
                np.any(acertos[mascara, :5], axis=1)
            )
        ),
        "hit10": float(
            np.mean(
                np.any(acertos[mascara, :10], axis=1)
            )
        )
    }

# ------------------------------------------------------------
# 6. Persistir resultados
# ------------------------------------------------------------

print("\n5/5 — Salvando métricas...", flush=True)

resultados_f3 = {
    "fold": 3,
    "imagens_validacao": int(n),
    "identidades_validacao": int(
        len(np.unique(identidades))
    ),
    "verificacao": {
        "pares_positivos": int(len(dist_pos)),
        "pares_negativos": int(len(dist_neg)),
        "roc_auc": float(auc),
        "eer_aproximado": eer,
        "limiar_distancia_eer": limiar_distancia,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "matriz_confusao": matriz.tolist()
    },
    "recuperacao": {
        "hit1": hit1,
        "hit5": hit5,
        "hit10": hit10,
        "map": map_score,
        "mrr": mrr
    },
    "por_dataset": metricas_dataset
}

ARQUIVO_RESULTADOS_F3 = (
    DIR_AVALIACAO_F3 / "metricas_fold3.json"
)

with open(
    ARQUIVO_RESULTADOS_F3,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        resultados_f3,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 7. Relatório
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("FOLD 3 — AVALIAÇÃO CONCLUÍDA")
print("=" * 55)

print("\nVERIFICAÇÃO")
print("Pares positivos:", len(dist_pos))
print("Pares negativos:", len(dist_neg))
print(f"ROC-AUC: {auc:.6f}")
print(f"EER aproximado: {eer:.6f}")
print(f"Limiar EER: {limiar_distancia:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall: {recall:.6f}")
print(f"F1: {f1:.6f}")
print("Matriz de confusão:")
print(matriz)

print("\nRECUPERAÇÃO GLOBAL")
print(f"Hit@1: {hit1:.2%}")
print(f"Hit@5: {hit5:.2%}")
print(f"Hit@10: {hit10:.2%}")
print(f"mAP: {map_score:.2%}")
print(f"MRR: {mrr:.2%}")

print("\nPOR DATASET")

for dataset, dados in metricas_dataset.items():
    print(
        f"{dataset}: "
        f"Hit@1={dados['hit1']:.2%}, "
        f"Hit@5={dados['hit5']:.2%}, "
        f"Hit@10={dados['hit10']:.2%}"
    )

print("\nTempo total:",
      round((time.time() - inicio) / 60, 2),
      "minutos")

print("Resultados salvos em:")
print(ARQUIVO_RESULTADOS_F3)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:

# [76] Figura comparativa dos resultados preliminares
# Execução em CPU; não altera os modelos nem os checkpoints.

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

RAIZ = Path("/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold")
DESTINO = RAIZ / "figuras_tcc"
DESTINO.mkdir(parents=True, exist_ok=True)

# Resultados consolidados dos folds 1 e 2 (%)
metricas = ["Hit@1", "Hit@5", "Hit@10", "mAP"]
fold_1 = [92.3886, 97.3804, 98.5767, 68.3193]
fold_2 = [93.74, 97.93, 98.83, 70.13]

x = np.arange(len(metricas))
largura = 0.35

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "pdf.fonttype": 42
})

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - largura/2, fold_1, largura, label="Fold 1")
b2 = ax.bar(x + largura/2, fold_2, largura, label="Fold 2")

ax.set_ylabel("Desempenho (%)")
ax.set_xlabel("Métrica de recuperação")
ax.set_title("Resultados preliminares da validação cruzada")
ax.set_xticks(x, metricas)
ax.set_ylim(0, 110)
ax.legend()
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

for barras in (b1, b2):
    for barra in barras:
        valor = barra.get_height()
        ax.annotate(
            f"{valor:.2f}",
            (barra.get_x() + barra.get_width()/2, valor),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9
        )

fig.tight_layout()

pdf = DESTINO / "comparacao_metricas_folds_1_2.pdf"
png = DESTINO / "comparacao_metricas_folds_1_2.png"
fig.savefig(pdf, bbox_inches="tight")
fig.savefig(png, dpi=300, bbox_inches="tight")
plt.show()

print("PDF:", pdf)
print("PNG:", png)


## 7. Treinamento e avaliação — Fold 4

Preparação, treinamento em duas fases e avaliação.

In [ ]:

# Diagnóstico dos arquivos de divisão — Fold 4

for nome, df in [
    ("TREINO", df_treino_f4),
    ("VALIDAÇÃO", df_validacao_f4),
    ("TESTE", df_teste_f4)
]:
    print(f"\n{'=' * 45}")
    print(nome)
    print(f"{'=' * 45}")
    print("Dimensões:", df.shape)
    print("Colunas:", df.columns.tolist())
    print("\nPrimeiras 3 linhas:")
    print(df.head(3).to_string(index=False))


In [ ]:

# ============================================================
# FOLD 4 — PREPARAÇÃO E AUDITORIA CORRIGIDA
# Compatível com os CSVs reais do experimento
# ============================================================

from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import tensorflow as tf

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/"
    "experimentos/tcc1_kfold"
)

DIR_SPLITS = RAIZ / "splits"

DIR_RESULTADOS_FOLD4 = RAIZ / "resultados/fold_4"
DIR_CHECKPOINT_FOLD4 = RAIZ / "checkpoints/fold_4"

# ------------------------------------------------------------
# 1. Carregar divisões existentes
# ------------------------------------------------------------

arquivo_treino = (
    DIR_SPLITS / "fold_4_imagens_treino.csv"
)

arquivo_validacao = (
    DIR_SPLITS / "fold_4_imagens_validacao.csv"
)

arquivo_teste = (
    DIR_SPLITS / "identidades_teste_30.csv"
)

for arquivo in [
    arquivo_treino,
    arquivo_validacao,
    arquivo_teste
]:
    assert arquivo.is_file(), (
        f"Arquivo não encontrado: {arquivo}"
    )

df_treino_f4 = pd.read_csv(arquivo_treino)
df_validacao_f4 = pd.read_csv(arquivo_validacao)
df_teste_f4 = pd.read_csv(arquivo_teste)

# ------------------------------------------------------------
# 2. Verificar colunas conforme o tipo de arquivo
# ------------------------------------------------------------

colunas_imagens = {
    "dataset", "identidade", "caminho"
}

colunas_teste = {
    "dataset",
    "identidade",
    "quantidade_imagens"
}

for nome, df in [
    ("treino", df_treino_f4),
    ("validacao", df_validacao_f4)
]:
    assert colunas_imagens.issubset(df.columns), (
        f"Colunas ausentes em {nome}: "
        f"{colunas_imagens - set(df.columns)}"
    )

    assert not df.empty

    assert not df[
        list(colunas_imagens)
    ].isna().any().any()

assert colunas_teste.issubset(
    df_teste_f4.columns
)

assert not df_teste_f4.empty

assert not df_teste_f4[
    list(colunas_teste)
].isna().any().any()

# ------------------------------------------------------------
# 3. Auditoria de identidades
# ------------------------------------------------------------

def obter_identidades(df):
    return set(
        df["identidade"].astype(str)
    )

ids_treino_f4 = obter_identidades(
    df_treino_f4
)

ids_validacao_f4 = obter_identidades(
    df_validacao_f4
)

ids_teste_f4 = obter_identidades(
    df_teste_f4
)

assert ids_treino_f4.isdisjoint(
    ids_validacao_f4
), "Vazamento entre treino e validação!"

assert ids_treino_f4.isdisjoint(
    ids_teste_f4
), "Vazamento entre treino e teste!"

assert ids_validacao_f4.isdisjoint(
    ids_teste_f4
), "Vazamento entre validação e teste!"

# Conferir consistência do identificador.
for df in [
    df_treino_f4,
    df_validacao_f4,
    df_teste_f4
]:
    prefixos = (
        df["identidade"].astype(str)
        .str.split("::", regex=False)
        .str[0]
    )

    assert (
        prefixos == df["dataset"].astype(str)
    ).all(), "Inconsistência entre dataset e identidade!"

# ------------------------------------------------------------
# 4. Verificar imagens sem duplicações
# ------------------------------------------------------------

assert not df_treino_f4[
    "caminho"
].duplicated().any(), (
    "Imagens duplicadas no treinamento!"
)

assert not df_validacao_f4[
    "caminho"
].duplicated().any(), (
    "Imagens duplicadas na validação!"
)

assert set(
    df_treino_f4["caminho"]
).isdisjoint(
    set(df_validacao_f4["caminho"])
), "Imagens compartilhadas entre treino e validação!"

# ------------------------------------------------------------
# 5. Resolver caminhos locais das imagens
# ------------------------------------------------------------

# Primeiro, procurar o caminho original.
# Se não existir, procurar o caminho já extraído
# no ambiente local do Colab.

RAIZES_LOCAIS = [
    Path("/content/datasets_extraido"),
    Path("/content/tcc1_kfold/datasets"),
    Path("/content")
]

def localizar_imagem_f4(caminho):

    p = Path(str(caminho))

    if p.is_file():
        return str(p)

    partes = p.parts

    marcadores = [
        "DogFaceNet-master",
        "YTBB",
        "YTBB-Dog"
    ]

    for marcador in marcadores:

        if marcador not in partes:
            continue

        indice = partes.index(marcador)
        sufixo = Path(*partes[indice:])

        for raiz in RAIZES_LOCAIS:

            candidato = raiz / sufixo

            if candidato.is_file():
                return str(candidato)

    raise FileNotFoundError(
        f"Imagem não encontrada: {caminho}"
    )

print("Localizando imagens de treinamento...")

df_treino_f4["caminho_local"] = [
    localizar_imagem_f4(c)
    for c in tqdm(
        df_treino_f4["caminho"],
        desc="Treino",
        unit="imagem"
    )
]

print("\nLocalizando imagens de validação...")

df_validacao_f4["caminho_local"] = [
    localizar_imagem_f4(c)
    for c in tqdm(
        df_validacao_f4["caminho"],
        desc="Validação",
        unit="imagem"
    )
]

# ------------------------------------------------------------
# 6. Índices para amostragem por identidade
# ------------------------------------------------------------

def construir_indice_f4(df):

    return {
        str(identidade):
            grupo["caminho_local"].tolist()
        for identidade, grupo in df.groupby(
            "identidade"
        )
    }

indice_treino_f4 = construir_indice_f4(
    df_treino_f4
)

indice_validacao_f4 = construir_indice_f4(
    df_validacao_f4
)

# P=8 identidades por lote.
assert len(indice_treino_f4) >= 8
assert len(indice_validacao_f4) >= 8

# ------------------------------------------------------------
# 7. Verificar checkpoints
# ------------------------------------------------------------

DIR_RESULTADOS_FOLD4.mkdir(
    parents=True,
    exist_ok=True
)

DIR_CHECKPOINT_FOLD4.mkdir(
    parents=True,
    exist_ok=True
)

checkpoint_fase1 = tf.train.latest_checkpoint(
    str(
        DIR_CHECKPOINT_FOLD4 / "estado_fase1"
    )
)

checkpoint_fase2 = tf.train.latest_checkpoint(
    str(
        DIR_CHECKPOINT_FOLD4 / "estado_fase2"
    )
)

# ------------------------------------------------------------
# 8. Relatório de auditoria
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FOLD 4 — AUDITORIA CONCLUÍDA")
print("=" * 60)

print(
    "Imagens treino:",
    len(df_treino_f4)
)

print(
    "Imagens validação:",
    len(df_validacao_f4)
)

print(
    "Identidades treino:",
    len(ids_treino_f4)
)

print(
    "Identidades validação:",
    len(ids_validacao_f4)
)

print(
    "Identidades teste independente:",
    len(ids_teste_f4)
)

print(
    "Imagens previstas no teste:",
    int(df_teste_f4["quantidade_imagens"].sum())
)

print("\nVazamento de identidades: NÃO")
print("Imagens duplicadas: NÃO")

print("\nCheckpoints existentes:")
print("Fase 1:", checkpoint_fase1)
print("Fase 2:", checkpoint_fase2)

print("\nConfiguração prevista:")
print("P=8 | K=4 | Batch=32")
print("Triplet Loss: margem=0.2")
print("Fase 1: LR=1e-4")
print("Fase 2: LR=1e-5")

print(
    "\nGPU:",
    tf.config.list_physical_devices("GPU")
)

print("\nFold 4 pronto para inicialização.")


In [ ]:

# ============================================================
# FOLD 4 — INICIALIZAÇÃO INDEPENDENTE DO MODELO
# VGG16 ImageNet + embeddings de 128 dimensões
# ============================================================

import json
import random
import numpy as np
import tensorflow as tf
from pathlib import Path

print("=" * 65)
print("FOLD 4 — INICIALIZAÇÃO INDEPENDENTE")
print("=" * 65)

# ------------------------------------------------------------
# 1. Reprodutibilidade
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

P = 8
K = 4
BATCH_SIZE = P * K
MARGEM = 0.2

EPOCAS_FASE1 = 10
EPOCAS_FASE2 = 10

PASSOS_POR_EPOCA = int(
    np.ceil(len(df_treino_f4) / BATCH_SIZE)
)


# ------------------------------------------------------------
# 2. Recuperação segura do Fold 4
# ------------------------------------------------------------

checkpoint_fase2 = tf.train.latest_checkpoint(
    str(DIR_CHECKPOINT_FOLD4 / "estado_fase2")
)

assert checkpoint_fase2 is not None, (
    "Checkpoint da fase 2 não encontrado."
)

leitor = tf.train.load_checkpoint(checkpoint_fase2)

epoca_salva = int(
    leitor.get_tensor(
        "epoch/.ATTRIBUTES/VARIABLE_VALUE"
    )
)

assert epoca_salva == 7, (
    f"Esperada época 7, encontrada {epoca_salva}."
)

print("\nMODO RECUPERAÇÃO")
print("Checkpoint:", checkpoint_fase2)
print("Época salva:", epoca_salva)
print("Próxima época:", epoca_salva + 1)


# ------------------------------------------------------------
# 3. Criar backbone VGG16 com pesos ImageNet
# ------------------------------------------------------------

entradas_f4 = tf.keras.Input(
    shape=(224, 224, 3),
    name="entrada_imagem"
)

backbone_f4 = tf.keras.applications.VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

# Fase 1: somente a cabeça de embeddings será treinada.
backbone_f4.trainable = False

x = backbone_f4(
    entradas_f4,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dense(
    256,
    activation="relu",
    name="dense_256"
)(x)

x = tf.keras.layers.Dense(
    128,
    activation="linear",
    name="dense_128"
)(x)

saidas_f4 = tf.keras.layers.UnitNormalization(
    axis=-1,
    name="normalizacao_l2"
)(x)

modelo_embeddings_f4 = tf.keras.Model(
    inputs=entradas_f4,
    outputs=saidas_f4,
    name="modelo_embeddings_fold4"
)

assert modelo_embeddings_f4.output_shape == (
    None, 128
)

# ------------------------------------------------------------
# 4. Mapear identidades
# ------------------------------------------------------------

ids_treino_ordenados_f4 = sorted(
    indice_treino_f4.keys()
)

ids_validacao_ordenados_f4 = sorted(
    indice_validacao_f4.keys()
)

mapa_treino_f4 = {
    identidade: i
    for i, identidade in enumerate(
        ids_treino_ordenados_f4
    )
}

mapa_validacao_f4 = {
    identidade: i
    for i, identidade in enumerate(
        ids_validacao_ordenados_f4
    )
}

assert len(mapa_treino_f4) == 2305
assert len(mapa_validacao_f4) == 576

# ------------------------------------------------------------
# 5. Preparar diretórios de treinamento
# ------------------------------------------------------------

DIR_ESTADO_FASE1_F4 = (
    DIR_CHECKPOINT_FOLD4 / "estado_fase1"
)

DIR_ESTADO_FASE2_F4 = (
    DIR_CHECKPOINT_FOLD4 / "estado_fase2"
)

DIR_ESTADO_FASE1_F4.mkdir(
    parents=True, exist_ok=True
)

DIR_ESTADO_FASE2_F4.mkdir(
    parents=True, exist_ok=True
)

# ------------------------------------------------------------
# 6. Resumo
# ------------------------------------------------------------

print("\nMODELO")
print("Arquitetura: VGG16 + GAP + Dense 256/128")
print("Pesos iniciais: ImageNet")
print("Backbone congelado:", not backbone_f4.trainable)
print("Parâmetros totais:", modelo_embeddings_f4.count_params())
print("Saída:", modelo_embeddings_f4.output_shape)

print("\nTREINAMENTO")
print("Identidades treino:", len(mapa_treino_f4))
print("Identidades validação:", len(mapa_validacao_f4))
print("P:", P)
print("K:", K)
print("Batch:", BATCH_SIZE)
print("Passos por época:", PASSOS_POR_EPOCA)
print("Margem:", MARGEM)
print("Épocas fase 1:", EPOCAS_FASE1)
print("Épocas fase 2:", EPOCAS_FASE2)

print("\nGPU:")
print(tf.config.list_physical_devices("GPU"))

print("\n" + "=" * 65)
print("FOLD 4 — MODELO INICIALIZADO")
print("=" * 65)


In [ ]:

# ============================================================
# FOLD 4 — LOTES P×K, VALIDAÇÃO FIXA E TESTE DA LOSS
# ============================================================

import json
import numpy as np
import tensorflow as tf
from tqdm.auto import tqdm

SEED = 42
P, K = 8, 4
BATCH_SIZE = 32
MARGEM = 0.2
LOTES_VALIDACAO = 150

ids_treino_f4 = sorted(indice_treino_f4)
ids_val_f4 = sorted(indice_validacao_f4)

# ------------------------------------------------------------
# 1. Validação fixa e persistente
# ------------------------------------------------------------

ARQUIVO_LOTES_F4 = (
    DIR_RESULTADOS_FOLD4 / "lotes_validacao.json"
)

if ARQUIVO_LOTES_F4.exists():
    print("Carregando validação fixa existente...")
    with open(ARQUIVO_LOTES_F4, encoding="utf-8") as f:
        lotes_val_f4 = json.load(f)
else:
    print("Gerando validação fixa do Fold 4...")

    rng_val = np.random.default_rng(SEED + 400)
    lotes_val_f4 = []

    for numero in tqdm(
        range(LOTES_VALIDACAO),
        desc="Lotes de validação"
    ):
        escolhidas = rng_val.choice(
            ids_val_f4, size=P, replace=False
        )

        caminhos, identidades = [], []

        for identidade in escolhidas:
            fotos = indice_validacao_f4[identidade]

            selecionadas = rng_val.choice(
                fotos,
                size=K,
                replace=len(fotos) < K
            )

            caminhos.extend(map(str, selecionadas))
            identidades.extend([identidade] * K)

        lotes_val_f4.append({
            "lote": numero + 1,
            "caminhos": caminhos,
            "identidades": identidades
        })

    with open(ARQUIVO_LOTES_F4, "w", encoding="utf-8") as f:
        json.dump(lotes_val_f4, f, ensure_ascii=False)

assert len(lotes_val_f4) == LOTES_VALIDACAO

for lote in lotes_val_f4:
    assert len(lote["caminhos"]) == BATCH_SIZE
    assert len(lote["identidades"]) == BATCH_SIZE

# ------------------------------------------------------------
# 2. Pré-processamento VGG16
# ------------------------------------------------------------

def preparar_imagem_f4(caminho, rotulo):
    conteudo = tf.io.read_file(caminho)

    imagem = tf.io.decode_image(
        conteudo,
        channels=3,
        expand_animations=False
    )

    imagem.set_shape([None, None, 3])
    imagem = tf.image.resize(imagem, [224, 224])

    imagem = tf.keras.applications.vgg16.preprocess_input(
        tf.cast(imagem, tf.float32)
    )

    return imagem, rotulo


def dataset_f4(caminhos, rotulos):
    return (
        tf.data.Dataset.from_tensor_slices((
            tf.constant(caminhos, dtype=tf.string),
            tf.constant(rotulos, dtype=tf.int32)
        ))
        .map(
            preparar_imagem_f4,
            num_parallel_calls=4
        )
        .batch(BATCH_SIZE, drop_remainder=True)
        .prefetch(2)
    )

# ------------------------------------------------------------
# 3. Dataset de validação fixa
# ------------------------------------------------------------

caminhos_val_f4 = []
rotulos_val_f4 = []

for lote in lotes_val_f4:
    caminhos_val_f4.extend(lote["caminhos"])
    rotulos_val_f4.extend([
        mapa_validacao_f4[identidade]
        for identidade in lote["identidades"]
    ])

dataset_validacao_f4 = dataset_f4(
    caminhos_val_f4,
    rotulos_val_f4
)

# ------------------------------------------------------------
# 4. Gerador de treinamento por época
# ------------------------------------------------------------

def criar_dataset_treino_f4(epoca):
    rng = np.random.default_rng(SEED + epoca)

    caminhos, rotulos = [], []

    for _ in range(PASSOS_POR_EPOCA):
        escolhidas = rng.choice(
            ids_treino_f4,
            size=P,
            replace=False
        )

        for identidade in escolhidas:
            fotos = indice_treino_f4[identidade]

            selecionadas = rng.choice(
                fotos,
                size=K,
                replace=len(fotos) < K
            )

            caminhos.extend(map(str, selecionadas))
            rotulos.extend(
                [mapa_treino_f4[identidade]] * K
            )

    return dataset_f4(caminhos, rotulos)

# ------------------------------------------------------------
# 5. Triplet Loss semi-hard
# ------------------------------------------------------------

@tf.function
def triplet_loss_f4(embeddings, rotulos):
    rotulos = tf.reshape(rotulos, [-1])

    produto = tf.matmul(
        embeddings,
        embeddings,
        transpose_b=True
    )

    normas = tf.reduce_sum(
        tf.square(embeddings),
        axis=1,
        keepdims=True
    )

    dist = tf.maximum(
        normas - 2 * produto + tf.transpose(normas),
        0.0
    )

    n = tf.shape(dist)[0]

    iguais = tf.equal(
        rotulos[:, None],
        rotulos[None, :]
    )

    positivos = iguais & ~tf.eye(n, dtype=tf.bool)
    negativos = ~iguais

    d_ap = dist[:, :, None]
    d_an = dist[:, None, :]

    candidatos = (
        negativos[:, None, :]
        & (d_an > d_ap)
        & (d_an < d_ap + MARGEM)
    )

    inf = tf.constant(float("inf"), dtype=dist.dtype)

    melhor_semi = tf.reduce_min(
        tf.where(candidatos, d_an, inf),
        axis=2
    )

    negativo_proximo = tf.reduce_min(
        tf.where(negativos, dist, inf),
        axis=1
    )[:, None]

    d_neg = tf.where(
        tf.math.is_finite(melhor_semi),
        melhor_semi,
        negativo_proximo
    )

    perdas = tf.where(
        positivos,
        tf.maximum(dist + MARGEM - d_neg, 0.0),
        0.0
    )

    return tf.reduce_sum(perdas) / tf.maximum(
        tf.reduce_sum(
            tf.cast(positivos, tf.float32)
        ),
        1.0
    )

# ------------------------------------------------------------
# 6. Teste de integridade
# ------------------------------------------------------------

print("\nTestando leitura de um lote...")

imagens_teste, rotulos_teste = next(
    iter(criar_dataset_treino_f4(1))
)

assert tuple(imagens_teste.shape) == (
    32, 224, 224, 3
)

assert len(np.unique(rotulos_teste.numpy())) == P

embeddings_teste = modelo_embeddings_f4(
    imagens_teste,
    training=False
)

loss_teste = float(
    triplet_loss_f4(
        embeddings_teste,
        rotulos_teste
    ).numpy()
)

assert np.isfinite(loss_teste)

print("\n" + "=" * 60)
print("FOLD 4 — PIPELINE VALIDADO")
print("=" * 60)
print("Lotes de treino por época:", PASSOS_POR_EPOCA)
print("Lotes fixos de validação:", len(lotes_val_f4))
print("Formato das imagens:", imagens_teste.shape)
print("Formato dos embeddings:", embeddings_teste.shape)
print("Identidades no lote:", len(np.unique(rotulos_teste)))
print(f"Triplet Loss inicial: {loss_teste:.6f}")
print("Arquivo de validação:", ARQUIVO_LOTES_F4)



In [ ]:

componentes = [
    "modelo_embeddings_f4",
    "backbone_f4",
    "triplet_loss_f4",
    "criar_dataset_treino_f4",
    "dataset_validacao_f4",
    "PASSOS_POR_EPOCA",
    "LOTES_VALIDACAO",
    "EPOCAS_FASE1"
]

faltantes = [
    nome for nome in componentes
    if nome not in globals()
]

if faltantes:
    print("Componentes ausentes:", faltantes)
else:
    print("Todos os componentes estão disponíveis.")
    print("Pronto para restaurar o checkpoint da época 7.")


In [ ]:

# ============================================================
# FOLD 4 — FASE 1
# VGG16 congelada | Adam 1e-4 | Triplet semi-hard
# 10 épocas | tqdm | checkpoints recuperáveis
# ============================================================

import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm

FASE = 1
EPOCAS = 10

ARQUIVO_HISTORICO_F4_F1 = (
    DIR_RESULTADOS_FOLD4 / "historico_fase1.csv"
)

ARQUIVO_MELHOR_F4_F1 = (
    DIR_CHECKPOINT_FOLD4 / "melhor_fase1.weights.h5"
)

DIR_ESTADO_F4_F1 = (
    DIR_CHECKPOINT_FOLD4 / "estado_fase1"
)

DIR_ESTADO_F4_F1.mkdir(
    parents=True, exist_ok=True
)

# ------------------------------------------------------------
# 1. Congelar backbone e configurar otimizador
# ------------------------------------------------------------

backbone_f4.trainable = False

otimizador_f4_f1 = tf.keras.optimizers.Adam(
    learning_rate=1e-4
)

@tf.function
def passo_treino_f4_f1(imagens, rotulos):

    with tf.GradientTape() as tape:
        embeddings = modelo_embeddings_f4(
            imagens, training=True
        )

        perda = triplet_loss_f4(
            embeddings, rotulos
        )

    variaveis = modelo_embeddings_f4.trainable_variables

    gradientes = tape.gradient(
        perda, variaveis
    )

    pares = [
        (g, v)
        for g, v in zip(gradientes, variaveis)
        if g is not None
    ]

    otimizador_f4_f1.apply_gradients(pares)

    return perda


@tf.function
def passo_validacao_f4_f1(imagens, rotulos):

    embeddings = modelo_embeddings_f4(
        imagens, training=False
    )

    return triplet_loss_f4(
        embeddings, rotulos
    )

# ------------------------------------------------------------
# 2. Estado persistente
# ------------------------------------------------------------

epoca_salva_f4_f1 = tf.Variable(
    0, dtype=tf.int64, trainable=False
)

estado_f4_f1 = tf.train.Checkpoint(
    model=modelo_embeddings_f4,
    optimizer=otimizador_f4_f1,
    epoch=epoca_salva_f4_f1
)

gestor_f4_f1 = tf.train.CheckpointManager(
    estado_f4_f1,
    str(DIR_ESTADO_F4_F1),
    max_to_keep=3
)

if ARQUIVO_HISTORICO_F4_F1.exists():
    historico_f4_f1 = pd.read_csv(
        ARQUIVO_HISTORICO_F4_F1
    ).to_dict("records")
else:
    historico_f4_f1 = []

if gestor_f4_f1.latest_checkpoint:

    estado_f4_f1.restore(
        gestor_f4_f1.latest_checkpoint
    ).expect_partial()

    epoca_inicial = int(
        epoca_salva_f4_f1.numpy()
    ) + 1

    print(
        "Checkpoint recuperado:",
        gestor_f4_f1.latest_checkpoint
    )

else:
    if historico_f4_f1:
        raise RuntimeError(
            "Histórico encontrado sem checkpoint. "
            "Verifique o estado antes de continuar."
        )

    epoca_inicial = 1

# Descartar entradas de histórico que não correspondem
# ao checkpoint restaurado, se houve interrupção ao salvar.
historico_f4_f1 = [
    h for h in historico_f4_f1
    if int(h["epoca"]) < epoca_inicial
]

if historico_f4_f1:
    melhor_loss_f4_f1 = min(
        float(h["loss_validacao"])
        for h in historico_f4_f1
    )
else:
    melhor_loss_f4_f1 = float("inf")

if (
    melhor_loss_f4_f1 < float("inf")
    and not ARQUIVO_MELHOR_F4_F1.exists()
):
    raise RuntimeError(
        "Histórico existe, mas os melhores pesos "
        "não foram encontrados."
    )

print("\n" + "=" * 65)
print("FOLD 4 — FASE 1")
print("=" * 65)
print("Época inicial:", epoca_inicial)
print("Época final:", EPOCAS)
print("Lotes treino:", PASSOS_POR_EPOCA)
print("Lotes validação:", LOTES_VALIDACAO)
print("Learning rate: 1e-4")
print("Backbone congelado:", not backbone_f4.trainable)
print("=" * 65, flush=True)

# ------------------------------------------------------------
# 3. Treinamento com progresso
# ------------------------------------------------------------

for epoca in range(epoca_inicial, EPOCAS + 1):

    inicio = time.time()
    perdas_treino = []

    print(
        f"\nFOLD 4 | FASE 1 | ÉPOCA {epoca}/{EPOCAS}",
        flush=True
    )

    dataset_treino = criar_dataset_treino_f4(epoca)

    barra = tqdm(
        dataset_treino,
        total=PASSOS_POR_EPOCA,
        desc=f"Treino {epoca:02d}/{EPOCAS}",
        unit="lote"
    )

    for imagens, rotulos in barra:

        perda = float(
            passo_treino_f4_f1(
                imagens, rotulos
            ).numpy()
        )

        if not np.isfinite(perda):
            raise FloatingPointError(
                f"Loss inválida na época {epoca}"
            )

        perdas_treino.append(perda)

        barra.set_postfix(
            loss=f"{np.mean(perdas_treino):.5f}"
        )

    # --------------------------------------------------------
    # 4. Validação fixa
    # --------------------------------------------------------

    perdas_val = []

    barra_val = tqdm(
        dataset_validacao_f4,
        total=LOTES_VALIDACAO,
        desc=f"Validação {epoca:02d}",
        unit="lote"
    )

    for imagens, rotulos in barra_val:

        perda = float(
            passo_validacao_f4_f1(
                imagens, rotulos
            ).numpy()
        )

        if not np.isfinite(perda):
            raise FloatingPointError(
                "Loss de validação inválida"
            )

        perdas_val.append(perda)

        barra_val.set_postfix(
            loss=f"{np.mean(perdas_val):.5f}"
        )

    loss_treino = float(
        np.mean(perdas_treino)
    )

    loss_validacao = float(
        np.mean(perdas_val)
    )

    # --------------------------------------------------------
    # 5. Melhor checkpoint
    # --------------------------------------------------------

    melhorou = (
        loss_validacao < melhor_loss_f4_f1
    )

    if melhorou:

        melhor_loss_f4_f1 = loss_validacao

        modelo_embeddings_f4.save_weights(
            ARQUIVO_MELHOR_F4_F1
        )

    duracao = time.time() - inicio

    historico_f4_f1.append({
        "epoca": epoca,
        "loss_treino": loss_treino,
        "loss_validacao": loss_validacao,
        "melhor_checkpoint": melhorou,
        "duracao_segundos": duracao
    })

    # --------------------------------------------------------
    # 6. Salvar estado e histórico
    # --------------------------------------------------------

    epoca_salva_f4_f1.assign(epoca)

    gestor_f4_f1.save(
        checkpoint_number=epoca
    )

    pd.DataFrame(
        historico_f4_f1
    ).to_csv(
        ARQUIVO_HISTORICO_F4_F1,
        index=False
    )

    print(
        f"\nÉPOCA {epoca:02d}/{EPOCAS} | "
        f"Treino: {loss_treino:.6f} | "
        f"Val: {loss_validacao:.6f} | "
        f"Tempo: {duracao/60:.2f} min"
        + (" | MELHOR" if melhorou else ""),
        flush=True
    )

# ------------------------------------------------------------
# 7. Recuperar os melhores pesos da fase 1
# ------------------------------------------------------------

assert ARQUIVO_MELHOR_F4_F1.exists(), (
    "Checkpoint dos melhores pesos não encontrado."
)

modelo_embeddings_f4.load_weights(
    ARQUIVO_MELHOR_F4_F1
)

print("\n" + "=" * 65)
print("FOLD 4 — FASE 1 CONCLUÍDA")
print("=" * 65)

print("Épocas registradas:", len(historico_f4_f1))
print(f"Melhor loss: {melhor_loss_f4_f1:.8f}")
print("Melhores pesos:", ARQUIVO_MELHOR_F4_F1)
print("Estado:", gestor_f4_f1.latest_checkpoint)
print("Histórico:", ARQUIVO_HISTORICO_F4_F1)


In [ ]:

# ============================================================
# FOLD 4 — DIAGNÓSTICO INDEPENDENTE
# Não altera os pesos nem reinicia o treinamento
# ============================================================

from pathlib import Path
import pandas as pd
import tensorflow as tf

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/"
    "experimentos/tcc1_kfold"
)

DIR_CHECKPOINT_FOLD4 = RAIZ / "checkpoints/fold_4"
DIR_RESULTADOS_FOLD4 = RAIZ / "resultados/fold_4"

DIR_F2 = DIR_CHECKPOINT_FOLD4 / "estado_fase2"
ARQUIVO_HIST = DIR_RESULTADOS_FOLD4 / "historico_fase2.csv"
ARQUIVO_MELHOR = DIR_CHECKPOINT_FOLD4 / "melhor_fase2.weights.h5"

print("=" * 60)
print("FOLD 4 — DIAGNÓSTICO")
print("=" * 60)

print("Pasta de checkpoints:", DIR_F2.exists())
print("Histórico existe:", ARQUIVO_HIST.exists())
print("Melhores pesos existem:", ARQUIVO_MELHOR.exists())

ultimo_checkpoint = tf.train.latest_checkpoint(str(DIR_F2))
print("\nÚltimo checkpoint:", ultimo_checkpoint)

if ARQUIVO_HIST.exists():
    historico = pd.read_csv(ARQUIVO_HIST)

    print("\nÚltimas épocas registradas:")
    print(historico.tail(5).to_string(index=False))

    print("\nÚltima época:", int(historico["epoca"].max()))
    print("Melhor loss:", historico["loss_validacao"].min())

if ultimo_checkpoint:
    print("\nVariáveis de época no checkpoint:")
    for nome, forma in tf.train.list_variables(ultimo_checkpoint):
        if "epoch" in nome.lower():
            print(nome, forma)

print("=" * 60)


In [ ]:

componentes = [
    "modelo_embeddings_f4",
    "backbone_f4",
    "triplet_loss_f4",
    "criar_dataset_treino_f4",
    "dataset_validacao_f4",
    "PASSOS_POR_EPOCA",
    "LOTES_VALIDACAO",
    "EPOCAS_FASE1"
]

print("VERIFICAÇÃO ANTES DA RETOMADA\n")

todos_ok = True

for nome in componentes:
    existe = nome in globals()
    print(f"{nome:32s} {'OK' if existe else 'AUSENTE'}")
    todos_ok &= existe

print("\nPronto para restaurar:", todos_ok)


In [ ]:

# ============================================================
# FOLD 4 — FASE 2: FINE-TUNING DO BLOCO 5
# Adam 1e-5 | 10 épocas | Checkpoints recuperáveis
# ============================================================

import time
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm

EPOCAS_F4_F2 = 10

ARQUIVO_HIST_F4_F2 = (
    DIR_RESULTADOS_FOLD4 / "historico_fase2.csv"
)

ARQUIVO_MELHOR_F4_F2 = (
    DIR_CHECKPOINT_FOLD4 / "melhor_fase2.weights.h5"
)

ARQUIVO_FINAL_F4 = (
    DIR_CHECKPOINT_FOLD4 /
    "modelo_final_fold4.weights.h5"
)

DIR_ESTADO_F4_F2 = (
    DIR_CHECKPOINT_FOLD4 / "estado_fase2"
)

DIR_ESTADO_F4_F2.mkdir(
    parents=True, exist_ok=True
)

# ------------------------------------------------------------
# 1. Descongelar somente o bloco 5
# ------------------------------------------------------------

backbone_f4.trainable = True

for camada in backbone_f4.layers:
    camada.trainable = camada.name.startswith("block5_")

camadas_liberadas = [
    camada.name
    for camada in backbone_f4.layers
    if camada.trainable
]

print("Camadas liberadas:", camadas_liberadas)

assert camadas_liberadas == [
    "block5_conv1",
    "block5_conv2",
    "block5_conv3",
    "block5_pool"
], "Configuração inesperada do backbone."

# ------------------------------------------------------------
# 2. Otimizador e funções de treinamento
# ------------------------------------------------------------

otimizador_f4_f2 = tf.keras.optimizers.Adam(
    learning_rate=1e-5
)

@tf.function
def passo_treino_f4_f2(imagens, rotulos):
    with tf.GradientTape() as tape:
        embeddings = modelo_embeddings_f4(
            imagens, training=True
        )
        perda = triplet_loss_f4(
            embeddings, rotulos
        )

    variaveis = modelo_embeddings_f4.trainable_variables
    gradientes = tape.gradient(perda, variaveis)

    pares = [
        (gradiente, variavel)
        for gradiente, variavel in zip(
            gradientes, variaveis
        )
        if gradiente is not None
    ]

    otimizador_f4_f2.apply_gradients(pares)
    return perda


@tf.function
def passo_validacao_f4_f2(imagens, rotulos):
    embeddings = modelo_embeddings_f4(
        imagens, training=False
    )
    return triplet_loss_f4(embeddings, rotulos)


# ------------------------------------------------------------
# 3. Checkpoint de modelo + otimizador + época
# ------------------------------------------------------------

epoca_salva_f4_f2 = tf.Variable(
    0, dtype=tf.int64, trainable=False
)

# Preparar variáveis internas do Adam
otimizador_f4_f2.build(
    modelo_embeddings_f4.trainable_variables
)

estado_f4_f2 = tf.train.Checkpoint(
    model=modelo_embeddings_f4,
    optimizer=otimizador_f4_f2,
    epoch=epoca_salva_f4_f2
)

gestor_f4_f2 = tf.train.CheckpointManager(
    estado_f4_f2,
    str(DIR_ESTADO_F4_F2),
    max_to_keep=3
)

if gestor_f4_f2.latest_checkpoint:

    print(
        "Restaurando fase 2:",
        gestor_f4_f2.latest_checkpoint
    )

    status_restauracao = estado_f4_f2.restore(
        gestor_f4_f2.latest_checkpoint
    )

    # Verificar integridade da restauração
    status_restauracao.assert_existing_objects_matched()
    status_restauracao.assert_consumed()

    epoca_recuperada = int(
        epoca_salva_f4_f2.numpy()
    )

    assert epoca_recuperada == 7, (
        f"Esperada época 7, "
        f"encontrada {epoca_recuperada}."
    )

    iteracoes_adam = int(
        otimizador_f4_f2.iterations.numpy()
    )

    assert iteracoes_adam == 7 * PASSOS_POR_EPOCA, (
        f"Iterações inesperadas do Adam: "
        f"{iteracoes_adam}"
    )

    epoca_inicial_f4_f2 = epoca_recuperada + 1

    print("\nRESTAURAÇÃO CONCLUÍDA")
    print("Época recuperada:", epoca_recuperada)
    print("Iterações do Adam:", iteracoes_adam)
    print("Próxima época:", epoca_inicial_f4_f2)

else:

    # Iniciar fase 2 a partir dos melhores
    # pesos da fase 1, somente se não houver
    # checkpoint anterior da fase 2.

    assert ARQUIVO_MELHOR_F4_F1.exists(), (
        "Melhores pesos da fase 1 não encontrados."
    )

    modelo_embeddings_f4.load_weights(
        ARQUIVO_MELHOR_F4_F1
    )

    epoca_inicial_f4_f2 = 1

    print(
        "Pesos da melhor época da fase 1 carregados."
    )


# ------------------------------------------------------------
# 4. Histórico
# ------------------------------------------------------------

if ARQUIVO_HIST_F4_F2.exists():
    historico_f4_f2 = pd.read_csv(
        ARQUIVO_HIST_F4_F2
    ).to_dict("records")
else:
    historico_f4_f2 = []

if (
    not gestor_f4_f2.latest_checkpoint
    and historico_f4_f2
):
    raise RuntimeError(
        "Existe histórico da fase 2 sem checkpoint. "
        "Verifique os arquivos antes de prosseguir."
    )

historico_f4_f2 = [
    linha
    for linha in historico_f4_f2
    if int(linha["epoca"]) < epoca_inicial_f4_f2
]

melhor_loss_f4_f2 = (
    min(
        float(linha["loss_validacao"])
        for linha in historico_f4_f2
    )
    if historico_f4_f2
    else float("inf")
)

if (
    historico_f4_f2
    and not ARQUIVO_MELHOR_F4_F2.exists()
):
    raise RuntimeError(
        "Histórico existente, mas os melhores "
        "pesos da fase 2 não foram encontrados."
    )

print("\n" + "=" * 65)
print("FOLD 4 — FASE 2")
print("=" * 65)
print("Época inicial:", epoca_inicial_f4_f2)
print("Época final:", EPOCAS_F4_F2)
print("Learning rate: 1e-5")
print("Camadas treináveis:", camadas_liberadas)
print("Passos por época:", PASSOS_POR_EPOCA)
print("Lotes de validação:", LOTES_VALIDACAO)
print("=" * 65)

# ------------------------------------------------------------
# 5. Treinamento
# ------------------------------------------------------------

for epoca in range(
    epoca_inicial_f4_f2,
    EPOCAS_F4_F2 + 1
):
    inicio = time.time()
    perdas_treino = []

    print(
        f"\nFOLD 4 | FASE 2 | ÉPOCA "
        f"{epoca}/{EPOCAS_F4_F2}",
        flush=True
    )

    dataset_treino = criar_dataset_treino_f4(
        EPOCAS_FASE1 + epoca
    )

    barra_treino = tqdm(
        dataset_treino,
        total=PASSOS_POR_EPOCA,
        desc=f"Treino F2 {epoca:02d}",
        unit="lote"
    )

    for imagens, rotulos in barra_treino:
        perda = float(
            passo_treino_f4_f2(
                imagens, rotulos
            ).numpy()
        )

        if not np.isfinite(perda):
            raise FloatingPointError(
                f"Loss inválida na época {epoca}"
            )

        perdas_treino.append(perda)

        barra_treino.set_postfix(
            loss=f"{np.mean(perdas_treino):.5f}"
        )

    # Validação fixa
    perdas_val = []

    barra_val = tqdm(
        dataset_validacao_f4,
        total=LOTES_VALIDACAO,
        desc=f"Validação F2 {epoca:02d}",
        unit="lote"
    )

    for imagens, rotulos in barra_val:
        perda = float(
            passo_validacao_f4_f2(
                imagens, rotulos
            ).numpy()
        )

        if not np.isfinite(perda):
            raise FloatingPointError(
                "Loss de validação inválida"
            )

        perdas_val.append(perda)

        barra_val.set_postfix(
            loss=f"{np.mean(perdas_val):.5f}"
        )

    loss_treino = float(np.mean(perdas_treino))
    loss_validacao = float(np.mean(perdas_val))

    # Salvar melhores pesos
    melhorou = (
        loss_validacao < melhor_loss_f4_f2
    )

    if melhorou:
        melhor_loss_f4_f2 = loss_validacao

        modelo_embeddings_f4.save_weights(
            ARQUIVO_MELHOR_F4_F2
        )

    duracao = time.time() - inicio

    historico_f4_f2.append({
        "epoca": epoca,
        "loss_treino": loss_treino,
        "loss_validacao": loss_validacao,
        "melhor_checkpoint": melhorou,
        "duracao_segundos": duracao
    })

    # Checkpoint completo
    epoca_salva_f4_f2.assign(epoca)

    gestor_f4_f2.save(
        checkpoint_number=epoca
    )

    pd.DataFrame(
        historico_f4_f2
    ).to_csv(
        ARQUIVO_HIST_F4_F2,
        index=False
    )

    print(
        f"\nÉPOCA {epoca:02d}/{EPOCAS_F4_F2} | "
        f"Treino: {loss_treino:.6f} | "
        f"Val: {loss_validacao:.6f} | "
        f"Tempo: {duracao/60:.2f} min"
        + (" | MELHOR" if melhorou else ""),
        flush=True
    )

# ------------------------------------------------------------
# 6. Restaurar melhor modelo e salvar pesos finais
# ------------------------------------------------------------

assert ARQUIVO_MELHOR_F4_F2.exists(), (
    "Melhores pesos da fase 2 não encontrados."
)

modelo_embeddings_f4.load_weights(
    ARQUIVO_MELHOR_F4_F2
)

modelo_embeddings_f4.save_weights(
    ARQUIVO_FINAL_F4
)

print("\n" + "=" * 65)
print("FOLD 4 — FASE 2 CONCLUÍDA")
print("=" * 65)
print("Épocas:", len(historico_f4_f2))
print(f"Melhor loss: {melhor_loss_f4_f2:.8f}")
print("Pesos finais:", ARQUIVO_FINAL_F4)
print("Histórico:", ARQUIVO_HIST_F4_F2)
print("Checkpoint:", gestor_f4_f2.latest_checkpoint)


In [ ]:

# ============================================================
# FOLD 4 — EXTRAÇÃO DE EMBEDDINGS DA VALIDAÇÃO
# Mesmo protocolo de extração do Fold 3
# GPU recomendada | Não modifica checkpoints
# ============================================================

import time
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Diretórios e arquivos
# ------------------------------------------------------------

DIR_AVALIACAO_F4 = DIR_RESULTADOS_FOLD4 / "avaliacao"
DIR_AVALIACAO_F4.mkdir(parents=True, exist_ok=True)

ARQUIVO_EMBEDDINGS_F4 = (
    DIR_AVALIACAO_F4 / "embeddings_validacao.npy"
)

ARQUIVO_METADADOS_F4 = (
    DIR_AVALIACAO_F4 / "metadados_validacao.csv"
)

ARQUIVO_MODELO_FINAL_F4 = (
    DIR_CHECKPOINT_FOLD4 /
    "modelo_final_fold4.weights.h5"
)

assert ARQUIVO_MODELO_FINAL_F4.is_file(), (
    "Pesos finais do Fold 4 não encontrados."
)

# ------------------------------------------------------------
# 2. Carregar os melhores pesos
# ------------------------------------------------------------

modelo_embeddings_f4.load_weights(
    str(ARQUIVO_MODELO_FINAL_F4)
)

print("Melhores pesos do Fold 4 carregados.")

# ------------------------------------------------------------
# 3. Preparar imagens da validação
# ------------------------------------------------------------

df_avaliacao_f4 = (
    df_validacao_f4.copy()
    .sort_values(
        ["dataset", "identidade", "caminho_local"]
    )
    .reset_index(drop=True)
)

caminhos_f4 = (
    df_avaliacao_f4["caminho_local"]
    .astype(str)
    .tolist()
)

assert len(caminhos_f4) == 4899, (
    f"Quantidade inesperada: {len(caminhos_f4)}"
)

assert len(set(caminhos_f4)) == len(caminhos_f4), (
    "Existem caminhos duplicados."
)

# ------------------------------------------------------------
# 4. Pré-processamento VGG16
# ------------------------------------------------------------

def carregar_para_inferencia_f4(caminho):

    conteudo = tf.io.read_file(caminho)

    imagem = tf.io.decode_image(
        conteudo,
        channels=3,
        expand_animations=False
    )

    imagem.set_shape([None, None, 3])

    imagem = tf.image.resize(
        imagem, [224, 224]
    )

    return tf.keras.applications.vgg16.preprocess_input(
        tf.cast(imagem, tf.float32)
    )

dataset_inferencia_f4 = (
    tf.data.Dataset.from_tensor_slices(caminhos_f4)
    .map(
        carregar_para_inferencia_f4,
        num_parallel_calls=4
    )
    .batch(32)
    .prefetch(2)
)

total_lotes_f4 = (len(caminhos_f4) + 31) // 32

# ------------------------------------------------------------
# 5. Extrair embeddings
# ------------------------------------------------------------

print("\nFOLD 4 — EXTRAÇÃO DE EMBEDDINGS")
print("Imagens:", len(caminhos_f4))
print("Lotes:", total_lotes_f4)
print("Dimensões esperadas: 128")

inicio_f4 = time.time()
partes_f4 = []

for imagens in tqdm(
    dataset_inferencia_f4,
    total=total_lotes_f4,
    desc="Fold 4 | Extração",
    unit="lote"
):

    embeddings = modelo_embeddings_f4(
        imagens,
        training=False
    )

    partes_f4.append(embeddings.numpy())

embeddings_f4 = np.concatenate(
    partes_f4,
    axis=0
).astype(np.float32)

# ------------------------------------------------------------
# 6. Validar embeddings
# ------------------------------------------------------------

assert embeddings_f4.shape == (
    len(df_avaliacao_f4), 128
)

assert np.isfinite(embeddings_f4).all(), (
    "Embeddings contêm valores inválidos."
)

normas_f4 = np.linalg.norm(
    embeddings_f4,
    axis=1
)

assert np.allclose(
    normas_f4,
    1.0,
    atol=1e-4
), "Embeddings não estão normalizados."

# ------------------------------------------------------------
# 7. Salvar resultados
# ------------------------------------------------------------

np.save(
    ARQUIVO_EMBEDDINGS_F4,
    embeddings_f4
)

df_avaliacao_f4.to_csv(
    ARQUIVO_METADADOS_F4,
    index=False
)

resumo_f4 = {
    "fold": 4,
    "imagens": int(len(embeddings_f4)),
    "dimensoes": int(embeddings_f4.shape[1]),
    "identidades": int(
        (
            df_avaliacao_f4["dataset"].astype(str)
            + "::"
            + df_avaliacao_f4["identidade"].astype(str)
        ).nunique()
    ),
    "norma_minima": float(normas_f4.min()),
    "norma_maxima": float(normas_f4.max()),
    "tempo_segundos": float(
        time.time() - inicio_f4
    )
}

with open(
    DIR_AVALIACAO_F4 / "resumo_extracao.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        resumo_f4,
        arquivo,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 8. Relatório
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FOLD 4 — EXTRAÇÃO CONCLUÍDA")
print("=" * 60)

print("Formato:", embeddings_f4.shape)
print("Identidades:", resumo_f4["identidades"])
print("Norma mínima:", resumo_f4["norma_minima"])
print("Norma máxima:", resumo_f4["norma_maxima"])
print(
    "Tempo (min):",
    round(resumo_f4["tempo_segundos"] / 60, 2)
)
print("Arquivo:", ARQUIVO_EMBEDDINGS_F4)
print("=" * 60)


In [ ]:

# ============================================================
# FOLD 4 — AVALIAÇÃO QUANTITATIVA COMPLETA
# Mesmo protocolo do Fold 2 — Bloco [72]
# ============================================================

import json
import time
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

inicio = time.time()
SEED = 42
rng = np.random.default_rng(SEED)

DIR_AVALIACAO_F4 = (
    DIR_RESULTADOS_FOLD4 / "avaliacao"
)

ARQUIVO_EMBEDDINGS_F4 = (
    DIR_AVALIACAO_F4 / "embeddings_validacao.npy"
)

ARQUIVO_METADADOS_F4 = (
    DIR_AVALIACAO_F4 / "metadados_validacao.csv"
)

# ------------------------------------------------------------
# 1. Carregar embeddings e metadados
# ------------------------------------------------------------

print("1/5 — Carregando embeddings...", flush=True)

emb = np.load(ARQUIVO_EMBEDDINGS_F4)
meta = pd.read_csv(ARQUIVO_METADADOS_F4)

assert len(emb) == len(meta) == 4899
assert emb.shape[1] == 128
assert np.isfinite(emb).all()

identidades = (
    meta["dataset"].astype(str)
    + "::"
    + meta["identidade"].astype(str)
).to_numpy()

n = len(emb)
assert len(np.unique(identidades)) == 576

print("Imagens:", n)
print("Identidades:", len(np.unique(identidades)))

# ------------------------------------------------------------
# 2. Construir pares balanceados
# ------------------------------------------------------------

print("\n2/5 — Construindo pares...", flush=True)

grupos = {
    identidade: np.where(
        identidades == identidade
    )[0]
    for identidade in np.unique(identidades)
}

pares_positivos = []

for indices in tqdm(
    grupos.values(),
    total=len(grupos),
    desc="Pares positivos"
):
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            pares_positivos.append(
                (indices[i], indices[j])
            )

pares_positivos = np.asarray(
    pares_positivos,
    dtype=np.int32
).reshape(-1, 2)

quantidade_positivos = len(pares_positivos)

assert quantidade_positivos > 0

pares_negativos = set()

with tqdm(
    total=quantidade_positivos,
    desc="Pares negativos"
) as barra:

    while len(pares_negativos) < quantidade_positivos:

        i = int(rng.integers(n))
        j = int(rng.integers(n))

        if i == j or identidades[i] == identidades[j]:
            continue

        par = (min(i, j), max(i, j))

        if par not in pares_negativos:
            pares_negativos.add(par)
            barra.update(1)

pares_negativos = np.asarray(
    sorted(pares_negativos),
    dtype=np.int32
)

# ------------------------------------------------------------
# 3. Métricas de verificação
# ------------------------------------------------------------

print("\n3/5 — Calculando verificação...", flush=True)

def distancias_pares(pares):
    diferenca = (
        emb[pares[:, 0]] - emb[pares[:, 1]]
    )
    return np.linalg.norm(diferenca, axis=1)

dist_pos = distancias_pares(pares_positivos)
dist_neg = distancias_pares(pares_negativos)

y_true = np.concatenate([
    np.ones(len(dist_pos), dtype=np.int32),
    np.zeros(len(dist_neg), dtype=np.int32)
])

distancias = np.concatenate([
    dist_pos, dist_neg
])

scores = -distancias

auc = roc_auc_score(y_true, scores)

fpr, tpr, thresholds = roc_curve(
    y_true, scores
)

fnr = 1.0 - tpr
indice_eer = np.argmin(np.abs(fpr - fnr))

eer = float(
    (fpr[indice_eer] + fnr[indice_eer]) / 2
)

limiar_distancia = float(
    -thresholds[indice_eer]
)

y_pred = (
    distancias <= limiar_distancia
).astype(np.int32)

precision = precision_score(
    y_true, y_pred, zero_division=0
)

recall = recall_score(
    y_true, y_pred, zero_division=0
)

f1 = f1_score(
    y_true, y_pred, zero_division=0
)

matriz = confusion_matrix(
    y_true, y_pred
)

print(f"ROC-AUC: {auc:.6f}")
print(f"EER aproximado: {eer:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall: {recall:.6f}")
print(f"F1: {f1:.6f}")
print("Verificação concluída.", flush=True)

# 4. Recuperação global leave-one-out
# ------------------------------------------------------------

print("\n4/5 — Calculando recuperação...", flush=True)

# Embeddings L2-normalizados:
# cosseno e distância euclidiana produzem
# a mesma ordenação dos vizinhos.

similaridades = emb @ emb.T

np.fill_diagonal(
    similaridades, -np.inf
)

print("Ordenando vizinhos...", flush=True)

ordem = np.argsort(
    -similaridades,
    axis=1
)

del similaridades

acertos = (
    identidades[ordem]
    == identidades[:, None]
)

del ordem

hit1 = float(
    np.mean(acertos[:, 0])
)

hit5 = float(
    np.mean(np.any(acertos[:, :5], axis=1))
)

hit10 = float(
    np.mean(np.any(acertos[:, :10], axis=1))
)

aps = []
rrs = []

for linha in tqdm(
    acertos,
    total=n,
    desc="mAP e MRR",
    unit="imagem"
):

    posicoes = np.flatnonzero(linha)

    if len(posicoes) == 0:
        aps.append(0.0)
        rrs.append(0.0)
        continue

    precisao_acumulada = (
        np.arange(1, len(posicoes) + 1)
        / (posicoes + 1)
    )

    aps.append(
        float(np.mean(precisao_acumulada))
    )

    rrs.append(
        float(1.0 / (posicoes[0] + 1))
    )

map_score = float(np.mean(aps))
mrr = float(np.mean(rrs))

# ------------------------------------------------------------
# 5. Métricas por dataset
# ------------------------------------------------------------

metricas_dataset = {}

for dataset in sorted(meta["dataset"].unique()):

    mascara = (
        meta["dataset"].to_numpy() == dataset
    )

    metricas_dataset[str(dataset)] = {
        "imagens": int(mascara.sum()),
        "hit1": float(
            np.mean(acertos[mascara, 0])
        ),
        "hit5": float(
            np.mean(
                np.any(acertos[mascara, :5], axis=1)
            )
        ),
        "hit10": float(
            np.mean(
                np.any(acertos[mascara, :10], axis=1)
            )
        )
    }

# ------------------------------------------------------------
# 6. Persistir resultados
# ------------------------------------------------------------

print("\n5/5 — Salvando métricas...", flush=True)

resultados_f4 = {
    "fold": 4,
    "imagens_validacao": int(n),
    "identidades_validacao": int(
        len(np.unique(identidades))
    ),
    "verificacao": {
        "pares_positivos": int(len(dist_pos)),
        "pares_negativos": int(len(dist_neg)),
        "roc_auc": float(auc),
        "eer_aproximado": eer,
        "limiar_distancia_eer": limiar_distancia,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "matriz_confusao": matriz.tolist()
    },
    "recuperacao": {
        "hit1": hit1,
        "hit5": hit5,
        "hit10": hit10,
        "map": map_score,
        "mrr": mrr
    },
    "por_dataset": metricas_dataset
}

ARQUIVO_RESULTADOS_F4 = (
    DIR_AVALIACAO_F4 / "metricas_fold4.json"
)

with open(
    ARQUIVO_RESULTADOS_F4,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        resultados_f4,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 7. Relatório
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("FOLD 4 — AVALIAÇÃO CONCLUÍDA")
print("=" * 55)

print("\nVERIFICAÇÃO")
print("Pares positivos:", len(dist_pos))
print("Pares negativos:", len(dist_neg))
print(f"ROC-AUC: {auc:.6f}")
print(f"EER aproximado: {eer:.6f}")
print(f"Limiar EER: {limiar_distancia:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall: {recall:.6f}")
print(f"F1: {f1:.6f}")
print("Matriz de confusão:")
print(matriz)

print("\nRECUPERAÇÃO GLOBAL")
print(f"Hit@1: {hit1:.2%}")
print(f"Hit@5: {hit5:.2%}")
print(f"Hit@10: {hit10:.2%}")
print(f"mAP: {map_score:.2%}")
print(f"MRR: {mrr:.2%}")

print("\nPOR DATASET")

for dataset, dados in metricas_dataset.items():
    print(
        f"{dataset}: "
        f"Hit@1={dados['hit1']:.2%}, "
        f"Hit@5={dados['hit5']:.2%}, "
        f"Hit@10={dados['hit10']:.2%}"
    )

print("\nTempo total:")

## 8. Treinamento e avaliação — Fold 5

Inclui auditorias, restauração de checkpoints e treinamento em duas fases.

In [ ]:

# ============================================================
# FOLD 5 — PREPARAÇÃO E AUDITORIA
# Não altera modelos nem checkpoints anteriores
# ============================================================

from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import tensorflow as tf

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/"
    "experimentos/tcc1_kfold"
)

DIR_SPLITS = RAIZ / "splits"

DIR_RESULTADOS_FOLD5 = RAIZ / "resultados/fold_5"
DIR_CHECKPOINT_FOLD5 = RAIZ / "checkpoints/fold_5"

# ------------------------------------------------------------
# 1. Carregar divisões
# ------------------------------------------------------------

arquivo_treino_f5 = (
    DIR_SPLITS / "fold_5_imagens_treino.csv"
)

arquivo_validacao_f5 = (
    DIR_SPLITS / "fold_5_imagens_validacao.csv"
)

arquivo_teste_f5 = (
    DIR_SPLITS / "identidades_teste_30.csv"
)

for arquivo in (
    arquivo_treino_f5,
    arquivo_validacao_f5,
    arquivo_teste_f5
):
    assert arquivo.is_file(), (
        f"Arquivo não encontrado: {arquivo}"
    )

df_treino_f5 = pd.read_csv(arquivo_treino_f5)
df_validacao_f5 = pd.read_csv(arquivo_validacao_f5)
df_teste_f5 = pd.read_csv(arquivo_teste_f5)

# ------------------------------------------------------------
# 2. Verificar estrutura
# ------------------------------------------------------------

colunas_imagens = {
    "dataset", "identidade", "caminho"
}

colunas_teste = {
    "dataset", "identidade", "quantidade_imagens"
}

for nome, df in (
    ("treino", df_treino_f5),
    ("validacao", df_validacao_f5)
):
    assert not df.empty, f"{nome} vazio"
    assert colunas_imagens.issubset(df.columns)
    assert not df[list(colunas_imagens)].isna().any().any()
    assert not df["caminho"].duplicated().any()

assert not df_teste_f5.empty
assert colunas_teste.issubset(df_teste_f5.columns)
assert not df_teste_f5[
    list(colunas_teste)
].isna().any().any()

# ------------------------------------------------------------
# 3. Auditoria de identidades
# ------------------------------------------------------------

def identidades_f5(df):
    return set(df["identidade"].astype(str))

ids_treino_f5 = identidades_f5(df_treino_f5)
ids_validacao_f5 = identidades_f5(df_validacao_f5)
ids_teste_f5 = identidades_f5(df_teste_f5)

assert ids_treino_f5.isdisjoint(ids_validacao_f5), (
    "Vazamento treino/validação"
)

assert ids_treino_f5.isdisjoint(ids_teste_f5), (
    "Vazamento treino/teste"
)

assert ids_validacao_f5.isdisjoint(ids_teste_f5), (
    "Vazamento validação/teste"
)

assert set(df_treino_f5["caminho"]).isdisjoint(
    set(df_validacao_f5["caminho"])
), "Imagens compartilhadas entre treino e validação"

for df in (
    df_treino_f5,
    df_validacao_f5,
    df_teste_f5
):
    prefixos = (
        df["identidade"].astype(str)
        .str.split("::", regex=False)
        .str[0]
    )

    assert (
        prefixos == df["dataset"].astype(str)
    ).all(), "Inconsistência entre dataset e identidade"

# ------------------------------------------------------------
# 4. Localizar imagens
# ------------------------------------------------------------

RAIZES_LOCAIS_F5 = [
    Path("/content/datasets_extraido"),
    Path("/content/tcc1_kfold/datasets"),
    Path("/content")
]

def localizar_imagem_f5(caminho):
    p = Path(str(caminho))

    if p.is_file():
        return str(p)

    partes = p.parts

    for marcador in (
        "DogFaceNet-master",
        "YTBB",
        "YTBB-Dog"
    ):
        if marcador not in partes:
            continue

        indice = partes.index(marcador)
        sufixo = Path(*partes[indice:])

        for raiz in RAIZES_LOCAIS_F5:
            candidato = raiz / sufixo

            if candidato.is_file():
                return str(candidato)

    raise FileNotFoundError(
        f"Imagem não encontrada: {caminho}"
    )

print("Localizando imagens de treino...")

df_treino_f5["caminho_local"] = [
    localizar_imagem_f5(c)
    for c in tqdm(
        df_treino_f5["caminho"],
        desc="Treino Fold 5",
        unit="imagem"
    )
]

print("\nLocalizando imagens de validação...")

df_validacao_f5["caminho_local"] = [
    localizar_imagem_f5(c)
    for c in tqdm(
        df_validacao_f5["caminho"],
        desc="Validação Fold 5",
        unit="imagem"
    )
]

# ------------------------------------------------------------
# 5. Índices para amostragem P×K
# ------------------------------------------------------------

def construir_indice_f5(df):
    return {
        str(identidade): grupo["caminho_local"].tolist()
        for identidade, grupo in df.groupby("identidade")
    }

indice_treino_f5 = construir_indice_f5(df_treino_f5)
indice_validacao_f5 = construir_indice_f5(
    df_validacao_f5
)

assert len(indice_treino_f5) >= 8
assert len(indice_validacao_f5) >= 8

assert all(
    len(caminhos) >= 4
    for caminhos in indice_treino_f5.values()
), "Identidade de treino com menos de 4 imagens"

assert all(
    len(caminhos) >= 4
    for caminhos in indice_validacao_f5.values()
), "Identidade de validação com menos de 4 imagens"

# ------------------------------------------------------------
# 6. Verificar estados anteriores
# ------------------------------------------------------------

checkpoint_fase1_f5 = tf.train.latest_checkpoint(
    str(DIR_CHECKPOINT_FOLD5 / "estado_fase1")
)

checkpoint_fase2_f5 = tf.train.latest_checkpoint(
    str(DIR_CHECKPOINT_FOLD5 / "estado_fase2")
)

modelo_final_f5 = (
    DIR_CHECKPOINT_FOLD5 /
    "modelo_final_fold5.weights.h5"
)

# Não sobrescrever um experimento existente.
assert checkpoint_fase1_f5 is None, (
    "Já existe checkpoint da fase 1 do Fold 5."
)

assert checkpoint_fase2_f5 is None, (
    "Já existe checkpoint da fase 2 do Fold 5."
)

assert not modelo_final_f5.exists(), (
    "Já existe modelo final do Fold 5."
)

# ------------------------------------------------------------
# 7. Relatório
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FOLD 5 — AUDITORIA CONCLUÍDA")
print("=" * 60)

print("Imagens treino:", len(df_treino_f5))
print("Imagens validação:", len(df_validacao_f5))

print("Identidades treino:", len(ids_treino_f5))
print("Identidades validação:", len(ids_validacao_f5))
print("Identidades teste:", len(ids_teste_f5))

print(
    "Imagens previstas no teste:",
    int(df_teste_f5["quantidade_imagens"].sum())
)

print("\nVazamento de identidades: NÃO")
print("Imagens duplicadas: NÃO")
print("Imagens locais: LOCALIZADAS")

print("\nCheckpoints existentes:")
print("Fase 1:", checkpoint_fase1_f5)
print("Fase 2:", checkpoint_fase2_f5)

print("\nConfiguração prevista:")
print("P=8 | K=4 | Batch=32")
print("Triplet Loss: margem=0.2")
print("Fase 1: 10 épocas | LR=1e-4")
print("Fase 2: 10 épocas | LR=1e-5")

print(
    "\nGPU:",
    tf.config.list_physical_devices("GPU")
)

print("\nFold 5 pronto para inicialização.")
print("=" * 60)


In [ ]:

# ============================================================
# FOLD 5 — DIAGNÓSTICO DE IMAGENS POR IDENTIDADE
# ============================================================

import pandas as pd

def diagnosticar_identidades(indice, nome):
    quantidades = pd.Series({
        identidade: len(caminhos)
        for identidade, caminhos in indice.items()
    })

    insuficientes = quantidades[quantidades < 4]

    print(f"\n{nome}")
    print("=" * 55)
    print("Total de identidades:", len(quantidades))
    print("Mínimo de imagens:", quantidades.min())
    print("Máximo de imagens:", quantidades.max())
    print("Identidades com menos de 4:", len(insuficientes))

    print("\nDistribuição:")
    print(
        quantidades.value_counts()
        .sort_index()
        .head(15)
    )

    if len(insuficientes):
        print("\nIdentidades insuficientes:")
        print(insuficientes.to_string())

    return insuficientes

insuficientes_treino_f5 = diagnosticar_identidades(
    indice_treino_f5,
    "TREINAMENTO — FOLD 5"
)

insuficientes_validacao_f5 = diagnosticar_identidades(
    indice_validacao_f5,
    "VALIDAÇÃO — FOLD 5"
)


In [ ]:

# ============================================================
# FOLD 5 — VERIFICAÇÃO DO PROTOCOLO P×K
# ============================================================

import inspect

print("=" * 60)
print("PROTOCOLO DE AMOSTRAGEM DO FOLD 4")
print("=" * 60)

print("\nFunção de treinamento:")

try:
    print(
        inspect.getsource(
            criar_dataset_treino_f4
        )
    )
except (OSError, TypeError):
    print(
        "Não foi possível recuperar o código-fonte."
    )

print("\nFunções relacionadas disponíveis:")

for nome in sorted(globals()):
    if (
        "amostr" in nome.lower()
        or "lote" in nome.lower()
        or "batch" in nome.lower()
    ):
        if callable(globals()[nome]):
            print(nome)

print("\nResumo Fold 5:")
print(
    "Treino com menos de 4 imagens:",
    len(insuficientes_treino_f5)
)
print(
    "Validação com menos de 4 imagens:",
    len(insuficientes_validacao_f5)
)


In [ ]:

# ============================================================
# FOLD 5 — FINALIZAÇÃO DA AUDITORIA
# ============================================================

import tensorflow as tf

# 1. Verificar mínimo de duas imagens por identidade
assert all(
    len(fotos) >= 2
    for fotos in indice_treino_f5.values()
), "Identidade de treino com menos de 2 imagens"

assert all(
    len(fotos) >= 2
    for fotos in indice_validacao_f5.values()
), "Identidade de validação com menos de 2 imagens"

# 2. Confirmar isolamento das identidades
assert ids_treino_f5.isdisjoint(ids_validacao_f5)
assert ids_treino_f5.isdisjoint(ids_teste_f5)
assert ids_validacao_f5.isdisjoint(ids_teste_f5)

# 3. Verificar checkpoints existentes
checkpoint_fase1_f5 = tf.train.latest_checkpoint(
    str(DIR_CHECKPOINT_FOLD5 / "estado_fase1")
)

checkpoint_fase2_f5 = tf.train.latest_checkpoint(
    str(DIR_CHECKPOINT_FOLD5 / "estado_fase2")
)

modelo_final_f5 = (
    DIR_CHECKPOINT_FOLD5 /
    "modelo_final_fold5.weights.h5"
)

assert checkpoint_fase1_f5 is None, (
    "Já existe checkpoint da fase 1 do Fold 5"
)

assert checkpoint_fase2_f5 is None, (
    "Já existe checkpoint da fase 2 do Fold 5"
)

assert not modelo_final_f5.exists(), (
    "Já existe modelo final do Fold 5"
)

# 4. Preparar diretórios
DIR_RESULTADOS_FOLD5.mkdir(
    parents=True, exist_ok=True
)

DIR_CHECKPOINT_FOLD5.mkdir(
    parents=True, exist_ok=True
)

# 5. Relatório
print("=" * 60)
print("FOLD 5 — AUDITORIA CONCLUÍDA")
print("=" * 60)

print(f"Imagens treino: {len(df_treino_f5)}")
print(f"Imagens validação: {len(df_validacao_f5)}")

print(f"Identidades treino: {len(ids_treino_f5)}")
print(f"Identidades validação: {len(ids_validacao_f5)}")
print(f"Identidades teste: {len(ids_teste_f5)}")

print("\nProtocolo P×K:")
print("P = 8")
print("K = 4")
print("Reposição para identidades com menos de K imagens")

print("\nIdentidades que exigem reposição:")
print(f"Treino: {len(insuficientes_treino_f5)}")
print(f"Validação: {len(insuficientes_validacao_f5)}")

print("\nVazamento de identidades: NÃO")
print("Imagens locais: LOCALIZADAS")
print("Checkpoints anteriores do Fold 5: NÃO")

print(
    "\nGPU:",
    tf.config.list_physical_devices("GPU")
)

print("\nFOLD 5 PRONTO PARA INICIALIZAÇÃO")
print("=" * 60)


In [ ]:

# ============================================================
# FOLD 5 — INSPEÇÃO DA ARQUITETURA ORIGINAL
# ============================================================

import inspect
import tensorflow as tf

print("=" * 60)
print("FOLD 5 — REFERÊNCIA DA ARQUITETURA")
print("=" * 60)

print("\n1. Funções relacionadas ao modelo:")

funcoes_modelo = []

for nome, objeto in list(globals().items()):
    if not callable(objeto):
        continue

    if any(termo in nome.lower() for termo in (
        "modelo", "vgg", "embedding", "triplet"
    )):
        funcoes_modelo.append(nome)

for nome in sorted(funcoes_modelo):
    print(" -", nome)

print("\n2. Configurações disponíveis:")

parametros = [
    "SEED",
    "P",
    "K",
    "PASSOS_POR_EPOCA",
    "MARGEM",
    "IMG_SIZE",
    "TAMANHO_IMAGEM",
    "DIM_EMBEDDING",
    "EMBEDDING_DIM",
    "BATCH_SIZE"
]

for nome in parametros:
    if nome in globals():
        print(f"{nome}: {globals()[nome]}")

print("\n3. Modelos disponíveis:")

for nome, objeto in list(globals().items()):
    if isinstance(objeto, tf.keras.Model):
        print(
            f"{nome}: "
            f"{objeto.name} | "
            f"{objeto.count_params():,} parâmetros"
        )

print("\n4. Código das funções de criação:")

for nome in sorted(funcoes_modelo):
    if not any(
        termo in nome.lower()
        for termo in ("criar_modelo", "construir_modelo")
    ):
        continue

    try:
        print(f"\n--- {nome} ---")
        print(inspect.getsource(globals()[nome]))
    except (OSError, TypeError):
        print(
            f"Fonte indisponível para {nome}"
        )

print("\nInspeção concluída.")


In [ ]:

# ============================================================
# FOLD 5 — INSPEÇÃO DETALHADA DO MODELO ORIGINAL
# ============================================================

import tensorflow as tf
import inspect

print("=" * 65)
print("ARQUITETURA ORIGINAL — FOLD 4")
print("=" * 65)

modelo_embeddings_f4.summary(expand_nested=True)

print("\n" + "=" * 65)
print("CAMADAS EXTERNAS")
print("=" * 65)

for i, camada in enumerate(modelo_embeddings_f4.layers):
    print(
        f"{i:02d} | "
        f"{camada.name:35s} | "
        f"{camada.__class__.__name__:25s} | "
        f"Treinável: {camada.trainable}"
    )

    config = camada.get_config()

    for chave in (
        "units",
        "activation",
        "rate",
        "axis",
        "epsilon",
        "kernel_regularizer"
    ):
        if chave in config:
            print(f"     {chave}: {config[chave]}")

print("\n" + "=" * 65)
print("CONFIGURAÇÃO DO BACKBONE")
print("=" * 65)

print("Nome:", backbone_f4.name)
print("Treinável:", backbone_f4.trainable)

print("\nÚltimas 12 camadas:")

for camada in backbone_f4.layers[-12:]:
    print(
        f"{camada.name:25s} "
        f"trainable={camada.trainable}"
    )

print("\n" + "=" * 65)
print("FUNÇÃO DE PERDA")
print("=" * 65)

try:
    print(inspect.getsource(triplet_loss_f4))
except (OSError, TypeError):
    print("Código-fonte indisponível.")

print("\n" + "=" * 65)
print("DIMENSÕES")
print("=" * 65)

print("Entrada:", modelo_embeddings_f4.input_shape)
print("Saída:", modelo_embeddings_f4.output_shape)
print("Parâmetros:", modelo_embeddings_f4.count_params())

print("\nINSPEÇÃO CONCLUÍDA")


In [ ]:

# ============================================================
# FOLD 5 — CRIAÇÃO INDEPENDENTE DO MODELO
# ============================================================

import random
import numpy as np
import tensorflow as tf

# Reprodutibilidade da inicialização
tf.keras.utils.set_random_seed(SEED)

# Verificar se não há modelo Fold 5 existente
assert "modelo_embeddings_f5" not in globals(), (
    "Já existe um modelo Fold 5 na memória. "
    "Não reinicialize um treinamento em andamento."
)

# ------------------------------------------------------------
# 1. Backbone VGG16 — pesos originais ImageNet
# ------------------------------------------------------------

backbone_f5 = tf.keras.applications.VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

backbone_f5.trainable = False

# ------------------------------------------------------------
# 2. Cabeça de embeddings
# ------------------------------------------------------------

entrada_f5 = tf.keras.Input(
    shape=(224, 224, 3),
    name="entrada_imagem"
)

x = backbone_f5(entrada_f5)

x = tf.keras.layers.GlobalAveragePooling2D(
    name="global_average_pooling2d"
)(x)

x = tf.keras.layers.Dense(
    256,
    activation="relu",
    name="dense_256"
)(x)

x = tf.keras.layers.Dense(
    128,
    activation="linear",
    name="dense_128"
)(x)

saida_f5 = tf.keras.layers.UnitNormalization(
    axis=-1,
    name="normalizacao_l2"
)(x)

modelo_embeddings_f5 = tf.keras.Model(
    inputs=entrada_f5,
    outputs=saida_f5,
    name="modelo_embeddings_fold5"
)

# ------------------------------------------------------------
# 3. Verificações de arquitetura
# ------------------------------------------------------------

assert modelo_embeddings_f5.count_params() == 14878912

assert modelo_embeddings_f5.input_shape == (
    None, 224, 224, 3
)

assert modelo_embeddings_f5.output_shape == (
    None, 128
)

assert backbone_f5 is not backbone_f4

assert modelo_embeddings_f5 is not modelo_embeddings_f4

assert not any(
    camada.trainable
    for camada in backbone_f5.layers
    if camada.weights
), "Backbone deveria estar congelado na fase 1"

# ------------------------------------------------------------
# 4. Comparação com Fold 4
# ------------------------------------------------------------

print("=" * 60)
print("FOLD 5 — MODELO INICIALIZADO")
print("=" * 60)

print(
    "Parâmetros Fold 4:",
    modelo_embeddings_f4.count_params()
)

print(
    "Parâmetros Fold 5:",
    modelo_embeddings_f5.count_params()
)

print(
    "Parâmetros treináveis na fase 1:",
    sum(
        int(np.prod(p.shape))
        for p in modelo_embeddings_f5.trainable_weights
    )
)

print(
    "Backbone congelado:",
    not backbone_f5.trainable
)

print(
    "Modelo independente:",
    modelo_embeddings_f5 is not modelo_embeddings_f4
)

print(
    "Backbone independente:",
    backbone_f5 is not backbone_f4
)

print("\nResumo do modelo:")
modelo_embeddings_f5.summary()

print("\nFOLD 5 PRONTO PARA PREPARAR A FASE 1")


In [ ]:

# ============================================================
# FOLD 5 — INSPEÇÃO DO PIPELINE DE TREINAMENTO
# ============================================================

import inspect
import tensorflow as tf

print("=" * 60)
print("FOLD 5 — REFERÊNCIA DO PIPELINE")
print("=" * 60)

nomes_funcoes = [
    "dataset_f4",
    "criar_dataset_treino_f4",
    "criar_dataset_validacao_f4",
    "passo_treino_f4",
    "passo_validacao_f4",
    "triplet_loss_f4"
]

for nome in nomes_funcoes:
    print(f"\n{'=' * 60}")
    print(nome)
    print("=" * 60)

    objeto = globals().get(nome)

    if objeto is None:
        print("Não encontrado.")
        continue

    try:
        print(inspect.getsource(objeto))
    except (OSError, TypeError):
        print("Código-fonte indisponível.")

print("\n" + "=" * 60)
print("VARIÁVEIS DE TREINAMENTO")
print("=" * 60)

for nome in sorted(globals()):
    if any(termo in nome.lower() for termo in (
        "otimizador",
        "optimizer",
        "passos_validacao",
        "ids_validacao",
        "mapa_treino",
        "mapa_validacao",
        "taxa_aprendizado"
    )):
        objeto = globals()[nome]

        if isinstance(
            objeto,
            (int, float, str, dict, list, tuple)
        ):
            if isinstance(objeto, (dict, list, tuple)):
                print(f"{nome}: {len(objeto)} elementos")
            else:
                print(f"{nome}: {objeto}")
        else:
            print(f"{nome}: {type(objeto).__name__}")

print("\nINSPEÇÃO DO PIPELINE CONCLUÍDA")


In [ ]:

# ============================================================
# FOLD 5 — INSPEÇÃO FINAL DO PIPELINE ORIGINAL
# ============================================================

import inspect
import tensorflow as tf

print("=" * 60)
print("1. PRÉ-PROCESSAMENTO DO FOLD 4")
print("=" * 60)

try:
    print(inspect.getsource(preparar_imagem_f4))
except (OSError, TypeError):
    print("Código-fonte indisponível.")

print("\n" + "=" * 60)
print("2. FUNÇÕES DISPONÍVEIS PARA VALIDAÇÃO")
print("=" * 60)

for nome, objeto in list(globals().items()):
    if callable(objeto) and any(
        termo in nome.lower()
        for termo in (
            "valid",
            "avaliar",
            "treinar",
            "train_step",
            "val_step"
        )
    ):
        print(nome)

print("\n" + "=" * 60)
print("3. CONFIGURAÇÃO DOS OTIMIZADORES")
print("=" * 60)

for nome in (
    "otimizador_f4_f1",
    "otimizador_f4_f2"
):
    otimizador = globals().get(nome)

    if otimizador is None:
        continue

    print(f"\n{nome}")
    print("Classe:", type(otimizador).__name__)
    print(
        "Learning rate:",
        float(
            tf.keras.backend.get_value(
                otimizador.learning_rate
            )
        )
    )
    print(
        "Iterações:",
        int(
            tf.keras.backend.get_value(
                otimizador.iterations
            )
        )
    )

print("\n" + "=" * 60)
print("4. VARIÁVEIS DE VALIDAÇÃO")
print("=" * 60)

for nome, objeto in list(globals().items()):
    if "valid" not in nome.lower():
        continue

    if isinstance(objeto, (int, float, str)):
        print(f"{nome}: {objeto}")
    elif isinstance(objeto, (list, tuple, dict, set)):
        print(f"{nome}: {len(objeto)} elementos")
    elif isinstance(objeto, tf.data.Dataset):
        print(f"{nome}: tf.data.Dataset")

print("\nINSPEÇÃO FINAL CONCLUÍDA")


In [ ]:

# ============================================================
# FOLD 5 — DATASETS DE TREINO E VALIDAÇÃO
# ============================================================

import numpy as np
import tensorflow as tf

# ------------------------------------------------------------
# 1. Identidades e rótulos independentes
# ------------------------------------------------------------

ids_treino_ordenados_f5 = sorted(
    indice_treino_f5.keys()
)

ids_validacao_ordenados_f5 = sorted(
    indice_validacao_f5.keys()
)

mapa_treino_f5 = {
    identidade: i
    for i, identidade in enumerate(
        ids_treino_ordenados_f5
    )
}

mapa_validacao_f5 = {
    identidade: i
    for i, identidade in enumerate(
        ids_validacao_ordenados_f5
    )
}

# ------------------------------------------------------------
# 2. Pré-processamento idêntico ao Fold 4
# ------------------------------------------------------------

def preparar_imagem_f5(caminho, rotulo):
    conteudo = tf.io.read_file(caminho)

    imagem = tf.io.decode_image(
        conteudo,
        channels=3,
        expand_animations=False
    )

    imagem.set_shape([None, None, 3])

    imagem = tf.image.resize(
        imagem, [224, 224]
    )

    imagem = tf.keras.applications.vgg16.preprocess_input(
        tf.cast(imagem, tf.float32)
    )

    return imagem, rotulo


def dataset_f5(caminhos, rotulos):
    return (
        tf.data.Dataset.from_tensor_slices((
            tf.constant(caminhos, dtype=tf.string),
            tf.constant(rotulos, dtype=tf.int32)
        ))
        .map(
            preparar_imagem_f5,
            num_parallel_calls=4
        )
        .batch(
            BATCH_SIZE,
            drop_remainder=True
        )
        .prefetch(2)
    )

# ------------------------------------------------------------
# 3. Treinamento — amostragem P×K
# ------------------------------------------------------------

def criar_dataset_treino_f5(epoca):
    rng = np.random.default_rng(SEED + epoca)

    caminhos, rotulos = [], []

    for _ in range(PASSOS_POR_EPOCA):
        escolhidas = rng.choice(
            ids_treino_ordenados_f5,
            size=P,
            replace=False
        )

        for identidade in escolhidas:
            fotos = indice_treino_f5[identidade]

            selecionadas = rng.choice(
                fotos,
                size=K,
                replace=len(fotos) < K
            )

            caminhos.extend(
                map(str, selecionadas)
            )

            rotulos.extend(
                [mapa_treino_f5[identidade]] * K
            )

    return dataset_f5(caminhos, rotulos)

# ------------------------------------------------------------
# 4. Validação fixa — P×K
# ------------------------------------------------------------

def criar_dataset_validacao_f5():
    rng = np.random.default_rng(SEED + 10000 + 5)

    caminhos, rotulos = [], []

    for _ in range(LOTES_VALIDACAO):
        escolhidas = rng.choice(
            ids_validacao_ordenados_f5,
            size=P,
            replace=False
        )

        for identidade in escolhidas:
            fotos = indice_validacao_f5[identidade]

            selecionadas = rng.choice(
                fotos,
                size=K,
                replace=len(fotos) < K
            )

            caminhos.extend(
                map(str, selecionadas)
            )

            rotulos.extend(
                [mapa_validacao_f5[identidade]] * K
            )

    return dataset_f5(caminhos, rotulos)


dataset_validacao_f5 = criar_dataset_validacao_f5()

# ------------------------------------------------------------
# 5. Verificação dos lotes
# ------------------------------------------------------------

dataset_treino_teste_f5 = criar_dataset_treino_f5(
    epoca=0
)

imagens, rotulos = next(
    iter(dataset_treino_teste_f5)
)

assert imagens.shape == (32, 224, 224, 3)
assert rotulos.shape == (32,)

assert len(np.unique(rotulos.numpy())) == P

assert all(
    np.sum(rotulos.numpy() == rotulo) == K
    for rotulo in np.unique(rotulos.numpy())
)

print("=" * 60)
print("FOLD 5 — DATASETS PREPARADOS")
print("=" * 60)

print("Identidades treino:", len(mapa_treino_f5))
print("Identidades validação:", len(mapa_validacao_f5))

print("Batch:", imagens.shape)
print("Rótulos:", rotulos.shape)

print(
    "Identidades distintas no lote:",
    len(np.unique(rotulos.numpy()))
)

print("Passos treino:", PASSOS_POR_EPOCA)
print("Lotes validação:", LOTES_VALIDACAO)

print("\nAmostragem P×K: OK")
print("Pré-processamento VGG16: OK")
print("Validação fixa: OK")

print("\nFOLD 5 PRONTO PARA CONFIGURAR O TREINAMENTO")


In [ ]:

# ============================================================
# FOLD 5 — AUDITORIA FINAL DA VALIDAÇÃO E TREINAMENTO
# ============================================================

import inspect
import tensorflow as tf

print("=" * 60)
print("1. PASSO DE VALIDAÇÃO — FASE 1")
print("=" * 60)

try:
    print(inspect.getsource(passo_validacao_f4_f1))
except (OSError, TypeError):
    print("Código-fonte indisponível.")

print("\n" + "=" * 60)
print("2. PASSO DE VALIDAÇÃO — FASE 2")
print("=" * 60)

try:
    print(inspect.getsource(passo_validacao_f4_f2))
except (OSError, TypeError):
    print("Código-fonte indisponível.")

print("\n" + "=" * 60)
print("3. FUNÇÕES DE TREINAMENTO DISPONÍVEIS")
print("=" * 60)

for nome, objeto in list(globals().items()):
    if not callable(objeto):
        continue

    if any(termo in nome.lower() for termo in (
        "passo_treino",
        "passo_treinamento",
        "train_step",
        "treinar_f4",
        "validacao_fixa"
    )):
        print(nome)

print("\n" + "=" * 60)
print("4. DATASET DE VALIDAÇÃO ORIGINAL")
print("=" * 60)

print("Objeto:", type(dataset_validacao_f4).__name__)

print(
    "Cardinalidade:",
    int(tf.data.experimental.cardinality(
        dataset_validacao_f4
    ).numpy())
)

imagens_f4, rotulos_f4 = next(
    iter(dataset_validacao_f4)
)

print("Formato das imagens:", imagens_f4.shape)
print("Formato dos rótulos:", rotulos_f4.shape)

print("\n" + "=" * 60)
print("5. ESTADO DO FOLD 5")
print("=" * 60)

print(
    "Modelo inicializado:",
    "modelo_embeddings_f5" in globals()
)

print(
    "Dataset validação:",
    "dataset_validacao_f5" in globals()
)

print(
    "Checkpoints fase 1:",
    tf.train.latest_checkpoint(
        str(DIR_CHECKPOINT_FOLD5 / "estado_fase1")
    )
)

print("\nAUDITORIA FINAL CONCLUÍDA")


In [ ]:

# ============================================================
# FOLD 5 — CONFIGURAÇÃO DA FASE 1
# ============================================================

import tensorflow as tf
import inspect

# ------------------------------------------------------------
# 1. Triplet Loss — mesma implementação do Fold 4
# ------------------------------------------------------------

# Reutilizamos a função matemática, não os pesos do modelo.
triplet_loss_f5 = triplet_loss_f4

# ------------------------------------------------------------
# 2. Otimizador independente
# ------------------------------------------------------------

otimizador_f5_f1 = tf.keras.optimizers.Adam(
    learning_rate=1e-4
)

# ------------------------------------------------------------
# 3. Passo de treinamento
# ------------------------------------------------------------

@tf.function
def passo_treino_f5_f1(imagens, rotulos):

    with tf.GradientTape() as tape:
        embeddings = modelo_embeddings_f5(
            imagens,
            training=True
        )

        perda = triplet_loss_f5(
            embeddings,
            rotulos
        )

    variaveis = modelo_embeddings_f5.trainable_variables

    gradientes = tape.gradient(
        perda,
        variaveis
    )

    pares_gradientes = [
        (gradiente, variavel)
        for gradiente, variavel in zip(
            gradientes,
            variaveis
        )
        if gradiente is not None
    ]

    otimizador_f5_f1.apply_gradients(
        pares_gradientes
    )

    return perda

# ------------------------------------------------------------
# 4. Passo de validação
# ------------------------------------------------------------

@tf.function
def passo_validacao_f5_f1(imagens, rotulos):

    embeddings = modelo_embeddings_f5(
        imagens,
        training=False
    )

    return triplet_loss_f5(
        embeddings,
        rotulos
    )

# ------------------------------------------------------------
# 5. Checkpoint de recuperação
# ------------------------------------------------------------

DIR_ESTADO_F5_F1 = (
    DIR_CHECKPOINT_FOLD5 / "estado_fase1"
)

DIR_ESTADO_F5_F1.mkdir(
    parents=True,
    exist_ok=True
)

epoca_f5_f1 = tf.Variable(
    0,
    dtype=tf.int64,
    trainable=False,
    name="epoca_concluida"
)

checkpoint_f5_f1 = tf.train.Checkpoint(
    modelo=modelo_embeddings_f5,
    otimizador=otimizador_f5_f1,
    epoca=epoca_f5_f1
)

gerenciador_f5_f1 = tf.train.CheckpointManager(
    checkpoint_f5_f1,
    directory=str(DIR_ESTADO_F5_F1),
    max_to_keep=3
)

# ------------------------------------------------------------
# 6. Verificações
# ------------------------------------------------------------

assert not backbone_f5.trainable

assert modelo_embeddings_f5.count_params() == 14878912

assert int(otimizador_f5_f1.iterations.numpy()) == 0

assert gerenciador_f5_f1.latest_checkpoint is None, (
    "Já existe checkpoint da fase 1."
)

print("=" * 60)
print("FOLD 5 — FASE 1 CONFIGURADA")
print("=" * 60)

print("Otimizador: Adam")
print("Learning rate: 0.0001")
print("Épocas previstas: 10")
print("Passos por época:", PASSOS_POR_EPOCA)
print("Lotes de validação:", LOTES_VALIDACAO)

print(
    "Parâmetros treináveis:",
    sum(
        int(tf.size(v).numpy())
        for v in modelo_embeddings_f5.trainable_variables
    )
)

print("Iterações do otimizador: 0")
print("Checkpoint:", DIR_ESTADO_F5_F1)

print("\nFASE 1 PRONTA PARA TREINAMENTO")


In [ ]:

import inspect

print("=" * 60)
print("PASSO DE TREINAMENTO ORIGINAL — FOLD 4")
print("=" * 60)

print(inspect.getsource(passo_treino_f4_f1))

print("\n" + "=" * 60)
print("VALIDAÇÃO FIXA — INFORMAÇÕES DISPONÍVEIS")
print("=" * 60)

print("Lotes Fold 4:", len(dataset_validacao_f4))
print("Lotes Fold 5:", len(dataset_validacao_f5))

print("\nArquivos do notebook no ambiente:")

from pathlib import Path

for pasta in (
    Path("/content"),
    Path("/content/drive/MyDrive/TCC_Caes")
):
    if pasta.exists():
        for arquivo in list(pasta.glob("*.ipynb"))[:10]:
            print(arquivo)


In [ ]:

# ============================================================
# FOLD 5 — TREINAMENTO FASE 1
# VGG16 congelada | Adam 1e-4 | 10 épocas
# ============================================================

import time
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from tqdm.auto import tqdm

TOTAL_EPOCAS_F5_F1 = 10

ARQUIVO_HISTORICO_F5_F1 = (
    DIR_RESULTADOS_FOLD5 /
    "historico_fase1.csv"
)

ARQUIVO_MELHOR_F5_F1 = (
    DIR_CHECKPOINT_FOLD5 /
    "melhor_fase1_fold5.weights.h5"
)

ARQUIVO_CONFIG_F5_F1 = (
    DIR_RESULTADOS_FOLD5 /
    "configuracao_fase1.json"
)

# ------------------------------------------------------------
# 1. Configuração do experimento
# ------------------------------------------------------------

configuracao_f5_f1 = {
    "fold": 5,
    "fase": 1,
    "epocas": TOTAL_EPOCAS_F5_F1,
    "passos_por_epoca": PASSOS_POR_EPOCA,
    "lotes_validacao": LOTES_VALIDACAO,
    "batch_size": BATCH_SIZE,
    "P": P,
    "K": K,
    "seed": SEED,
    "seed_validacao": SEED + 10000 + 5,
    "learning_rate": 1e-4,
    "margem_triplet": MARGEM,
    "backbone": "VGG16",
    "pesos_iniciais": "ImageNet",
    "backbone_congelado": True
}

with open(
    ARQUIVO_CONFIG_F5_F1,
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        configuracao_f5_f1,
        arquivo,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 2. Recuperação de checkpoint
# ------------------------------------------------------------

ultimo_checkpoint = gerenciador_f5_f1.latest_checkpoint

if ultimo_checkpoint:
    checkpoint_f5_f1.restore(
        ultimo_checkpoint
    ).expect_partial()

    print(
        "Checkpoint restaurado:",
        ultimo_checkpoint
    )

epoca_inicial = int(
    epoca_f5_f1.numpy()
)

# ------------------------------------------------------------
# 3. Histórico anterior
# ------------------------------------------------------------

if ARQUIVO_HISTORICO_F5_F1.exists():
    historico_f5_f1 = pd.read_csv(
        ARQUIVO_HISTORICO_F5_F1
    ).to_dict("records")
else:
    historico_f5_f1 = []

historico_f5_f1 = [
    linha
    for linha in historico_f5_f1
    if int(linha["epoca"]) <= epoca_inicial
]

melhor_loss_f5_f1 = min(
    (
        float(linha["loss_validacao"])
        for linha in historico_f5_f1
    ),
    default=float("inf")
)

# ------------------------------------------------------------
# 4. Treinamento
# ------------------------------------------------------------

print("=" * 60)
print("FOLD 5 — TREINAMENTO FASE 1")
print("=" * 60)

print("Época inicial:", epoca_inicial)
print("Época final:", TOTAL_EPOCAS_F5_F1)
print("Melhor loss anterior:", melhor_loss_f5_f1)

inicio_total = time.time()

for epoca in range(
    epoca_inicial,
    TOTAL_EPOCAS_F5_F1
):

    inicio_epoca = time.time()

    dataset_treino = criar_dataset_treino_f5(
        epoca
    )

    perdas_treino = []

    barra = tqdm(
        dataset_treino,
        total=PASSOS_POR_EPOCA,
        desc=f"Fold 5 | Fase 1 | Época {epoca+1}/10",
        unit="lote"
    )

    for imagens, rotulos in barra:
        perda = passo_treino_f5_f1(
            imagens,
            rotulos
        )

        valor = float(perda.numpy())
        perdas_treino.append(valor)

        barra.set_postfix(
            loss=f"{np.mean(perdas_treino):.5f}"
        )

    loss_treino = float(
        np.mean(perdas_treino)
    )

    # --------------------------------------------------------
    # Validação fixa
    # --------------------------------------------------------

    perdas_validacao = []

    for imagens, rotulos in dataset_validacao_f5:
        perda = passo_validacao_f5_f1(
            imagens,
            rotulos
        )

        perdas_validacao.append(
            float(perda.numpy())
        )

    loss_validacao = float(
        np.mean(perdas_validacao)
    )

    assert np.isfinite(loss_treino)
    assert np.isfinite(loss_validacao)

    # --------------------------------------------------------
    # Salvar melhores pesos
    # --------------------------------------------------------

    if loss_validacao < melhor_loss_f5_f1:
        melhor_loss_f5_f1 = loss_validacao

        modelo_embeddings_f5.save_weights(
            ARQUIVO_MELHOR_F5_F1
        )

        print(
            f"Novo melhor modelo: "
            f"val_loss={loss_validacao:.6f}"
        )

    # --------------------------------------------------------
    # Salvar histórico
    # --------------------------------------------------------

    tempo_epoca = time.time() - inicio_epoca

    historico_f5_f1.append({
        "epoca": epoca + 1,
        "loss_treino": loss_treino,
        "loss_validacao": loss_validacao,
        "tempo_segundos": tempo_epoca
    })

    pd.DataFrame(
        historico_f5_f1
    ).to_csv(
        ARQUIVO_HISTORICO_F5_F1,
        index=False
    )

    # --------------------------------------------------------
    # Checkpoint de recuperação
    # --------------------------------------------------------

    epoca_f5_f1.assign(epoca + 1)

    caminho_checkpoint = gerenciador_f5_f1.save(
        checkpoint_number=epoca + 1
    )

    print(
        f"\nÉpoca {epoca+1}/10 concluída"
    )

    print(f"Loss treino: {loss_treino:.6f}")
    print(f"Loss validação: {loss_validacao:.6f}")
    print(f"Tempo: {tempo_epoca/60:.2f} min")
    print("Checkpoint:", caminho_checkpoint)

    print("-" * 60)

# ------------------------------------------------------------
# 5. Relatório final
# ------------------------------------------------------------

tempo_total = time.time() - inicio_total

print("\n" + "=" * 60)
print("FOLD 5 — FASE 1 CONCLUÍDA")
print("=" * 60)

print(
    "Épocas concluídas:",
    int(epoca_f5_f1.numpy())
)

print(
    "Iterações do Adam:",
    int(otimizador_f5_f1.iterations.numpy())
)

print(
    "Melhor loss validação:",
    melhor_loss_f5_f1
)

print(
    "Tempo total:",
    round(tempo_total / 60, 2),
    "minutos"
)

print(
    "Melhores pesos:",
    ARQUIVO_MELHOR_F5_F1
)

print(
    "Histórico:",
    ARQUIVO_HISTORICO_F5_F1
)

print("\nFASE 1 FINALIZADA")


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

pasta = Path(
    "/content/drive/MyDrive/TCC_Caes"
)

print("Pasta acessível:", pasta.exists())

if pasta.exists():
    print("Conteúdo:")
    for item in pasta.iterdir():
        print(" -", item.name)


In [ ]:

from pathlib import Path
import tensorflow as tf

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_CHECKPOINT_FOLD5 = RAIZ / "checkpoints/fold_5"
DIR_RESULTADOS_FOLD5 = RAIZ / "resultados/fold_5"

DIR_ESTADO_F5_F1 = (
    DIR_CHECKPOINT_FOLD5 / "estado_fase1"
)

print("=" * 60)
print("FOLD 5 — VERIFICAÇÃO NA TERCEIRA CONTA")
print("=" * 60)

print("Experimento acessível:", RAIZ.is_dir())

checkpoint = tf.train.latest_checkpoint(
    str(DIR_ESTADO_F5_F1)
)

print("Checkpoint encontrado:", checkpoint)

print(
    "Melhores pesos:",
    (
        DIR_CHECKPOINT_FOLD5 /
        "melhor_fase1_fold5.weights.h5"
    ).is_file()
)

print(
    "Histórico:",
    (
        DIR_RESULTADOS_FOLD5 /
        "historico_fase1.csv"
    ).is_file()
)

# Teste de escrita sem modificar arquivos existentes
arquivo_teste = RAIZ / "_teste_permissao_colab.txt"

try:
    if arquivo_teste.exists():
        print("Teste de escrita: nome já existente; não sobrescrito")
    else:
        arquivo_teste.write_text(
            "Teste de acesso pela terceira conta",
            encoding="utf-8"
        )
        print("Teste de escrita: OK")
        arquivo_teste.unlink()
        print("Arquivo temporário removido: OK")
except Exception as erro:
    print("Teste de escrita: FALHOU")
    print(type(erro).__name__, str(erro))

print("\nGPU:", tf.config.list_physical_devices("GPU"))


In [ ]:

# ============================================================
# FOLD 5 — RESTAURAÇÃO DA FASE 1
# Terceira conta Google Colab
# ============================================================

import tensorflow as tf
from pathlib import Path

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# 1. Reconstruir a arquitetura original
backbone_f5 = tf.keras.applications.VGG16(
    include_top=False,
    weights=None,
    input_shape=(224, 224, 3)
)

backbone_f5.trainable = False

entrada_f5 = tf.keras.Input(
    shape=(224, 224, 3),
    name="entrada_imagem"
)

x = backbone_f5(entrada_f5)

x = tf.keras.layers.GlobalAveragePooling2D(
    name="global_average_pooling2d"
)(x)

x = tf.keras.layers.Dense(
    256,
    activation="relu",
    name="dense_256"
)(x)

x = tf.keras.layers.Dense(
    128,
    activation="linear",
    name="dense_128"
)(x)

saida_f5 = tf.keras.layers.UnitNormalization(
    axis=-1,
    name="normalizacao_l2"
)(x)

modelo_embeddings_f5 = tf.keras.Model(
    entrada_f5,
    saida_f5,
    name="modelo_embeddings_fold5"
)

# 2. Reconstruir otimizador da fase 1
otimizador_f5_f1 = tf.keras.optimizers.Adam(
    learning_rate=1e-4
)

epoca_f5_f1 = tf.Variable(
    0,
    dtype=tf.int64,
    trainable=False,
    name="epoca_concluida"
)

# 3. Restaurar checkpoint
DIR_ESTADO_F5_F1 = (
    DIR_CHECKPOINT_FOLD5 / "estado_fase1"
)

checkpoint_f5_f1 = tf.train.Checkpoint(
    modelo=modelo_embeddings_f5,
    otimizador=otimizador_f5_f1,
    epoca=epoca_f5_f1
)

caminho_checkpoint = tf.train.latest_checkpoint(
    str(DIR_ESTADO_F5_F1)
)

assert caminho_checkpoint is not None, (
    "Checkpoint não encontrado."
)

checkpoint_f5_f1.restore(
    caminho_checkpoint
).expect_partial()

# 4. Verificar restauração
print("=" * 60)
print("FOLD 5 — MODELO RESTAURADO")
print("=" * 60)

print("Checkpoint:", caminho_checkpoint)

print(
    "Épocas concluídas:",
    int(epoca_f5_f1.numpy())
)

print(
    "Iterações do Adam:",
    int(otimizador_f5_f1.iterations.numpy())
)

print(
    "Parâmetros totais:",
    modelo_embeddings_f5.count_params()
)

print(
    "Parâmetros treináveis:",
    sum(
        int(tf.size(v).numpy())
        for v in modelo_embeddings_f5.trainable_variables
    )
)

assert int(epoca_f5_f1.numpy()) == 10
assert int(otimizador_f5_f1.iterations.numpy()) == 6250
assert modelo_embeddings_f5.count_params() == 14878912

print("\nRESTAURAÇÃO CONCLUÍDA COM SUCESSO")


In [ ]:

# ============================================================
# FOLD 5 — FASE 2: CONFIGURAÇÃO DO FINE-TUNING
# ============================================================

import tensorflow as tf

# 1. Garantir que a fase 1 foi restaurada
assert int(epoca_f5_f1.numpy()) == 10
assert int(otimizador_f5_f1.iterations.numpy()) == 6250

# 2. Liberar o bloco 5 da VGG16
backbone_f5.trainable = True

camadas_liberadas = {
    "block5_conv1",
    "block5_conv2",
    "block5_conv3"
}

for camada in backbone_f5.layers:
    camada.trainable = camada.name in camadas_liberadas

# 3. Criar otimizador independente da fase 2
otimizador_f5_f2 = tf.keras.optimizers.Adam(
    learning_rate=1e-5
)

# 4. Verificar parâmetros treináveis
parametros_treinaveis = sum(
    int(tf.size(variavel).numpy())
    for variavel in modelo_embeddings_f5.trainable_variables
)

camadas_ativas = [
    camada.name
    for camada in backbone_f5.layers
    if camada.trainable
]

print("=" * 60)
print("FOLD 5 — FASE 2 CONFIGURADA")
print("=" * 60)

print("Camadas liberadas:", camadas_ativas)
print("Parâmetros treináveis:", parametros_treinaveis)
print("Parâmetros totais:", modelo_embeddings_f5.count_params())
print("Learning rate:", float(otimizador_f5_f2.learning_rate.numpy()))
print("Iterações do Adam:", int(otimizador_f5_f2.iterations.numpy()))

assert set(camadas_ativas) == camadas_liberadas
assert parametros_treinaveis == 7243648
assert modelo_embeddings_f5.count_params() == 14878912

print("\nFINE-TUNING CONFIGURADO COM SUCESSO")


In [ ]:

# ============================================================
# FOLD 5 — VERIFICAÇÃO DO PIPELINE NA NOVA SESSÃO
# ============================================================

import tensorflow as tf

componentes = [
    "df_treino_f5",
    "df_validacao_f5",
    "indice_treino_f5",
    "indice_validacao_f5",
    "ids_treino_ordenados_f5",
    "ids_validacao_ordenados_f5",
    "mapa_treino_f5",
    "mapa_validacao_f5",
    "preparar_imagem_f5",
    "dataset_f5",
    "criar_dataset_treino_f5",
    "criar_dataset_validacao_f5",
    "dataset_validacao_f5",
    "triplet_loss_f4",
    "triplet_loss_f5",
]

print("=" * 60)
print("FOLD 5 — AUDITORIA DO PIPELINE")
print("=" * 60)

faltantes = []

for nome in componentes:
    disponivel = nome in globals()

    print(
        f"{nome}:",
        "OK" if disponivel else "NÃO CARREGADO"
    )

    if not disponivel:
        faltantes.append(nome)

print("\n" + "=" * 60)
print("HIPERPARÂMETROS")
print("=" * 60)

for nome in [
    "SEED",
    "P",
    "K",
    "BATCH_SIZE",
    "PASSOS_POR_EPOCA",
    "LOTES_VALIDACAO",
    "MARGEM"
]:
    print(
        f"{nome}:",
        globals().get(nome, "NÃO CARREGADO")
    )

print("\n" + "=" * 60)
print("MODELO RESTAURADO")
print("=" * 60)

print(
    "Parâmetros treináveis:",
    sum(
        int(tf.size(v).numpy())
        for v in modelo_embeddings_f5.trainable_variables
    )
)

print(
    "Iterações Adam fase 2:",
    int(otimizador_f5_f2.iterations.numpy())
)

print("\nComponentes faltantes:", len(faltantes))

if faltantes:
    print("Precisamos recuperar o pipeline.")
else:
    print("Pipeline disponível para validação.")


In [ ]:

# ============================================================
# FOLD 5 — RECUPERAÇÃO DO PIPELINE DE DADOS
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

# 1. Hiperparâmetros originais
SEED = 42
P = 8
K = 4
BATCH_SIZE = 32
PASSOS_POR_EPOCA = 625
LOTES_VALIDACAO = 150
MARGEM = 0.2

DIR_SPLITS = RAIZ / "splits"

# 2. Carregar splits
df_treino_f5 = pd.read_csv(
    DIR_SPLITS / "fold_5_imagens_treino.csv"
)

df_validacao_f5 = pd.read_csv(
    DIR_SPLITS / "fold_5_imagens_validacao.csv"
)

print("Colunas treino:", df_treino_f5.columns.tolist())
print("Colunas validação:", df_validacao_f5.columns.tolist())

# 3. Resolver caminhos locais
# Preserva caminho_local caso já exista no CSV.

def resolver_caminho_f5(caminho):
    caminho = str(caminho)

    if caminho.startswith("/content/drive/"):
        return caminho

    return str(
        Path("/content/drive/MyDrive/TCC_Caes") /
        caminho.lstrip("/")
    )

for df in (df_treino_f5, df_validacao_f5):
    if "caminho_local" not in df.columns:
        assert "caminho" in df.columns, (
            "CSV sem coluna caminho ou caminho_local."
        )
        df["caminho_local"] = df["caminho"].map(
            resolver_caminho_f5
        )

# 4. Índices por identidade
indice_treino_f5 = {
    str(identidade): grupo["caminho_local"].tolist()
    for identidade, grupo in df_treino_f5.groupby(
        "identidade"
    )
}

indice_validacao_f5 = {
    str(identidade): grupo["caminho_local"].tolist()
    for identidade, grupo in df_validacao_f5.groupby(
        "identidade"
    )
}

ids_treino_ordenados_f5 = sorted(
    indice_treino_f5.keys()
)

ids_validacao_ordenados_f5 = sorted(
    indice_validacao_f5.keys()
)

mapa_treino_f5 = {
    identidade: i
    for i, identidade in enumerate(
        ids_treino_ordenados_f5
    )
}

mapa_validacao_f5 = {
    identidade: i
    for i, identidade in enumerate(
        ids_validacao_ordenados_f5
    )
}

# 5. Pré-processamento VGG16
def preparar_imagem_f5(caminho, rotulo):
    conteudo = tf.io.read_file(caminho)

    imagem = tf.io.decode_image(
        conteudo,
        channels=3,
        expand_animations=False
    )

    imagem.set_shape([None, None, 3])

    imagem = tf.image.resize(
        imagem, [224, 224]
    )

    imagem = tf.keras.applications.vgg16.preprocess_input(
        tf.cast(imagem, tf.float32)
    )

    return imagem, rotulo


def dataset_f5(caminhos, rotulos):
    return (
        tf.data.Dataset.from_tensor_slices((
            tf.constant(caminhos, dtype=tf.string),
            tf.constant(rotulos, dtype=tf.int32)
        ))
        .map(
            preparar_imagem_f5,
            num_parallel_calls=4
        )
        .batch(
            BATCH_SIZE,
            drop_remainder=True
        )
        .prefetch(2)
    )

# 6. Amostragem de treinamento
def criar_dataset_treino_f5(epoca):
    rng = np.random.default_rng(SEED + epoca)

    caminhos, rotulos = [], []

    for _ in range(PASSOS_POR_EPOCA):
        escolhidas = rng.choice(
            ids_treino_ordenados_f5,
            size=P,
            replace=False
        )

        for identidade in escolhidas:
            fotos = indice_treino_f5[identidade]

            selecionadas = rng.choice(
                fotos,
                size=K,
                replace=len(fotos) < K
            )

            caminhos.extend(map(str, selecionadas))
            rotulos.extend(
                [mapa_treino_f5[identidade]] * K
            )

    return dataset_f5(caminhos, rotulos)

# 7. Validação fixa
def criar_dataset_validacao_f5():
    rng = np.random.default_rng(SEED + 10000 + 5)

    caminhos, rotulos = [], []

    for _ in range(LOTES_VALIDACAO):
        escolhidas = rng.choice(
            ids_validacao_ordenados_f5,
            size=P,
            replace=False
        )

        for identidade in escolhidas:
            fotos = indice_validacao_f5[identidade]

            selecionadas = rng.choice(
                fotos,
                size=K,
                replace=len(fotos) < K
            )

            caminhos.extend(map(str, selecionadas))
            rotulos.extend(
                [mapa_validacao_f5[identidade]] * K
            )

    return dataset_f5(caminhos, rotulos)


dataset_validacao_f5 = criar_dataset_validacao_f5()

# 8. Auditoria
print("\n" + "=" * 60)
print("FOLD 5 — PIPELINE RECUPERADO")
print("=" * 60)

print("Imagens treino:", len(df_treino_f5))
print("Imagens validação:", len(df_validacao_f5))
print("Identidades treino:", len(indice_treino_f5))
print("Identidades validação:", len(indice_validacao_f5))

print(
    "Lotes validação:",
    int(tf.data.experimental.cardinality(
        dataset_validacao_f5
    ).numpy())
)

print(
    "Parâmetros treináveis preservados:",
    sum(
        int(tf.size(v).numpy())
        for v in modelo_embeddings_f5.trainable_variables
    )
)

assert len(df_treino_f5) == 19934
assert len(df_validacao_f5) == 4937
assert len(indice_treino_f5) == 2305
assert len(indice_validacao_f5) == 576

assert int(
    tf.data.experimental.cardinality(
        dataset_validacao_f5
    ).numpy()
) == 150

assert sum(
    int(tf.size(v).numpy())
    for v in modelo_embeddings_f5.trainable_variables
) == 7243648

print("\nPIPELINE RECUPERADO COM SUCESSO")


In [ ]:

from pathlib import Path

print("=" * 60)
print("FOLD 5 — LOCALIZAÇÃO DOS DATASETS")
print("=" * 60)

bases = [
    Path("/content/drive/MyDrive/datasets_extraido"),
    Path("/content/drive/MyDrive/TCC_Caes/datasets"),
    Path("/content/drive/MyDrive/TCC_Caes"),
]

for base in bases:
    print(f"\nPasta: {base}")
    print("Existe:", base.is_dir())

    if base.is_dir():
        print("Conteúdo:")
        for item in sorted(base.iterdir()):
            print(" -", item.name)

print("\n" + "=" * 60)
print("AMOSTRAS DOS CAMINHOS ORIGINAIS")
print("=" * 60)

for nome, df in [
    ("Treinamento", df_treino_f5),
    ("Validação", df_validacao_f5)
]:
    print(f"\n{nome}:")
    print(df["caminho"].head(3).to_string(index=False))

print("\nLOCALIZAÇÃO CONCLUÍDA")


In [ ]:

# ============================================================
# FOLD 5 — PREPARAÇÃO FINAL DA FASE 2
# ============================================================

import tensorflow as tf
import numpy as np

print("=" * 60)
print("FOLD 5 — VALIDAÇÃO FINAL DO PIPELINE")
print("=" * 60)

# 1. Confirmar integridade dos dados
assert len(df_treino_f5) == 19934
assert len(df_validacao_f5) == 4937
assert len(faltantes_f5) == 0

# 2. Confirmar hiperparâmetros
assert P == 8
assert K == 4
assert BATCH_SIZE == 32
assert PASSOS_POR_EPOCA == 625
assert LOTES_VALIDACAO == 150
assert MARGEM == 0.2

# 3. Testar lote de treinamento
dataset_teste_f5 = criar_dataset_treino_f5(epoca=10)

imagens, rotulos = next(iter(dataset_teste_f5))

print("Imagens:", imagens.shape)
print("Rótulos:", rotulos.shape)

assert imagens.shape == (32, 224, 224, 3)
assert len(np.unique(rotulos.numpy())) == P

# 4. Testar inferência
embeddings = modelo_embeddings_f5(
    imagens,
    training=False
)

print("Embeddings:", embeddings.shape)

assert embeddings.shape == (32, 128)

# 5. Calcular Triplet Loss
perda = triplet_loss_f5(embeddings, rotulos)

print("Triplet Loss:", float(perda.numpy()))

assert np.isfinite(float(perda.numpy()))

# 6. Preparar funções de treinamento
@tf.function
def passo_treino_f5_f2(imagens, rotulos):
    with tf.GradientTape() as tape:
        embeddings = modelo_embeddings_f5(
            imagens,
            training=True
        )

        perda = triplet_loss_f5(
            embeddings,
            rotulos
        )

    variaveis = modelo_embeddings_f5.trainable_variables

    gradientes = tape.gradient(
        perda,
        variaveis
    )

    pares = [
        (g, v)
        for g, v in zip(gradientes, variaveis)
        if g is not None
    ]

    otimizador_f5_f2.apply_gradients(pares)

    return perda


@tf.function
def passo_validacao_f5_f2(imagens, rotulos):
    embeddings = modelo_embeddings_f5(
        imagens,
        training=False
    )

    return triplet_loss_f5(
        embeddings,
        rotulos
    )

# 7. Auditoria do modelo
parametros = sum(
    int(tf.size(v).numpy())
    for v in modelo_embeddings_f5.trainable_variables
)

print("\nParâmetros treináveis:", parametros)
print(
    "Iterações Adam fase 2:",
    int(otimizador_f5_f2.iterations.numpy())
)

assert parametros == 7243648
assert int(otimizador_f5_f2.iterations.numpy()) == 0

print("\n" + "=" * 60)
print("FOLD 5 — PRONTO PARA TREINAMENTO DA FASE 2")
print("=" * 60)


In [ ]:

# ============================================================
# FOLD 5 — FASE 2: FINE-TUNING COMPLETO
# VGG16 + Triplet Loss Semi-Hard
# Checkpoints, histórico CSV e barras de progresso
# ============================================================

from tqdm.auto import tqdm
import tensorflow as tf
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path

# ------------------------------------------------------------
# 0. Configurações
# ------------------------------------------------------------

EPOCAS_FASE2_F5 = 10

DIR_ESTADO_F5_F2 = (
    DIR_CHECKPOINT_FOLD5 / "estado_fase2"
)

DIR_ESTADO_F5_F2.mkdir(
    parents=True,
    exist_ok=True
)

DIR_RESULTADOS_FOLD5.mkdir(
    parents=True,
    exist_ok=True
)

ARQUIVO_HISTORICO_F5_F2 = (
    DIR_RESULTADOS_FOLD5 / "historico_fase2.csv"
)

ARQUIVO_MELHOR_F5_F2 = (
    DIR_CHECKPOINT_FOLD5 /
    "melhor_fase2_fold5.weights.h5"
)

ARQUIVO_FINAL_F5 = (
    DIR_CHECKPOINT_FOLD5 /
    "modelo_final_fold5.weights.h5"
)

ARQUIVO_CONFIG_F5_F2 = (
    DIR_RESULTADOS_FOLD5 /
    "configuracao_fase2.json"
)

# ------------------------------------------------------------
# 1. Auditoria inicial
# ------------------------------------------------------------

assert P == 8
assert K == 4
assert BATCH_SIZE == 32
assert PASSOS_POR_EPOCA == 625
assert LOTES_VALIDACAO == 150
assert MARGEM == 0.2

assert len(df_treino_f5) == 19934
assert len(df_validacao_f5) == 4937

parametros_treinaveis = sum(
    int(tf.size(v).numpy())
    for v in modelo_embeddings_f5.trainable_variables
)

assert parametros_treinaveis == 7243648

camadas_ativas = {
    camada.name
    for camada in backbone_f5.layers
    if camada.trainable
}

assert camadas_ativas == {
    "block5_conv1",
    "block5_conv2",
    "block5_conv3"
}

assert np.isclose(
    float(otimizador_f5_f2.learning_rate.numpy()),
    1e-5
)

# ------------------------------------------------------------
# 2. Estado persistente da fase 2
# ------------------------------------------------------------

epoca_f5_f2 = tf.Variable(
    0,
    dtype=tf.int64,
    trainable=False
)

checkpoint_f5_f2 = tf.train.Checkpoint(
    modelo=modelo_embeddings_f5,
    otimizador=otimizador_f5_f2,
    epoca=epoca_f5_f2
)

gerenciador_f5_f2 = tf.train.CheckpointManager(
    checkpoint_f5_f2,
    directory=str(DIR_ESTADO_F5_F2),
    max_to_keep=3
)

checkpoint_anterior = (
    gerenciador_f5_f2.latest_checkpoint
)

if checkpoint_anterior:

    print("Restaurando checkpoint da fase 2...")

    # Preparar variáveis do Adam para restaurar
    # corretamente os momentos do otimizador.
    otimizador_f5_f2.build(
        modelo_embeddings_f5.trainable_variables
    )

    status = checkpoint_f5_f2.restore(
        checkpoint_anterior
    )

    status.assert_existing_objects_matched()

    print("Checkpoint:", checkpoint_anterior)

else:

    assert int(
        otimizador_f5_f2.iterations.numpy()
    ) == 0, (
        "O Adam já possui iterações, mas não há "
        "checkpoint da fase 2. Não reinicie "
        "a partir deste estado."
    )

    print(
        "Iniciando fase 2 a partir dos pesos "
        "restaurados da fase 1."
    )

epoca_atual = int(epoca_f5_f2.numpy())

iteracoes_atuais = int(
    otimizador_f5_f2.iterations.numpy()
)

assert 0 <= epoca_atual <= EPOCAS_FASE2_F5

assert iteracoes_atuais == (
    epoca_atual * PASSOS_POR_EPOCA
), (
    "Inconsistência entre a época restaurada "
    "e as iterações do Adam."
)

# ------------------------------------------------------------
# 3. Recuperar histórico
# ------------------------------------------------------------

colunas_historico = [
    "epoca",
    "loss_treino",
    "loss_validacao",
    "tempo_segundos"
]

if ARQUIVO_HISTORICO_F5_F2.exists():

    historico_f5_f2 = pd.read_csv(
        ARQUIVO_HISTORICO_F5_F2
    )

else:

    historico_f5_f2 = pd.DataFrame(
        columns=colunas_historico
    )

if not historico_f5_f2.empty:

    historico_f5_f2 = historico_f5_f2[
        historico_f5_f2["epoca"] <= epoca_atual
    ].copy()

    historico_f5_f2 = (
        historico_f5_f2
        .drop_duplicates(
            subset=["epoca"],
            keep="last"
        )
        .sort_values("epoca")
        .reset_index(drop=True)
    )

if epoca_atual > 0:

    epocas_registradas = set(
        historico_f5_f2["epoca"].astype(int)
    )

    assert epocas_registradas == set(
        range(1, epoca_atual + 1)
    ), (
        "O histórico CSV não corresponde às "
        "épocas do checkpoint. Verifique os "
        "arquivos antes de continuar."
    )

if historico_f5_f2.empty:

    melhor_loss_f5_f2 = float("inf")

else:

    melhor_loss_f5_f2 = float(
        historico_f5_f2["loss_validacao"].min()
    )

# ------------------------------------------------------------
# 4. Registrar configuração
# ------------------------------------------------------------

config_f5_f2 = {
    "fold": 5,
    "fase": 2,
    "arquitetura": "VGG16",
    "epocas": EPOCAS_FASE2_F5,
    "passos_por_epoca": PASSOS_POR_EPOCA,
    "lotes_validacao": LOTES_VALIDACAO,
    "P": P,
    "K": K,
    "batch_size": BATCH_SIZE,
    "margem": MARGEM,
    "learning_rate": 1e-5,
    "camadas_liberadas": [
        "block5_conv1",
        "block5_conv2",
        "block5_conv3"
    ],
    "seed": SEED,
    "epoca_inicial_amostragem": 10
}

with open(
    ARQUIVO_CONFIG_F5_F2,
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        config_f5_f2,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 5. Resumo inicial
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FOLD 5 — FASE 2: FINE-TUNING")
print("=" * 60)

print("Épocas concluídas:", epoca_atual)
print("Épocas previstas:", EPOCAS_FASE2_F5)
print("Passos por época:", PASSOS_POR_EPOCA)
print("Lotes de validação:", LOTES_VALIDACAO)
print("Parâmetros treináveis:", parametros_treinaveis)

print(
    "Learning rate:",
    float(otimizador_f5_f2.learning_rate.numpy())
)

print("Iterações do Adam:", iteracoes_atuais)
print("Melhor loss anterior:", melhor_loss_f5_f2)

print("\nIniciando treinamento...")

# ------------------------------------------------------------
# 6. Treinamento
# ------------------------------------------------------------

for epoca in range(
    epoca_atual,
    EPOCAS_FASE2_F5
):

    inicio = time.time()

    print("\n" + "=" * 60)
    print(
        f"FOLD 5 — FASE 2 — "
        f"ÉPOCA {epoca + 1}/{EPOCAS_FASE2_F5}"
    )
    print("=" * 60)

    # --------------------------------------------------------
    # 6.1. Dataset de treinamento
    # --------------------------------------------------------

    dataset_treino = criar_dataset_treino_f5(
        epoca=10 + epoca
    )

    metrica_treino = tf.keras.metrics.Mean()

    # --------------------------------------------------------
    # 6.2. Treinamento com barra de progresso
    # --------------------------------------------------------

    with tqdm(
        dataset_treino,
        total=PASSOS_POR_EPOCA,
        desc=(
            f"Fold 5 | Treino | "
            f"Época {epoca + 1}/10"
        ),
        unit="lote",
        leave=True
    ) as barra_treino:

        for passo, (imagens, rotulos) in enumerate(
            barra_treino,
            start=1
        ):

            perda = passo_treino_f5_f2(
                imagens,
                rotulos
            )

            metrica_treino.update_state(perda)

            if (
                passo % 10 == 0
                or passo == PASSOS_POR_EPOCA
            ):

                barra_treino.set_postfix(
                    loss=(
                        f"{float(metrica_treino.result().numpy()):.6f}"
                    )
                )

    loss_treino = float(
        metrica_treino.result().numpy()
    )

    # --------------------------------------------------------
    # 6.3. Validação com barra de progresso
    # --------------------------------------------------------

    metrica_validacao = tf.keras.metrics.Mean()

    with tqdm(
        dataset_validacao_f5,
        total=LOTES_VALIDACAO,
        desc=(
            f"Fold 5 | Validação | "
            f"Época {epoca + 1}/10"
        ),
        unit="lote",
        leave=False
    ) as barra_validacao:

        for passo, (imagens, rotulos) in enumerate(
            barra_validacao,
            start=1
        ):

            perda = passo_validacao_f5_f2(
                imagens,
                rotulos
            )

            metrica_validacao.update_state(perda)

            if (
                passo % 10 == 0
                or passo == LOTES_VALIDACAO
            ):

                barra_validacao.set_postfix(
                    loss=(
                        f"{float(metrica_validacao.result().numpy()):.6f}"
                    )
                )

    loss_validacao = float(
        metrica_validacao.result().numpy()
    )

    # --------------------------------------------------------
    # 6.4. Auditoria dos resultados
    # --------------------------------------------------------

    assert np.isfinite(loss_treino), (
        "Loss de treinamento inválida."
    )

    assert np.isfinite(loss_validacao), (
        "Loss de validação inválida."
    )

    iteracoes_esperadas = (
        (epoca + 1) * PASSOS_POR_EPOCA
    )

    assert int(
        otimizador_f5_f2.iterations.numpy()
    ) == iteracoes_esperadas, (
        "Número inesperado de atualizações do Adam."
    )

    tempo_epoca = time.time() - inicio

    print("\n" + "-" * 60)

    print(
        f"Época concluída: "
        f"{epoca + 1}/{EPOCAS_FASE2_F5}"
    )

    print(f"Loss treino: {loss_treino:.6f}")
    print(f"Loss validação: {loss_validacao:.6f}")
    print(f"Tempo: {tempo_epoca / 60:.2f} min")

    # --------------------------------------------------------
    # 6.5. Atualizar histórico CSV
    # --------------------------------------------------------

    novo_registro = pd.DataFrame([{
        "epoca": epoca + 1,
        "loss_treino": loss_treino,
        "loss_validacao": loss_validacao,
        "tempo_segundos": tempo_epoca
    }])

    historico_f5_f2 = pd.concat(
        [
            historico_f5_f2,
            novo_registro
        ],
        ignore_index=True
    )

    historico_f5_f2.to_csv(
        ARQUIVO_HISTORICO_F5_F2,
        index=False
    )

    # --------------------------------------------------------
    # 6.6. Salvar melhor modelo
    # --------------------------------------------------------

    if loss_validacao < melhor_loss_f5_f2:

        melhor_loss_f5_f2 = loss_validacao

        modelo_embeddings_f5.save_weights(
            str(ARQUIVO_MELHOR_F5_F2)
        )

        print("Novo melhor modelo salvo.")

    # --------------------------------------------------------
    # 6.7. Salvar checkpoint completo
    # --------------------------------------------------------

    epoca_f5_f2.assign(epoca + 1)

    caminho_checkpoint = (
        gerenciador_f5_f2.save(
            checkpoint_number=epoca + 1
        )
    )

    print(
        "Checkpoint salvo:",
        caminho_checkpoint
    )

    print(
        "Iterações do Adam:",
        int(otimizador_f5_f2.iterations.numpy())
    )

    print("-" * 60)

# ------------------------------------------------------------
# 7. Finalização
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FOLD 5 — FASE 2 CONCLUÍDA")
print("=" * 60)

print(
    "Épocas concluídas:",
    int(epoca_f5_f2.numpy())
)

print(
    "Melhor loss validação:",
    melhor_loss_f5_f2
)

print(
    "Iterações finais do Adam:",
    int(otimizador_f5_f2.iterations.numpy())
)

# Salvar os pesos da última época
modelo_embeddings_f5.save_weights(
    str(ARQUIVO_FINAL_F5)
)

print("\nArquivos gerados:")
print("Melhor modelo:", ARQUIVO_MELHOR_F5_F2)
print("Modelo final:", ARQUIVO_FINAL_F5)
print("Histórico:", ARQUIVO_HISTORICO_F5_F2)
print("Checkpoints:", DIR_ESTADO_F5_F2)

print("\nTREINAMENTO FINALIZADO COM SUCESSO")


In [ ]:

# ============================================================
# FOLD 5 — RECUPERAÇÃO DO INÍCIO DA FASE 2
# ============================================================

import tensorflow as tf

# Garantir que não existe checkpoint da fase 2
assert tf.train.latest_checkpoint(
    str(DIR_CHECKPOINT_FOLD5 / "estado_fase2")
) is None, (
    "Já existe checkpoint da fase 2. "
    "Não sobrescrever o progresso."
)

# Restaurar somente os pesos do modelo a partir da fase 1.
# Não restaurar o otimizador da fase 1.
checkpoint_fase1_pesos = tf.train.Checkpoint(
    modelo=modelo_embeddings_f5
)

caminho_fase1 = tf.train.latest_checkpoint(
    str(DIR_CHECKPOINT_FOLD5 / "estado_fase1")
)

assert caminho_fase1 is not None

checkpoint_fase1_pesos.restore(
    caminho_fase1
).expect_partial()

# Criar um novo Adam para a fase 2
otimizador_f5_f2 = tf.keras.optimizers.Adam(
    learning_rate=1e-5
)

# Conferir o estado
assert int(
    otimizador_f5_f2.iterations.numpy()
) == 0

parametros = sum(
    int(tf.size(v).numpy())
    for v in modelo_embeddings_f5.trainable_variables
)

assert parametros == 7243648

print("=" * 60)
print("FOLD 5 — FASE 2 REINICIALIZADA")
print("=" * 60)

print("Checkpoint de origem:", caminho_fase1)
print("Parâmetros treináveis:", parametros)
print(
    "Learning rate:",
    float(otimizador_f5_f2.learning_rate.numpy())
)
print(
    "Iterações do novo Adam:",
    int(otimizador_f5_f2.iterations.numpy())
)

print("\nPRONTO PARA REINICIAR A FASE 2")


In [ ]:

# ============================================================
# FOLD 5 — DIAGNÓSTICO APÓS RECONEXÃO DO COLAB
# ============================================================

from pathlib import Path
import tensorflow as tf

RAIZ_DIAG = Path(
    "/content/drive/MyDrive/TCC_Caes/"
    "experimentos/tcc1_kfold"
)

DIR_CKPT_DIAG = (
    RAIZ_DIAG / "checkpoints/fold_5"
)

DIR_RESULT_DIAG = (
    RAIZ_DIAG / "resultados/fold_5"
)

print("=" * 60)
print("FOLD 5 — RECUPERAÇÃO APÓS RECONEXÃO")
print("=" * 60)

print("\n1. Estado da sessão Python")

for nome in [
    "modelo_embeddings_f5",
    "otimizador_f5_f2",
    "passo_treino_f5_f2",
    "loss_treino",
    "loss_validacao"
]:
    print(
        f"{nome}:",
        "DISPONÍVEL" if nome in globals()
        else "NÃO DISPONÍVEL"
    )

if "otimizador_f5_f2" in globals():
    print(
        "Iterações Adam atual:",
        int(otimizador_f5_f2.iterations.numpy())
    )

print("\n2. Estado do Google Drive")

print(
    "Projeto acessível:",
    RAIZ_DIAG.exists()
)

print("\n3. Checkpoints")

for fase in ["estado_fase1", "estado_fase2"]:
    pasta = DIR_CKPT_DIAG / fase

    checkpoint = (
        tf.train.latest_checkpoint(str(pasta))
        if pasta.exists()
        else None
    )

    print(f"{fase}:", checkpoint)

print("\n4. Arquivos de pesos")

for nome in [
    "melhor_fase1_fold5.weights.h5",
    "resgate_fase2_epoca1.weights.h5",
    "melhor_fase2_fold5.weights.h5",
    "modelo_final_fold5.weights.h5"
]:
    caminho = DIR_CKPT_DIAG / nome

    print(
        nome,
        "ENCONTRADO" if caminho.exists()
        else "NÃO ENCONTRADO"
    )

print("\n5. Histórico")

for nome in [
    "historico_fase1.csv",
    "historico_fase2.csv",
    "resgate_fase2_epoca1.json"
]:
    caminho = DIR_RESULT_DIAG / nome

    print(
        nome,
        "ENCONTRADO" if caminho.exists()
        else "NÃO ENCONTRADO"
    )

print("\n" + "=" * 60)
print("DIAGNÓSTICO FINALIZADO")


In [ ]:

# ============================================================
# FOLD 5 — PREPARAÇÃO SEGURA PARA RETOMADA
# ============================================================

import tensorflow as tf
import json

# ------------------------------------------------------------
# 1. Verificar o backup
# ------------------------------------------------------------

arquivo_pesos = (
    DIR_CHECKPOINT_FOLD5 /
    "resgate_fase2_epoca1.weights.h5"
)

arquivo_info = (
    DIR_RESULTADOS_FOLD5 /
    "resgate_fase2_epoca1.json"
)

assert arquivo_pesos.exists(), (
    "Backup dos pesos não encontrado."
)

assert arquivo_info.exists(), (
    "Metadados do resgate não encontrados."
)

with open(
    arquivo_info,
    "r",
    encoding="utf-8"
) as arquivo:
    dados_resgate = json.load(arquivo)

assert dados_resgate["epoca"] == 1
assert dados_resgate["lotes_processados"] == 625

# Não sobrescrever um treinamento já recuperado
assert tf.train.latest_checkpoint(
    str(DIR_CHECKPOINT_FOLD5 / "estado_fase2")
) is None, (
    "Já existe checkpoint da fase 2. "
    "Verifique antes de continuar."
)

# ------------------------------------------------------------
# 2. Restaurar os pesos da época 1
# ------------------------------------------------------------

modelo_embeddings_f5.load_weights(
    str(arquivo_pesos)
)

# ------------------------------------------------------------
# 3. Criar e inicializar um novo Adam
# ------------------------------------------------------------

otimizador_f5_f2 = tf.keras.optimizers.Adam(
    learning_rate=1e-5
)

variaveis_treinaveis = (
    modelo_embeddings_f5.trainable_variables
)

otimizador_f5_f2.build(
    variaveis_treinaveis
)

assert int(
    otimizador_f5_f2.iterations.numpy()
) == 0

# ------------------------------------------------------------
# 4. Redefinir a função de treinamento
# ------------------------------------------------------------

@tf.function
def passo_treino_f5_f2_novo(imagens, rotulos):

    with tf.GradientTape() as tape:

        embeddings = modelo_embeddings_f5(
            imagens,
            training=True
        )

        perda = triplet_loss_f5(
            embeddings,
            rotulos
        )

    variaveis = (
        modelo_embeddings_f5.trainable_variables
    )

    gradientes = tape.gradient(
        perda,
        variaveis
    )

    pares = [
        (gradiente, variavel)
        for gradiente, variavel in zip(
            gradientes,
            variaveis
        )
        if gradiente is not None
    ]

    otimizador_f5_f2.apply_gradients(
        pares
    )

    return perda

# ------------------------------------------------------------
# 5. Auditoria final
# ------------------------------------------------------------

parametros = sum(
    int(tf.size(v).numpy())
    for v in variaveis_treinaveis
)

assert parametros == 7243648

print("=" * 60)
print("FOLD 5 — RETOMADA PREPARADA")
print("=" * 60)

print("Pesos restaurados:", arquivo_pesos)
print("Época concluída:", dados_resgate["epoca"])

print(
    "Loss treino anterior:",
    dados_resgate["loss_treino"]
)

print(
    "Loss validação anterior:",
    dados_resgate["loss_validacao"]
)

print("Parâmetros treináveis:", parametros)

print(
    "Learning rate:",
    float(otimizador_f5_f2.learning_rate.numpy())
)

print(
    "Iterações do novo Adam:",
    int(otimizador_f5_f2.iterations.numpy())
)

print(
    "Nova função de treinamento:",
    passo_treino_f5_f2_novo.__name__
)

print("\nPRONTO PARA CONFIGURAR A RETOMADA")


In [ ]:

# ============================================================
# FOLD 5 — FASE 2: CONTINUAÇÃO DAS ÉPOCAS 2 A 10
# VGG16 + Triplet Loss Semi-Hard
# Checkpoint completo, histórico e barras de progresso
# ============================================================

from tqdm.auto import tqdm
import tensorflow as tf
import numpy as np
import pandas as pd
import json
import time

# ------------------------------------------------------------
# 1. Configurações e caminhos
# ------------------------------------------------------------

EPOCAS_FASE2_F5 = 10
EPOCA_REINICIO_ADAM_F5 = 1

DIR_ESTADO_F5_F2 = (
    DIR_CHECKPOINT_FOLD5 / "estado_fase2"
)

ARQUIVO_HISTORICO_F5_F2 = (
    DIR_RESULTADOS_FOLD5 / "historico_fase2.csv"
)

ARQUIVO_MELHOR_F5_F2 = (
    DIR_CHECKPOINT_FOLD5 /
    "melhor_fase2_fold5.weights.h5"
)

ARQUIVO_FINAL_F5 = (
    DIR_CHECKPOINT_FOLD5 /
    "modelo_final_fold5.weights.h5"
)

ARQUIVO_REGISTRO_F5 = (
    DIR_RESULTADOS_FOLD5 /
    "registro_recuperacao_fase2.json"
)

assert ARQUIVO_REGISTRO_F5.exists()
assert ARQUIVO_HISTORICO_F5_F2.exists()

# ------------------------------------------------------------
# 2. Recuperar checkpoint completo
# ------------------------------------------------------------

# O otimizador precisa ter suas variáveis criadas
# antes da restauração do checkpoint.
otimizador_f5_f2.build(
    modelo_embeddings_f5.trainable_variables
)

# Criar objetos próprios deste bloco.
# Não reutilizar a função antiga de treinamento.
epoca_controle_f5 = tf.Variable(
    0,
    dtype=tf.int64,
    trainable=False
)

checkpoint_controle_f5 = tf.train.Checkpoint(
    modelo=modelo_embeddings_f5,
    otimizador=otimizador_f5_f2,
    epoca=epoca_controle_f5
)

gerenciador_controle_f5 = tf.train.CheckpointManager(
    checkpoint_controle_f5,
    directory=str(DIR_ESTADO_F5_F2),
    max_to_keep=3
)

caminho_checkpoint = (
    gerenciador_controle_f5.latest_checkpoint
)

assert caminho_checkpoint is not None, (
    "Checkpoint da fase 2 não encontrado."
)

status = checkpoint_controle_f5.restore(
    caminho_checkpoint
)

status.assert_existing_objects_matched()

epoca_atual = int(
    epoca_controle_f5.numpy()
)

iteracoes_atuais = int(
    otimizador_f5_f2.iterations.numpy()
)

assert 1 <= epoca_atual <= EPOCAS_FASE2_F5

iteracoes_esperadas = (
    (epoca_atual - EPOCA_REINICIO_ADAM_F5)
    * PASSOS_POR_EPOCA
)

assert iteracoes_atuais == iteracoes_esperadas, (
    f"Adam inconsistente: {iteracoes_atuais} "
    f"iterações; esperado: {iteracoes_esperadas}."
)

# ------------------------------------------------------------
# 3. Recuperar histórico
# ------------------------------------------------------------

historico_f5_f2 = pd.read_csv(
    ARQUIVO_HISTORICO_F5_F2
)

historico_f5_f2 = historico_f5_f2[
    historico_f5_f2["epoca"] <= epoca_atual
].copy()

historico_f5_f2 = (
    historico_f5_f2
    .drop_duplicates(
        subset=["epoca"],
        keep="last"
    )
    .sort_values("epoca")
    .reset_index(drop=True)
)

assert set(
    historico_f5_f2["epoca"].astype(int)
) == set(
    range(1, epoca_atual + 1)
), "Histórico incompatível com o checkpoint."

melhor_loss_f5_f2 = float(
    historico_f5_f2["loss_validacao"].min()
)

# ------------------------------------------------------------
# 4. Definir função de treinamento estável
# ------------------------------------------------------------

# O Adam já está inicializado.
# Capturá-lo explicitamente evita utilizar outro
# otimizador após alterações nas variáveis globais.

modelo_treino_31k = modelo_embeddings_f5
adam_treino_31k = otimizador_f5_f2
loss_treino_31k = triplet_loss_f5

@tf.function
def passo_treino_31k(imagens, rotulos):

    with tf.GradientTape() as tape:

        embeddings = modelo_treino_31k(
            imagens,
            training=True
        )

        perda = loss_treino_31k(
            embeddings,
            rotulos
        )

    variaveis = modelo_treino_31k.trainable_variables

    gradientes = tape.gradient(
        perda,
        variaveis
    )

    pares = [
        (gradiente, variavel)
        for gradiente, variavel in zip(
            gradientes,
            variaveis
        )
        if gradiente is not None
    ]

    adam_treino_31k.apply_gradients(pares)

    return perda

@tf.function
def passo_validacao_31k(imagens, rotulos):

    embeddings = modelo_treino_31k(
        imagens,
        training=False
    )

    return loss_treino_31k(
        embeddings,
        rotulos
    )

# ------------------------------------------------------------
# 5. Auditoria inicial
# ------------------------------------------------------------

parametros_treinaveis = sum(
    int(tf.size(v).numpy())
    for v in modelo_embeddings_f5.trainable_variables
)

assert parametros_treinaveis == 7243648

print("=" * 60)
print("FOLD 5 — FASE 2: RETOMADA")
print("=" * 60)

print("Checkpoint:", caminho_checkpoint)
print("Épocas concluídas:", epoca_atual)
print("Épocas restantes:", EPOCAS_FASE2_F5 - epoca_atual)
print("Parâmetros treináveis:", parametros_treinaveis)
print("Iterações do Adam:", iteracoes_atuais)
print("Melhor loss anterior:", melhor_loss_f5_f2)

# ------------------------------------------------------------
# 6. Treinamento
# ------------------------------------------------------------

for epoca in range(
    epoca_atual,
    EPOCAS_FASE2_F5
):

    inicio = time.time()

    print("\n" + "=" * 60)
    print(
        f"FOLD 5 — FASE 2 — "
        f"ÉPOCA {epoca + 1}/{EPOCAS_FASE2_F5}"
    )
    print("=" * 60)

    dataset_treino = criar_dataset_treino_f5(
        epoca=10 + epoca
    )

    metrica_treino = tf.keras.metrics.Mean()

    # --------------------------------------------------------
    # 6.1. Treinamento
    # --------------------------------------------------------

    with tqdm(
        dataset_treino,
        total=PASSOS_POR_EPOCA,
        desc=f"Fold 5 | Treino | Época {epoca + 1}/10",
        unit="lote"
    ) as barra:

        for passo, (imagens, rotulos) in enumerate(
            barra,
            start=1
        ):

            perda = passo_treino_31k(
                imagens,
                rotulos
            )

            metrica_treino.update_state(perda)

            if (
                passo % 10 == 0
                or passo == PASSOS_POR_EPOCA
            ):

                barra.set_postfix(
                    loss=(
                        f"{float(metrica_treino.result().numpy()):.6f}"
                    )
                )

    loss_treino = float(
        metrica_treino.result().numpy()
    )

    # --------------------------------------------------------
    # 6.2. Validação
    # --------------------------------------------------------

    metrica_validacao = tf.keras.metrics.Mean()

    with tqdm(
        dataset_validacao_f5,
        total=LOTES_VALIDACAO,
        desc=f"Fold 5 | Validação | Época {epoca + 1}/10",
        unit="lote",
        leave=False
    ) as barra_validacao:

        for passo, (imagens, rotulos) in enumerate(
            barra_validacao,
            start=1
        ):

            perda = passo_validacao_31k(
                imagens,
                rotulos
            )

            metrica_validacao.update_state(perda)

            if (
                passo % 10 == 0
                or passo == LOTES_VALIDACAO
            ):

                barra_validacao.set_postfix(
                    loss=(
                        f"{float(metrica_validacao.result().numpy()):.6f}"
                    )
                )

    loss_validacao = float(
        metrica_validacao.result().numpy()
    )

    # --------------------------------------------------------
    # 6.3. Verificações
    # --------------------------------------------------------

    assert np.isfinite(loss_treino)
    assert np.isfinite(loss_validacao)

    iteracoes_esperadas = (
        (epoca + 1 - EPOCA_REINICIO_ADAM_F5)
        * PASSOS_POR_EPOCA
    )

    iteracoes_reais = int(
        otimizador_f5_f2.iterations.numpy()
    )

    assert iteracoes_reais == iteracoes_esperadas, (
        f"Adam inconsistente: {iteracoes_reais}; "
        f"esperado: {iteracoes_esperadas}."
    )

    tempo_epoca = time.time() - inicio

    print("\n" + "-" * 60)
    print(f"Época: {epoca + 1}/10")
    print(f"Loss treino: {loss_treino:.6f}")
    print(f"Loss validação: {loss_validacao:.6f}")
    print(f"Tempo: {tempo_epoca / 60:.2f} min")
    print(f"Iterações Adam: {iteracoes_reais}")

    # --------------------------------------------------------
    # 6.4. Histórico
    # --------------------------------------------------------

    novo_registro = pd.DataFrame([{
        "epoca": epoca + 1,
        "loss_treino": loss_treino,
        "loss_validacao": loss_validacao,
        "tempo_segundos": tempo_epoca
    }])

    historico_f5_f2 = pd.concat(
        [historico_f5_f2, novo_registro],
        ignore_index=True
    )

    historico_f5_f2.to_csv(
        ARQUIVO_HISTORICO_F5_F2,
        index=False
    )

    # --------------------------------------------------------
    # 6.5. Melhor modelo
    # --------------------------------------------------------

    if loss_validacao < melhor_loss_f5_f2:

        melhor_loss_f5_f2 = loss_validacao

        modelo_embeddings_f5.save_weights(
            str(ARQUIVO_MELHOR_F5_F2)
        )

        print("Novo melhor modelo salvo.")

    # --------------------------------------------------------
    # 6.6. Checkpoint completo
    # --------------------------------------------------------

    epoca_controle_f5.assign(epoca + 1)

    caminho_salvo = gerenciador_controle_f5.save(
        checkpoint_number=epoca + 1
    )

    print("Checkpoint salvo:", caminho_salvo)
    print("-" * 60)

# ------------------------------------------------------------
# 7. Finalização
# ------------------------------------------------------------

assert int(epoca_controle_f5.numpy()) == 10

assert int(
    otimizador_f5_f2.iterations.numpy()
) == 9 * PASSOS_POR_EPOCA

modelo_embeddings_f5.save_weights(
    str(ARQUIVO_FINAL_F5)
)

print("\n" + "=" * 60)
print("FOLD 5 — FASE 2 CONCLUÍDA")
print("=" * 60)

print("Épocas concluídas:", int(epoca_controle_f5.numpy()))
print("Melhor loss validação:", melhor_loss_f5_f2)

print(
    "Iterações do Adam após recuperação:",
    int(otimizador_f5_f2.iterations.numpy())
)

print("Melhor modelo:", ARQUIVO_MELHOR_F5_F2)
print("Modelo final:", ARQUIVO_FINAL_F5)
print("Histórico:", ARQUIVO_HISTORICO_F5_F2)

print("\nTREINAMENTO FINALIZADO COM SUCESSO")


## 9. Consolidação, exportação e teste

Auditoria dos cinco folds, exportação TFLite e avaliação no conjunto independente de teste.

In [ ]:

# ============================================================
# BLOCO 32 — AUDITORIA FINAL DO FOLD 5
# ============================================================

from pathlib import Path
import pandas as pd
import tensorflow as tf

print("=" * 60)
print("FOLD 5 — AUDITORIA FINAL")
print("=" * 60)

arquivos = {
    "Histórico fase 1":
        DIR_RESULTADOS_FOLD5 / "historico_fase1.csv",

    "Histórico fase 2":
        DIR_RESULTADOS_FOLD5 / "historico_fase2.csv",

    "Melhores pesos fase 1":
        DIR_CHECKPOINT_FOLD5 /
        "melhor_fase1_fold5.weights.h5",

    "Melhores pesos fase 2":
        DIR_CHECKPOINT_FOLD5 /
        "melhor_fase2_fold5.weights.h5",

    "Pesos finais":
        DIR_CHECKPOINT_FOLD5 /
        "modelo_final_fold5.weights.h5",

    "Registro de recuperação":
        DIR_RESULTADOS_FOLD5 /
        "registro_recuperacao_fase2.json",
}

print("\n1. Arquivos essenciais")

for nome, caminho in arquivos.items():

    existe = caminho.exists()

    tamanho = (
        caminho.stat().st_size / (1024 ** 2)
        if existe else 0
    )

    print(
        f"{'OK' if existe else 'AUSENTE'} | "
        f"{nome} | {tamanho:.2f} MB"
    )

assert all(
    caminho.exists()
    for caminho in arquivos.values()
), "Existem arquivos essenciais ausentes."

print("\n2. Histórico da fase 2")

historico = pd.read_csv(
    arquivos["Histórico fase 2"]
)

print(historico.to_string(index=False))

assert len(historico) == 10
assert historico["epoca"].tolist() == list(
    range(1, 11)
)

melhor_indice = (
    historico["loss_validacao"].idxmin()
)

melhor_epoca = int(
    historico.loc[melhor_indice, "epoca"]
)

melhor_loss = float(
    historico.loc[
        melhor_indice,
        "loss_validacao"
    ]
)

print("\n3. Melhor resultado")

print("Época:", melhor_epoca)
print("Loss validação:", melhor_loss)

print("\n4. Checkpoint final")

checkpoint = tf.train.latest_checkpoint(
    str(
        DIR_CHECKPOINT_FOLD5 /
        "estado_fase2"
    )
)

print("Checkpoint:", checkpoint)

assert checkpoint is not None

print("\n" + "=" * 60)
print("AUDITORIA CONCLUÍDA COM SUCESSO")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCO 33 — INVENTÁRIO DOS CINCO FOLDS
# ============================================================

from pathlib import Path
import pandas as pd

RAIZ_EXPERIMENTOS = (
    Path("/content/drive/MyDrive/TCC_Caes/")
    / "experimentos/tcc1_kfold"
)

DIR_SPLITS = RAIZ_EXPERIMENTOS / "splits"
DIR_CHECKPOINTS = RAIZ_EXPERIMENTOS / "checkpoints"
DIR_RESULTADOS = RAIZ_EXPERIMENTOS / "resultados"

print("=" * 65)
print("TCC — INVENTÁRIO DOS CINCO FOLDS")
print("=" * 65)

for fold in range(1, 6):

    print(f"\n{'-' * 65}")
    print(f"FOLD {fold}")
    print("-" * 65)

    diretorio_fold = (
        DIR_CHECKPOINTS / f"fold_{fold}"
    )

    # Pesos disponíveis
    if diretorio_fold.exists():

        arquivos_pesos = sorted(
            diretorio_fold.glob("*.weights.h5")
        )

        print("\nModelos disponíveis:")

        if arquivos_pesos:

            for arquivo in arquivos_pesos:
                print("  OK:", arquivo.name)

        else:
            print("  Nenhum arquivo de pesos encontrado.")

    else:
        print("Diretório de checkpoints ausente.")

    # Históricos disponíveis
    diretorio_resultados = (
        DIR_RESULTADOS / f"fold_{fold}"
    )

    print("\nHistóricos:")

    for fase in (1, 2):

        arquivo = (
            diretorio_resultados /
            f"historico_fase{fase}.csv"
        )

        if arquivo.exists():

            df = pd.read_csv(arquivo)

            print(
                f"  Fase {fase}: "
                f"{len(df)} épocas"
            )

        else:
            print(
                f"  Fase {fase}: não encontrado"
            )

print("\n" + "=" * 65)
print("ARQUIVOS DE DIVISÃO DOS DADOS")
print("=" * 65)

if DIR_SPLITS.exists():

    arquivos_csv = sorted(
        DIR_SPLITS.glob("*.csv")
    )

    for arquivo in arquivos_csv:

        print("\nArquivo:", arquivo.name)

        try:

            df = pd.read_csv(arquivo)

            print("  Registros:", len(df))
            print("  Colunas:", list(df.columns))

        except Exception as erro:

            print("  Erro:", str(erro))

else:
    print("Diretório de splits não encontrado.")

print("\n" + "=" * 65)
print("INVENTÁRIO CONCLUÍDO")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCO 34 — EXPORTAÇÃO DO FOLD 5 PARA TENSORFLOW LITE
# ============================================================

from pathlib import Path
import tensorflow as tf
import json
import numpy as np

# ------------------------------------------------------------
# 1. Diretórios
# ------------------------------------------------------------

DIR_EXPORTACAO = (
    RAIZ_EXPERIMENTOS / "exportacao_tflite"
)

DIR_EXPORTACAO.mkdir(
    parents=True,
    exist_ok=True
)

ARQUIVO_PESOS = (
    DIR_CHECKPOINT_FOLD5 /
    "melhor_fase2_fold5.weights.h5"
)

ARQUIVO_TFLITE = (
    DIR_EXPORTACAO /
    "perdigueiro_fold5_float32.tflite"
)

ARQUIVO_METADADOS = (
    DIR_EXPORTACAO /
    "perdigueiro_fold5_metadados.json"
)

assert ARQUIVO_PESOS.exists(), (
    "Pesos do melhor Fold 5 não encontrados."
)

# ------------------------------------------------------------
# 2. Restaurar melhores pesos
# ------------------------------------------------------------

# Usamos a arquitetura já construída no notebook.
# A operação não altera os arquivos de checkpoint.

modelo_embeddings_f5.load_weights(
    str(ARQUIVO_PESOS)
)

print("Melhores pesos carregados.")
print("Arquivo:", ARQUIVO_PESOS)

# ------------------------------------------------------------
# 3. Criar modelo de inferência
# ------------------------------------------------------------

entrada = tf.keras.Input(
    shape=(224, 224, 3),
    dtype=tf.float32,
    name="imagem"
)

saida = modelo_embeddings_f5(
    entrada,
    training=False
)

modelo_inferencia = tf.keras.Model(
    inputs=entrada,
    outputs=saida,
    name="perdigueiro_embeddings"
)

assert modelo_inferencia.output_shape[-1] == 128

# ------------------------------------------------------------
# 4. Converter para TensorFlow Lite
# ------------------------------------------------------------

print("\nConvertendo modelo para TFLite...")

conversor = tf.lite.TFLiteConverter.from_keras_model(
    modelo_inferencia
)

# Primeira versão: float32, sem quantização.
# Mantém a precisão numérica original dos pesos.
conversor.optimizations = []

modelo_tflite = conversor.convert()

with open(ARQUIVO_TFLITE, "wb") as arquivo:
    arquivo.write(modelo_tflite)

print("Conversão concluída.")

# ------------------------------------------------------------
# 5. Verificar modelo exportado
# ------------------------------------------------------------

interpretador = tf.lite.Interpreter(
    model_path=str(ARQUIVO_TFLITE)
)

interpretador.allocate_tensors()

entradas = interpretador.get_input_details()
saidas = interpretador.get_output_details()

print("\nEntrada:")
for item in entradas:
    print("Nome:", item["name"])
    print("Shape:", item["shape"])
    print("Tipo:", item["dtype"])

print("\nSaída:")
for item in saidas:
    print("Nome:", item["name"])
    print("Shape:", item["shape"])
    print("Tipo:", item["dtype"])

assert tuple(entradas[0]["shape"]) == (
    1, 224, 224, 3
)

assert tuple(saidas[0]["shape"]) == (
    1, 128
)

# ------------------------------------------------------------
# 6. Teste de inferência e comparação
# ------------------------------------------------------------

imagem_teste = np.zeros(
    (1, 224, 224, 3),
    dtype=np.float32
)

embedding_keras = modelo_inferencia(
    imagem_teste,
    training=False
).numpy()

interpretador.set_tensor(
    entradas[0]["index"],
    imagem_teste
)

interpretador.invoke()

embedding_tflite = interpretador.get_tensor(
    saidas[0]["index"]
)

diferenca_maxima = float(
    np.max(
        np.abs(
            embedding_keras - embedding_tflite
        )
    )
)

assert np.all(
    np.isfinite(embedding_tflite)
)

assert np.allclose(
    embedding_keras,
    embedding_tflite,
    atol=1e-4,
    rtol=1e-4
), (
    "Diferença acima da tolerância "
    "entre Keras e TFLite."
)

# ------------------------------------------------------------
# 7. Registrar metadados
# ------------------------------------------------------------

metadados = {
    "projeto": "Perdigueiro",
    "arquitetura": "VGG16",
    "fold": 5,
    "fase": 2,
    "melhor_epoca": 8,
    "formato": "TensorFlow Lite",
    "precisao": "float32",
    "entrada": [1, 224, 224, 3],
    "saida": [1, 128],
    "normalizacao_embedding": "L2",
    "diferenca_maxima_keras_tflite": diferenca_maxima,
    "observacao": (
        "O pré-processamento da imagem deve "
        "ser idêntico ao utilizado no treinamento."
    )
}

with open(
    ARQUIVO_METADADOS,
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        metadados,
        arquivo,
        indent=4,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 8. Resumo
# ------------------------------------------------------------

tamanho_mb = (
    ARQUIVO_TFLITE.stat().st_size /
    (1024 ** 2)
)

print("\n" + "=" * 60)
print("PERDIGUEIRO — EXPORTAÇÃO TFLITE")
print("=" * 60)

print("Modelo:", ARQUIVO_TFLITE)
print("Tamanho:", f"{tamanho_mb:.2f} MB")
print("Entrada:", entradas[0]["shape"])
print("Saída:", saidas[0]["shape"])

print(
    "Diferença máxima Keras/TFLite:",
    diferenca_maxima
)

print("Metadados:", ARQUIVO_METADADOS)

print("\nEXPORTAÇÃO CONCLUÍDA COM SUCESSO")


In [ ]:

# ============================================================
# BLOCO 35 — AUDITORIA DO CONJUNTO DE TESTE
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

DIR_SPLITS = RAIZ_EXPERIMENTOS / "splits"

df_teste = pd.read_csv(
    DIR_SPLITS / "imagens_teste_30.csv"
)

df_desenvolvimento = pd.read_csv(
    DIR_SPLITS / "identidades_desenvolvimento_70.csv"
)

df_identidades_teste = pd.read_csv(
    DIR_SPLITS / "identidades_teste_30.csv"
)

print("=" * 65)
print("TCC — AUDITORIA DO TESTE INDEPENDENTE")
print("=" * 65)

# ------------------------------------------------------------
# 1. Estatísticas gerais
# ------------------------------------------------------------

print("\n1. Dimensões do conjunto")

print("Imagens:", len(df_teste))

print(
    "Identidades:",
    df_teste["identidade"].nunique()
)

contagem = (
    df_teste.groupby("identidade")
    .size()
)

print("\nImagens por identidade:")
print(contagem.describe().to_string())

print(
    "\nIdentidades com pelo menos 2 imagens:",
    int((contagem >= 2).sum())
)

print(
    "Identidades com pelo menos 5 imagens:",
    int((contagem >= 5).sum())
)

print(
    "Identidades com apenas 1 imagem:",
    int((contagem == 1).sum())
)

# ------------------------------------------------------------
# 2. Verificar separação entre conjuntos
# ------------------------------------------------------------

ids_dev = set(
    df_desenvolvimento["identidade"]
)

ids_teste = set(
    df_identidades_teste["identidade"]
)

sobreposicao = ids_dev.intersection(ids_teste)

print("\n2. Separação das identidades")
print("Identidades compartilhadas:", len(sobreposicao))

assert len(sobreposicao) == 0, (
    "Há identidades compartilhadas entre "
    "desenvolvimento e teste."
)

# ------------------------------------------------------------
# 3. Distribuição por dataset
# ------------------------------------------------------------

print("\n3. Distribuição por dataset")

print(
    df_teste["dataset"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 4. Conferir integridade dos registros
# ------------------------------------------------------------

print("\n4. Integridade dos registros")

print(
    "Caminhos nulos:",
    int(df_teste["caminho"].isna().sum())
)

print(
    "Caminhos duplicados:",
    int(df_teste["caminho"].duplicated().sum())
)

print("\nExemplos de caminhos:")

for caminho in df_teste["caminho"].head(5):
    print(" ", caminho)

print("\n" + "=" * 65)
print("AUDITORIA DO TESTE CONCLUÍDA")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCO 36 — VERIFICAÇÃO DE CAMINHOS E PRÉ-PROCESSAMENTO
# ============================================================

import os
import inspect
import tensorflow as tf
import pandas as pd

print("=" * 65)
print("TCC — PREPARAÇÃO DA AVALIAÇÃO")
print("=" * 65)

# ------------------------------------------------------------
# 1. Verificar acessibilidade das imagens
# ------------------------------------------------------------

print("\n1. Acessibilidade das imagens")

amostra = df_teste.sample(
    n=min(30, len(df_teste)),
    random_state=42
)

existentes = sum(
    os.path.isfile(caminho)
    for caminho in amostra["caminho"]
)

print("Imagens verificadas:", len(amostra))
print("Arquivos encontrados:", existentes)

if existentes != len(amostra):
    print(
        "ATENÇÃO: alguns caminhos não estão "
        "diretamente acessíveis."
    )

# ------------------------------------------------------------
# 2. Verificar funções disponíveis
# ------------------------------------------------------------

print("\n2. Funções de processamento disponíveis")

nomes = [
    "carregar_imagem",
    "carregar_imagem_f5",
    "preprocessar_imagem",
    "preprocessar_imagem_f5",
    "decodificar_imagem",
    "criar_dataset_treino_f5",
    "criar_dataset_validacao_f5"
]

for nome in nomes:

    objeto = globals().get(nome)

    if callable(objeto):
        print("DISPONÍVEL:", nome)

        try:
            codigo = inspect.getsource(objeto)
            print(codigo[:1800])
        except (OSError, TypeError):
            print(
                "  Código-fonte não disponível."
            )

    else:
        print("Não encontrado:", nome)

# ------------------------------------------------------------
# 3. Inspecionar configuração do modelo
# ------------------------------------------------------------

print("\n3. Configuração do modelo")

print(
    "Entrada:",
    modelo_embeddings_f5.input_shape
)

print(
    "Saída:",
    modelo_embeddings_f5.output_shape
)

print("\n4. Pré-processamento VGG16")

print(
    "Referência Keras:",
    tf.keras.applications.vgg16.preprocess_input
)

print("\n" + "=" * 65)
print("VERIFICAÇÃO CONCLUÍDA")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCO 37 — AUDITORIA DO PRÉ-PROCESSAMENTO ORIGINAL
# ============================================================

import inspect
import tensorflow as tf

print("=" * 65)
print("TCC — PRÉ-PROCESSAMENTO ORIGINAL")
print("=" * 65)

# ------------------------------------------------------------
# 1. Inspecionar dataset_f5
# ------------------------------------------------------------

print("\n1. IMPLEMENTAÇÃO DE dataset_f5")

funcao_dataset = globals().get("dataset_f5")

if callable(funcao_dataset):

    try:
        print(inspect.getsource(funcao_dataset))

    except (OSError, TypeError) as erro:
        print("Código-fonte indisponível:", erro)

else:
    print("Função dataset_f5 não encontrada.")

# ------------------------------------------------------------
# 2. Identificar funções auxiliares
# ------------------------------------------------------------

print("\n2. FUNÇÕES AUXILIARES REFERENCIADAS")

if callable(funcao_dataset):

    nomes = funcao_dataset.__code__.co_names

    for nome in nomes:

        objeto = funcao_dataset.__globals__.get(
            nome
        )

        if inspect.isfunction(objeto):

            print("\n" + "-" * 50)
            print("FUNÇÃO:", nome)
            print("-" * 50)

            try:
                print(inspect.getsource(objeto))

            except (OSError, TypeError):
                print("Código-fonte indisponível.")

# ------------------------------------------------------------
# 3. Inspecionar camadas iniciais
# ------------------------------------------------------------

print("\n3. PRIMEIRAS CAMADAS DO MODELO")

for camada in modelo_embeddings_f5.layers[:12]:

    print(
        f"{camada.name:35s} "
        f"{type(camada).__name__}"
    )

# ------------------------------------------------------------
# 4. Procurar camadas de pré-processamento
# ------------------------------------------------------------

print("\n4. CAMADAS DE NORMALIZAÇÃO")

tipos = (
    tf.keras.layers.Rescaling,
    tf.keras.layers.Normalization
)

encontradas = []

for camada in modelo_embeddings_f5._flatten_layers(
    include_self=False,
    recursive=True
):

    if isinstance(camada, tipos):

        encontradas.append(camada)

        print(
            camada.name,
            type(camada).__name__,
            camada.get_config()
        )

if not encontradas:
    print(
        "Nenhuma camada Rescaling/Normalization "
        "encontrada."
    )

print("\n" + "=" * 65)
print("AUDITORIA CONCLUÍDA")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCO 38 — VALIDAÇÃO COM IMAGENS REAIS
# ============================================================

import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 65)
print("TCC — VALIDAÇÃO KERAS × TFLITE")
print("=" * 65)

# ------------------------------------------------------------
# 1. Selecionar imagens reais
# ------------------------------------------------------------

amostra = df_teste.sample(
    n=16,
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Reutilizar pré-processamento original
# ------------------------------------------------------------

def preparar_imagem_inferencia(caminho):

    conteudo = tf.io.read_file(caminho)

    imagem = tf.io.decode_image(
        conteudo,
        channels=3,
        expand_animations=False
    )

    imagem.set_shape([None, None, 3])

    imagem = tf.image.resize(
        imagem,
        [224, 224]
    )

    imagem = tf.keras.applications.vgg16.preprocess_input(
        tf.cast(imagem, tf.float32)
    )

    return imagem

dataset_amostra = (
    tf.data.Dataset.from_tensor_slices(
        amostra["caminho"].astype(str).tolist()
    )
    .map(
        preparar_imagem_inferencia,
        num_parallel_calls=4
    )
    .batch(16)
    .prefetch(2)
)

imagens = next(iter(dataset_amostra))

print("\nImagens:", imagens.shape)
print("Tipo:", imagens.dtype)

# ------------------------------------------------------------
# 3. Inferência Keras
# ------------------------------------------------------------

embeddings_keras = modelo_embeddings_f5(
    imagens,
    training=False
).numpy()

# ------------------------------------------------------------
# 4. Inferência TFLite
# ------------------------------------------------------------

ARQUIVO_TFLITE = (
    RAIZ_EXPERIMENTOS /
    "exportacao_tflite" /
    "perdigueiro_fold5_float32.tflite"
)

assert ARQUIVO_TFLITE.exists()

interpretador = tf.lite.Interpreter(
    model_path=str(ARQUIVO_TFLITE)
)

# Ajustar entrada para 16 imagens
entrada_info = interpretador.get_input_details()[0]

interpretador.resize_tensor_input(
    entrada_info["index"],
    [16, 224, 224, 3]
)

interpretador.allocate_tensors()

entrada_info = interpretador.get_input_details()[0]
saida_info = interpretador.get_output_details()[0]

interpretador.set_tensor(
    entrada_info["index"],
    imagens.numpy()
)

interpretador.invoke()

embeddings_tflite = interpretador.get_tensor(
    saida_info["index"]
)

# ------------------------------------------------------------
# 5. Comparar embeddings
# ------------------------------------------------------------

diferencas = np.abs(
    embeddings_keras - embeddings_tflite
)

diferenca_maxima = float(
    diferencas.max()
)

diferenca_media = float(
    diferencas.mean()
)

normas_keras = np.linalg.norm(
    embeddings_keras,
    axis=1
)

normas_tflite = np.linalg.norm(
    embeddings_tflite,
    axis=1
)

equivalentes = np.allclose(
    embeddings_keras,
    embeddings_tflite,
    atol=1e-4,
    rtol=1e-4
)

# ------------------------------------------------------------
# 6. Resultados
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RESULTADOS")
print("-" * 65)

print("Embeddings Keras:", embeddings_keras.shape)
print("Embeddings TFLite:", embeddings_tflite.shape)

print(
    "Diferença máxima:",
    f"{diferenca_maxima:.10f}"
)

print(
    "Diferença média:",
    f"{diferenca_media:.10f}"
)

print(
    "Norma L2 média Keras:",
    float(normas_keras.mean())
)

print(
    "Norma L2 média TFLite:",
    float(normas_tflite.mean())
)

print(
    "Equivalência numérica:",
    equivalentes
)

assert embeddings_keras.shape == (16, 128)
assert embeddings_tflite.shape == (16, 128)

assert np.all(np.isfinite(embeddings_keras))
assert np.all(np.isfinite(embeddings_tflite))

assert equivalentes, (
    "Diferenças superiores à tolerância."
)

assert np.allclose(
    normas_tflite,
    1.0,
    atol=1e-4
), "Os embeddings não estão normalizados em L2."

print("\n" + "=" * 65)
print("VALIDAÇÃO CONCLUÍDA COM SUCESSO")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCO 39 — EXTRAÇÃO DE EMBEDDINGS DO TESTE (FOLD 5)
# ============================================================

import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path
import time

print("=" * 65)
print("TCC — EXTRAÇÃO DE EMBEDDINGS — FOLD 5")
print("=" * 65)

# ------------------------------------------------------------
# 1. Configurações
# ------------------------------------------------------------

DIR_EMBEDDINGS = (
    RAIZ_EXPERIMENTOS / "resultados" / "embeddings_teste"
)

DIR_EMBEDDINGS.mkdir(
    parents=True,
    exist_ok=True
)

ARQUIVO_EMBEDDINGS = (
    DIR_EMBEDDINGS / "embeddings_teste_fold5.npz"
)

BATCH_INFERENCIA = 32

# Usar a ordem original do CSV
df_avaliacao = df_teste.reset_index(drop=True).copy()

caminhos = (
    df_avaliacao["caminho"]
    .astype(str)
    .tolist()
)

print("Imagens:", len(caminhos))
print("Lote:", BATCH_INFERENCIA)
print("Destino:", ARQUIVO_EMBEDDINGS)

# ------------------------------------------------------------
# 2. Construir dataset de inferência
# ------------------------------------------------------------

dataset_inferencia = (
    tf.data.Dataset.from_tensor_slices(caminhos)
    .map(
        preparar_imagem_inferencia,
        num_parallel_calls=4
    )
    .batch(
        BATCH_INFERENCIA,
        drop_remainder=False
    )
    .prefetch(2)
)

# ------------------------------------------------------------
# 3. Restaurar melhores pesos do Fold 5
# ------------------------------------------------------------

ARQUIVO_PESOS = (
    RAIZ_EXPERIMENTOS /
    "checkpoints" /
    "fold_5" /
    "melhor_fase2_fold5.weights.h5"
)

assert ARQUIVO_PESOS.exists()

modelo_embeddings_f5.load_weights(
    str(ARQUIVO_PESOS)
)

print("\nPesos restaurados:", ARQUIVO_PESOS.name)

# ------------------------------------------------------------
# 4. Extrair embeddings
# ------------------------------------------------------------

inicio = time.time()

lista_embeddings = []

total_lotes = int(
    np.ceil(len(caminhos) / BATCH_INFERENCIA)
)

for indice, lote in enumerate(dataset_inferencia, 1):

    embeddings_lote = modelo_embeddings_f5(
        lote,
        training=False
    ).numpy()

    lista_embeddings.append(
        embeddings_lote.astype(np.float32)
    )

    if indice % 50 == 0 or indice == total_lotes:

        print(
            f"Lote {indice}/{total_lotes} "
            f"— {min(indice * BATCH_INFERENCIA, len(caminhos))}"
            f"/{len(caminhos)} imagens"
        )

embeddings = np.concatenate(
    lista_embeddings,
    axis=0
)

duracao = time.time() - inicio

# ------------------------------------------------------------
# 5. Verificar integridade
# ------------------------------------------------------------

assert embeddings.shape == (
    len(df_avaliacao), 128
)

assert np.all(np.isfinite(embeddings))

normas = np.linalg.norm(
    embeddings,
    axis=1
)

assert np.allclose(
    normas,
    1.0,
    atol=1e-4
)

# ------------------------------------------------------------
# 6. Salvar resultados
# ------------------------------------------------------------

np.savez_compressed(
    ARQUIVO_EMBEDDINGS,
    embeddings=embeddings,
    identidades=df_avaliacao[
        "identidade"
    ].astype(str).to_numpy(),
    datasets=df_avaliacao[
        "dataset"
    ].astype(str).to_numpy(),
    caminhos=df_avaliacao[
        "caminho"
    ].astype(str).to_numpy()
)

# ------------------------------------------------------------
# 7. Resumo
# ------------------------------------------------------------

tamanho_mb = (
    ARQUIVO_EMBEDDINGS.stat().st_size /
    (1024 ** 2)
)

print("\n" + "=" * 65)
print("EXTRAÇÃO CONCLUÍDA")
print("=" * 65)

print("Formato:", embeddings.shape)
print("Tipo:", embeddings.dtype)
print("Norma L2 média:", float(normas.mean()))
print("Tempo total:", f"{duracao:.1f} segundos")
print("Tamanho do arquivo:", f"{tamanho_mb:.2f} MB")
print("Arquivo:", ARQUIVO_EMBEDDINGS)


In [ ]:

# ============================================================
# BLOCO 40 — REIDENTIFICAÇÃO GALLERY–QUERY — FOLD 5
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 65)
print("TCC — AVALIAÇÃO DE REIDENTIFICAÇÃO — FOLD 5")
print("=" * 65)

# ------------------------------------------------------------
# 1. Carregar embeddings
# ------------------------------------------------------------

arquivo = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "embeddings_teste" /
    "embeddings_teste_fold5.npz"
)

with np.load(arquivo, allow_pickle=True) as dados:
    embeddings = dados["embeddings"].astype(np.float32)
    identidades = dados["identidades"].astype(str)
    caminhos = dados["caminhos"].astype(str)

assert embeddings.shape == (10528, 128)
assert len(identidades) == len(embeddings)

# ------------------------------------------------------------
# 2. Separar galeria e consultas
# ------------------------------------------------------------

rng = np.random.default_rng(42)

indices_galeria = []
indices_consulta = []

for identidade in np.unique(identidades):

    indices = np.flatnonzero(
        identidades == identidade
    )

    assert len(indices) >= 2

    escolhido = rng.choice(indices)

    indices_galeria.append(int(escolhido))

    indices_consulta.extend(
        int(i) for i in indices
        if i != escolhido
    )

indices_galeria = np.array(
    indices_galeria,
    dtype=np.int64
)

indices_consulta = np.array(
    indices_consulta,
    dtype=np.int64
)

galeria = embeddings[indices_galeria]
consultas = embeddings[indices_consulta]

ids_galeria = identidades[indices_galeria]
ids_consulta = identidades[indices_consulta]

assert len(np.unique(ids_galeria)) == len(ids_galeria)
assert len(indices_galeria) == 1235
assert len(indices_consulta) == 9293

print("\nGaleria:", galeria.shape)
print("Consultas:", consultas.shape)

# ------------------------------------------------------------
# 3. Calcular similaridade cosseno
# ------------------------------------------------------------

# Como os embeddings são normalizados em L2,
# o produto escalar equivale à similaridade cosseno.

similaridades = consultas @ galeria.T

# ------------------------------------------------------------
# 4. Calcular posição da identidade correta
# ------------------------------------------------------------

# Há exatamente uma imagem correta por consulta
# na galeria.

mapa_galeria = {
    identidade: indice
    for indice, identidade in enumerate(ids_galeria)
}

indices_corretos = np.array([
    mapa_galeria[identidade]
    for identidade in ids_consulta
])

similaridades_corretas = similaridades[
    np.arange(len(consultas)),
    indices_corretos
]

# Empates são resolvidos pela ordem dos índices
# da galeria, para manter resultados determinísticos.

ranks = np.empty(
    len(consultas),
    dtype=np.int32
)

for inicio in range(0, len(consultas), 256):

    fim = min(inicio + 256, len(consultas))

    bloco = similaridades[inicio:fim]
    corretos = indices_corretos[inicio:fim]
    scores_corretos = similaridades_corretas[inicio:fim]

    maiores = bloco > scores_corretos[:, None]

    empates_anteriores = (
        (bloco == scores_corretos[:, None]) &
        (
            np.arange(len(galeria))[None, :] <
            corretos[:, None]
        )
    )

    ranks[inicio:fim] = (
        1 +
        maiores.sum(axis=1) +
        empates_anteriores.sum(axis=1)
    )

# ------------------------------------------------------------
# 5. Métricas
# ------------------------------------------------------------

rank1 = float(np.mean(ranks <= 1))
rank5 = float(np.mean(ranks <= 5))
rank10 = float(np.mean(ranks <= 10))

# Uma única imagem relevante por consulta:
# AP = 1 / rank, logo mAP = MRR.

map_score = float(np.mean(1.0 / ranks))

print("\n" + "-" * 65)
print("RESULTADOS — FOLD 5")
print("-" * 65)

print(f"Rank-1:  {rank1:.4%}")
print(f"Rank-5:  {rank5:.4%}")
print(f"Rank-10: {rank10:.4%}")
print(f"mAP:     {map_score:.4%}")

print("\nPosição mediana:", float(np.median(ranks)))
print("Posição média:", float(np.mean(ranks)))

# ------------------------------------------------------------
# 6. Salvar protocolo e resultados
# ------------------------------------------------------------

DIR_AVALIACAO = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "avaliacao_reidentificacao"
)

DIR_AVALIACAO.mkdir(
    parents=True,
    exist_ok=True
)

arquivo_protocolo = (
    DIR_AVALIACAO / "protocolo_gallery_query_seed42.npz"
)

np.savez_compressed(
    arquivo_protocolo,
    indices_galeria=indices_galeria,
    indices_consulta=indices_consulta,
    caminhos_galeria=caminhos[indices_galeria],
    caminhos_consulta=caminhos[indices_consulta]
)

arquivo_resultados = (
    DIR_AVALIACAO / "resultados_fold5.csv"
)

pd.DataFrame({
    "caminho_consulta": caminhos[indices_consulta],
    "identidade": ids_consulta,
    "rank_correto": ranks,
    "similaridade_correta": similaridades_corretas
}).to_csv(
    arquivo_resultados,
    index=False
)

print("\nProtocolo:", arquivo_protocolo)
print("Resultados:", arquivo_resultados)

print("\n" + "=" * 65)
print("AVALIAÇÃO CONCLUÍDA")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCO 41 — DESEMPENHO POR DATASET — FOLD 5
# ============================================================

import numpy as np
import pandas as pd

print("=" * 65)
print("TCC — RESULTADOS POR DATASET — FOLD 5")
print("=" * 65)

# ------------------------------------------------------------
# 1. Recuperar dataset de cada consulta
# ------------------------------------------------------------

# Variáveis geradas pelo Bloco 40:
# indices_consulta, ranks e df_teste

origens_consulta = (
    df_teste.iloc[indices_consulta]["dataset"]
    .astype(str)
    .to_numpy()
)

assert len(origens_consulta) == len(ranks)

# ------------------------------------------------------------
# 2. Calcular métricas por origem
# ------------------------------------------------------------

resultados = []

for origem in sorted(np.unique(origens_consulta)):

    mascara = origens_consulta == origem
    ranks_origem = ranks[mascara]

    resultados.append({
        "dataset": origem,
        "consultas": len(ranks_origem),
        "rank1": np.mean(ranks_origem <= 1),
        "rank5": np.mean(ranks_origem <= 5),
        "rank10": np.mean(ranks_origem <= 10),
        "map": np.mean(1.0 / ranks_origem),
        "rank_medio": np.mean(ranks_origem),
        "rank_mediano": np.median(ranks_origem)
    })

df_resultados_dataset = pd.DataFrame(resultados)

# ------------------------------------------------------------
# 3. Exibir resultados
# ------------------------------------------------------------

print("\nResultados por origem:\n")

print(
    df_resultados_dataset.to_string(
        index=False,
        formatters={
            "rank1": "{:.2%}".format,
            "rank5": "{:.2%}".format,
            "rank10": "{:.2%}".format,
            "map": "{:.2%}".format,
            "rank_medio": "{:.2f}".format,
            "rank_mediano": "{:.1f}".format
        }
    )
)

# ------------------------------------------------------------
# 4. Salvar análise
# ------------------------------------------------------------

arquivo_saida = (
    DIR_AVALIACAO /
    "resultados_por_dataset_fold5.csv"
)

df_resultados_dataset.to_csv(
    arquivo_saida,
    index=False
)

print("\nArquivo salvo:", arquivo_saida)

print("\n" + "=" * 65)
print("ANÁLISE CONCLUÍDA")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCO 42 — EXTRAÇÃO DOS EMBEDDINGS — FOLDS 1 A 4
# ============================================================

from pathlib import Path
import tensorflow as tf
import numpy as np
import pandas as pd
import time
import gc

RAIZ_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/TCC_Caes/"
    "experimentos/tcc1_kfold"
)

DIR_CHECKPOINTS = RAIZ_EXPERIMENTOS / "checkpoints"
DIR_SPLITS = RAIZ_EXPERIMENTOS / "splits"

DIR_EMBEDDINGS = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "embeddings_teste"
)

DIR_EMBEDDINGS.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Verificar ambiente
# ------------------------------------------------------------

print("=" * 65)
print("TCC — EXTRAÇÃO DOS FOLDS 1 A 4")
print("=" * 65)

print("\nGPUs disponíveis:")
print(tf.config.list_physical_devices("GPU"))

assert tf.config.list_physical_devices("GPU"), (
    "GPU não encontrada. Ative a GPU no Colab."
)

# ------------------------------------------------------------
# 2. Carregar conjunto de teste
# ------------------------------------------------------------

df_teste = pd.read_csv(
    DIR_SPLITS / "imagens_teste_30.csv"
).reset_index(drop=True)

caminhos = df_teste["caminho"].astype(str).tolist()

assert len(caminhos) == 10528

# ------------------------------------------------------------
# 3. Pré-processamento original
# ------------------------------------------------------------

def preparar_imagem_avaliacao(caminho):

    conteudo = tf.io.read_file(caminho)

    imagem = tf.io.decode_image(
        conteudo,
        channels=3,
        expand_animations=False
    )

    imagem.set_shape([None, None, 3])

    imagem = tf.image.resize(
        imagem,
        [224, 224]
    )

    imagem = tf.keras.applications.vgg16.preprocess_input(
        tf.cast(imagem, tf.float32)
    )

    return imagem

dataset = (
    tf.data.Dataset.from_tensor_slices(caminhos)
    .map(
        preparar_imagem_avaliacao,
        num_parallel_calls=4
    )
    .batch(32, drop_remainder=False)
    .prefetch(2)
)

# ------------------------------------------------------------
# 4. Reconstruir arquitetura
# ------------------------------------------------------------

def construir_modelo_avaliacao():

    entrada = tf.keras.Input(
        shape=(224, 224, 3),
        name="entrada_imagem"
    )

    base = tf.keras.applications.VGG16(
        include_top=False,
        weights=None,
        input_shape=(224, 224, 3)
    )

    x = base(entrada)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dense(
        256,
        activation="relu",
        name="dense_256"
    )(x)

    x = tf.keras.layers.Dense(
        128,
        activation="linear",
        name="dense_128"
    )(x)

    saida = tf.keras.layers.UnitNormalization(
        axis=-1,
        name="normalizacao_l2"
    )(x)

    return tf.keras.Model(
        entrada,
        saida,
        name="modelo_embeddings"
    )

modelo = construir_modelo_avaliacao()

print("\nArquitetura reconstruída.")
print("Parâmetros:", modelo.count_params())

assert modelo.count_params() == 14878912

# ------------------------------------------------------------
# 5. Checkpoints selecionados
# ------------------------------------------------------------

checkpoints = {
    1: "melhor_fase2.weights.h5",
    2: "melhor_fase2.weights.h5",
    3: "melhor_fase2.weights.h5",
    4: "melhor_fase2.weights.h5"
}

# ------------------------------------------------------------
# 6. Processar cada fold
# ------------------------------------------------------------

for fold, nome_checkpoint in checkpoints.items():

    arquivo_saida = (
        DIR_EMBEDDINGS /
        f"embeddings_teste_fold{fold}.npz"
    )

    if arquivo_saida.exists():

        print(
            f"\nFold {fold}: arquivo existente, "
            "extração ignorada."
        )

        continue

    arquivo_pesos = (
        DIR_CHECKPOINTS /
        f"fold_{fold}" /
        nome_checkpoint
    )

    assert arquivo_pesos.exists(), (
        f"Checkpoint não encontrado: {arquivo_pesos}"
    )

    print("\n" + "-" * 65)
    print(f"PROCESSANDO FOLD {fold}")
    print("-" * 65)

    modelo.load_weights(str(arquivo_pesos))

    inicio = time.time()
    lista_embeddings = []

    for indice, lote in enumerate(dataset, 1):

        resultado = modelo(
            lote,
            training=False
        ).numpy()

        lista_embeddings.append(
            resultado.astype(np.float32)
        )

        if indice % 50 == 0 or indice == 329:

            print(
                f"Fold {fold}: "
                f"lote {indice}/329"
            )

    embeddings = np.concatenate(
        lista_embeddings,
        axis=0
    )

    assert embeddings.shape == (10528, 128)
    assert np.all(np.isfinite(embeddings))

    normas = np.linalg.norm(
        embeddings,
        axis=1
    )

    assert np.allclose(
        normas,
        1.0,
        atol=1e-4
    )

    # Strings Unicode: dispensa allow_pickle=True
    np.savez_compressed(
        arquivo_saida,
        embeddings=embeddings,
        identidades=np.asarray(
            df_teste["identidade"].astype(str).tolist(),
            dtype=np.str_
        ),
        datasets=np.asarray(
            df_teste["dataset"].astype(str).tolist(),
            dtype=np.str_
        ),
        caminhos=np.asarray(
            caminhos,
            dtype=np.str_
        )
    )

    duracao = time.time() - inicio

    print(
        f"\nFold {fold} concluído "
        f"em {duracao / 60:.1f} minutos."
    )

    print("Arquivo:", arquivo_saida)

    del lista_embeddings, embeddings
    gc.collect()

print("\n" + "=" * 65)
print("EXTRAÇÃO DOS FOLDS CONCLUÍDA")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCO 43 — AVALIAÇÃO CONSOLIDADA DOS CINCO FOLDS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

RAIZ_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/TCC_Caes/"
    "experimentos/tcc1_kfold"
)

DIR_EMBEDDINGS = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "embeddings_teste"
)

DIR_AVALIACAO = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "avaliacao_reidentificacao"
)

DIR_AVALIACAO.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Carregar protocolo fixo
# ------------------------------------------------------------

arquivo_protocolo = (
    DIR_AVALIACAO /
    "protocolo_gallery_query_seed42.npz"
)

with np.load(
    arquivo_protocolo,
    allow_pickle=True
) as protocolo:

    indices_galeria = protocolo[
        "indices_galeria"
    ]

    indices_consulta = protocolo[
        "indices_consulta"
    ]

    caminhos_galeria = protocolo[
        "caminhos_galeria"
    ].astype(str)

    caminhos_consulta = protocolo[
        "caminhos_consulta"
    ].astype(str)

print("=" * 65)
print("TCC — AVALIAÇÃO DOS CINCO FOLDS")
print("=" * 65)

print("Galeria:", len(indices_galeria))
print("Consultas:", len(indices_consulta))

# ------------------------------------------------------------
# 2. Avaliar cada fold
# ------------------------------------------------------------

resultados = []

for fold in range(1, 6):

    arquivo = (
        DIR_EMBEDDINGS /
        f"embeddings_teste_fold{fold}.npz"
    )

    assert arquivo.exists(), (
        f"Arquivo ausente: {arquivo}"
    )

    with np.load(
        arquivo,
        allow_pickle=(fold == 5)
    ) as dados:

        embeddings = dados[
            "embeddings"
        ].astype(np.float32)

        identidades = dados[
            "identidades"
        ].astype(str)

        caminhos = dados[
            "caminhos"
        ].astype(str)

    # Conferir correspondência com o protocolo
    assert np.array_equal(
        caminhos[indices_galeria],
        caminhos_galeria
    )

    assert np.array_equal(
        caminhos[indices_consulta],
        caminhos_consulta
    )

    galeria = embeddings[indices_galeria]
    consultas = embeddings[indices_consulta]

    ids_galeria = identidades[indices_galeria]
    ids_consulta = identidades[indices_consulta]

    mapa_galeria = {
        identidade: indice
        for indice, identidade in enumerate(ids_galeria)
    }

    indices_corretos = np.array([
        mapa_galeria[identidade]
        for identidade in ids_consulta
    ])

    # Similaridade cosseno
    similaridades = consultas @ galeria.T

    scores_corretos = similaridades[
        np.arange(len(consultas)),
        indices_corretos
    ]

    # Rank exato, com desempate determinístico
    ranks = np.empty(
        len(consultas),
        dtype=np.int32
    )

    for inicio in range(0, len(consultas), 256):

        fim = min(
            inicio + 256,
            len(consultas)
        )

        bloco = similaridades[inicio:fim]
        corretos = indices_corretos[inicio:fim]
        scores = scores_corretos[inicio:fim]

        maiores = bloco > scores[:, None]

        empates = (
            (bloco == scores[:, None]) &
            (
                np.arange(len(galeria))[None, :] <
                corretos[:, None]
            )
        )

        ranks[inicio:fim] = (
            1 +
            maiores.sum(axis=1) +
            empates.sum(axis=1)
        )

    metricas = {
        "fold": fold,
        "rank1": float(np.mean(ranks <= 1)),
        "rank5": float(np.mean(ranks <= 5)),
        "rank10": float(np.mean(ranks <= 10)),
        "map": float(np.mean(1.0 / ranks)),
        "rank_medio": float(np.mean(ranks)),
        "rank_mediano": float(np.median(ranks))
    }

    resultados.append(metricas)

    print(
        f"\nFold {fold}: "
        f"Rank-1={metricas['rank1']:.2%} | "
        f"Rank-5={metricas['rank5']:.2%} | "
        f"mAP={metricas['map']:.2%}"
    )

    # Salvar resultados individuais
    pd.DataFrame({
        "caminho_consulta": caminhos[indices_consulta],
        "identidade": ids_consulta,
        "rank_correto": ranks,
        "similaridade_correta": scores_corretos
    }).to_csv(
        DIR_AVALIACAO /
        f"resultados_fold{fold}.csv",
        index=False
    )

# ------------------------------------------------------------
# 3. Consolidar métricas
# ------------------------------------------------------------

df_metricas = pd.DataFrame(resultados)

print("\n" + "=" * 65)
print("TABELA CONSOLIDADA")
print("=" * 65)

print(
    df_metricas.to_string(
        index=False,
        formatters={
            "rank1": "{:.2%}".format,
            "rank5": "{:.2%}".format,
            "rank10": "{:.2%}".format,
            "map": "{:.2%}".format,
            "rank_medio": "{:.2f}".format,
            "rank_mediano": "{:.1f}".format
        }
    )
)

# ------------------------------------------------------------
# 4. Média e desvio-padrão
# ------------------------------------------------------------

colunas = [
    "rank1",
    "rank5",
    "rank10",
    "map"
]

print("\n" + "=" * 65)
print("MÉDIA ± DESVIO-PADRÃO ENTRE FOLDS")
print("=" * 65)

for coluna in colunas:

    media = df_metricas[coluna].mean()
    desvio = df_metricas[coluna].std(ddof=1)

    print(
        f"{coluna:8s}: "
        f"{media:.2%} ± {desvio:.2%}"
    )

# ------------------------------------------------------------
# 5. Salvar consolidação
# ------------------------------------------------------------

arquivo_saida = (
    DIR_AVALIACAO /
    "metricas_cinco_folds.csv"
)

df_metricas.to_csv(
    arquivo_saida,
    index=False
)

print("\nArquivo salvo:", arquivo_saida)

print("\n" + "=" * 65)
print("AVALIAÇÃO CONSOLIDADA CONCLUÍDA")
print("=" * 65)


## 10. Métricas finais, gráficos e análise de erros

Verificação ROC-AUC, reidentificação, avaliação com múltiplas referências e visualizações finais.

In [ ]:

# ============================================================
# BLOCO 44 — CRIAÇÃO DE PARES DE VERIFICAÇÃO
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

RAIZ_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/TCC_Caes/"
    "experimentos/tcc1_kfold"
)

DIR_AVALIACAO = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "avaliacao_verificacao"
)

DIR_AVALIACAO.mkdir(
    parents=True,
    exist_ok=True
)

DIR_SPLITS = RAIZ_EXPERIMENTOS / "splits"

df_teste = pd.read_csv(
    DIR_SPLITS / "imagens_teste_30.csv"
).reset_index(drop=True)

identidades = df_teste["identidade"].astype(str).to_numpy()
caminhos = df_teste["caminho"].astype(str).to_numpy()

rng = np.random.default_rng(42)

print("=" * 65)
print("TCC — PREPARAÇÃO DOS PARES DE VERIFICAÇÃO")
print("=" * 65)

# ------------------------------------------------------------
# 1. Criar pares positivos
# ------------------------------------------------------------

pares_positivos = []

ids_unicas = np.unique(identidades)

for identidade in ids_unicas:

    indices = np.flatnonzero(
        identidades == identidade
    )

    assert len(indices) >= 2

    escolhidos = rng.choice(
        indices,
        size=2,
        replace=False
    )

    pares_positivos.append(
        (int(escolhidos[0]), int(escolhidos[1]))
    )

pares_positivos = np.asarray(
    pares_positivos,
    dtype=np.int32
)

# ------------------------------------------------------------
# 2. Criar pares negativos
# ------------------------------------------------------------

pares_negativos = []

for identidade in ids_unicas:

    indices_origem = np.flatnonzero(
        identidades == identidade
    )

    indice_a = int(
        rng.choice(indices_origem)
    )

    # Selecionar outra identidade
    outras_ids = ids_unicas[
        ids_unicas != identidade
    ]

    outra_identidade = rng.choice(
        outras_ids
    )

    indices_destino = np.flatnonzero(
        identidades == outra_identidade
    )

    indice_b = int(
        rng.choice(indices_destino)
    )

    pares_negativos.append(
        (indice_a, indice_b)
    )

pares_negativos = np.asarray(
    pares_negativos,
    dtype=np.int32
)

# ------------------------------------------------------------
# 3. Consolidar pares
# ------------------------------------------------------------

pares = np.concatenate(
    [pares_positivos, pares_negativos],
    axis=0
)

rotulos = np.concatenate([
    np.ones(
        len(pares_positivos),
        dtype=np.int32
    ),
    np.zeros(
        len(pares_negativos),
        dtype=np.int32
    )
])

# ------------------------------------------------------------
# 4. Validar integridade
# ------------------------------------------------------------

assert len(pares) == 2470

for a, b in pares_positivos:
    assert identidades[a] == identidades[b]

for a, b in pares_negativos:
    assert identidades[a] != identidades[b]

# ------------------------------------------------------------
# 5. Salvar protocolo
# ------------------------------------------------------------

arquivo_saida = (
    DIR_AVALIACAO /
    "pares_verificacao_seed42.npz"
)

np.savez_compressed(
    arquivo_saida,
    pares=pares,
    rotulos=rotulos,
    caminhos=np.asarray(
        caminhos,
        dtype=np.str_
    )
)

print("\nPares positivos:", len(pares_positivos))
print("Pares negativos:", len(pares_negativos))
print("Total de pares:", len(pares))

print("\nArquivo salvo:", arquivo_saida)

print("\n" + "=" * 65)
print("PROTOCOLO DE VERIFICAÇÃO CONCLUÍDO")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCO 45 — ROC-AUC DOS CINCO FOLDS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

RAIZ_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_EMBEDDINGS = (
    RAIZ_EXPERIMENTOS / "resultados" / "embeddings_teste"
)

DIR_VERIFICACAO = (
    RAIZ_EXPERIMENTOS / "resultados" / "avaliacao_verificacao"
)

DIR_VERIFICACAO.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("TCC — ROC-AUC DOS CINCO FOLDS")
print("=" * 65)

# ------------------------------------------------------------
# 1. Carregar protocolo fixo
# ------------------------------------------------------------

arquivo_pares = (
    DIR_VERIFICACAO / "pares_verificacao_seed42.npz"
)

with np.load(arquivo_pares, allow_pickle=False) as dados:
    pares = dados["pares"].astype(np.int64)
    rotulos = dados["rotulos"].astype(np.int32)
    caminhos_referencia = dados["caminhos"].astype(str)

assert pares.shape == (2470, 2)
assert rotulos.shape == (2470,)
assert set(np.unique(rotulos)) == {0, 1}

print("\nPares positivos:", np.sum(rotulos == 1))
print("Pares negativos:", np.sum(rotulos == 0))

# ------------------------------------------------------------
# 2. Avaliar os cinco folds
# ------------------------------------------------------------

resultados = []

for fold in range(1, 6):

    arquivo = (
        DIR_EMBEDDINGS / f"embeddings_teste_fold{fold}.npz"
    )

    assert arquivo.exists(), f"Arquivo ausente: {arquivo}"

    with np.load(arquivo, allow_pickle=(fold == 5)) as dados:
        embeddings = dados["embeddings"].astype(np.float32)
        caminhos = dados["caminhos"].astype(str)

    assert embeddings.shape == (10528, 128)
    assert np.array_equal(caminhos, caminhos_referencia), (
        f"Ordem das imagens diferente no Fold {fold}"
    )

    # Similaridade cosseno entre cada par
    # Os embeddings já estão normalizados em L2.
    a = embeddings[pares[:, 0]]
    b = embeddings[pares[:, 1]]

    similaridades = np.sum(a * b, axis=1)

    assert np.all(np.isfinite(similaridades))

    # Métrica independente de limiar
    auc = roc_auc_score(rotulos, similaridades)

    # Pontos da curva ROC
    fpr, tpr, limiares = roc_curve(
        rotulos,
        similaridades,
        drop_intermediate=False
    )

    media_positivos = float(
        np.mean(similaridades[rotulos == 1])
    )

    media_negativos = float(
        np.mean(similaridades[rotulos == 0])
    )

    resultados.append({
        "fold": fold,
        "roc_auc": float(auc),
        "similaridade_media_positivos": media_positivos,
        "similaridade_media_negativos": media_negativos
    })

    # Salvar pontuações dos pares
    pd.DataFrame({
        "indice_imagem_a": pares[:, 0],
        "indice_imagem_b": pares[:, 1],
        "rotulo": rotulos,
        "similaridade": similaridades
    }).to_csv(
        DIR_VERIFICACAO / f"pontuacoes_fold{fold}.csv",
        index=False
    )

    # Salvar curva ROC
    pd.DataFrame({
        "fpr": fpr,
        "tpr": tpr,
        "limiar": limiares
    }).to_csv(
        DIR_VERIFICACAO / f"curva_roc_fold{fold}.csv",
        index=False
    )

    print(
        f"\nFold {fold}: "
        f"ROC-AUC={auc:.4%} | "
        f"Sim. positivos={media_positivos:.4f} | "
        f"Sim. negativos={media_negativos:.4f}"
    )

# ------------------------------------------------------------
# 3. Consolidar resultados
# ------------------------------------------------------------

df_auc = pd.DataFrame(resultados)

print("\n" + "=" * 65)
print("RESULTADOS CONSOLIDADOS")
print("=" * 65)

print(
    df_auc.to_string(
        index=False,
        formatters={
            "roc_auc": "{:.4%}".format,
            "similaridade_media_positivos": "{:.4f}".format,
            "similaridade_media_negativos": "{:.4f}".format
        }
    )
)

media_auc = df_auc["roc_auc"].mean()
desvio_auc = df_auc["roc_auc"].std(ddof=1)

print("\nROC-AUC média: "
      f"{media_auc:.4%} ± {desvio_auc:.4%}")

# ------------------------------------------------------------
# 4. Salvar consolidação
# ------------------------------------------------------------

arquivo_saida = (
    DIR_VERIFICACAO / "metricas_roc_auc_cinco_folds.csv"
)

df_auc.to_csv(arquivo_saida, index=False)

print("\nArquivo salvo:", arquivo_saida)

print("\n" + "=" * 65)
print("AVALIAÇÃO ROC-AUC CONCLUÍDA")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCO 46 — GRÁFICOS ROC E SIMILARIDADES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

RAIZ_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_VERIFICACAO = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "avaliacao_verificacao"
)

DIR_GRAFICOS = DIR_VERIFICACAO / "graficos"
DIR_GRAFICOS.mkdir(parents=True, exist_ok=True)

df_metricas = pd.read_csv(
    DIR_VERIFICACAO /
    "metricas_roc_auc_cinco_folds.csv"
)

# ------------------------------------------------------------
# 1. Curvas ROC dos cinco folds
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

for fold in range(1, 6):

    df_roc = pd.read_csv(
        DIR_VERIFICACAO /
        f"curva_roc_fold{fold}.csv"
    )

    auc = df_metricas.loc[
        df_metricas["fold"] == fold,
        "roc_auc"
    ].iloc[0]

    plt.plot(
        df_roc["fpr"],
        df_roc["tpr"],
        linewidth=1.7,
        label=f"Fold {fold} (AUC={auc:.4f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    label="Classificador aleatório"
)

plt.xlabel("Taxa de falsos positivos (FPR)")
plt.ylabel("Taxa de verdadeiros positivos (TPR)")
plt.title("Curvas ROC — Verificação de identidade")
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.tight_layout()

arquivo_roc = (
    DIR_GRAFICOS / "curvas_roc_cinco_folds.png"
)

plt.savefig(arquivo_roc, dpi=300, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# 2. Distribuição das similaridades — Fold 5
# ------------------------------------------------------------

df_pares = pd.read_csv(
    DIR_VERIFICACAO / "pontuacoes_fold5.csv"
)

positivos = df_pares.loc[
    df_pares["rotulo"] == 1,
    "similaridade"
].to_numpy()

negativos = df_pares.loc[
    df_pares["rotulo"] == 0,
    "similaridade"
].to_numpy()

plt.figure(figsize=(8, 6))

plt.hist(
    negativos,
    bins=50,
    density=True,
    alpha=0.55,
    label="Cães diferentes"
)

plt.hist(
    positivos,
    bins=50,
    density=True,
    alpha=0.55,
    label="Mesmo cão"
)

plt.axvline(
    negativos.mean(),
    linestyle="--",
    linewidth=1.5,
    label=f"Média negativos: {negativos.mean():.3f}"
)

plt.axvline(
    positivos.mean(),
    linestyle="--",
    linewidth=1.5,
    label=f"Média positivos: {positivos.mean():.3f}"
)

plt.xlabel("Similaridade cosseno")
plt.ylabel("Densidade")
plt.title("Distribuição das similaridades — Fold 5")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

arquivo_distribuicao = (
    DIR_GRAFICOS /
    "distribuicao_similaridades_fold5.png"
)

plt.savefig(
    arquivo_distribuicao,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 3. Confirmar arquivos
# ------------------------------------------------------------

print("=" * 65)
print("GRÁFICOS GERADOS")
print("=" * 65)

print("ROC:", arquivo_roc)
print("Distribuições:", arquivo_distribuicao)

print("\nAUC Fold 5:", roc_auc_score(
    df_pares["rotulo"],
    df_pares["similaridade"]
))

print("\nBloco 46 concluído.")


In [ ]:

# ============================================================
# BLOCO 47 — ANÁLISE DOS ERROS DE REIDENTIFICAÇÃO
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

RAIZ_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_AVALIACAO = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "avaliacao_reidentificacao"
)

DIR_ANALISE = DIR_AVALIACAO / "analise_erros"
DIR_ANALISE.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Carregar resultados do Fold 5
# ------------------------------------------------------------

df = pd.read_csv(
    DIR_AVALIACAO / "resultados_fold5.csv"
)

df_teste = pd.read_csv(
    RAIZ_EXPERIMENTOS /
    "splits" /
    "imagens_teste_30.csv"
)

# Associar a origem das imagens
mapa_dataset = (
    df_teste[["caminho", "dataset"]]
    .drop_duplicates(subset="caminho")
)

assert mapa_dataset["caminho"].is_unique

df = df.merge(
    mapa_dataset,
    left_on="caminho_consulta",
    right_on="caminho",
    how="left",
    validate="many_to_one"
)

assert df["dataset"].notna().all()
assert len(df) == 9293

# ------------------------------------------------------------
# 2. Categorizar resultados
# ------------------------------------------------------------

df["categoria"] = np.select(
    [
        df["rank_correto"] == 1,
        df["rank_correto"].between(2, 5),
        df["rank_correto"].between(6, 10),
        df["rank_correto"] > 10
    ],
    [
        "Rank 1",
        "Rank 2–5",
        "Rank 6–10",
        "Fora do Top 10"
    ],
    default="Não classificado"
)

ordem = [
    "Rank 1",
    "Rank 2–5",
    "Rank 6–10",
    "Fora do Top 10"
]

resumo = (
    df["categoria"]
    .value_counts()
    .reindex(ordem, fill_value=0)
    .rename_axis("categoria")
    .reset_index(name="consultas")
)

resumo["percentual"] = (
    resumo["consultas"] / len(df) * 100
)

print("=" * 65)
print("TCC — ANÁLISE DE ERROS — FOLD 5")
print("=" * 65)

print("\nDistribuição das consultas:")
print(
    resumo.to_string(
        index=False,
        formatters={
            "percentual": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 3. Erros por dataset
# ------------------------------------------------------------

df["erro_rank1"] = df["rank_correto"] > 1
df["erro_top10"] = df["rank_correto"] > 10

por_dataset = (
    df.groupby("dataset")
    .agg(
        consultas=("rank_correto", "size"),
        erros_rank1=("erro_rank1", "sum"),
        erros_top10=("erro_top10", "sum"),
        rank_medio=("rank_correto", "mean")
    )
    .reset_index()
)

por_dataset["taxa_erro_rank1"] = (
    por_dataset["erros_rank1"] /
    por_dataset["consultas"] * 100
)

por_dataset["taxa_erro_top10"] = (
    por_dataset["erros_top10"] /
    por_dataset["consultas"] * 100
)

print("\nErros por dataset:")
print(
    por_dataset.to_string(
        index=False,
        formatters={
            "rank_medio": "{:.2f}".format,
            "taxa_erro_rank1": "{:.2f}%".format,
            "taxa_erro_top10": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 4. Identidades com maior dificuldade
# ------------------------------------------------------------

por_identidade = (
    df.groupby(["dataset", "identidade"])
    .agg(
        consultas=("rank_correto", "size"),
        erros_rank1=("erro_rank1", "sum"),
        erros_top10=("erro_top10", "sum"),
        rank_medio=("rank_correto", "mean"),
        rank_maximo=("rank_correto", "max")
    )
    .reset_index()
)

por_identidade["taxa_erro_rank1"] = (
    por_identidade["erros_rank1"] /
    por_identidade["consultas"]
)

# Priorizar identidades com pelo menos 3 consultas
identidades_dificeis = (
    por_identidade[
        por_identidade["consultas"] >= 3
    ]
    .sort_values(
        ["taxa_erro_rank1", "rank_medio", "consultas"],
        ascending=[False, False, False]
    )
)

print("\nDez identidades com maior dificuldade:")
print(
    identidades_dificeis.head(10).to_string(
        index=False,
        formatters={
            "taxa_erro_rank1": "{:.2%}".format,
            "rank_medio": "{:.2f}".format
        }
    )
)

# ------------------------------------------------------------
# 5. Salvar resultados
# ------------------------------------------------------------

resumo.to_csv(
    DIR_ANALISE / "distribuicao_ranks_fold5.csv",
    index=False
)

por_dataset.to_csv(
    DIR_ANALISE / "erros_por_dataset_fold5.csv",
    index=False
)

por_identidade.to_csv(
    DIR_ANALISE / "erros_por_identidade_fold5.csv",
    index=False
)

df.sort_values(
    "rank_correto",
    ascending=False
).to_csv(
    DIR_ANALISE / "consultas_ordenadas_por_erro_fold5.csv",
    index=False
)

print("\nArquivos salvos em:", DIR_ANALISE)
print("\nANÁLISE CONCLUÍDA")


In [ ]:

# ============================================================
# BLOCO 48 — VISUALIZAÇÃO DOS ERROS MAIS GRAVES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

RAIZ_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_EMBEDDINGS = (
    RAIZ_EXPERIMENTOS / "resultados" / "embeddings_teste"
)

DIR_AVALIACAO = (
    RAIZ_EXPERIMENTOS / "resultados" /
    "avaliacao_reidentificacao"
)

DIR_ANALISE = DIR_AVALIACAO / "analise_erros"
DIR_ANALISE.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("TCC — VISUALIZAÇÃO DE ERROS — FOLD 5")
print("=" * 65)

# ------------------------------------------------------------
# 1. Carregar embeddings
# ------------------------------------------------------------

with np.load(
    DIR_EMBEDDINGS / "embeddings_teste_fold5.npz",
    allow_pickle=True
) as dados:

    embeddings = dados["embeddings"].astype(np.float32)
    identidades = dados["identidades"].astype(str)
    caminhos = dados["caminhos"].astype(str)

# ------------------------------------------------------------
# 2. Carregar protocolo Gallery–Query
# ------------------------------------------------------------

with np.load(
    DIR_AVALIACAO / "protocolo_gallery_query_seed42.npz",
    allow_pickle=True
) as dados:

    indices_galeria = dados["indices_galeria"]
    indices_consulta = dados["indices_consulta"]

# ------------------------------------------------------------
# 3. Recuperar consultas com maiores erros
# ------------------------------------------------------------

df_erros = pd.read_csv(
    DIR_ANALISE /
    "consultas_ordenadas_por_erro_fold5.csv"
)

piores = df_erros[
    df_erros["rank_correto"] > 10
].head(6)

assert len(piores) == 6

mapa_caminhos = {
    caminho: indice
    for indice, caminho in enumerate(caminhos)
}

galeria = embeddings[indices_galeria]

mapa_identidades_galeria = {
    identidades[indice]: indice
    for indice in indices_galeria
}

# ------------------------------------------------------------
# 4. Função para carregar imagem
# ------------------------------------------------------------

def carregar_imagem(caminho):

    with Image.open(caminho) as imagem:
        return imagem.convert("RGB").copy()

# ------------------------------------------------------------
# 5. Montar figura comparativa
# ------------------------------------------------------------

fig, eixos = plt.subplots(
    6, 3,
    figsize=(12, 22)
)

registros = []

for linha, (_, caso) in enumerate(piores.iterrows()):

    caminho_consulta = str(caso["caminho_consulta"])
    indice_consulta = mapa_caminhos[caminho_consulta]

    identidade_correta = identidades[indice_consulta]

    # Comparar consulta com toda a galeria
    similaridades = (
        embeddings[indice_consulta] @ galeria.T
    )

    posicao_top1 = int(np.argmax(similaridades))

    indice_top1 = int(
        indices_galeria[posicao_top1]
    )

    indice_correto = int(
        mapa_identidades_galeria[identidade_correta]
    )

    imagens = [
        carregar_imagem(caminhos[indice_consulta]),
        carregar_imagem(caminhos[indice_top1]),
        carregar_imagem(caminhos[indice_correto])
    ]

    titulos = [
        "Consulta",
        "Top 1 incorreto",
        "Identidade correta"
    ]

    for coluna in range(3):

        eixos[linha, coluna].imshow(imagens[coluna])
        eixos[linha, coluna].set_title(titulos[coluna])
        eixos[linha, coluna].axis("off")

    eixos[linha, 0].set_ylabel(
        f"Rank correto: {int(caso['rank_correto'])}",
        fontsize=10
    )

    registros.append({
        "identidade_consulta": identidade_correta,
        "identidade_top1": identidades[indice_top1],
        "rank_correto": int(caso["rank_correto"]),
        "similaridade_top1": float(similaridades[posicao_top1]),
        "similaridade_correta": float(
            embeddings[indice_consulta] @
            embeddings[indice_correto]
        ),
        "caminho_consulta": caminhos[indice_consulta],
        "caminho_top1": caminhos[indice_top1],
        "caminho_correto": caminhos[indice_correto]
    })

plt.suptitle(
    "Análise qualitativa dos erros — Fold 5",
    fontsize=15
)

plt.tight_layout(rect=[0, 0, 1, 0.98])

arquivo_figura = (
    DIR_ANALISE / "erros_graves_fold5.png"
)

plt.savefig(
    arquivo_figura,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 6. Salvar informações dos casos
# ------------------------------------------------------------

df_casos = pd.DataFrame(registros)

arquivo_csv = (
    DIR_ANALISE / "erros_graves_fold5.csv"
)

df_casos.to_csv(arquivo_csv, index=False)

print("\nFigura salva:", arquivo_figura)
print("Dados salvos:", arquivo_csv)

print("\nResumo dos casos:")
print(
    df_casos[
        [
            "identidade_consulta",
            "identidade_top1",
            "rank_correto",
            "similaridade_top1",
            "similaridade_correta"
        ]
    ].to_string(index=False)
)

print("\nANÁLISE VISUAL CONCLUÍDA")


In [ ]:

# ============================================================
# BLOCO 49 — AUDITORIA DOS ERROS GRAVES
# ============================================================

from pathlib import Path
import pandas as pd

RAIZ_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

arquivo = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "avaliacao_reidentificacao" /
    "analise_erros" /
    "erros_graves_fold5.csv"
)

df = pd.read_csv(arquivo)

print("=" * 65)
print("TCC — AUDITORIA DOS SEIS ERROS GRAVES")
print("=" * 65)

df["diferenca_similaridade"] = (
    df["similaridade_top1"] -
    df["similaridade_correta"]
)

colunas = [
    "identidade_consulta",
    "identidade_top1",
    "rank_correto",
    "similaridade_top1",
    "similaridade_correta",
    "diferenca_similaridade"
]

print(
    df[colunas].to_string(
        index=False,
        formatters={
            "similaridade_top1": "{:.4f}".format,
            "similaridade_correta": "{:.4f}".format,
            "diferenca_similaridade": "{:.4f}".format
        }
    )
)

print("\nIdentidades consultadas:")
print(df["identidade_consulta"].value_counts())

print("\nIdentidades incorretas no Top 1:")
print(df["identidade_top1"].value_counts())

print("\nDiferença média entre Top 1 e correto:")
print(f"{df['diferenca_similaridade'].mean():.4f}")

print("\nAUDITORIA CONCLUÍDA")


In [ ]:

# ============================================================
# BLOCO 50 — PROTOCOLO COM 1, 3 E 5 REFERÊNCIAS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

RAIZ_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_MULTIPLAS = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "avaliacao_multiplas_referencias"
)

DIR_MULTIPLAS.mkdir(parents=True, exist_ok=True)

df_teste = pd.read_csv(
    RAIZ_EXPERIMENTOS /
    "splits" /
    "imagens_teste_30.csv"
).reset_index(drop=True)

identidades = df_teste["identidade"].astype(str).to_numpy()
caminhos = df_teste["caminho"].astype(str).to_numpy()

rng = np.random.default_rng(42)

print("=" * 65)
print("TCC — PROTOCOLO DE MÚLTIPLAS REFERÊNCIAS")
print("=" * 65)

# ------------------------------------------------------------
# 1. Selecionar identidades com pelo menos 6 imagens
# ------------------------------------------------------------

contagens = pd.Series(identidades).value_counts()

ids_elegiveis = sorted(
    contagens[contagens >= 6].index.tolist()
)

print("\nIdentidades elegíveis:", len(ids_elegiveis))

# ------------------------------------------------------------
# 2. Fixar cinco referências e consultas restantes
# ------------------------------------------------------------

referencias = []
consultas = []

for identidade in ids_elegiveis:

    indices = np.flatnonzero(
        identidades == identidade
    )

    indices = rng.permutation(indices)

    referencias.append(indices[:5])
    consultas.extend(indices[5:].tolist())

referencias = np.asarray(
    referencias,
    dtype=np.int64
)

consultas = np.asarray(
    consultas,
    dtype=np.int64
)

assert referencias.shape == (len(ids_elegiveis), 5)
assert len(consultas) > 0

assert not np.intersect1d(
    referencias.ravel(),
    consultas
).size

# ------------------------------------------------------------
# 3. Construir cenários aninhados
# ------------------------------------------------------------

# As mesmas identidades e consultas serão usadas.
# Apenas o número de referências varia.

cenarios = {
    1: referencias[:, :1],
    3: referencias[:, :3],
    5: referencias[:, :5]
}

for quantidade, indices in cenarios.items():

    assert indices.shape == (
        len(ids_elegiveis),
        quantidade
    )

    print(
        f"\nCenário {quantidade}: "
        f"{len(ids_elegiveis)} identidades, "
        f"{indices.size} imagens na galeria"
    )

print("\nConsultas fixas:", len(consultas))

# ------------------------------------------------------------
# 4. Salvar protocolo
# ------------------------------------------------------------

arquivo_saida = (
    DIR_MULTIPLAS /
    "protocolo_multiplas_referencias_seed42.npz"
)

np.savez_compressed(
    arquivo_saida,
    identidades_elegiveis=np.asarray(
        ids_elegiveis,
        dtype=np.str_
    ),
    referencias_5=referencias,
    indices_consulta=consultas,
    caminhos=np.asarray(
        caminhos,
        dtype=np.str_
    )
)

print("\nProtocolo salvo:", arquivo_saida)

print("\n" + "=" * 65)
print("PREPARAÇÃO CONCLUÍDA")
print("=" * 65)



In [ ]:

# ============================================================
# BLOCO 51 — AVALIAÇÃO COM 1, 3 E 5 REFERÊNCIAS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import time

RAIZ_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_EMBEDDINGS = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "embeddings_teste"
)

DIR_MULTIPLAS = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "avaliacao_multiplas_referencias"
)

DIR_MULTIPLAS.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("TCC — AVALIAÇÃO COM MÚLTIPLAS REFERÊNCIAS")
print("=" * 65)

inicio_total = time.time()

# ------------------------------------------------------------
# 1. Carregar protocolo fixo do Bloco 50
# ------------------------------------------------------------

arquivo_protocolo = (
    DIR_MULTIPLAS /
    "protocolo_multiplas_referencias_seed42.npz"
)

with np.load(arquivo_protocolo, allow_pickle=False) as dados:

    ids_elegiveis = dados[
        "identidades_elegiveis"
    ].astype(str)

    referencias_5 = dados[
        "referencias_5"
    ].astype(np.int64)

    indices_consulta = dados[
        "indices_consulta"
    ].astype(np.int64)

    caminhos_referencia = dados[
        "caminhos"
    ].astype(str)

n_identidades = len(ids_elegiveis)
n_consultas = len(indices_consulta)

assert referencias_5.shape == (n_identidades, 5)
assert n_identidades == 897
assert n_consultas == 4548

assert not np.intersect1d(
    referencias_5.ravel(),
    indices_consulta
).size

print("\nIdentidades:", n_identidades)
print("Consultas fixas:", n_consultas)
print("Cenários:", [1, 3, 5])

# ------------------------------------------------------------
# 2. Preparar rótulos corretos
# ------------------------------------------------------------

df_teste = pd.read_csv(
    RAIZ_EXPERIMENTOS /
    "splits" /
    "imagens_teste_30.csv"
).reset_index(drop=True)

identidades_teste = (
    df_teste["identidade"]
    .astype(str)
    .to_numpy()
)

caminhos_teste = (
    df_teste["caminho"]
    .astype(str)
    .to_numpy()
)

assert np.array_equal(
    caminhos_teste,
    caminhos_referencia
), "Ordem das imagens diferente do protocolo."

# Cada linha de referencias_5 corresponde a uma identidade.
for i, identidade in enumerate(ids_elegiveis):

    assert np.all(
        identidades_teste[referencias_5[i]] == identidade
    )

ids_consultas = identidades_teste[indices_consulta]

mapa_identidades = {
    identidade: i
    for i, identidade in enumerate(ids_elegiveis)
}

indices_corretos = np.array(
    [mapa_identidades[identidade]
     for identidade in ids_consultas],
    dtype=np.int32
)

# ------------------------------------------------------------
# 3. Avaliar cada fold e cenário
# ------------------------------------------------------------

resultados = []

for fold in range(1, 6):

    arquivo_embeddings = (
        DIR_EMBEDDINGS /
        f"embeddings_teste_fold{fold}.npz"
    )

    assert arquivo_embeddings.exists(), (
        f"Embeddings ausentes: {arquivo_embeddings}"
    )

    with np.load(
        arquivo_embeddings,
        allow_pickle=(fold == 5)
    ) as dados:

        embeddings = dados[
            "embeddings"
        ].astype(np.float32)

        caminhos = dados[
            "caminhos"
        ].astype(str)

        identidades_embeddings = dados[
            "identidades"
        ].astype(str)

    assert embeddings.shape == (10528, 128)

    assert np.array_equal(
        caminhos,
        caminhos_referencia
    ), f"Caminhos divergentes no Fold {fold}"

    assert np.array_equal(
        identidades_embeddings,
        identidades_teste
    ), f"Identidades divergentes no Fold {fold}"

    assert np.allclose(
        np.linalg.norm(embeddings, axis=1),
        1.0,
        atol=1e-4
    )

    print("\n" + "-" * 65)
    print(f"FOLD {fold}")
    print("-" * 65)

    consultas = embeddings[indices_consulta]

    for n_referencias in [1, 3, 5]:

        inicio_cenario = time.time()

        # Shape:
        # (897 identidades, N referências, 128)
        galeria = embeddings[
            referencias_5[:, :n_referencias]
        ]

        # Calcular similaridades por blocos de consultas
        ranks = np.empty(
            n_consultas,
            dtype=np.int32
        )

        # Guardar a similaridade da identidade correta
        scores_corretos = np.empty(
            n_consultas,
            dtype=np.float32
        )

        for inicio in range(0, n_consultas, 128):

            fim = min(
                inicio + 128,
                n_consultas
            )

            bloco_consultas = consultas[inicio:fim]

            # Produto matricial contra todas as imagens
            # da galeria: (bloco, 897*N)
            similares_imagens = (
                bloco_consultas @
                galeria.reshape(-1, 128).T
            )

            # (bloco, 897 identidades, N referências)
            similares_imagens = (
                similares_imagens.reshape(
                    fim - inicio,
                    n_identidades,
                    n_referencias
                )
            )

            # Uma pontuação por identidade:
            # maior similaridade entre suas referências
            scores_identidades = np.max(
                similares_imagens,
                axis=2
            )

            corretos = indices_corretos[inicio:fim]

            scores = scores_identidades[
                np.arange(fim - inicio),
                corretos
            ]

            scores_corretos[inicio:fim] = scores

            # Ranking exato com desempate determinístico
            maiores = (
                scores_identidades > scores[:, None]
            )

            empates_anteriores = (
                (scores_identidades == scores[:, None]) &
                (
                    np.arange(n_identidades)[None, :] <
                    corretos[:, None]
                )
            )

            ranks[inicio:fim] = (
                1 +
                maiores.sum(axis=1) +
                empates_anteriores.sum(axis=1)
            )

        # ----------------------------------------------------
        # 4. Métricas do cenário
        # ----------------------------------------------------

        rank1 = float(np.mean(ranks <= 1))
        rank5 = float(np.mean(ranks <= 5))
        rank10 = float(np.mean(ranks <= 10))

        # Com uma identidade relevante por consulta,
        # AP = 1 / rank da identidade correta.
        map_score = float(np.mean(1.0 / ranks))

        resultados.append({
            "fold": fold,
            "referencias_por_identidade": n_referencias,
            "identidades": n_identidades,
            "consultas": n_consultas,
            "rank1": rank1,
            "rank5": rank5,
            "rank10": rank10,
            "map": map_score,
            "rank_medio": float(np.mean(ranks)),
            "rank_mediano": float(np.median(ranks))
        })

        # Salvar resultados individuais
        df_consultas = pd.DataFrame({
            "caminho_consulta": caminhos[indices_consulta],
            "identidade": ids_consultas,
            "rank_correto": ranks,
            "similaridade_correta": scores_corretos
        })

        arquivo_individual = (
            DIR_MULTIPLAS /
            f"resultados_fold{fold}_refs{n_referencias}.csv"
        )

        df_consultas.to_csv(
            arquivo_individual,
            index=False
        )

        duracao = time.time() - inicio_cenario

        print(
            f"{n_referencias} referência(s): "
            f"Rank-1={rank1:.2%} | "
            f"Rank-5={rank5:.2%} | "
            f"Rank-10={rank10:.2%} | "
            f"mAP={map_score:.2%} | "
            f"{duracao:.1f}s"
        )

# ------------------------------------------------------------
# 5. Tabela consolidada
# ------------------------------------------------------------

df_resultados = pd.DataFrame(resultados)

arquivo_resultados = (
    DIR_MULTIPLAS /
    "metricas_multiplas_referencias_cinco_folds.csv"
)

df_resultados.to_csv(
    arquivo_resultados,
    index=False
)

print("\n" + "=" * 65)
print("RESULTADOS CONSOLIDADOS")
print("=" * 65)

print(
    df_resultados[
        [
            "fold",
            "referencias_por_identidade",
            "rank1",
            "rank5",
            "rank10",
            "map"
        ]
    ].to_string(
        index=False,
        formatters={
            "rank1": "{:.2%}".format,
            "rank5": "{:.2%}".format,
            "rank10": "{:.2%}".format,
            "map": "{:.2%}".format
        }
    )
)

# ------------------------------------------------------------
# 6. Média e desvio-padrão por cenário
# ------------------------------------------------------------

resumo = (
    df_resultados
    .groupby("referencias_por_identidade")
    .agg(
        rank1_media=("rank1", "mean"),
        rank1_desvio=("rank1", "std"),
        rank5_media=("rank5", "mean"),
        rank5_desvio=("rank5", "std"),
        rank10_media=("rank10", "mean"),
        rank10_desvio=("rank10", "std"),
        map_media=("map", "mean"),
        map_desvio=("map", "std")
    )
    .reset_index()
)

arquivo_resumo = (
    DIR_MULTIPLAS /
    "resumo_multiplas_referencias.csv"
)

resumo.to_csv(
    arquivo_resumo,
    index=False
)

print("\n" + "=" * 65)
print("MÉDIA ± DESVIO-PADRÃO POR CENÁRIO")
print("=" * 65)

for _, linha in resumo.iterrows():

    print(
        f"\n{int(linha['referencias_por_identidade'])} "
        "referência(s):"
    )

    for metrica in ["rank1", "rank5", "rank10", "map"]:

        media = linha[f"{metrica}_media"]
        desvio = linha[f"{metrica}_desvio"]

        print(
            f"  {metrica:7s}: "
            f"{media:.2%} ± {desvio:.2%}"
        )

print("\nArquivos salvos:")
print(arquivo_resultados)
print(arquivo_resumo)

print(
    f"\nTempo total: "
    f"{time.time() - inicio_total:.1f} segundos"
)

print("\n" + "=" * 65)
print("AVALIAÇÃO DE MÚLTIPLAS REFERÊNCIAS CONCLUÍDA")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCO 52 — FIGURAS FINAIS DO EXPERIMENTO
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAIZ_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_MULTIPLAS = (
    RAIZ_EXPERIMENTOS /
    "resultados" /
    "avaliacao_multiplas_referencias"
)

DIR_GRAFICOS = DIR_MULTIPLAS / "graficos"
DIR_GRAFICOS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    DIR_MULTIPLAS /
    "metricas_multiplas_referencias_cinco_folds.csv"
)

resumo = pd.read_csv(
    DIR_MULTIPLAS /
    "resumo_multiplas_referencias.csv"
).sort_values("referencias_por_identidade")

print("=" * 65)
print("TCC — FIGURAS FINAIS")
print("=" * 65)

# ------------------------------------------------------------
# 1. Evolução das métricas médias
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 6))

x = resumo["referencias_por_identidade"]

for metrica, nome in [
    ("rank1_media", "Rank-1"),
    ("rank5_media", "Rank-5"),
    ("rank10_media", "Rank-10"),
    ("map_media", "mAP")
]:
    ax.plot(
        x,
        resumo[metrica] * 100,
        marker="o",
        linewidth=2,
        markersize=7,
        label=nome
    )

ax.set_xlabel("Fotografias de referência por identidade")
ax.set_ylabel("Desempenho (%)")
ax.set_title("Desempenho com múltiplas referências")
ax.set_xticks([1, 3, 5])
ax.set_ylim(65, 100)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()

arquivo1 = (
    DIR_GRAFICOS /
    "evolucao_metricas_multiplas_referencias.png"
)

fig.savefig(arquivo1, dpi=300, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# 2. Rank-1 por fold e cenário
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 6))

for quantidade in [1, 3, 5]:

    dados = df[
        df["referencias_por_identidade"] == quantidade
    ].sort_values("fold")

    ax.plot(
        dados["fold"],
        dados["rank1"] * 100,
        marker="o",
        linewidth=2,
        label=f"{quantidade} referência(s)"
    )

ax.set_xlabel("Fold")
ax.set_ylabel("Rank-1 (%)")
ax.set_title("Rank-1 por fold e quantidade de referências")
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_ylim(65, 100)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()

arquivo2 = (
    DIR_GRAFICOS /
    "rank1_por_fold_multiplas_referencias.png"
)

fig.savefig(arquivo2, dpi=300, bbox_inches="tight")
plt.show()

print("\nFiguras salvas:")
print(arquivo1)
print(arquivo2)

print("\nBLOCO 52 CONCLUÍDO")


## 11. Auditoria final

Conferência dos resultados experimentais.

In [ ]:

# ============================================================
# BLOCO 53 — AUDITORIA FINAL DOS EXPERIMENTOS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = Path(
    "/content/drive/MyDrive/TCC_Caes/experimentos/tcc1_kfold"
)

DIR_RESULTADOS = RAIZ / "resultados"
DIR_EMB = DIR_RESULTADOS / "embeddings_teste"
DIR_REID = DIR_RESULTADOS / "avaliacao_reidentificacao"
DIR_VER = DIR_RESULTADOS / "avaliacao_verificacao"
DIR_MULTI = DIR_RESULTADOS / "avaliacao_multiplas_referencias"

verificacoes = []

def verificar(nome, condicao):
    ok = bool(condicao)
    verificacoes.append((nome, ok))
    print(f"{'OK' if ok else 'FALHA'} — {nome}")

print("=" * 65)
print("TCC — AUDITORIA FINAL")
print("=" * 65)

# ------------------------------------------------------------
# 1. Integridade dos embeddings
# ------------------------------------------------------------

print("\n1. EMBEDDINGS")

caminhos_base = None
identidades_base = None

for fold in range(1, 6):

    arquivo = DIR_EMB / f"embeddings_teste_fold{fold}.npz"

    verificar(f"Arquivo Fold {fold}", arquivo.exists())

    if not arquivo.exists():
        continue

    with np.load(arquivo, allow_pickle=(fold == 5)) as dados:
        emb = dados["embeddings"].astype(np.float32)
        ids = dados["identidades"].astype(str)
        caminhos = dados["caminhos"].astype(str)

    verificar(
        f"Dimensões Fold {fold}",
        emb.shape == (10528, 128)
    )

    verificar(
        f"Valores finitos Fold {fold}",
        np.all(np.isfinite(emb))
    )

    verificar(
        f"Normalização L2 Fold {fold}",
        np.allclose(
            np.linalg.norm(emb, axis=1),
            1.0,
            atol=1e-4
        )
    )

    if caminhos_base is None:
        caminhos_base = caminhos
        identidades_base = ids
    else:
        verificar(
            f"Ordem das imagens Fold {fold}",
            np.array_equal(caminhos, caminhos_base)
        )
        verificar(
            f"Identidades Fold {fold}",
            np.array_equal(ids, identidades_base)
        )

# ------------------------------------------------------------
# 2. Protocolo Gallery–Query
# ------------------------------------------------------------

print("\n2. PROTOCOLO GALLERY–QUERY")

with np.load(
    DIR_REID / "protocolo_gallery_query_seed42.npz",
    allow_pickle=True
) as dados:
    galeria = dados["indices_galeria"]
    consultas = dados["indices_consulta"]

verificar("Galeria com 1.235 imagens", len(galeria) == 1235)
verificar("Consultas com 9.293 imagens", len(consultas) == 9293)

verificar(
    "Sem sobreposição galeria/consultas",
    np.intersect1d(galeria, consultas).size == 0
)

verificar(
    "Todas as imagens utilizadas",
    len(np.unique(np.concatenate([galeria, consultas]))) == 10528
)

verificar(
    "Uma identidade por referência",
    len(np.unique(identidades_base[galeria])) == len(galeria)
)

# ------------------------------------------------------------
# 3. Protocolo de verificação
# ------------------------------------------------------------

print("\n3. PARES DE VERIFICAÇÃO")

with np.load(
    DIR_VER / "pares_verificacao_seed42.npz",
    allow_pickle=False
) as dados:
    pares = dados["pares"]
    rotulos = dados["rotulos"]

verificar("Total de 2.470 pares", len(pares) == 2470)
verificar("1.235 pares positivos", np.sum(rotulos == 1) == 1235)
verificar("1.235 pares negativos", np.sum(rotulos == 0) == 1235)

verificar(
    "Rótulos dos pares consistentes",
    np.all(
        (identidades_base[pares[:, 0]] ==
         identidades_base[pares[:, 1]]) ==
        (rotulos == 1)
    )
)

# ------------------------------------------------------------
# 4. Protocolo de múltiplas referências
# ------------------------------------------------------------

print("\n4. MÚLTIPLAS REFERÊNCIAS")

with np.load(
    DIR_MULTI / "protocolo_multiplas_referencias_seed42.npz",
    allow_pickle=False
) as dados:
    referencias = dados["referencias_5"]
    consultas_multi = dados["indices_consulta"]
    ids_multi = dados["identidades_elegiveis"].astype(str)

verificar(
    "897 identidades elegíveis",
    len(ids_multi) == 897
)

verificar(
    "Cinco referências por identidade",
    referencias.shape == (897, 5)
)

verificar(
    "4.548 consultas fixas",
    len(consultas_multi) == 4548
)

verificar(
    "Sem sobreposição entre referências e consultas",
    np.intersect1d(
        referencias.ravel(),
        consultas_multi
    ).size == 0
)

verificar(
    "Identidades das referências consistentes",
    np.all(
        identidades_base[referencias] ==
        ids_multi[:, None]
    )
)

# ------------------------------------------------------------
# 5. Tabelas e figuras
# ------------------------------------------------------------

print("\n5. RESULTADOS E FIGURAS")

arquivos_obrigatorios = [
    DIR_REID / "metricas_cinco_folds.csv",
    DIR_VER / "metricas_roc_auc_cinco_folds.csv",
    DIR_MULTI / "metricas_multiplas_referencias_cinco_folds.csv",
    DIR_MULTI / "resumo_multiplas_referencias.csv",
    DIR_VER / "graficos" / "curvas_roc_cinco_folds.png",
    DIR_VER / "graficos" / "distribuicao_similaridades_fold5.png",
    DIR_MULTI / "graficos" /
        "evolucao_metricas_multiplas_referencias.png",
    DIR_MULTI / "graficos" /
        "rank1_por_fold_multiplas_referencias.png"
]

for arquivo in arquivos_obrigatorios:
    verificar(
        arquivo.name,
        arquivo.exists() and arquivo.stat().st_size > 0
    )

# ------------------------------------------------------------
# 6. Consistência das médias
# ------------------------------------------------------------

print("\n6. CONSISTÊNCIA DAS MÉTRICAS")

df_reid = pd.read_csv(
    DIR_REID / "metricas_cinco_folds.csv"
)

df_auc = pd.read_csv(
    DIR_VER / "metricas_roc_auc_cinco_folds.csv"
)

df_multi = pd.read_csv(
    DIR_MULTI / "metricas_multiplas_referencias_cinco_folds.csv"
)

df_resumo = pd.read_csv(
    DIR_MULTI / "resumo_multiplas_referencias.csv"
)

verificar("Cinco folds na reidentificação", len(df_reid) == 5)
verificar("Cinco folds na ROC-AUC", len(df_auc) == 5)
verificar("Quinze avaliações multi-ref", len(df_multi) == 15)

for _, linha in df_resumo.iterrows():

    n = int(linha["referencias_por_identidade"])

    valores = df_multi.loc[
        df_multi["referencias_por_identidade"] == n,
        "rank1"
    ]

    verificar(
        f"Média Rank-1 ({n} referências)",
        len(valores) == 5 and np.isclose(
            valores.mean(),
            linha["rank1_media"],
            atol=1e-10
        )
    )

# ------------------------------------------------------------
# 7. Relatório de auditoria
# ------------------------------------------------------------

df_auditoria = pd.DataFrame(
    verificacoes,
    columns=["verificacao", "aprovado"]
)

arquivo_auditoria = (
    DIR_RESULTADOS / "auditoria_final_tcc.csv"
)

df_auditoria.to_csv(
    arquivo_auditoria,
    index=False
)

total = len(df_auditoria)
aprovadas = int(df_auditoria["aprovado"].sum())
falhas = total - aprovadas

print("\n" + "=" * 65)
print("RESULTADO DA AUDITORIA")
print("=" * 65)

print(f"Verificações: {total}")
print(f"Aprovadas: {aprovadas}")
print(f"Falhas: {falhas}")

print("\nRelatório:", arquivo_auditoria)

if falhas == 0:
    print("\nAUDITORIA AUTOMÁTICA APROVADA")
else:
    print("\nATENÇÃO: EXISTEM VERIFICAÇÕES PENDENTES")

print("=" * 65)
